<a href="https://colab.research.google.com/github/carmellius/AEMO-Energy-Market-Pipeline/blob/main/AEMO_Master_Platform_v1_Copy_26_08_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ENVIRONMENT SETUP

## Confirmation of Local run

In [ ]:
import sys
import os
import psutil
import pandas as pd
import nemosis
import importlib.metadata
from google.cloud import bigquery

# 1. Check Python Version
print(f"✅ Python Version: {sys.version}")

# 2. Check RAM
total_ram = psutil.virtual_memory().total / (1024**3)
print(f"✅ Total RAM detected: {total_ram:.2f} GB")

# 3. Check Imports
print(f"✅ Pandas Version: {pd.__version__}")
try:
    # Proper way to get version for some packages
    v = importlib.metadata.version('nemosis')
    print(f"✅ NEMOSIS Version: {v}")
except:
    print("✅ NEMOSIS imported successfully")

# 4. Check BigQuery Connection
# UPDATE THIS NAME TO YOUR ACTUAL JSON FILE
KEY_PATH = "C:/Users/carme/OneDrive/Desktop/AEMO Warehouse/master-aemo-0a9a3232a2a0.json"

if os.path.exists(KEY_PATH):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH
    client = bigquery.Client(project="master-aemo")
    datasets = list(client.list_datasets())
    print(f"✅ BigQuery Connected. Found {len(datasets)} datasets.")
else:
    print(f"❌ KEY_PATH NOT FOUND: {KEY_PATH}")
    print("Check your folder and make sure the filename matches exactly.")

In [ ]:
# =============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# =============================================================================

# Clean previous installations
!pip uninstall -y nemosis pandas -q

# Install stable validated versions
!pip install nemosis==3.8.1 -q
!pip install pandas==3.0.3 -q

print("Environment setup complete.")
print("IMPORTANT: Restart runtime before continuing.")

IMPORTS

In [ ]:
import gc


In [ ]:
# =============================================================================
# SECTION 2 — IMPORTS
# =============================================================================

# Standard libraries
import os
import io
import zipfile
import requests
import warnings
from datetime import datetime, timedelta

# Data handling
import pandas as pd
import numpy as np

# BigQuery
from google.cloud import bigquery
from google.oauth2 import service_account

# NEMOSIS
import nemosis
from nemosis import dynamic_data_compiler

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")

CONFIGURATION

In [ ]:
# =============================================================================
# SECTION 3 — CONFIGURATION (LOCAL TANK VERSION)
# =============================================================================

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

# 1. Update this to your ACTUAL Desktop folder
BASE_DIR = r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
CACHE_DIR = os.path.join(BASE_DIR, "nemosis_cache")
KEY_PATH = os.path.join(BASE_DIR, "master-aemo-0a9a3232a2a0.json")

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

DISPATCHIS_URL = "https://www.nemweb.com.au/REPORTS/CURRENT/DispatchIS_Reports/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

print("✅ Local Configuration loaded successfully.")

ENVIRONMENT VALIDATION

In [ ]:
# =============================================================================
# SECTION 4 — ENVIRONMENT VALIDATION
# =============================================================================

print("NEMOSIS imported successfully.")
print("Pandas version:", pd.__version__)

# Test NEMOSIS retrieval
test_price = dynamic_data_compiler(
    start_time="2023/01/01 00:00:00",
    end_time="2023/01/02 00:00:00",
    table_name="DISPATCHPRICE",
    raw_data_location=CACHE_DIR
)

print("\nDISPATCHPRICE TEST SUCCESS")
print(test_price.shape)

test_price.head()

UTILITY FUNCTIONS

In [ ]:
# =============================================================================
# SECTION 5 — UTILITY FUNCTIONS (THE TANK ENGINE)
# =============================================================================

def validate_required_columns(df, required_columns, table_name):
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"{table_name} missing required columns: {missing_columns}")
    print(f"✅ {table_name} validation passed.")

def prepare_master_table(df):
    """
    This is the core logic that prevents crashes.
    It forces all numbers to Floats and all dates to Timestamps.
    """
    if df is None or df.empty: return df

    # 1. Unified Datetime standardisation (Pipeline A, B, C, D all use this)
    dt_cols = ["SETTLEMENTDATE", "INTERVAL_DATETIME", "OFFERDATE", "EFFECTIVEDATE", "LASTCHANGED"]
    for col in dt_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 2. Force Numeric Consistency (The BigQuery Schema fix)
    for col in df.columns:
        if col not in dt_cols and not pd.api.types.is_datetime64_any_dtype(df[col]):
            if df[col].dtype == "object":
                try:
                    df[col] = pd.to_numeric(df[col], errors='ignore')
                except: pass

            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].astype("float64")

    return df.drop_duplicates()

# Keep your original function names for compatibility, but point them to the new logic
def standardise_datetime_columns(df): return prepare_master_table(df)
def convert_numeric_columns(df): return prepare_master_table(df)
def remove_duplicates(df): return df.drop_duplicates()

BIGQUERY SCHEMA DEFINITIONS

In [ ]:
# =============================================================================
# SECTION 6 — BIGQUERY SCHEMA DEFINITIONS
# =============================================================================

# -----------------------------------------------------------------------------
# Master Warehouse Table Schemas
# -----------------------------------------------------------------------------

MASTER_TABLE_SCHEMAS = {

    # -------------------------------------------------------------------------
    # DISPATCHPRICE
    # -------------------------------------------------------------------------

    "DISPATCHPRICE": [

        bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("REGIONID", "STRING"),

        bigquery.SchemaField("RRP", "FLOAT"),

        bigquery.SchemaField("RAISE6SECRRP", "FLOAT"),
        bigquery.SchemaField("RAISE60SECRRP", "FLOAT"),
        bigquery.SchemaField("RAISE5MINRRP", "FLOAT"),
        bigquery.SchemaField("RAISEREGRRP", "FLOAT"),

        bigquery.SchemaField("LOWER6SECRRP", "FLOAT"),
        bigquery.SchemaField("LOWER60SECRRP", "FLOAT"),
        bigquery.SchemaField("LOWER5MINRRP", "FLOAT"),
        bigquery.SchemaField("LOWERREGRRP", "FLOAT"),

        bigquery.SchemaField("PRICE_STATUS", "STRING")

    ],

    # -------------------------------------------------------------------------
    # DISPATCHLOAD
    # -------------------------------------------------------------------------

    "DISPATCHLOAD": [

        bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("DUID", "STRING"),

        bigquery.SchemaField("INITIALMW", "FLOAT"),
        bigquery.SchemaField("TOTALCLEARED", "FLOAT"),

        bigquery.SchemaField("RAMPDOWNRATE", "FLOAT"),
        bigquery.SchemaField("RAMPUPRATE", "FLOAT"),

        bigquery.SchemaField("AVAILABILITY", "FLOAT"),
        bigquery.SchemaField("SEMIDISPATCHCAP", "FLOAT"),

        bigquery.SchemaField("AGCSTATUS", "STRING"),
        bigquery.SchemaField("DISPATCHMODE", "STRING"),

        bigquery.SchemaField("RAISE6SEC", "FLOAT"),
        bigquery.SchemaField("RAISE60SEC", "FLOAT"),
        bigquery.SchemaField("RAISE5MIN", "FLOAT"),
        bigquery.SchemaField("RAISEREG", "FLOAT"),

        bigquery.SchemaField("LOWER6SEC", "FLOAT"),
        bigquery.SchemaField("LOWER60SEC", "FLOAT"),
        bigquery.SchemaField("LOWER5MIN", "FLOAT"),
        bigquery.SchemaField("LOWERREG", "FLOAT")

    ],

    # -------------------------------------------------------------------------
    # REGION
    # -------------------------------------------------------------------------

    "REGION": [

        bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("REGIONID", "STRING"),

        bigquery.SchemaField("TOTALDEMAND", "FLOAT"),
        bigquery.SchemaField("DEMANDFORECAST", "FLOAT"),

        bigquery.SchemaField("INITIALSUPPLY", "FLOAT"),
        bigquery.SchemaField("AVAILABLEGENERATION", "FLOAT")

    ],

    # -------------------------------------------------------------------------
    # CONSTRAINT
    # -------------------------------------------------------------------------

    "CONSTRAINT": [

        bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("CONSTRAINTID", "STRING"),

        bigquery.SchemaField("RHS", "FLOAT"),
        bigquery.SchemaField("LHS", "FLOAT"),

        bigquery.SchemaField("MARGINALVALUE", "FLOAT"),
        bigquery.SchemaField("VIOLATIONDEGREE", "FLOAT")

    ],

    # -------------------------------------------------------------------------
    # INTERCONNECTOR
    # -------------------------------------------------------------------------

    "INTERCONNECTOR": [

        bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("INTERCONNECTORID", "STRING"),

        bigquery.SchemaField("MWFLOW", "FLOAT"),
        bigquery.SchemaField("MWLOSSES", "FLOAT"),

        bigquery.SchemaField("EXPORTLIMIT", "FLOAT"),
        bigquery.SchemaField("IMPORTLIMIT", "FLOAT")

    ],

    # -------------------------------------------------------------------------
    # DUDETAIL
    # -------------------------------------------------------------------------

    "DUDETAIL": [

    bigquery.SchemaField("EFFECTIVEDATE", "TIMESTAMP"),

    bigquery.SchemaField("DUID", "STRING"),

    bigquery.SchemaField("VERSIONNO", "INTEGER"),

    bigquery.SchemaField("CONNECTIONPOINTID", "STRING"),

    bigquery.SchemaField("REGISTEREDCAPACITY", "FLOAT"),

    bigquery.SchemaField("AGCCAPABILITY", "STRING"),

    bigquery.SchemaField("DISPATCHTYPE", "STRING"),

    bigquery.SchemaField("MAXCAPACITY", "FLOAT"),

    bigquery.SchemaField("STARTTYPE", "STRING"),

    bigquery.SchemaField("NORMALLYONFLAG", "STRING"),

    bigquery.SchemaField("LASTCHANGED", "TIMESTAMP")

],
    # -------------------------------------------------------------------------
    # DATA_QUALITY
    # -------------------------------------------------------------------------

    "DATA_QUALITY": [

        bigquery.SchemaField("TABLE_NAME", "STRING"),
        bigquery.SchemaField("CHECK_TIME", "TIMESTAMP"),

        bigquery.SchemaField("ROW_COUNT", "INTEGER"),
        bigquery.SchemaField("DUPLICATE_ROWS", "INTEGER"),

        bigquery.SchemaField("MISSING_VALUES", "INTEGER"),
        bigquery.SchemaField("STATUS", "STRING")

    ]

}

print("Master schema definitions loaded successfully.")

GOOGLE AUTHENTICATION

In [ ]:
# =============================================================================
# SECTION 6.5 — LOCAL GOOGLE AUTHENTICATION
# =============================================================================

import os
from google.cloud import bigquery

# We check if the key exists before starting
if os.path.exists(KEY_PATH):
    # This tells Google libraries to use your local JSON key
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

    # Initialize the client
    client = bigquery.Client(project=PROJECT_ID)

    # Test the connection
    datasets = list(client.list_datasets())
    print(f"✅ Local Authentication successful. Connected to project: {PROJECT_ID}")
else:
    print(f"❌ ERROR: Key file not found at {KEY_PATH}")
    print("Please check Section 3 to ensure your KEY_PATH is correct.")

BIGQUERY TABLE CREATION

In [ ]:
# =============================================================================
# SECTION 7 — BIGQUERY TABLE CREATION
# =============================================================================

# -----------------------------------------------------------------------------
# Create BigQuery Client
# -----------------------------------------------------------------------------

client = bigquery.Client(project=PROJECT_ID)

print("BigQuery client initialised.")


# -----------------------------------------------------------------------------
# Create Dataset If Not Exists
# -----------------------------------------------------------------------------

def create_dataset_if_not_exists():

    dataset_ref = bigquery.Dataset(
        f"{PROJECT_ID}.{DATASET_ID}"
    )

    try:

        client.get_dataset(dataset_ref)

        print(f"Dataset already exists: {DATASET_ID}")

    except:

        client.create_dataset(dataset_ref)

        print(f"Created dataset: {DATASET_ID}")


# -----------------------------------------------------------------------------
# Create Warehouse Tables
# -----------------------------------------------------------------------------

def create_master_schema_tables():

    create_dataset_if_not_exists()

    for table_name, schema in MASTER_TABLE_SCHEMAS.items():

        table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

        table = bigquery.Table(
            table_id,
            schema=schema
        )

        try:

            client.get_table(table_id)

            print(f"Table already exists: {table_name}")

        except:

            client.create_table(table)

            print(f"Created table: {table_name}")


# -----------------------------------------------------------------------------
# Execute Table Creation
# -----------------------------------------------------------------------------

create_master_schema_tables()

print("\nWarehouse schema creation complete.")

WAREHOUSE RESET

In [ ]:
# =============================================================================
# SECTION 7.5 — WAREHOUSE RESET
# =============================================================================

def reset_master_tables():

    for table_name in MASTER_TABLE_SCHEMAS.keys():

        table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

        try:

            client.delete_table(
                table_id,
                not_found_ok=True
            )

            print(f"Deleted: {table_name}")

        except Exception as e:

            print(f"Failed deleting {table_name}")

            print(e)


## reset_master_tables()

print("\nWarehouse reset complete.")

Realtime_dispatch (Pipline A)

In [ ]:
# =============================================================================
# SECTION 8 — REALTIME DISPATCH INGESTION
# =============================================================================

# -----------------------------------------------------------------------------
# Discover Latest DispatchIS File
# -----------------------------------------------------------------------------

def discover_latest_dispatchis_file():

    response = requests.get(
        DISPATCHIS_URL,
        headers=HEADERS
    )

    if response.status_code != 200:

        raise Exception(
            f"Failed to access DispatchIS page: {response.status_code}"
        )

    zip_files = []

    for line in response.text.splitlines():

        if ".zip" in line and "PUBLIC_DISPATCHIS" in line:

            start = line.find("PUBLIC_DISPATCHIS")

            end = line.find(".zip") + 4

            filename = line[start:end]

            zip_files.append(filename)

    if not zip_files:

        raise Exception("No DispatchIS zip files discovered.")

    latest_file = sorted(zip_files)[-1]

    print(f"Latest file discovered: {latest_file}")

    return latest_file


# -----------------------------------------------------------------------------
# Download DispatchIS Zip
# -----------------------------------------------------------------------------

def download_dispatchis_zip(filename):

    file_url = DISPATCHIS_URL + filename

    response = requests.get(
        file_url,
        headers=HEADERS
    )

    if response.status_code != 200:

        raise Exception(
            f"Failed to download file: {response.status_code}"
        )

    print(f"Downloaded: {filename}")

    return io.BytesIO(response.content)


# -----------------------------------------------------------------------------
# Extract DispatchIS CSV
# -----------------------------------------------------------------------------

def extract_dispatchis_csv(zip_buffer):
    """
    Extract CSV from zip, skip the metadata header row,
    and return a DataFrame with no header (raw data).
    This handles the actual DispatchIS format where the first row is metadata.
    """
    with zipfile.ZipFile(zip_buffer) as z:
        csv_name = [f for f in z.namelist() if f.endswith(".CSV")][0]
        print(f"Processing CSV: {csv_name}")
        with z.open(csv_name) as csv_file:
            # Read all lines, decode, skip the first metadata row
            lines = [line.decode("utf-8", errors="ignore") for line in csv_file.readlines()]
            if not lines:
                raise ValueError("CSV file is empty")
            # First line is metadata (e.g., "C          NEMP.WORLD ..."), skip it
            data_lines = lines[1:]
            # Convert to DataFrame using StringIO, no header
            from io import StringIO
            df = pd.read_csv(StringIO(''.join(data_lines)), header=None, low_memory=False, on_bad_lines="skip")
    print(f"Loaded DataFrame: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df


# -----------------------------------------------------------------------------
# Parse DispatchIS Tables
# -----------------------------------------------------------------------------

def parse_dispatchis_tables(df):
    """
    Parse the raw DispatchIS DataFrame into separate tables based on the report type
    found in the first column (e.g., 'I DISPATCH CASE_SOLUTION').
    This replaces the old method that incorrectly looked at column index 2.
    """
    print("\nDEBUG: Input DataFrame shape:", df.shape)

    raw_tables = {}

    # Map from report name (as it appears after "DISPATCH " in first column) to output key
    report_type_map = {
        "PRICE": "PRICE",
        "REGIONSUM": "REGIONSUM",
        "CONSTRAINT": "CONSTRAINT",
        "INTERCONNECTORRES": "INTERCONNECTORRES",
        "INTERCONNECTION": "INTERCONNECTION",
        "LOCAL_PRICE": "LOCAL_PRICE",
        "CASE_SOLUTION": "CASE_SOLUTION"
    }

    # Extract the first column (index 0) as strings for pattern matching
    first_col = df.iloc[:, 0].astype(str)

    for report_name, table_name in report_type_map.items():
        # Create a mask: rows where first column contains "DISPATCH <report_name>"
        mask = first_col.str.contains(f"DISPATCH {report_name}", na=False)
        subset = df[mask].copy()

        if not subset.empty:
            raw_tables[table_name] = subset
            print(f"{table_name}: {subset.shape[0]:,} rows × {subset.shape[1]} cols")
        else:
            print(f"WARNING: {report_name} not found in DataFrame")

    print(f"\nTotal tables extracted: {len(raw_tables)}")
    return raw_tables


# -----------------------------------------------------------------------------
# Execute Realtime Dispatch Pipeline
# -----------------------------------------------------------------------------

def realtime_dispatch():

    print("\n" + "=" * 70)
    print("REALTIME DISPATCH INGESTION")
    print("=" * 70)

    latest_file = discover_latest_dispatchis_file()

    zip_buffer = download_dispatchis_zip(latest_file)

    raw_df = extract_dispatchis_csv(zip_buffer)

    raw_tables = parse_dispatchis_tables(raw_df)

    print(f"\nTotal tables extracted: {len(raw_tables)}")

    return raw_tables


print("Realtime dispatch ingestion module loaded.")

WAREHOUSE TRANSFORMS

In [ ]:
# =============================================================================
# SECTION 9 — WAREHOUSE TRANSFORMS (UPDATED FOR BIDDING)
# =============================================================================

def warehouse_transforms(raw_tables):
    master_tables = {}

    # 1. DISPATCHPRICE
    if "DISPATCHPRICE" in raw_tables and not raw_tables["DISPATCHPRICE"].empty:
        master_tables["DISPATCHPRICE"] = transform_dispatchprice(raw_tables["DISPATCHPRICE"])

    # 2. DISPATCHLOAD
    if "DISPATCHLOAD" in raw_tables and not raw_tables["DISPATCHLOAD"].empty:
        master_tables["DISPATCHLOAD"] = transform_dispatchload(raw_tables["DISPATCHLOAD"])

    # 3. REGION
    if "REGION" in raw_tables and not raw_tables["REGION"].empty:
        master_tables["REGION"] = transform_region(raw_tables["REGION"])

    # 4. BIDDING DATA (NEW LOGIC)
    if "BIDDAYOFFER" in raw_tables and not raw_tables["BIDDAYOFFER"].empty:
        master_tables["BIDDAYOFFER"] = prepare_master_table(raw_tables["BIDDAYOFFER"])

    if "BIDPEROFFER" in raw_tables and not raw_tables["BIDPEROFFER"].empty:
        master_tables["BIDPEROFFER"] = prepare_master_table(raw_tables["BIDPEROFFER"])

    print(f"✅ Warehouse transforms complete for: {list(master_tables.keys())}")
    return master_tables

Historical_market (Pipeline C)

In [ ]:
# =============================================================================
# SECTION 10 — HISTORICAL MARKET INGESTION (REVISED)
# =============================================================================

HISTORICAL_TABLE_MAP = {
    "DISPATCHPRICE": "DISPATCHPRICE",
    "DISPATCHREGIONSUM": "REGION",
    "DISPATCHCONSTRAINT": "CONSTRAINT",
    "DISPATCHINTERCONNECTORRES": "INTERCONNECTOR",
    "DISPATCHLOAD": "DISPATCHLOAD"
}

def fetch_historical_table(table_name, start_time, end_time):
    print(f"   📥 FETCHING: {table_name}...")
    try:
        df = dynamic_data_compiler(
            start_time=start_time,
            end_time=end_time,
            table_name=table_name,
            raw_data_location=CACHE_DIR
        )
        return df if df is not None else pd.DataFrame()
    except Exception as e:
        print(f"   ⚠️ Could not fetch {table_name}: {e}")
        return pd.DataFrame()

def historical_market(start_time, end_time, table_map=None):
    # If no specific map is provided, use the full default map
    if table_map is None:
        table_map = HISTORICAL_TABLE_MAP

    print(f"\n--- Ingesting {len(table_map)} tables for period: {start_time} ---")
    raw_tables = {}

    for source_table, warehouse_table in table_map.items():
        df = fetch_historical_table(source_table, start_time, end_time)
        if not df.empty:
            raw_tables[warehouse_table] = df
            print(f"   ✅ SUCCESS: {source_table}")

    return raw_tables

In [ ]:
historical_raw_tables = historical_market(
    start_time="2023/01/01 00:00:00",
    end_time="2023/01/02 00:00:00"
)

BIGQUERY LOADERS

In [ ]:
# =============================================================================
# SECTION 11 — BIGQUERY LOADERS (FLEXIBLE)
# =============================================================================

def load_table_to_bigquery(df, table_name):
    if df.empty: return
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    # These flags allow Pipeline A & C to change the table schema automatically
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        autodetect=True,
        schema_update_options=[
            bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
            bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION
        ]
    )
    print(f"LOADING: {table_name} ({len(df):,} rows)")
    client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
    print(f"✅ SUCCESS: {table_name} loaded.")

def load_master_tables(master_tables):
    for table_name, df in master_tables.items():
        try:
            load_table_to_bigquery(df, table_name)
        except Exception as e:
            print(f"❌ FAILED {table_name}: {e}")

PIPELINE ORCHESTRATION

In [ ]:
# =============================================================================
# SECTION 12 — PIPELINE ORCHESTRATION
# =============================================================================

# -----------------------------------------------------------------------------
# Run Historical Pipeline
# -----------------------------------------------------------------------------

def run_realtime_pipeline():

    print("\n" + "=" * 80)
    print("RUNNING REALTIME PIPELINE")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Ingestion
    # -------------------------------------------------------------------------

    realtime_raw_tables = realtime_dispatch()

    # -------------------------------------------------------------------------
    # Standardisation
    # -------------------------------------------------------------------------

    realtime_standardised = standardise_realtime_tables(
        realtime_raw_tables
    )

    # -------------------------------------------------------------------------
    # Transforms
    # -------------------------------------------------------------------------

    realtime_master_tables = warehouse_transforms(
        raw_tables=realtime_standardised
    )

    # DEBUG: print table names and row counts before loading
    print("\nDEBUG: realtime_master_tables keys:", realtime_master_tables.keys())
    for name, df in realtime_master_tables.items():
        print(f"DEBUG: {name} has {len(df):,} rows")

    # -------------------------------------------------------------------------
    # BigQuery Load
    # -------------------------------------------------------------------------

    load_master_tables(
        realtime_master_tables
    )

    print("\nRealtime pipeline complete.")

    return realtime_master_tables

ASSET METADATA (pipeline D)

In [ ]:
# =============================================================================
# SECTION 13 — ASSET METADATA
# =============================================================================

# -----------------------------------------------------------------------------
# Asset Metadata Table Map
# -----------------------------------------------------------------------------

ASSET_METADATA_TABLE_MAP = {

    "DUDETAIL": "DUDETAIL"

}


# -----------------------------------------------------------------------------
# Fetch Asset Metadata Table
# -----------------------------------------------------------------------------

def fetch_asset_metadata_table(
    table_name,
    start_time,
    end_time
):

    print("\n" + "=" * 60)
    print(f"FETCHING ASSET METADATA: {table_name}")
    print("=" * 60)

    df = dynamic_data_compiler(

        start_time=start_time,

        end_time=end_time,

        table_name=table_name,

        raw_data_location=CACHE_DIR

    )

    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} cols")

    return df


# -----------------------------------------------------------------------------
# Asset Metadata Ingestion
# -----------------------------------------------------------------------------

def asset_metadata(
    start_time,
    end_time
):

    raw_tables = {}

    print("\n" + "=" * 70)
    print("ASSET METADATA INGESTION")
    print("=" * 70)

    for source_table, warehouse_table in ASSET_METADATA_TABLE_MAP.items():

        try:

            df = fetch_asset_metadata_table(
                source_table,
                start_time,
                end_time
            )

            raw_tables[warehouse_table] = df

            print(f"SUCCESS: {source_table}")

        except Exception as e:

            print(f"FAILED: {source_table}")

            print(e)

    return raw_tables


print("Asset metadata module loaded.")

In [ ]:
asset_raw_tables = asset_metadata(
    start_time="2023/01/01 00:00:00",
    end_time="2023/01/02 00:00:00"
)

DUDETAIL TRANSFORMS

In [ ]:
# =============================================================================
# SECTION 14 — DUDETAIL TRANSFORMS (REVISED)
# =============================================================================

def transform_dudetail(dudetail_df):
    if dudetail_df is None or dudetail_df.empty:
        print("⚠️ No DUDETAIL data to transform.")
        return pd.DataFrame()

    required_columns = ["EFFECTIVEDATE", "DUID"]
    validate_required_columns(dudetail_df, required_columns, "DUDETAIL")

    available_columns = [
        col for col in [
            "EFFECTIVEDATE", "DUID", "VERSIONNO", "CONNECTIONPOINTID",
            "REGISTEREDCAPACITY", "AGCCAPABILITY", "DISPATCHTYPE",
            "MAXCAPACITY", "STARTTYPE", "NORMALLYONFLAG", "LASTCHANGED"
        ]
        if col in dudetail_df.columns
    ]

    master_df = dudetail_df[available_columns].copy()
    return prepare_master_table(master_df)

print("✅ DUDETAIL transform module loaded.")

In [ ]:
dudetail_master = transform_dudetail(
    asset_raw_tables["DUDETAIL"]
)

In [ ]:
load_master_tables({"DUDETAIL": dudetail_master})

BIDDING DATA (Pipeline E)

In [ ]:
# =============================================================================
# SECTION 15 — THE BIDDING MARATHON (FINAL SURGICAL VERSION)
# =============================================================================
import calendar

def run_monthly_bidding_load(year, months):
    """
    Surgically pulls 5-minute bidding data.
    Uses the exact keys required by NEMOSIS v3.8.1.
    """
    # These are the ONLY keys NEMOSIS v3.8.1 accepts for 5-minute bids
    BID_TARGETS = {
        "BIDDAYOFFER_D": "BIDDAYOFFER",
        "BIDPEROFFER_D": "BIDPEROFFER"
    }

    print(f"🚀 TANK BIDDING MISSION: {year} | Target Tables: {list(BID_TARGETS.keys())}")

    for month in months:
        last_day = calendar.monthrange(year, month)[1]
        # We use a slightly wider range to ensure NEMOSIS catches the 'overlap' files
        start_str = f"{year}/{month:02d}/01 00:00:00"
        end_str = f"{year}/{month:02d}/{last_day:02d} 23:59:59"

        print(f"\n--- 📅 PROCESSING: {year}-{month:02d} ---")
        try:
            # We call historical_market using the verified _D keys
            raw_data = historical_market(start_str, end_str, table_map=BID_TARGETS)

            if raw_data and any(not df.empty for df in raw_data.values()):
                master_tables = warehouse_transforms(raw_data)
                load_master_tables(master_tables)

                # RAM & SSD Cleanup
                del raw_data, master_tables
                gc.collect()
                for f in os.listdir(CACHE_DIR):
                    path = os.path.join(CACHE_DIR, f)
                    if os.path.isfile(path):
                        try: os.unlink(path)
                        except: pass
            else:
                print(f"⚠️ AEMO Archive not responding for {year}-{month:02d}. Skipping.")
        except Exception as e:
            print(f"❌ Error in {year}-{month:02d}: {e}")

print("✅ Surgical Bidding Orchestrator Loaded.")

In [ ]:
# EXECUTE THE BIDDING MARATHON
# We do 2023, 2024, and 2025.
# Total row count expected: ~300 million records.
run_bidding_marathon([2023, 2024, 2025])

In [ ]:
bidding_raw_tables = bidding_data(
    start_time="2023/01/01 00:00:00",
    end_time="2023/01/02 00:00:00"
)

REALTIME STANDARDISATION

In [ ]:
# =============================================================================
# SECTION 16 — REALTIME STANDARDISATION
# =============================================================================

# -----------------------------------------------------------------------------
# Realtime Table Mapping
# -----------------------------------------------------------------------------

def standardise_realtime_tables(realtime_raw_tables):

    print("\n" + "=" * 70)
    print("STANDARDISING REALTIME TABLES")
    print("=" * 70)

    # DEBUG: show what keys came from the raw tables
    print("DEBUG: keys in realtime_raw_tables:", list(realtime_raw_tables.keys()))

    standardised_tables = {}

    REALTIME_TABLE_MAP = {
        "PRICE": "DISPATCHPRICE",
        "REGIONSUM": "REGION",
        "CONSTRAINT": "CONSTRAINT",
        "INTERCONNECTORRES": "INTERCONNECTOR"
    }

    for source_table, warehouse_table in REALTIME_TABLE_MAP.items():
        if source_table in realtime_raw_tables:
            standardised_tables[warehouse_table] = realtime_raw_tables[source_table]
            print(f"{source_table} → {warehouse_table} (rows: {len(realtime_raw_tables[source_table]):,})")
        else:
            print(f"WARNING: {source_table} not found in raw tables")

    return standardised_tables

In [ ]:
realtime_master_tables = run_realtime_pipeline()

In [ ]:
from nemosis import defaults

print("\nDynamic tables:")
for table in sorted(defaults.dynamic_tables):
    if any(
        keyword in table.upper()
        for keyword in [
            "DU",
            "LOAD",
            "FCAS",
            "BID",
            "SCADA"
        ]
    ):
        print(table)

In [ ]:
import nemosis.defaults as defaults

print(dir(defaults))

In [ ]:
[x for x in dir(defaults) if "table" in x.lower()]

In [ ]:
from nemosis import defaults

print(type(defaults.table_types))
print(list(defaults.table_types.keys())[:20])

In [ ]:
print(defaults.table_types.get("DISPATCHLOAD"))
print(defaults.table_types.get("DUDETAIL"))
print(defaults.table_types.get("BIDDAYOFFER_D"))
print(defaults.table_types.get("FCAS_4_SECOND"))

In [ ]:
print(defaults.table_columns["DISPATCHLOAD"][:20])

In [ ]:
print(defaults.table_primary_keys["DISPATCHLOAD"])

In [ ]:
from nemosis import dynamic_data_compiler

test = dynamic_data_compiler(
    start_time="2024/01/01 00:00:00",
    end_time="2024/01/01 00:05:00",
    table_name="DISPATCHPRICE",
    raw_data_location=CACHE_DIR
)

print(test.shape)
print(test.head())

In [ ]:
dynamic_data_compiler(
    start_time="2024/01/01 00:00:00",
    end_time="2024/01/01 00:05:00",
    table_name="DISPATCHLOAD",
    raw_data_location=CACHE_DIR
)

In [ ]:
dynamic_data_compiler(
    start_time="2024/01/01 00:00:00",
    end_time="2024/01/01 00:05:00",
    table_name="DISPATCH_UNIT_SCADA",
    raw_data_location=CACHE_DIR
)

In [ ]:
dynamic_data_compiler(
    start_time="2024/01/01 00:00:00",
    end_time="2024/01/02 00:00:00",
    table_name="DUDETAIL",
    raw_data_location=CACHE_DIR
)

In [ ]:
print("DISPATCHPRICE rows:", client.query("SELECT COUNT(*) as c FROM `master-aemo.aemo_master_schema.DISPATCHPRICE`").to_dataframe()['c'][0])
print("REGION rows:", client.query("SELECT COUNT(*) as c FROM `master-aemo.aemo_master_schema.REGION`").to_dataframe()['c'][0])
print("DISPATCHLOAD rows:", client.query("SELECT COUNT(*) as c FROM `master-aemo.aemo_master_schema.DISPATCHLOAD`").to_dataframe()['c'][0])
print("DUDETAIL rows:", client.query("SELECT COUNT(*) as c FROM `master-aemo.aemo_master_schema.DUDETAIL`").to_dataframe()['c'][0])

In [ ]:
# Manual test of realtime extraction
latest = discover_latest_dispatchis_file()
print("Latest file:", latest)
zip_buffer = download_dispatchis_zip(latest)
df = extract_dispatchis_csv(zip_buffer)
print("Full DataFrame shape:", df.shape)
print("First 3 rows:")
print(df.head(3))
print("\nUnique report types (column 2):")
print(df.iloc[:,2].unique())

In [ ]:
# Check multiple recent DispatchIS files for PRICE and REGIONSUM
from datetime import datetime, timedelta

def check_recent_files(hours_back=1):
    found_price = False
    found_region = False
    for i in range(hours_back * 6):  # assume file every 10 min
        try:
            latest = discover_latest_dispatchis_file()
            zip_buffer = download_dispatchis_zip(latest)
            df = extract_dispatchis_csv(zip_buffer)
            first_col = df.iloc[:, 0].astype(str)
            if first_col.str.contains("DISPATCH PRICE", na=False).any():
                found_price = True
            if first_col.str.contains("DISPATCH REGIONSUM", na=False).any():
                found_region = True
            if found_price and found_region:
                break
        except Exception as e:
            print(f"Error: {e}")
        time.sleep(2)  # be gentle
    print(f"PRICE found: {found_price}, REGIONSUM found: {found_region}")


## Production Workflows

In [ ]:
# =============================================================================
# SECTION 17 — PRODUCTION DAILY UPDATER
# =============================================================================

def run_daily_update(days_back=1):
    """
    Automated task to fetch data from the past X days and append to BigQuery.
    Use days_back=1 for daily maintenance.
    Use days_back=7 or 30 for initial backfilling.
    """
    # 1. Calculate Date Range (AEMO format)
    end_dt = datetime.now()
    start_dt = end_dt - timedelta(days=days_back)

    start_str = start_dt.strftime("%Y/%m/%d 00:00:00")
    end_str = end_dt.strftime("%Y/%m/%d %H:%M:%S")

    print(f"\nPROD: Starting Ingestion for period: {start_str} to {end_str}")

    try:
        # 2. Fetch via NEMOSIS (Historical is more stable for daily totals)
        raw_data = historical_market(start_str, end_str)

        # 3. Transform to Master Schema
        master_tables = warehouse_transforms(raw_data)

        # 4. Load to BigQuery (APPEND mode)
        load_master_tables(master_tables)

        print(f"PROD SUCCESS: {days_back} day(s) of data added to Warehouse.")
    except Exception as e:
        print(f"PROD ERROR: Daily update failed. {e}")

# To keep your data updating daily, simply run this line:
# run_daily_update(days_back=1)

In [ ]:
# =============================================================================
# SECTION 17.5 — BACKFILL ORCHESTRATION (TANK POWERED)
# =============================================================================

def run_backfill_years(start_year, end_year, tables_to_run=None):
    """
    Orchestrates the backfill. Uses 'tables_to_run' to target
    specific gaps (e.g., just DISPATCHLOAD).
    """
    for year in range(start_year, end_year + 1):
        print(f"\n🚀 TANK STARTING YEAR: {year}")

        start_str = f"{year}/01/01 00:00:00"
        end_str = f"{year}/12/31 23:59:59"

        # 1. Ingest only what we requested
        raw_data = historical_market(start_str, end_str, table_map=tables_to_run)

        if raw_data:
            # 2. Transform (Pipeline Logic)
            master_tables = warehouse_transforms(raw_data)

            # 3. Load (BigQuery Flexible Loader)
            load_master_tables(master_tables)

            # --- TANK RAM & DISK CLEANUP ---
            del raw_data, master_tables
            gc.collect()

            # Clear SSD Cache to keep the Tank lean
            for f in os.listdir(CACHE_DIR):
                path = os.path.join(CACHE_DIR, f)
                if os.path.isfile(path):
                    try: os.unlink(path)
                    except: pass
        else:
            print(f"⚠️ No data found for {year} targets.")

## Tank Control Room

In [ ]:
# =============================================================================
# SECTION 18 — THE GRAND BACKFILL ORCHESTRATOR (2015 - 2026)
# =============================================================================
"""
PURPOSE: This is the 'Control Room' of the Tank. It utilizes 96GB of local RAM
to ingest high-resolution AEMO data directly into Google BigQuery.

ARCHITECTURAL NOTE:
- Years 2023-2026 are already verified and complete in our warehouse.
- This run targets the 'Deep History' (2015-2022) to support decadal
  climate-correlation and BESS revenue modeling.
"""

# --- 1. SET TARGETS ---
# We fetch Price, Region (Demand), and the massive DispatchLoad table.
HISTORICAL_TARGETS = {
    "DISPATCHPRICE": "DISPATCHPRICE",
    "DISPATCHREGIONSUM": "REGION",
    "DISPATCHLOAD": "DISPATCHLOAD"
}

# --- 2. SET CHRONOLOGY ---
# Select the decade of interest.
# Current Mission: The 2015-2022 Gap.
YEARS_TO_PROCESS = [2018, 2019, 2020, 2021, 2022]

# --- 3. EXECUTE MARATHON ---
print(f"🚀 TANK MISSION STARTING: {min(YEARS_TO_PROCESS)} -> {max(YEARS_TO_PROCESS)}")
print(f"📊 Target Dataset: {PROJECT_ID}.{DATASET_ID}")

for year in YEARS_TO_PROCESS:
    print(f"\n" + "#" * 60)
    print(f"### CURRENT YEAR TARGET: {year}")
    print("#" * 60)

    try:
        # Calls the modular orchestrator from Section 17.5
        # This function handles the 96GB RAM loading and BigQuery Push
        run_backfill_years(
            start_year=year,
            end_year=year,
            tables_to_run=HISTORICAL_TARGETS
        )

        print(f"\n✅ YEAR COMPLETE: {year}")

    except Exception as e:
        print(f"❌ CRITICAL ERROR in Year {year}: {e}")
        print("Continuing to next year...")
        continue

print(f"\n" + "=" * 60)
print("🏆 GRAND BACKFILL ACCOMPLISHED")
print(f"Processed {len(YEARS_TO_PROCESS)} years of high-resolution data.")
print("=" * 60)

In [ ]:
# =============================================================================
# SECTION 19 — UNIFIED BIDDING INGESTOR (2023 - 2025)
# =============================================================================
import requests
import zipfile
import io
import time

def ingest_bidding_mission_complete(year, months):
    """
    Directly targets AEMO Archives for maximum reliability.
    Handles both modern folders and deep archive folders.
    """
    for month in months:
        # AEMO uses two different URL patterns. We check both.
        # Pattern 1: MMSDM (Deep Archive - 2023)
        # Pattern 2: MMS (Recent Archive - 2024+)
        url_patterns = [
            f"https://nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/{year}/MMSDM_{year}_{month:02d}/MMSDM_Historical_Data_SQLLoader/DATA/",
            f"https://nemweb.com.au/Data_Archive/Wholesale_Electricity/MMS/{year}/MMS_Data_{year}_{month:02d}_01.zip"
        ]

        print(f"\n📂 MISSION: {year}-{month:02d}")

        # We try to find the BIDDAYOFFER and BIDPEROFFER (including split files)
        file_prefixes = ["PUBLIC_DVD_BIDDAYOFFER", "PUBLIC_DVD_BIDPEROFFER1", "PUBLIC_DVD_BIDPEROFFER2", "PUBLIC_DVD_BIDPEROFFER_D"]

        for prefix in file_prefixes:
            # Construct the possible filenames
            zip_name = f"{prefix}_{year}{month:02d}010000.zip"
            # Some months omit the day '01' in the zip name, we handle that too
            possible_urls = [url_patterns[0] + zip_name]

            for file_url in possible_urls:
                for attempt in range(3): # 3 Retries for stability
                    try:
                        r = requests.get(file_url, stream=True, timeout=60)
                        if r.status_code == 200:
                            print(f"   📥 Downloading: {zip_name}")
                            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                                for csv_name in z.namelist():
                                    with z.open(csv_name) as f:
                                        df = pd.read_csv(f, skiprows=1, low_memory=False)
                                        bq_name = "BIDDAYOFFER" if "BIDDAYOFFER" in csv_name else "BIDPEROFFER"
                                        df_clean = prepare_master_table(df)
                                        load_table_to_bigquery(df_clean, bq_name)
                                        del df, df_clean
                                        gc.collect()
                            print(f"   ✅ SUCCESS: {zip_name} loaded.")
                            break
                        elif r.status_code == 404:
                            break # Silent skip if file doesn't exist
                    except Exception as e:
                        print(f"   ⚠️ Attempt {attempt+1} failed for {zip_name}. Retrying in 10s...")
                        time.sleep(10)
                        if attempt == 2: print(f"   ❌ Network failure on {zip_name}: {e}")

# --- THE EXECUTION CONTROL ---
# 1. Fill the 2023 Void (The hardest year)
print("🚀 STARTING THE 2023 ARCHIVE INGESTION...")
ingest_bidding_mission_complete(2023, range(1, 13))

# 2. Fill the early 2024 Gap (Jan - July)
print("\n🚀 STARTING THE 2024 ARCHIVE INGESTION...")
ingest_bidding_mission_complete(2024, range(1, 8))

print("\n🏆 MISSION ACCOMPLISHED: ALL DATA SECURED.")

In [ ]:
# =============================================================================
# SECTION 19A — AEMO BIDDING ARCHIVE DISCOVERY
# =============================================================================

import requests
from bs4 import BeautifulSoup


def discover_bidding_files(year, month):
    """
    Inspects an AEMO MMSDM monthly archive and lists all files
    related to bidding data.

    This function DOES NOT download or load anything.
    It is purely a discovery and diagnostic tool.
    """

    # -------------------------------------------------------------------------
    # Build the AEMO monthly archive URL
    # -------------------------------------------------------------------------

    archive_url = (
        f"https://www.nemweb.com.au/"
        f"Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/"
        f"MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    print("=" * 80)
    print("AEMO BIDDING ARCHIVE DISCOVERY")
    print("=" * 80)

    print(f"\n📅 Target month: {year}-{month:02d}")
    print(f"🌐 Archive URL:")
    print(archive_url)

    # -------------------------------------------------------------------------
    # Connect to AEMO
    # -------------------------------------------------------------------------

    try:

        response = requests.get(
            archive_url,
            timeout=60
        )

        print(f"\n📡 HTTP Status: {response.status_code}")

        if response.status_code != 200:

            print("\n❌ Could not access this archive directory.")

            return []

    except Exception as e:

        print(f"\n❌ CONNECTION FAILED: {e}")

        return []

    # -------------------------------------------------------------------------
    # Parse the directory listing
    # -------------------------------------------------------------------------

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    links = [
        link.get("href")
        for link in soup.find_all("a")
        if link.get("href")
    ]

    # -------------------------------------------------------------------------
    # Search specifically for bidding-related files
    # -------------------------------------------------------------------------

    bidding_keywords = [

        "BIDDAYOFFER",
        "BIDOFFERPERIOD",
        "BIDPEROFFER"

    ]

    bidding_files = []

    for link in links:

        link_upper = link.upper()

        if any(
            keyword in link_upper
            for keyword in bidding_keywords
        ):

            bidding_files.append(link)

    # -------------------------------------------------------------------------
    # Display Results
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("BIDDING FILES FOUND")
    print("=" * 80)

    if not bidding_files:

        print("\n⚠️ NO BIDDING FILES FOUND.")

        print(
            "\nThis means either:"
            "\n1. The archive structure is different for this month, or"
            "\n2. The bidding files are stored elsewhere."
        )

    else:

        print(f"\n✅ FOUND {len(bidding_files)} BIDDING FILE(S):\n")

        for file in sorted(bidding_files):

            print(f"📦 {file}")

    print("\n" + "=" * 80)

    return bidding_files


# =============================================================================
# TEST — START WITH SEPTEMBER 2024
# =============================================================================

bidding_files = discover_bidding_files(
    year=2024,
    month=9
)

In [ ]:
# =============================================================================
# BIDDING FILE RECONNAISSANCE — INSPECT ONE REAL AEMO FILE
# =============================================================================

import requests
import zipfile
import io
import pandas as pd

url = (
    "https://www.nemweb.com.au"
    "/Data_Archive/Wholesale_Electricity/MMSDM/2024/"
    "MMSDM_2024_09/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    "PUBLIC_ARCHIVE%23BIDDAYOFFER%23FILE01%23202409010000.zip"
)

print("=" * 80)
print("DOWNLOADING REAL BIDDAYOFFER FILE")
print("=" * 80)

response = requests.get(url, timeout=120)

print(f"\nHTTP Status: {response.status_code}")
print(f"Download size: {len(response.content):,} bytes")

response.raise_for_status()

zip_buffer = io.BytesIO(response.content)

with zipfile.ZipFile(zip_buffer) as z:

    print("\nFILES INSIDE ZIP:")
    print("-" * 80)

    for name in z.namelist():
        print(name)

    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    print("\n" + "=" * 80)
    print("READING FIRST CSV")
    print("=" * 80)

    csv_name = csv_files[0]

    print(f"\nCSV: {csv_name}")

    with z.open(csv_name) as f:

        # First inspect raw lines
        print("\nFIRST 5 RAW LINES:")
        print("-" * 80)

        raw_lines = []

        for _ in range(5):
            line = f.readline().decode(
                "utf-8",
                errors="replace"
            )
            raw_lines.append(line)
            print(line[:500])

    # Re-open because we already consumed the file
    with z.open(csv_name) as f:

        df_test = pd.read_csv(
            f,
            skiprows=1,
            low_memory=False
        )

print("\n" + "=" * 80)
print("DATAFRAME INSPECTION")
print("=" * 80)

print(f"\nShape: {df_test.shape}")

print("\nCOLUMNS:")
print(df_test.columns.tolist())

print("\nDATA TYPES:")
print(df_test.dtypes)

print("\nFIRST 5 ROWS:")
display(df_test.head())

print("\nMISSING VALUES:")
print(df_test.isnull().sum())

print("\nDONE — FILE RECONNAISSANCE COMPLETE")

In [ ]:
# =============================================================================
# SECTION 19A — BIDDAYOFFER CLEAN PARSER
# =============================================================================

def parse_biddayoffer_archive(csv_file):
    """
    Parse an AEMO BIDDAYOFFER archive CSV correctly.

    AEMO structure:

    Row 1: C = archive metadata
    Row 2: I = column definitions
    Row 3+: D = actual data
    """

    # -------------------------------------------------------------------------
    # READ CSV
    # -------------------------------------------------------------------------

    df = pd.read_csv(
        csv_file,
        skiprows=1,
        low_memory=False
    )

    print(f"Raw rows: {len(df):,}")
    print(f"Raw columns: {len(df.columns)}")

    # -------------------------------------------------------------------------
    # RENAME THE FIRST FOUR AEMO STRUCTURAL COLUMNS
    # -------------------------------------------------------------------------

    df = df.rename(columns={
        df.columns[0]: "RECORDTYPE",
        df.columns[1]: "DATASET",
        df.columns[2]: "TABLE_NAME",
        df.columns[3]: "VERSION"
    })

    # -------------------------------------------------------------------------
    # KEEP ONLY ACTUAL DATA RECORDS
    # -------------------------------------------------------------------------

    df = df[
        df["RECORDTYPE"] == "D"
    ].copy()

    print(f"Data rows after filtering D records: {len(df):,}")

    # -------------------------------------------------------------------------
    # REMOVE AEMO STRUCTURAL COLUMNS
    # -------------------------------------------------------------------------

    structural_columns = [
        "RECORDTYPE",
        "DATASET",
        "TABLE_NAME",
        "VERSION"
    ]

    df = df.drop(
        columns=structural_columns,
        errors="ignore"
    )

    # -------------------------------------------------------------------------
    # CLEAN COLUMN NAMES
    # -------------------------------------------------------------------------

    df.columns = [
        str(col).strip().upper()
        for col in df.columns
    ]

    # -------------------------------------------------------------------------
    # DATE COLUMNS
    # -------------------------------------------------------------------------

    date_columns = [
        "SETTLEMENTDATE",
        "OFFERDATE",
        "LASTCHANGED"
    ]

    for col in date_columns:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    # -------------------------------------------------------------------------
    # NUMERIC COLUMNS
    # -------------------------------------------------------------------------

    numeric_columns = [

        "VERSIONNO",

        "PRICEBAND1",
        "PRICEBAND2",
        "PRICEBAND3",
        "PRICEBAND4",
        "PRICEBAND5",
        "PRICEBAND6",
        "PRICEBAND7",
        "PRICEBAND8",
        "PRICEBAND9",
        "PRICEBAND10",

        "MINIMUMLOAD",

        "T1",
        "T2",
        "T3",
        "T4",

        "MR_FACTOR"

    ]

    for col in numeric_columns:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    print("\nFINAL STRUCTURE")
    print("-" * 60)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nColumns:")
    print(df.columns.tolist())

    return df

In [ ]:
# =============================================================================
# TEST BIDDAYOFFER PARSER
# =============================================================================

url = (
    "https://www.nemweb.com.au"
    "/Data_Archive/Wholesale_Electricity/MMSDM/2024/"
    "MMSDM_2024_09/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    "PUBLIC_ARCHIVE%23BIDDAYOFFER%23FILE01%23202409010000.zip"
)

print("Downloading September 2024 BIDDAYOFFER...")

response = requests.get(
    url,
    timeout=180
)

response.raise_for_status()

with zipfile.ZipFile(
    io.BytesIO(response.content)
) as z:

    csv_name = z.namelist()[0]

    print(f"\nProcessing: {csv_name}")

    with z.open(csv_name) as f:

        biddayoffer_test = parse_biddayoffer_archive(f)


print("\n" + "=" * 80)
print("BIDDAYOFFER TEST COMPLETE")
print("=" * 80)

display(biddayoffer_test.head())

print("\nDATA TYPES:")
print(biddayoffer_test.dtypes)

print("\nMISSING VALUES:")
print(
    biddayoffer_test.isnull().sum()
)

## The Bidding War

In [ ]:
# =============================================================================
# SECTION 19A — BIDDAYOFFER ARCHIVE LOADER
# =============================================================================

import requests
import zipfile
import io
import gc
import pandas as pd


def load_biddayoffer_month(year, month):
    """
    Downloads and parses one month of BIDDAYOFFER data
    from the AEMO MMSDM historical archive.
    """

    print("\n" + "=" * 80)
    print(f"BIDDAYOFFER INGESTION: {year}-{month:02d}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Build AEMO archive URL
    # -------------------------------------------------------------------------

    base_url = (
        f"https://www.nemweb.com.au"
        f"/Data_Archive/Wholesale_Electricity/MMSDM/{year}/"
        f"MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    filename = (
        f"PUBLIC_ARCHIVE%23BIDDAYOFFER%23FILE01"
        f"%23{year}{month:02d}010000.zip"
    )

    url = base_url + filename

    print(f"\nDownloading:")
    print(url)

    # -------------------------------------------------------------------------
    # Download
    # -------------------------------------------------------------------------

    response = requests.get(
        url,
        timeout=300
    )

    print(f"\nHTTP Status: {response.status_code}")

    response.raise_for_status()

    # -------------------------------------------------------------------------
    # Open ZIP
    # -------------------------------------------------------------------------

    with zipfile.ZipFile(
        io.BytesIO(response.content)
    ) as z:

        files = z.namelist()

        print("\nFiles inside ZIP:")

        for file in files:
            print(f"   📄 {file}")

        # Find CSV file
        csv_files = [
            file for file in files
            if file.upper().endswith(".CSV")
        ]

        if not csv_files:
            raise ValueError(
                "No CSV file found inside the archive."
            )

        csv_name = csv_files[0]

        print(f"\nProcessing:")
        print(csv_name)

        # ---------------------------------------------------------------------
        # Parse using your EXISTING working parser
        # ---------------------------------------------------------------------

        with z.open(csv_name) as f:

            df = parse_biddayoffer_archive(f)

    # -------------------------------------------------------------------------
    # Results
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("BIDDAYOFFER MONTH COMPLETE")
    print("=" * 80)

    print(f"\nRows loaded into memory: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    return df

In [ ]:
# =============================================================================
# TEST THE REUSABLE BIDDAYOFFER LOADER
# =============================================================================

biddayoffer_test = load_biddayoffer_month(
    year=2024,
    month=9
)

display(biddayoffer_test.head())

print("\nDATA TYPES:")
print(biddayoffer_test.dtypes)

In [ ]:
# =============================================================================
# CHECK BIDDAYOFFER BIGQUERY TABLE
# =============================================================================

from google.cloud import bigquery

dataset_id = "aemo_master_schema"
table_id = "BIDDAYOFFER"

full_table_id = (
    f"{client.project}.{dataset_id}.{table_id}"
)

print("=" * 80)
print("CHECKING BIGQUERY TABLE")
print("=" * 80)

try:
    table = client.get_table(full_table_id)

    print(f"\n✅ Table exists: {full_table_id}")
    print(f"Rows currently in table: {table.num_rows:,}")

    print("\nCURRENT BIGQUERY SCHEMA:")
    print("-" * 80)

    for field in table.schema:
        print(f"{field.name:<30} {field.field_type}")

except Exception as e:

    print(f"\n⚠️ Could not find/read table:")
    print(e)

In [ ]:
# =============================================================================
# CREATE BIDDAYOFFER ARCHIVE TABLE
# =============================================================================

from google.cloud import bigquery

dataset_id = "aemo_master_schema"
table_name = "BIDDAYOFFER_ARCHIVE"

table_id = f"{client.project}.{dataset_id}.{table_name}"

schema = [

    bigquery.SchemaField("DUID", "STRING"),
    bigquery.SchemaField("BIDTYPE", "STRING"),
    bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
    bigquery.SchemaField("OFFERDATE", "DATETIME"),
    bigquery.SchemaField("DIRECTION", "STRING"),
    bigquery.SchemaField("VERSIONNO", "FLOAT"),
    bigquery.SchemaField("PARTICIPANTID", "STRING"),
    bigquery.SchemaField("DAILYENERGYCONSTRAINT", "FLOAT"),
    bigquery.SchemaField("REBIDEXPLANATION", "STRING"),

    bigquery.SchemaField("PRICEBAND1", "FLOAT"),
    bigquery.SchemaField("PRICEBAND2", "FLOAT"),
    bigquery.SchemaField("PRICEBAND3", "FLOAT"),
    bigquery.SchemaField("PRICEBAND4", "FLOAT"),
    bigquery.SchemaField("PRICEBAND5", "FLOAT"),
    bigquery.SchemaField("PRICEBAND6", "FLOAT"),
    bigquery.SchemaField("PRICEBAND7", "FLOAT"),
    bigquery.SchemaField("PRICEBAND8", "FLOAT"),
    bigquery.SchemaField("PRICEBAND9", "FLOAT"),
    bigquery.SchemaField("PRICEBAND10", "FLOAT"),

    bigquery.SchemaField("MINIMUMLOAD", "FLOAT"),

    bigquery.SchemaField("T1", "FLOAT"),
    bigquery.SchemaField("T2", "FLOAT"),
    bigquery.SchemaField("T3", "FLOAT"),
    bigquery.SchemaField("T4", "FLOAT"),

    bigquery.SchemaField("NORMALSTATUS", "FLOAT"),
    bigquery.SchemaField("LASTCHANGED", "DATETIME"),
    bigquery.SchemaField("MR_FACTOR", "FLOAT"),

    bigquery.SchemaField("ENTRYTYPE", "STRING"),

    bigquery.SchemaField("REBID_EVENT_TIME", "STRING"),
    bigquery.SchemaField("REBID_AWARE_TIME", "STRING"),
    bigquery.SchemaField("REBID_DECISION_TIME", "STRING"),

    bigquery.SchemaField("REBID_CATEGORY", "STRING"),
    bigquery.SchemaField("REFERENCE_ID", "STRING"),

    # Pipeline metadata
    bigquery.SchemaField("_LOAD_DATE", "TIMESTAMP"),
    bigquery.SchemaField("_SOURCE", "STRING"),
    bigquery.SchemaField("_PIPELINE", "STRING")

]

table = bigquery.Table(
    table_id,
    schema=schema
)

table = client.create_table(
    table,
    exists_ok=True
)

print("=" * 80)
print("BIDDAYOFFER ARCHIVE TABLE READY")
print("=" * 80)

print(f"\nTable:")
print(table.full_table_id)

print(f"\nColumns: {len(table.schema)}")

for field in table.schema:
    print(f"{field.name:<30} {field.field_type}")

In [ ]:
# =============================================================================
# SECTION 19 — BIDDAYOFFER PRODUCTION ARCHIVE BACKFILL
# =============================================================================

import requests
import zipfile
import io
import gc
import re
import time
from bs4 import BeautifulSoup
from datetime import datetime

# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

BIDDAYOFFER_ARCHIVE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BIDDAYOFFER_ARCHIVE"
)

BASE_URL = "https://www.nemweb.com.au"

# =============================================================================
# 1. DISCOVER BIDDAYOFFER FILE FOR A MONTH
# =============================================================================

def discover_biddayoffer_file(year, month):

    archive_url = (
        f"{BASE_URL}"
        f"/Data_Archive/Wholesale_Electricity/MMSDM/{year}/"
        f"MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    print("\n" + "=" * 80)
    print(f"DISCOVERING BIDDAYOFFER: {year}-{month:02d}")
    print("=" * 80)

    try:

        response = requests.get(
            archive_url,
            timeout=120
        )

        print(f"Archive HTTP Status: {response.status_code}")

        if response.status_code != 200:

            print("❌ Archive folder unavailable.")

            return None

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        links = []

        for link in soup.find_all("a"):

            href = link.get("href")

            if href:

                # We specifically want the main BIDDAYOFFER archive.
                # Exclude BIDDAYOFFER_D because it is a different table.
                if (
                    "BIDDAYOFFER" in href.upper()
                    and "BIDDAYOFFER_D" not in href.upper()
                    and href.lower().endswith(".zip")
                ):

                    links.append(href)

        if not links:

            print("⚠️ No BIDDAYOFFER archive found.")

            return None

        # Remove duplicates
        links = list(set(links))

        print(f"✅ Found {len(links)} candidate file(s):")

        for link in links:

            print(f"   📦 {link}")

        # Normally there should be FILE01
        selected_file = sorted(links)[0]

        # Make absolute URL
        if selected_file.startswith("http"):

            file_url = selected_file

        elif selected_file.startswith("/"):

            file_url = BASE_URL + selected_file

        else:

            file_url = archive_url + selected_file

        print("\n🎯 Selected:")

        print(file_url)

        return file_url

    except Exception as e:

        print(f"❌ Discovery failed: {e}")

        return None


# =============================================================================
# 2. DOWNLOAD AND PARSE ONE MONTH
# =============================================================================

def download_and_parse_biddayoffer(year, month):

    print("\n" + "=" * 80)
    print(f"BIDDAYOFFER INGESTION: {year}-{month:02d}")
    print("=" * 80)

    file_url = discover_biddayoffer_file(
        year,
        month
    )

    if file_url is None:

        return None

    try:

        print("\n📥 Downloading archive...")

        response = requests.get(
            file_url,
            timeout=300
        )

        print(
            f"Download HTTP Status: "
            f"{response.status_code}"
        )

        response.raise_for_status()

        print(
            f"Downloaded size: "
            f"{len(response.content) / 1024 / 1024:.2f} MB"
        )

        with zipfile.ZipFile(
            io.BytesIO(response.content)
        ) as z:

            files = z.namelist()

            print("\nFiles inside ZIP:")

            for file_name in files:

                print(f"   📄 {file_name}")

            # Find CSV
            csv_files = [

                f for f in files

                if (
                    f.upper().endswith(".CSV")
                    and "BIDDAYOFFER" in f.upper()
                )
            ]

            if not csv_files:

                print("❌ No BIDDAYOFFER CSV found.")

                return None

            csv_name = csv_files[0]

            print("\nProcessing:")

            print(csv_name)

            with z.open(csv_name) as f:

                df = parse_biddayoffer_archive(f)

        print("\n" + "-" * 80)

        print("PARSE COMPLETE")

        print("-" * 80)

        print(f"Rows: {len(df):,}")

        print(f"Columns: {len(df.columns)}")

        return df

    except Exception as e:

        print(
            f"\n❌ FAILED: {year}-{month:02d}"
        )

        print(e)

        return None


# =============================================================================
# 3. PREPARE FOR BIGQUERY
# =============================================================================

def prepare_biddayoffer_for_bigquery(df):

    print("\nPreparing BIDDAYOFFER for BigQuery...")

    df = df.copy()

    # -------------------------------------------------------------------------
    # NUMERIC COLUMNS
    # -------------------------------------------------------------------------

    numeric_columns = [

        "VERSIONNO",

        "DAILYENERGYCONSTRAINT",

        "PRICEBAND1",
        "PRICEBAND2",
        "PRICEBAND3",
        "PRICEBAND4",
        "PRICEBAND5",
        "PRICEBAND6",
        "PRICEBAND7",
        "PRICEBAND8",
        "PRICEBAND9",
        "PRICEBAND10",

        "MINIMUMLOAD",

        "T1",
        "T2",
        "T3",
        "T4",

        "NORMALSTATUS",

        "MR_FACTOR"
    ]

    for column in numeric_columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            ).astype("float64")

    # -------------------------------------------------------------------------
    # DATETIME COLUMNS
    # -------------------------------------------------------------------------

    datetime_columns = [

        "SETTLEMENTDATE",

        "OFFERDATE",

        "LASTCHANGED"
    ]

    for column in datetime_columns:

        if column in df.columns:

            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )

    # -------------------------------------------------------------------------
    # STRING COLUMNS
    # -------------------------------------------------------------------------

    for column in df.columns:

        if column not in numeric_columns + datetime_columns:

            df[column] = df[column].astype("string")

    # -------------------------------------------------------------------------
    # PIPELINE METADATA
    # -------------------------------------------------------------------------

    df["_LOAD_DATE"] = pd.Timestamp.utcnow()

    df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"

    df["_PIPELINE"] = "BIDDAYOFFER_ARCHIVE_BACKFILL"

    print("✅ Preparation complete.")

    return df


# =============================================================================
# 4. LOAD ONE MONTH
# =============================================================================

def load_biddayoffer_month(year, month):

    print("\n" + "█" * 80)

    print(
        f"🚀 STARTING MONTH: "
        f"{year}-{month:02d}"
    )

    print("█" * 80)

    df = download_and_parse_biddayoffer(
        year,
        month
    )

    if df is None:

        print(
            f"⚠️ Skipping "
            f"{year}-{month:02d}"
        )

        return False

    try:

        df_clean = prepare_biddayoffer_for_bigquery(
            df
        )

        print("\nLoading into BigQuery...")

        load_table_to_bigquery(
            df_clean,
            "BIDDAYOFFER_ARCHIVE"
        )

        print("\n" + "=" * 80)

        print(
            f"✅ SUCCESS: "
            f"{year}-{month:02d}"
        )

        print(
            f"Rows processed: "
            f"{len(df_clean):,}"
        )

        print("=" * 80)

        return True

    except Exception as e:

        print("\n❌ BIGQUERY LOAD FAILED")

        print(e)

        return False

    finally:

        # ---------------------------------------------------------------------
        # MEMORY CLEANUP
        # ---------------------------------------------------------------------

        try:

            del df

        except:

            pass

        try:

            del df_clean

        except:

            pass

        gc.collect()

        print("\n🧹 Memory cleared.")


# =============================================================================
# 5. RUN A RANGE OF MONTHS
# =============================================================================

def run_biddayoffer_backfill(
    start_year,
    end_year
):

    print("\n" + "🚀" * 20)

    print("BIDDAYOFFER ARCHIVE BACKFILL")

    print(
        f"Period: "
        f"{start_year} → {end_year}"
    )

    print("🚀" * 20)

    successful_months = []

    failed_months = []

    current_year = datetime.now().year

    current_month = datetime.now().month

    for year in range(
        start_year,
        end_year + 1
    ):

        for month in range(1, 13):

            # Don't attempt future months
            if (
                year == current_year
                and month > current_month
            ):

                break

            success = load_biddayoffer_month(
                year,
                month
            )

            if success:

                successful_months.append(
                    f"{year}-{month:02d}"
                )

            else:

                failed_months.append(
                    f"{year}-{month:02d}"
                )

            # Small pause to be polite to AEMO servers
            time.sleep(2)

    # =========================================================================
    # FINAL REPORT
    # =========================================================================

    print("\n" + "=" * 80)

    print("🏁 BIDDAYOFFER BACKFILL COMPLETE")

    print("=" * 80)

    print(
        f"\nSuccessful months: "
        f"{len(successful_months)}"
    )

    print(
        f"Failed months: "
        f"{len(failed_months)}"
    )

    print("\n✅ SUCCESSFUL:")

    print(successful_months)

    print("\n⚠️ FAILED:")

    print(failed_months)

    return {

        "successful": successful_months,

        "failed": failed_months
    }

In [ ]:
# =============================================================================
# PRODUCTION TEST — SEPTEMBER 2024
# =============================================================================

test_result = load_biddayoffer_month(
    2024,
    9
)

In [ ]:
# =============================================================================
# SECTION 19.5 — BIDDAYOFFER WAREHOUSE-AWARE CHECKPOINT SYSTEM
# =============================================================================

from google.cloud import bigquery


# =============================================================================
# CHECK WHETHER A MONTH IS ALREADY LOADED
# =============================================================================

def biddayoffer_month_exists(year, month):

    month_start = (
        f"{year}-{month:02d}-01"
    )

    # Calculate the first day of the next month
    if month == 12:

        next_month = (
            f"{year + 1}-01-01"
        )

    else:

        next_month = (
            f"{year}-{month + 1:02d}-01"
        )


    query = f"""
    SELECT COUNT(*) AS row_count
    FROM `{BIDDAYOFFER_ARCHIVE_TABLE}`
    WHERE SETTLEMENTDATE >= TIMESTAMP('{month_start}')
      AND SETTLEMENTDATE < TIMESTAMP('{next_month}')
    """


    try:

        query_job = bq_client.query(query)

        results = query_job.result()

        row_count = list(results)[0].row_count


        if row_count > 0:

            print(
                f"🔒 CHECKPOINT FOUND: "
                f"{year}-{month:02d}"
            )

            print(
                f"   Existing rows: "
                f"{row_count:,}"
            )

            return True


        print(
            f"🟢 NOT YET LOADED: "
            f"{year}-{month:02d}"
        )

        return False


    except Exception as e:

        print(
            f"⚠️ Checkpoint query failed: {e}"
        )

        # Important:
        # If BigQuery cannot be checked,
        # DO NOT automatically assume the month exists.

        return False


# =============================================================================
# WAREHOUSE-AWARE MONTH LOADER
# =============================================================================

def load_biddayoffer_month_safe(year, month):

    print("\n" + "█" * 80)

    print(
        f"📅 CHECKING MONTH: "
        f"{year}-{month:02d}"
    )

    print("█" * 80)


    # -------------------------------------------------------------------------
    # CHECK BIGQUERY FIRST
    # -------------------------------------------------------------------------

    if biddayoffer_month_exists(year, month):

        print(
            f"\n⏭️ SKIPPING {year}-{month:02d}"
        )

        print(
            "Reason: Data already exists in BigQuery."
        )

        return "SKIPPED"


    # -------------------------------------------------------------------------
    # LOAD THE MONTH
    # -------------------------------------------------------------------------

    success = load_biddayoffer_month(
        year,
        month
    )


    if success:

        return "SUCCESS"


    return "FAILED"


# =============================================================================
# WAREHOUSE-AWARE BACKFILL
# =============================================================================

def run_biddayoffer_backfill_safe(
    start_year,
    end_year
):

    print("\n" + "=" * 80)

    print("🚀 BIDDAYOFFER WAREHOUSE-AWARE BACKFILL")

    print("=" * 80)

    print(
        f"Period: "
        f"{start_year} → {end_year}"
    )


    successful = []

    skipped = []

    failed = []


    current_year = datetime.now().year

    current_month = datetime.now().month


    # =========================================================================
    # LOOP THROUGH YEARS
    # =========================================================================

    for year in range(
        start_year,
        end_year + 1
    ):


        print("\n" + "#" * 80)

        print(
            f"📆 PROCESSING YEAR: {year}"
        )

        print("#" * 80)


        # ---------------------------------------------------------------------
        # LOOP THROUGH MONTHS
        # ---------------------------------------------------------------------

        for month in range(1, 13):


            # Do not process future months
            if (
                year == current_year
                and month > current_month
            ):

                break


            result = load_biddayoffer_month_safe(
                year,
                month
            )


            month_label = (
                f"{year}-{month:02d}"
            )


            # -----------------------------------------------------------------
            # RECORD RESULT
            # -----------------------------------------------------------------

            if result == "SUCCESS":

                successful.append(
                    month_label
                )


            elif result == "SKIPPED":

                skipped.append(
                    month_label
                )


            else:

                failed.append(
                    month_label
                )


            # -----------------------------------------------------------------
            # SHORT PAUSE BETWEEN MONTHS
            # -----------------------------------------------------------------

            time.sleep(2)


    # =========================================================================
    # FINAL REPORT
    # =========================================================================

    print("\n" + "=" * 80)

    print("🏁 BIDDAYOFFER BACKFILL COMPLETE")

    print("=" * 80)


    print(
        f"\n✅ Newly loaded: "
        f"{len(successful)} months"
    )

    print(
        f"⏭️ Already existed: "
        f"{len(skipped)} months"
    )

    print(
        f"❌ Failed: "
        f"{len(failed)} months"
    )


    print("\nNEWLY LOADED:")

    print(successful)


    print("\nSKIPPED:")

    print(skipped)


    print("\nFAILED:")

    print(failed)


    return {

        "successful": successful,

        "skipped": skipped,

        "failed": failed

    }

In [ ]:
# =============================================================================
# BIGQUERY CONNECTION INITIALISATION
# =============================================================================

from google.cloud import bigquery
import os

# Confirm credentials exist
if not os.path.exists(KEY_PATH):
    raise FileNotFoundError(
        f"BigQuery credentials not found:\n{KEY_PATH}"
    )

# Create BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Test connection
datasets = list(client.list_datasets())

print("=" * 80)
print("✅ BIGQUERY CONNECTION READY")
print("=" * 80)

print(f"Project: {PROJECT_ID}")
print(f"Datasets found: {len(datasets)}")

for dataset in datasets:
    print(f"   📁 {dataset.dataset_id}")

In [ ]:
print(client)

In [ ]:
print(
    client.get_dataset(
        f"{PROJECT_ID}.{DATASET_ID}"
    )
)

In [ ]:
# =============================================================================
# RESTORE BIGQUERY CLIENT — LOCAL TANK RUNTIME
# =============================================================================

from google.cloud import bigquery
import os

print("=" * 80)
print("RESTORING BIGQUERY CONNECTION")
print("=" * 80)

# Check that the project configuration exists
print(f"Project ID: {PROJECT_ID}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Credential path: {KEY_PATH}")

# Confirm credential file exists on THE TANK
if not os.path.exists(KEY_PATH):
    raise FileNotFoundError(
        f"\n❌ Credential file not found on local machine:\n{KEY_PATH}"
    )

print("\n✅ Credential file found.")

# Create the BigQuery client
client = bigquery.Client(project=PROJECT_ID)

print("✅ BigQuery client created successfully.")

# Test access to the dataset
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

dataset = client.get_dataset(dataset_ref)

print(f"✅ Connected to dataset: {dataset.dataset_id}")

print("\n" + "=" * 80)
print("🏆 BIGQUERY CONNECTION READY")
print("=" * 80)

In [ ]:
# =============================================================================
# CHECKPOINT — VERIFY BIDDAYOFFER ARCHIVE LOAD
# =============================================================================

print("=" * 80)
print("BIDDAYOFFER ARCHIVE CHECKPOINT")
print("=" * 80)

table_id = (
    f"{PROJECT_ID}.{DATASET_ID}.BIDDAYOFFER_ARCHIVE"
)

table = client.get_table(table_id)

print(f"\nTable: {table_id}")
print(f"Rows in BigQuery: {table.num_rows:,}")

print("\nCURRENT DATE RANGE:")

query = f"""
SELECT
    MIN(SETTLEMENTDATE) AS earliest_date,
    MAX(SETTLEMENTDATE) AS latest_date,
    COUNT(*) AS total_rows
FROM `{table_id}`
"""

result = client.query(query).to_dataframe()

display(result)

In [ ]:
# =============================================================================
# DUPLICATE CHECK — BIDDAYOFFER ARCHIVE
# =============================================================================

table_id = (
    f"{PROJECT_ID}.{DATASET_ID}.BIDDAYOFFER_ARCHIVE"
)

query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT CONCAT(
        COALESCE(DUID, ''),
        '|',
        COALESCE(BIDTYPE, ''),
        '|',
        CAST(SETTLEMENTDATE AS STRING),
        '|',
        CAST(OFFERDATE AS STRING),
        '|',
        COALESCE(DIRECTION, ''),
        '|',
        CAST(VERSIONNO AS STRING),
        '|',
        COALESCE(REFERENCE_ID, '')
    )) AS estimated_unique_rows

FROM `{table_id}`
"""

duplicate_check = client.query(query).to_dataframe()

display(duplicate_check)

In [ ]:
# =============================================================================
# SECTION 20 — WAREHOUSE-AWARE BIDDAYOFFER BACKFILL CONTROLLER
# =============================================================================

from datetime import datetime
import pandas as pd
import gc
import time


# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"
TABLE_ID = "BIDDAYOFFER_ARCHIVE"

FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"


# -----------------------------------------------------------------------------
# CHECK WHETHER A MONTH IS ALREADY LOADED
# -----------------------------------------------------------------------------

def is_bid_month_loaded(year, month):

    query = f"""
    SELECT
        COUNT(*) AS row_count
    FROM `{FULL_TABLE_ID}`
    WHERE DATE(SETTLEMENTDATE) >= DATE('{year}-{month:02d}-01')
      AND DATE(SETTLEMENTDATE) <
          DATE_ADD(
              DATE('{year}-{month:02d}-01'),
              INTERVAL 1 MONTH
          )
    """

    result = bq_client.query(query).result()

    rows = list(result)

    row_count = rows[0].row_count

    return row_count > 0, row_count


# -----------------------------------------------------------------------------
# VERIFY MONTH AFTER LOADING
# -----------------------------------------------------------------------------

def verify_bid_month(year, month):

    query = f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(SETTLEMENTDATE) AS earliest_date,
        MAX(SETTLEMENTDATE) AS latest_date
    FROM `{FULL_TABLE_ID}`
    WHERE DATE(SETTLEMENTDATE) >= DATE('{year}-{month:02d}-01')
      AND DATE(SETTLEMENTDATE) <
          DATE_ADD(
              DATE('{year}-{month:02d}-01'),
              INTERVAL 1 MONTH
          )
    """

    result = list(
        bq_client.query(query).result()
    )

    row = result[0]

    return {
        "rows": row.row_count,
        "earliest": row.earliest_date,
        "latest": row.latest_date
    }


# -----------------------------------------------------------------------------
# PROCESS ONE MONTH
# -----------------------------------------------------------------------------

def process_bid_month(year, month):

    print("\n" + "=" * 80)
    print(f"📅 BIDDAYOFFER MONTH: {year}-{month:02d}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # STEP 1 — CHECK BIGQUERY
    # -------------------------------------------------------------------------

    loaded, existing_rows = is_bid_month_loaded(year, month)

    if loaded:

        print(
            f"⏭️  SKIPPING {year}-{month:02d}"
        )

        print(
            f"   Already in BigQuery: "
            f"{existing_rows:,} rows"
        )

        return {
            "status": "SKIPPED",
            "year": year,
            "month": month,
            "rows": existing_rows
        }

    print(
        f"📥 Month not found in warehouse."
    )

    print(
        f"🚀 Starting ingestion..."
    )

    # -------------------------------------------------------------------------
    # STEP 2 — RUN YOUR EXISTING INGESTOR
    # -------------------------------------------------------------------------

    try:

        bid_df = ingest_biddayoffer_month(
            year,
            month
        )

        if bid_df is None or bid_df.empty:

            raise ValueError(
                "Parser returned no data."
            )

        processed_rows = len(bid_df)

        print(
            f"\n📊 Parsed rows: "
            f"{processed_rows:,}"
        )

        # ---------------------------------------------------------------------
        # STEP 3 — PREPARE DATA
        # ---------------------------------------------------------------------

        print("\n🧹 Preparing for BigQuery...")

        bid_df = prepare_biddayoffer_for_bigquery(
            bid_df
        )

        # ---------------------------------------------------------------------
        # STEP 4 — LOAD BIGQUERY
        # ---------------------------------------------------------------------

        print("\n☁️ Loading into BigQuery...")

        load_biddayoffer_archive(
            bid_df
        )

        # Free memory immediately
        del bid_df
        gc.collect()

        # ---------------------------------------------------------------------
        # STEP 5 — VERIFY
        # ---------------------------------------------------------------------

        print("\n🔍 Verifying warehouse...")

        verification = verify_bid_month(
            year,
            month
        )

        if verification["rows"] == 0:

            raise RuntimeError(
                "Upload completed but verification "
                "found zero rows."
            )

        print("\n✅ MONTH VERIFIED")

        print(
            f"Rows: "
            f"{verification['rows']:,}"
        )

        print(
            f"Range: "
            f"{verification['earliest']} "
            f"→ "
            f"{verification['latest']}"
        )

        return {
            "status": "SUCCESS",
            "year": year,
            "month": month,
            "rows": verification["rows"]
        }

    except Exception as e:

        print("\n❌ MONTH FAILED")

        print(
            f"{year}-{month:02d}"
        )

        print(
            f"Error: {e}"
        )

        gc.collect()

        return {
            "status": "FAILED",
            "year": year,
            "month": month,
            "error": str(e)
        }


# -----------------------------------------------------------------------------
# RUN A RANGE OF MONTHS
# -----------------------------------------------------------------------------

def run_biddayoffer_backfill(
    start_year,
    start_month,
    end_year,
    end_month
):

    results = []

    current_year = start_year
    current_month = start_month

    print("\n" + "█" * 80)
    print("🚀 BIDDAYOFFER HISTORICAL BACKFILL")
    print("█" * 80)

    print(
        f"\nRange: "
        f"{start_year}-{start_month:02d} "
        f"→ "
        f"{end_year}-{end_month:02d}"
    )

    while True:

        result = process_bid_month(
            current_year,
            current_month
        )

        results.append(result)

        # Stop condition
        if (
            current_year == end_year
            and current_month == end_month
        ):
            break

        # Move to next month
        current_month += 1

        if current_month > 12:

            current_month = 1
            current_year += 1

        # Small pause between months
        time.sleep(2)

    # -------------------------------------------------------------------------
    # FINAL REPORT
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("🏁 BACKFILL COMPLETE")
    print("=" * 80)

    results_df = pd.DataFrame(results)

    if not results_df.empty:

        print(
            "\nSTATUS SUMMARY:"
        )

        print(
            results_df["status"]
            .value_counts()
        )

        display(results_df)

    return results_df

In [ ]:
# =============================================================================
# SECTION 20.1 — BIDDAYOFFER INGESTION ENGINE
# =============================================================================

import requests
import zipfile
import io
import gc
from bs4 import BeautifulSoup
from google.cloud import bigquery


# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

BIDDAYOFFER_ARCHIVE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BIDDAYOFFER_ARCHIVE"
)


# -----------------------------------------------------------------------------
# DISCOVER BIDDAYOFFER FILE FOR A MONTH
# -----------------------------------------------------------------------------

def discover_biddayoffer_file(year, month):

    print("\n" + "=" * 80)
    print(f"DISCOVERING BIDDAYOFFER: {year}-{month:02d}")
    print("=" * 80)

    archive_url = (
        f"https://www.nemweb.com.au"
        f"/Data_Archive/Wholesale_Electricity/MMSDM/{year}"
        f"/MMSDM_{year}_{month:02d}"
        f"/MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    response = requests.get(
        archive_url,
        timeout=180
    )

    print(f"Archive HTTP Status: {response.status_code}")

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    candidates = []

    for link in soup.find_all("a"):

        href = link.get("href")

        if not href:
            continue

        href_upper = href.upper()

        # We specifically want the BIDDAYOFFER archive file.
        # Exclude BIDDAYOFFER_D because that is a different table/version.
        if (
            "BIDDAYOFFER" in href_upper
            and "BIDDAYOFFER_D" not in href_upper
            and href_upper.endswith(".ZIP")
        ):
            candidates.append(href)

    print(
        f"✅ Found {len(candidates)} candidate file(s):"
    )

    for candidate in candidates:
        print(f"   📦 {candidate}")

    if not candidates:

        raise FileNotFoundError(
            f"No BIDDAYOFFER archive found for "
            f"{year}-{month:02d}"
        )

    selected = candidates[0]

    if selected.startswith("http"):
        file_url = selected
    else:
        file_url = (
            "https://www.nemweb.com.au"
            + selected
        )

    print("\n🎯 Selected:")
    print(file_url)

    return file_url


# -----------------------------------------------------------------------------
# DOWNLOAD + PARSE ONE MONTH
# -----------------------------------------------------------------------------

def ingest_biddayoffer_month(year, month):

    print("\n" + "=" * 80)
    print(f"BIDDAYOFFER INGESTION: {year}-{month:02d}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # DISCOVER
    # -------------------------------------------------------------------------

    file_url = discover_biddayoffer_file(
        year,
        month
    )

    # -------------------------------------------------------------------------
    # DOWNLOAD
    # -------------------------------------------------------------------------

    print("\n📥 Downloading archive...")

    response = requests.get(
        file_url,
        timeout=300
    )

    print(
        f"Download HTTP Status: "
        f"{response.status_code}"
    )

    response.raise_for_status()

    print(
        f"Downloaded size: "
        f"{len(response.content) / 1024 / 1024:.2f} MB"
    )

    # -------------------------------------------------------------------------
    # OPEN ZIP
    # -------------------------------------------------------------------------

    with zipfile.ZipFile(
        io.BytesIO(response.content)
    ) as z:

        csv_files = [
            name
            for name in z.namelist()
            if name.upper().endswith(".CSV")
        ]

        print("\nFiles inside ZIP:")

        for csv_name in csv_files:
            print(f"   📄 {csv_name}")

        if not csv_files:

            raise FileNotFoundError(
                "No CSV file found inside ZIP."
            )

        # BIDDAYOFFER archive normally contains one CSV
        csv_name = csv_files[0]

        print("\nProcessing:")
        print(csv_name)

        with z.open(csv_name) as f:

            bid_df = parse_biddayoffer_archive(f)

    # -------------------------------------------------------------------------
    # RESULT
    # -------------------------------------------------------------------------

    print("\n" + "-" * 80)
    print("PARSE COMPLETE")
    print("-" * 80)

    print(f"Rows: {len(bid_df):,}")
    print(f"Columns: {len(bid_df.columns)}")

    return bid_df


# -----------------------------------------------------------------------------
# LOAD BIDDAYOFFER ARCHIVE INTO BIGQUERY
# -----------------------------------------------------------------------------

def load_biddayoffer_archive(df):

    print(
        f"\nLOADING: BIDDAYOFFER_ARCHIVE "
        f"({len(df):,} rows)"
    )

    job_config = bigquery.LoadJobConfig(

        write_disposition=(
            bigquery.WriteDisposition.WRITE_APPEND
        ),

        schema_update_options=[

            bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,

            bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION

        ]

    )

    load_job = bq_client.load_table_from_dataframe(
        df,
        BIDDAYOFFER_ARCHIVE_TABLE,
        job_config=job_config
    )

    load_job.result()

    print(
        "✅ SUCCESS: BIDDAYOFFER_ARCHIVE loaded."
    )

    return True


print("✅ BIDDAYOFFER INGESTION ENGINE READY")

In [ ]:
# =============================================================================
# CHECK REQUIRED BIDDING FUNCTIONS
# =============================================================================

required_functions = [
    "ingest_biddayoffer_month",
    "prepare_biddayoffer_for_bigquery",
    "load_biddayoffer_archive",
    "parse_biddayoffer_archive"
]

for function_name in required_functions:

    if function_name in globals():
        print(f"✅ FOUND: {function_name}")
    else:
        print(f"❌ MISSING: {function_name}")

In [ ]:
# =============================================================================
# RESTORE BIGQUERY CLIENT
# =============================================================================

from google.cloud import bigquery

PROJECT_ID = "master-aemo"

bq_client = bigquery.Client(
    project=PROJECT_ID
)

print("✅ BigQuery client restored")
print(f"Project: {bq_client.project}")

In [ ]:
# =============================================================================
# SECTION 21 — FULL BIDDAYOFFER HISTORICAL BACKFILL
# =============================================================================
#
# Mission:
# January 2015 → September 2026
#
# The controller will:
#   ✅ Skip months already in BigQuery
#   ✅ Download missing months
#   ✅ Parse locally on The Tank
#   ✅ Upload to BIDDAYOFFER_ARCHIVE
#   ✅ Verify successful uploads
#   ✅ Continue if an individual month fails
#
# =============================================================================


FULL_BACKFILL_RESULTS = run_biddayoffer_backfill(

    start_year=2015,
    start_month=1,

    end_year=2026,
    end_month=9

)


print("\n" + "█" * 90)
print("🏆 FULL BIDDAYOFFER BACKFILL MISSION FINISHED")
print("█" * 90)

print("\nFinal status summary:")

print(
    FULL_BACKFILL_RESULTS["status"]
    .value_counts()
)

print("\nFinal results:")

display(FULL_BACKFILL_RESULTS)

In [ ]:
# =============================================================================
# SECTION 22 — BIDOFFERPERIOD ARCHIVE PIPELINE
# CHUNKED + BIGQUERY SAFE VERSION
# =============================================================================

import requests
import zipfile
import io
import csv
import gc
import re
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from google.cloud import bigquery


# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_ID = "master-aemo"

BIDPERIOD_TABLE = (
    "master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE"
)

# Start conservatively. Increase later only if everything is stable.
CHUNK_SIZE = 250_000

# Network timeout
DOWNLOAD_TIMEOUT = 300


# =============================================================================
# BIGQUERY CLIENT
# =============================================================================

bq_client = bigquery.Client(
    project=PROJECT_ID
)


# =============================================================================
# 1. DISCOVER BIDOFFERPERIOD FILES
# =============================================================================

def discover_bidperiod_files(year, month):

    print("\n" + "=" * 80)
    print(f"DISCOVERING BIDOFFERPERIOD FILES: {year}-{month:02d}")
    print("=" * 80)

    archive_url = (
        f"https://www.nemweb.com.au"
        f"/Data_Archive/Wholesale_Electricity/MMSDM/{year}"
        f"/MMSDM_{year}_{month:02d}"
        f"/MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    print("\n🔎 Checking archive:")
    print(archive_url)

    response = requests.get(
        archive_url,
        timeout=DOWNLOAD_TIMEOUT
    )

    print(f"\nArchive HTTP Status: {response.status_code}")

    if response.status_code != 200:

        raise RuntimeError(
            f"❌ Archive folder unavailable: HTTP {response.status_code}"
        )


    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    found_files = []


    for link in soup.find_all("a"):

        href = link.get("href")

        if not href:
            continue


        href_upper = href.upper()


        # ONLY genuine BIDOFFERPERIOD archive files
        if (
            "BIDOFFERPERIOD" in href_upper
            and href_upper.endswith(".ZIP")
            and "MNSP_" not in href_upper
        ):

            if href.startswith("http"):

                full_url = href

            elif href.startswith("/"):

                full_url = (
                    "https://www.nemweb.com.au"
                    + href
                )

            else:

                full_url = (
                    archive_url
                    + href
                )


            found_files.append(
                full_url
            )


    found_files = sorted(
        list(set(found_files))
    )


    print("\n📦 BIDOFFERPERIOD FILES FOUND:")

    print(
        f"✅ Found {len(found_files)} file(s)"
    )


    for file_url in found_files:

        print(f"   📦 {file_url}")


    return found_files


# =============================================================================
# 2. FIND THE AEMO HEADER
# =============================================================================

def get_bidperiod_columns(csv_file):

    """
    AEMO format:

    C,...                         metadata
    I,BIDS,BIDOFFERPERIOD,2,...   column definition
    D,BIDS,BIDOFFERPERIOD,2,...   data

    Returns the actual database column names beginning after:
    D,BIDS,BIDOFFERPERIOD,2
    """

    # Read a few lines from the beginning
    csv_file.seek(0)

    text_wrapper = io.TextIOWrapper(
        csv_file,
        encoding="utf-8",
        newline=""
    )

    reader = csv.reader(
        text_wrapper
    )


    target_columns = None


    for line_number, row in enumerate(reader):

        if line_number > 100:

            break


        if len(row) < 5:

            continue


        # Find:
        # I,BIDS,BIDOFFERPERIOD,2,...
        if (
            row[0].strip().upper() == "I"
            and len(row) >= 5
            and row[1].strip().upper() == "BIDS"
            and row[2].strip().upper() == "BIDOFFERPERIOD"
        ):

            target_columns = [

                col.strip()

                for col in row[4:]
            ]

            break


    # Important: detach wrapper without closing ZIP stream
    text_wrapper.detach()


    if target_columns is None:

        raise ValueError(
            "❌ Could not find BIDOFFERPERIOD column definition."
        )


    print("\n🔎 Found BIDOFFERPERIOD column definition")

    print("\nActual data columns:")

    print(target_columns)


    return target_columns


# =============================================================================
# 3. GENERATE CHUNKED DATAFRAMES
# =============================================================================

def iter_bidperiod_chunks(csv_file, chunk_size=CHUNK_SIZE):

    """
    Stream the AEMO CSV and yield only BIDOFFERPERIOD
    data rows in manageable DataFrame chunks.

    This avoids loading the entire 44-million-row file into memory.
    """


    columns = get_bidperiod_columns(
        csv_file
    )


    # Return to beginning
    csv_file.seek(0)


    text_wrapper = io.TextIOWrapper(
        csv_file,
        encoding="utf-8",
        newline=""
    )


    reader = csv.reader(
        text_wrapper
    )


    batch = []

    total_records = 0


    for row in reader:


        # We only want:
        # D,BIDS,BIDOFFERPERIOD,2,...
        if len(row) < 4:

            continue


        if not (
            row[0].strip().upper() == "D"
            and row[1].strip().upper() == "BIDS"
            and row[2].strip().upper() == "BIDOFFERPERIOD"
        ):

            continue


        # Remove AEMO prefix:
        #
        # D
        # BIDS
        # BIDOFFERPERIOD
        # 2
        #
        # Remaining fields = actual data
        data_row = row[4:]


        # Protect against malformed rows
        if len(data_row) != len(columns):

            continue


        batch.append(
            data_row
        )

        total_records += 1


        # -------------------------------------------------------------
        # YIELD FULL CHUNK
        # -------------------------------------------------------------

        if len(batch) >= chunk_size:


            chunk_df = pd.DataFrame(
                batch,
                columns=columns
            )


            print(
                f"\n📦 Yielding chunk: "
                f"{len(chunk_df):,} rows "
                f"(records scanned: {total_records:,})"
            )


            yield chunk_df


            # Release batch memory
            del chunk_df
            batch.clear()

            gc.collect()


    # -----------------------------------------------------------------
    # FINAL PARTIAL CHUNK
    # -----------------------------------------------------------------

    if batch:


        chunk_df = pd.DataFrame(
            batch,
            columns=columns
        )


        print(
            f"\n📦 Yielding final chunk: "
            f"{len(chunk_df):,} rows "
            f"(total records: {total_records:,})"
        )


        yield chunk_df


        del chunk_df


    batch.clear()

    gc.collect()


    # Detach without closing ZIP stream
    try:

        text_wrapper.detach()

    except Exception:

        pass


# =============================================================================
# 4. PREPARE ONE CHUNK
# =============================================================================

def prepare_bidperiod_chunk(df):


    # -------------------------------------------------------------
    # COPY
    # -------------------------------------------------------------

    df = df.copy()


    # -------------------------------------------------------------
    # RENAME TO WAREHOUSE COLUMN NAMES
    # -------------------------------------------------------------

    rename_map = {

        "TRADINGDATE": "SETTLEMENTDATE",

        "OFFERDATETIME": "OFFERDATE"
    }


    df.rename(
        columns=rename_map,
        inplace=True
    )


    # -------------------------------------------------------------
    # DATETIME CONVERSION
    # -------------------------------------------------------------

    if "SETTLEMENTDATE" in df.columns:

        df["SETTLEMENTDATE"] = pd.to_datetime(
            df["SETTLEMENTDATE"],
            errors="coerce",
            utc=True
        )


    if "OFFERDATE" in df.columns:

        df["OFFERDATE"] = pd.to_datetime(
            df["OFFERDATE"],
            errors="coerce",
            utc=True
        )


    # -------------------------------------------------------------
    # NUMERIC COLUMNS
    # -------------------------------------------------------------

    numeric_columns = [

        "PERIODID",
        "MAXAVAIL",
        "FIXEDLOAD",
        "RAMPUPRATE",
        "RAMPDOWNRATE",
        "ENABLEMENTMIN",
        "ENABLEMENTMAX",
        "LOWBREAKPOINT",
        "HIGHBREAKPOINT",

        "BANDAVAIL1",
        "BANDAVAIL2",
        "BANDAVAIL3",
        "BANDAVAIL4",
        "BANDAVAIL5",
        "BANDAVAIL6",
        "BANDAVAIL7",
        "BANDAVAIL8",
        "BANDAVAIL9",
        "BANDAVAIL10",

        "PASAAVAILABILITY",
        "ENERGYLIMIT",
        "PERIODIDTO"
    ]


    for column in numeric_columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )


    # -------------------------------------------------------------
    # METADATA
    # -------------------------------------------------------------

    df["_LOAD_DATE"] = pd.Timestamp.now(
        tz="UTC"
    )


    df["_SOURCE"] = (
        "AEMO_NEMWEB_ARCHIVE"
    )


    df["_PIPELINE"] = (
        "BIDDAYOFFER_ARCHIVE_BACKFILL"
    )


    return df


# =============================================================================
# 5. CREATE TABLE IF NEEDED
# =============================================================================

def ensure_bidperiod_table_exists(df):


    try:

        bq_client.get_table(
            BIDPERIOD_TABLE
        )


        print(
            "\n✅ BigQuery table already exists."
        )


        return


    except Exception:

        print(
            "\n⚠️ BigQuery table does not exist."
        )

        print(
            "Creating table from first chunk schema..."
        )


    # -------------------------------------------------------------
    # CREATE TABLE USING FIRST DATAFRAME
    # -------------------------------------------------------------

    job_config = bigquery.LoadJobConfig(

        write_disposition=(
            bigquery.WriteDisposition.WRITE_EMPTY
        ),

        autodetect=True
    )


    load_job = bq_client.load_table_from_dataframe(

        df.head(1),

        BIDPERIOD_TABLE,

        job_config=job_config
    )


    load_job.result()


    print(
        "✅ BigQuery table created."
    )


# =============================================================================
# 6. UPLOAD ONE CHUNK TO BIGQUERY
# =============================================================================

def upload_bidperiod_chunk(df):


    job_config = bigquery.LoadJobConfig(

        write_disposition=(
            bigquery.WriteDisposition.WRITE_APPEND
        )
    )


    load_job = bq_client.load_table_from_dataframe(

        df,

        BIDPERIOD_TABLE,

        job_config=job_config
    )


    load_job.result()


    print(
        f"✅ Uploaded {len(df):,} rows"
    )


# =============================================================================
# 7. PROCESS ONE ZIP FILE
# =============================================================================

def process_bidperiod_file(
    file_url,
    max_chunks=None
):


    print("\n" + "=" * 100)
    print("PROCESSING BIDPEROFFER ARCHIVE FILE")
    print("=" * 100)


    print("\n" + "-" * 80)
    print("DOWNLOADING FILE")
    print("-" * 80)

    print(file_url)


    response = requests.get(

        file_url,

        timeout=DOWNLOAD_TIMEOUT
    )


    print(
        f"\nHTTP Status: {response.status_code}"
    )


    response.raise_for_status()


    print(
        f"Downloaded: "
        f"{len(response.content):,} bytes"
    )


    # -------------------------------------------------------------
    # OPEN ZIP
    # -------------------------------------------------------------

    with zipfile.ZipFile(

        io.BytesIO(response.content)

    ) as zip_file:


        zip_names = zip_file.namelist()


        print("\n📂 Files inside ZIP:")

        for name in zip_names:

            print(f"   {name}")


        # ---------------------------------------------------------
        # SELECT CSV
        # ---------------------------------------------------------

        csv_candidates = [

            name

            for name in zip_names

            if (
                name.upper().endswith(".CSV")

                and "BIDOFFERPERIOD" in name.upper()
            )
        ]


        if not csv_candidates:

            raise ValueError(
                "❌ No BIDOFFERPERIOD CSV found inside ZIP."
            )


        csv_name = csv_candidates[0]


        print("\n✅ Selected CSV:")

        print(csv_name)


        # ---------------------------------------------------------
        # PROCESS IN CHUNKS
        # ---------------------------------------------------------

        total_uploaded = 0


        with zip_file.open(csv_name) as csv_file:


            for chunk_number, raw_chunk in enumerate(

                iter_bidperiod_chunks(
                    csv_file,
                    chunk_size=CHUNK_SIZE
                ),

                start=1
            ):


                print("\n" + "-" * 80)

                print(
                    f"PROCESSING CHUNK "
                    f"{chunk_number:,}"
                )

                print("-" * 80)


                print(
                    f"Raw chunk rows: "
                    f"{len(raw_chunk):,}"
                )


                # -------------------------------------------------
                # PREPARE
                # -------------------------------------------------

                prepared_chunk = prepare_bidperiod_chunk(
                    raw_chunk
                )


                # -------------------------------------------------
                # CREATE TABLE IF NEEDED
                # -------------------------------------------------

                if (
                    total_uploaded == 0
                    and chunk_number == 1
                ):

                    ensure_bidperiod_table_exists(
                        prepared_chunk
                    )


                # -------------------------------------------------
                # UPLOAD
                # -------------------------------------------------

                upload_bidperiod_chunk(
                    prepared_chunk
                )


                total_uploaded += len(
                    prepared_chunk
                )


                print(
                    f"📊 FILE TOTAL UPLOADED: "
                    f"{total_uploaded:,}"
                )


                # -------------------------------------------------
                # MEMORY CLEANUP
                # -------------------------------------------------

                del raw_chunk
                del prepared_chunk

                gc.collect()


                # -------------------------------------------------
                # OPTIONAL TEST LIMIT
                # -------------------------------------------------

                if (

                    max_chunks is not None

                    and chunk_number >= max_chunks

                ):

                    print(
                        "\n🧪 TEST LIMIT REACHED"
                    )

                    break


    # -------------------------------------------------------------
    # FINAL RESULT
    # -------------------------------------------------------------

    print("\n" + "=" * 100)

    print("FILE COMPLETE")

    print("=" * 100)

    print(
        f"TOTAL ROWS UPLOADED: "
        f"{total_uploaded:,}"
    )


    return total_uploaded


# =============================================================================
# 8. TEST ONE CHUNK
# =============================================================================

def test_bidperiod_one_chunk(
    year,
    month
):


    print("\n" + "#" * 100)

    print(
        f"TESTING ONE BIDPEROFFER CHUNK: "
        f"{year}-{month:02d}"
    )

    print("#" * 100)


    files = discover_bidperiod_files(
        year,
        month
    )


    if not files:

        raise ValueError(
            "❌ No BIDOFFERPERIOD files found."
        )


    test_file = files[0]


    print("\n🧪 TEST MODE")

    print(
        "Processing ONE CHUNK from:"
    )

    print(test_file)


    result = process_bidperiod_file(

        test_file,

        max_chunks=1
    )


    print("\n" + "#" * 100)

    print("TEST COMPLETE")

    print(
        f"ROWS UPLOADED: {result:,}"
    )

    print("#" * 100)


    return result


# =============================================================================
# 9. RUN FULL MONTH
# =============================================================================

def run_bidperiod_month(
    year,
    month
):


    print("\n" + "#" * 100)

    print(
        f"RUNNING FULL BIDPEROFFER BACKFILL: "
        f"{year}-{month:02d}"
    )

    print("#" * 100)


    files = discover_bidperiod_files(
        year,
        month
    )


    if not files:

        print(
            "⚠️ No files found."
        )

        return


    grand_total = 0


    for file_number, file_url in enumerate(

        files,

        start=1
    ):


        print("\n" + "#" * 100)

        print(
            f"FILE {file_number}/{len(files)}"
        )

        print("#" * 100)


        try:

            rows_uploaded = process_bidperiod_file(
                file_url
            )


            grand_total += rows_uploaded


            print(
                f"\n✅ FILE {file_number} COMPLETE"
            )


        except Exception as error:


            print(
                f"\n❌ FILE {file_number} FAILED"
            )

            print(file_url)

            print(error)


            # Continue to next file rather than
            # killing the entire unattended run.
            gc.collect()

            continue


    print("\n" + "#" * 100)

    print("MONTH COMPLETE")

    print("#" * 100)

    print(
        f"TOTAL ROWS UPLOADED: "
        f"{grand_total:,}"
    )


    return grand_total


# =============================================================================
# 10. RUN ONE-CHUNK UPLOAD TEST
# =============================================================================

# Run this FIRST:

test_result = test_bidperiod_one_chunk(
    2024,
    8
)

In [ ]:
####################################################################################################
# SECTION 22 — FULL BIDPEROFFER BACKFILL
####################################################################################################

import time

YEAR = 2024
MONTH = 8

CHUNK_SIZE = 250_000

print("\n" + "#" * 100)
print(f"RUNNING FULL BIDPEROFFER BACKFILL: {YEAR}-{MONTH:02d}")
print("#" * 100)

# -----------------------------------------------------------------------------------------------
# DISCOVER FILES
# -----------------------------------------------------------------------------------------------

bidperiod_files = discover_bidperiod_files(YEAR, MONTH)

if not bidperiod_files:
    raise RuntimeError(
        f"No BIDOFFERPERIOD files found for {YEAR}-{MONTH:02d}"
    )

print("\n" + "=" * 100)
print(f"FILES TO PROCESS: {len(bidperiod_files)}")
print("=" * 100)


# -----------------------------------------------------------------------------------------------
# PROCESS ALL FILES
# -----------------------------------------------------------------------------------------------

total_rows_uploaded = 0
successful_files = 0
failed_files = []

for file_number, file_url in enumerate(bidperiod_files, start=1):

    print("\n\n" + "#" * 100)
    print(f"FILE {file_number}/{len(bidperiod_files)}")
    print("#" * 100)
    print(file_url)
    print()

    try:

        # IMPORTANT:
        # Do NOT pass chunk_size here.
        # The current process_bidperiod_file() function already handles
        # chunking internally.

        rows_uploaded = process_bidperiod_file(
            file_url=file_url
        )

        # Safety in case the function returns None
        if rows_uploaded is None:
            rows_uploaded = 0

        total_rows_uploaded += rows_uploaded
        successful_files += 1

        print("\n" + "-" * 100)
        print(f"✅ FILE {file_number} COMPLETE")
        print(f"📊 Rows uploaded from this file: {rows_uploaded:,}")
        print(f"📊 Running total: {total_rows_uploaded:,}")
        print("-" * 100)

    except Exception as e:

        print("\n❌ FILE FAILED")
        print(file_url)

        print("\nERROR:")
        print(str(e))

        failed_files.append({
            "file_number": file_number,
            "file_url": file_url,
            "error": str(e)
        })

        print("\n⚠️ Continuing to the next file...")

    # Small pause between massive downloads
    time.sleep(2)


# -----------------------------------------------------------------------------------------------
# FINAL SUMMARY
# -----------------------------------------------------------------------------------------------

print("\n\n" + "#" * 100)
print(f"🎉 MONTH COMPLETE: {YEAR}-{MONTH:02d}")
print(f"✅ Successful files: {successful_files}/{len(bidperiod_files)}")
print(f"❌ Failed files: {len(failed_files)}")
print(f"📊 TOTAL ROWS UPLOADED THIS RUN: {total_rows_uploaded:,}")
print("#" * 100)


# -----------------------------------------------------------------------------------------------
# SHOW FAILED FILES
# -----------------------------------------------------------------------------------------------

if failed_files:

    print("\n⚠️ FAILED FILES:")
    print("-" * 100)

    for failed in failed_files:

        print(f"\nFILE {failed['file_number']}")
        print(f"URL: {failed['file_url']}")
        print(f"ERROR: {failed['error']}")

else:

    print("\n🎉 ALL FILES COMPLETED SUCCESSFULLY!")


total_rows_uploaded

In [ ]:
# =============================================================================
# SECTION 22A — READ-ONLY BIDPEROFFER TABLE AUDIT
# Does not modify, reload, delete, or query the table's data.
# =============================================================================

from google.cloud import bigquery

BIDPERIOD_TABLE = "master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE"

audit_client = bigquery.Client(project="master-aemo")
table = audit_client.get_table(BIDPERIOD_TABLE)

print("TABLE:", table.full_table_id)
print("CREATED:", table.created)
print("LAST MODIFIED:", table.modified)
print("METADATA ROW COUNT:", f"{table.num_rows:,}")
print("SIZE (GB):", round(table.num_bytes / (1024**3), 2))

print("\nSCHEMA:")
for field in table.schema:
    print(f"{field.name:<22} {field.field_type:<12} {field.mode}")

In [ ]:
# =============================================================================
# SECTION 22B — READ-ONLY BIDPEROFFER LOAD-JOB AUDIT
# Does not read table data or modify anything.
# =============================================================================

from datetime import datetime, timezone
from collections import Counter
import pandas as pd

TARGET_PROJECT = "master-aemo"
TARGET_DATASET = "aemo_master_schema"
TARGET_TABLE = "BIDPEROFFER_ARCHIVE"
START_TIME = datetime(2026, 9, 7, tzinfo=timezone.utc)

rows = []

for job in audit_client.list_jobs(
    project=TARGET_PROJECT,
    all_users=True,
    min_creation_time=START_TIME,
    max_results=10000
):
    destination = getattr(job, "destination", None)

    if (
        destination
        and destination.project == TARGET_PROJECT
        and destination.dataset_id == TARGET_DATASET
        and destination.table_id == TARGET_TABLE
    ):
        error = getattr(job, "error_result", None) or {}

        rows.append({
            "created": job.created,
            "ended": job.ended,
            "job_id": job.job_id,
            "job_type": job.job_type,
            "state": job.state,
            "output_rows": getattr(job, "output_rows", None),
            "error_reason": error.get("reason"),
            "error_message": error.get("message"),
        })

jobs_audit = pd.DataFrame(rows).sort_values("created")

print("Matching load jobs:", len(jobs_audit))
print("\nJOB SUMMARY:")
display(
    jobs_audit.groupby(
        ["state", "error_reason"],
        dropna=False
    ).agg(
        jobs=("job_id", "count"),
        reported_rows=("output_rows", "sum")
    ).reset_index()
)

print("\nFAILED JOBS:")
display(
    jobs_audit[jobs_audit["error_reason"].notna()]
    [["created", "job_id", "error_reason", "error_message"]]
)

print("\nLAST 20 JOBS:")
display(jobs_audit.tail(20))

In [ ]:
# =============================================================================
# SECTION 23A — V2 PIPELINE PREFLIGHT
# Read-only: creates, changes, and loads nothing.
# =============================================================================

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

LEGACY_TABLE = f"{PROJECT_ID}.{DATASET_ID}.BIDPEROFFER_ARCHIVE"
V2_TABLE = f"{PROJECT_ID}.{DATASET_ID}.BIDPEROFFER_ARCHIVE_V2"
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.BIDPEROFFER_INGEST_MANIFEST"

preflight_client = bigquery.Client(project=PROJECT_ID)

print("Legacy table — preserve unchanged:")
legacy = preflight_client.get_table(LEGACY_TABLE)
print(f"  {LEGACY_TABLE}")
print(f"  Rows: {legacy.num_rows:,}")
print(f"  Size (GB): {legacy.num_bytes / (1024**3):.2f}")

print("\nNew V2 objects:")
for table_id in [V2_TABLE, MANIFEST_TABLE]:
    try:
        existing = preflight_client.get_table(table_id)
        print(f"  ⚠️ Already exists: {table_id} ({existing.num_rows:,} rows)")
    except NotFound:
        print(f"  ✅ Available to create: {table_id}")

print("\nPreflight complete. No BigQuery data was changed.")

In [ ]:
# =============================================================================
# SECTION 23B — CREATE V2 INGEST MANIFEST
# Creates one empty metadata table. Does not load bid data.
# =============================================================================

from google.cloud import bigquery
from google.api_core.exceptions import Conflict

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"
MANIFEST_TABLE = f"{PROJECT_ID}.{DATASET_ID}.BIDPEROFFER_INGEST_MANIFEST"

manifest_schema = [
    bigquery.SchemaField("RUN_ID", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("ARCHIVE_YEAR", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("ARCHIVE_MONTH", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("SOURCE_FILE_NAME", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("SOURCE_URL", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("STAGE_TABLE", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("STATUS", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("SOURCE_ROWS", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("LOADED_ROWS", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("CHUNK_COUNT", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("STARTED_AT", "TIMESTAMP", mode="NULLABLE"),
    bigquery.SchemaField("COMPLETED_AT", "TIMESTAMP", mode="NULLABLE"),
    bigquery.SchemaField("ERROR_MESSAGE", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("PIPELINE_VERSION", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("CREATED_AT", "TIMESTAMP", mode="REQUIRED"),
]

manifest = bigquery.Table(MANIFEST_TABLE, schema=manifest_schema)

try:
    manifest = preflight_client.create_table(manifest)
    print("✅ Empty manifest table created:")
except Conflict:
    manifest = preflight_client.get_table(MANIFEST_TABLE)
    print("ℹ️ Manifest table already existed:")

print(manifest.full_table_id)
print("Rows:", manifest.num_rows)
print("Schema fields:", len(manifest.schema))

In [ ]:
# =============================================================================
# SECTION 23C — DISCOVER AND REGISTER AUGUST 2024 ARCHIVES
# Downloads no ZIPs and loads no bid data.
# =============================================================================

import os
import re
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urlparse
from google.cloud import bigquery

YEAR = 2024
MONTH = 8
RUN_ID = "bidperiod-v2-2024-08"
PIPELINE_VERSION = "v2-staged-file-ingestion"

archive_url = (
    f"https://www.nemweb.com.au/"
    f"Data_Archive/Wholesale_Electricity/MMSDM/{YEAR}/"
    f"MMSDM_{YEAR}_{MONTH:02d}/"
    f"MMSDM_Historical_Data_SQLLoader/DATA/"
)

response = requests.get(archive_url, timeout=120)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
source_urls = []

for link in soup.find_all("a"):
    href = link.get("href")
    if not href:
        continue

    href_upper = href.upper()

    if (
        href_upper.endswith(".ZIP")
        and ("BIDOFFERPERIOD" in href_upper or "BIDPEROFFER" in href_upper)
        and "MNSP_" not in href_upper
    ):
        source_urls.append(
            href if href.startswith("http") else archive_url + href
        )

source_urls = sorted(set(source_urls))

records = []
now_utc = pd.Timestamp.now(tz="UTC")

for source_url in source_urls:
    source_file = os.path.basename(urlparse(source_url).path)
    file_match = re.search(r"FILE(\d+)", source_file.upper())
    file_number = file_match.group(1) if file_match else "UNKNOWN"

    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_STAGE_{YEAR}{MONTH:02d}_FILE{file_number}"
    )

    records.append({
        "RUN_ID": RUN_ID,
        "ARCHIVE_YEAR": YEAR,
        "ARCHIVE_MONTH": MONTH,
        "SOURCE_FILE_NAME": source_file,
        "SOURCE_URL": source_url,
        "STAGE_TABLE": stage_table,
        "STATUS": "DISCOVERED",
        "SOURCE_ROWS": None,
        "LOADED_ROWS": None,
        "CHUNK_COUNT": None,
        "STARTED_AT": None,
        "COMPLETED_AT": None,
        "ERROR_MESSAGE": None,
        "PIPELINE_VERSION": PIPELINE_VERSION,
        "CREATED_AT": now_utc,
    })

inventory_df = pd.DataFrame(records)

existing_sql = f"""
SELECT SOURCE_FILE_NAME
FROM `{MANIFEST_TABLE}`
WHERE ARCHIVE_YEAR = @year
  AND ARCHIVE_MONTH = @month
"""

existing_job = preflight_client.query(
    existing_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("year", "INT64", YEAR),
            bigquery.ScalarQueryParameter("month", "INT64", MONTH),
        ]
    ),
)

existing_files = {
    row.SOURCE_FILE_NAME
    for row in existing_job.result()
}

new_inventory_df = inventory_df[
    ~inventory_df["SOURCE_FILE_NAME"].isin(existing_files)
].copy()

if new_inventory_df.empty:
    print("ℹ️ All discovered files were already registered. Nothing added.")
else:
    load_job = preflight_client.load_table_from_dataframe(
        new_inventory_df,
        MANIFEST_TABLE,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND
        ),
    )
    load_job.result()
    print(f"✅ Registered {len(new_inventory_df)} source archives.")

print(f"\nArchives discovered: {len(inventory_df)}")
print(f"New manifest rows: {len(new_inventory_df)}")

display(
    inventory_df[
        ["SOURCE_FILE_NAME", "STAGE_TABLE", "STATUS"]
    ].head(10)
)

In [ ]:
# =============================================================================
# SECTION 23D — MANIFEST INVENTORY CHECK
# Read-only: confirms the exact files registered in the manifest.
# =============================================================================

inventory_check_sql = f"""
SELECT
  SAFE_CAST(
    REGEXP_EXTRACT(SOURCE_FILE_NAME, r'FILE(\\d+)')
    AS INT64
  ) AS file_number,
  SOURCE_FILE_NAME,
  SOURCE_URL,
  STAGE_TABLE,
  STATUS
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = 'bidperiod-v2-2024-08'
ORDER BY file_number, SOURCE_FILE_NAME
"""

inventory_check = preflight_client.query(
    inventory_check_sql
).to_dataframe()

print("Manifest rows:", len(inventory_check))
print("Distinct filenames:", inventory_check["SOURCE_FILE_NAME"].nunique())
print("Distinct stage tables:", inventory_check["STAGE_TABLE"].nunique())

display(inventory_check)

In [ ]:
# =============================================================================
# SECTION 23E — EXCLUDE BIDPEROFFER_D FROM THIS BIDOFFERPERIOD PIPELINE
# Updates one manifest row only. No bid/archive data is changed or deleted.
# =============================================================================

from google.cloud import bigquery

RUN_ID = "bidperiod-v2-2024-08"

extra_sql = f"""
SELECT SOURCE_FILE_NAME
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME LIKE '%BIDPEROFFER_D%'
"""

extra_df = preflight_client.query(
    extra_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

if len(extra_df) != 1:
    raise RuntimeError(
        f"Expected exactly one BIDPEROFFER_D record; found {len(extra_df)}. "
        "No change made."
    )

non_target_file = extra_df.iloc[0]["SOURCE_FILE_NAME"]

update_sql = f"""
UPDATE `{MANIFEST_TABLE}`
SET
  STATUS = 'EXCLUDED_NON_TARGET',
  STAGE_TABLE = NULL,
  ERROR_MESSAGE = 'Separate BIDPEROFFER_D dataset; excluded from BIDOFFERPERIOD V2 ingestion.'
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME = @source_file
"""

update_job = preflight_client.query(
    update_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "source_file", "STRING", non_target_file
            ),
        ]
    ),
)
update_job.result()

print("✅ Excluded:", non_target_file)
print("Rows updated:", update_job.num_dml_affected_rows)

check_sql = f"""
SELECT STATUS, COUNT(*) AS files
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
GROUP BY STATUS
ORDER BY STATUS
"""

display(
    preflight_client.query(
        check_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
            ]
        ),
    ).to_dataframe()
)

In [ ]:
# =============================================================================
# SECTION 23G — REPAIR MANIFEST SOURCE URLS
# Corrects 34 malformed metadata URLs only. Does not load or delete bid data.
# =============================================================================

import requests
from google.cloud import bigquery

RUN_ID = "bidperiod-v2-2024-08"
DOMAIN = "https://www.nemweb.com.au"
MARKER = "/Data_Archive/"

url_rows_sql = f"""
SELECT SOURCE_FILE_NAME, SOURCE_URL
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND STATUS = 'DISCOVERED'
ORDER BY SOURCE_FILE_NAME
"""

url_rows = preflight_client.query(
    url_rows_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

if len(url_rows) != 34:
    raise RuntimeError(
        f"Expected 34 DISCOVERED records; found {len(url_rows)}. No change made."
    )

def repair_aemo_url(url):
    first_marker = url.find(MARKER)
    second_marker = url.find(MARKER, first_marker + len(MARKER))

    if second_marker == -1:
        raise ValueError(f"Expected duplicated archive path, but found: {url}")

    return DOMAIN + url[second_marker:]

url_rows["FIXED_SOURCE_URL"] = url_rows["SOURCE_URL"].apply(
    repair_aemo_url
)

for _, record in url_rows.iterrows():
    update_sql = f"""
    UPDATE `{MANIFEST_TABLE}`
    SET SOURCE_URL = @fixed_url
    WHERE RUN_ID = @run_id
      AND SOURCE_FILE_NAME = @source_file
    """

    update_job = preflight_client.query(
        update_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "fixed_url", "STRING", record["FIXED_SOURCE_URL"]
                ),
                bigquery.ScalarQueryParameter(
                    "run_id", "STRING", RUN_ID
                ),
                bigquery.ScalarQueryParameter(
                    "source_file", "STRING", record["SOURCE_FILE_NAME"]
                ),
            ]
        ),
    )
    update_job.result()

print(f"✅ Repaired {len(url_rows)} manifest URLs.")

# Validate FILE01 without downloading its ZIP body.
file01_url = url_rows.iloc[0]["FIXED_SOURCE_URL"]
check_response = requests.get(file01_url, stream=True, timeout=120)

print("FILE01 HTTP status:", check_response.status_code)
print("Corrected FILE01 URL:", file01_url)

check_response.close()

In [ ]:
# =============================================================================
# SECTION 23F — V2 ONE-CHUNK STAGING TEST
# Loads 250,000 rows to a dedicated TEST table only.
# =============================================================================

import io
import os
import csv
import zipfile
import requests
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

TEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_STAGE_TEST_202408_FILE01"
)
TEST_CHUNK_SIZE = 250_000
RUN_ID = "bidperiod-v2-2024-08"

# Refuse to overwrite a prior test result.
try:
    preflight_client.get_table(TEST_TABLE)
    raise RuntimeError(
        f"{TEST_TABLE} already exists. Stop here; do not overwrite it."
    )
except NotFound:
    pass

# Read FILE01's registered source URL from the manifest.
source_sql = f"""
SELECT SOURCE_FILE_NAME, SOURCE_URL
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND STATUS = 'DISCOVERED'
  AND SOURCE_FILE_NAME LIKE '%BIDOFFERPERIOD%FILE01%'
"""

source_df = preflight_client.query(
    source_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

if len(source_df) != 1:
    raise RuntimeError(
        f"Expected exactly one BIDOFFERPERIOD FILE01 record; found {len(source_df)}."
    )

source_file = source_df.iloc[0]["SOURCE_FILE_NAME"]
source_url = source_df.iloc[0]["SOURCE_URL"]

print("Test source:", source_file)
print("Downloading ZIP...")

response = requests.get(source_url, timeout=300)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    csv_name = next(
        name for name in zip_file.namelist()
        if name.upper().endswith(".CSV")
        and "BIDOFFERPERIOD" in name.upper()
    )

    with zip_file.open(csv_name) as csv_file:

        # Find the AEMO BIDOFFERPERIOD column-definition record.
        header_wrapper = io.TextIOWrapper(
            csv_file, encoding="utf-8", newline=""
        )
        header_reader = csv.reader(header_wrapper)

        columns = None
        for row in header_reader:
            if (
                len(row) >= 5
                and row[0].strip().upper() == "I"
                and row[1].strip().upper() == "BIDS"
                and row[2].strip().upper() == "BIDOFFERPERIOD"
            ):
                columns = [column.strip() for column in row[4:]]
                break

        header_wrapper.detach()

        if columns is None:
            raise ValueError("Could not find BIDOFFERPERIOD column definition.")

        # Reopen from the beginning and collect one chunk only.
        csv_file.seek(0)
        data_wrapper = io.TextIOWrapper(
            csv_file, encoding="utf-8", newline=""
        )
        data_reader = csv.reader(data_wrapper)

        rows = []
        for row in data_reader:
            if (
                len(row) >= 4
                and row[0].strip().upper() == "D"
                and row[1].strip().upper() == "BIDS"
                and row[2].strip().upper() == "BIDOFFERPERIOD"
            ):
                values = row[4:]
                if len(values) == len(columns):
                    rows.append(values)

            if len(rows) >= TEST_CHUNK_SIZE:
                break

        data_wrapper.detach()

test_df = pd.DataFrame(rows, columns=columns)

# Convert to the V2 warehouse structure.
test_df = test_df.rename(
    columns={
        "TRADINGDATE": "SETTLEMENTDATE",
        "OFFERDATETIME": "OFFERDATE",
    }
)

for column in ["SETTLEMENTDATE", "OFFERDATE"]:
    test_df[column] = pd.to_datetime(
        test_df[column], errors="coerce", utc=True
    )

numeric_columns = [
    "PERIODID", "MAXAVAIL", "FIXEDLOAD", "RAMPUPRATE", "RAMPDOWNRATE",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4", "BANDAVAIL5",
    "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8", "BANDAVAIL9", "BANDAVAIL10",
    "PASAAVAILABILITY", "ENERGYLIMIT", "PERIODIDTO",
]

for column in numeric_columns:
    test_df[column] = pd.to_numeric(test_df[column], errors="coerce")

# File-level provenance is the critical V2 improvement.
test_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
test_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
test_df["_PIPELINE"] = "BIDPEROFFER_V2_STAGED"
test_df["_RUN_ID"] = RUN_ID
test_df["_SOURCE_FILE"] = source_file
test_df["_SOURCE_URL"] = source_url

print(f"Rows prepared: {len(test_df):,}")
print(f"Columns prepared: {len(test_df.columns)}")

load_job = preflight_client.load_table_from_dataframe(
    test_df,
    TEST_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
        autodetect=True,
    ),
)
load_job.result()

test_table = preflight_client.get_table(TEST_TABLE)

print("\n✅ TEST COMPLETE")
print("Table:", test_table.full_table_id)
print("Rows:", f"{test_table.num_rows:,}")
print("Columns:", len(test_table.schema))

display(test_df.head(3))

In [ ]:
# =============================================================================
# SECTION 23H — CREATE EMPTY V2 ARCHIVE TABLE
# Creates an empty partitioned table only. Does not load or alter legacy data.
# =============================================================================

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

LEGACY_TABLE = "master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE"
V2_TABLE = "master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE_V2"

# Refuse to overwrite if this cell is accidentally run twice.
try:
    preflight_client.get_table(V2_TABLE)
    raise RuntimeError(
        f"{V2_TABLE} already exists. Stop here; do not overwrite it."
    )
except NotFound:
    pass

legacy_table = preflight_client.get_table(LEGACY_TABLE)
v2_schema = list(legacy_table.schema)

existing_fields = {field.name for field in v2_schema}

for field in [
    bigquery.SchemaField("_RUN_ID", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("_SOURCE_FILE", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("_SOURCE_URL", "STRING", mode="NULLABLE"),
]:
    if field.name not in existing_fields:
        v2_schema.append(field)

v2_table = bigquery.Table(V2_TABLE, schema=v2_schema)

# Partitioning makes future monthly audits and analytics cheaper/faster.
v2_table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="SETTLEMENTDATE",
)

# Useful access pattern for unit/bid-type analysis.
v2_table.clustering_fields = ["DUID", "BIDTYPE"]

created_v2 = preflight_client.create_table(v2_table)

print("✅ Empty V2 archive table created:")
print("Table:", created_v2.full_table_id)
print("Rows:", created_v2.num_rows)
print("Columns:", len(created_v2.schema))
print("Partition field:", created_v2.time_partitioning.field)
print("Clustering:", created_v2.clustering_fields)

In [ ]:
# =============================================================================
# SECTION 23I — STAGE FULL FILE01
# Loads one complete archive into its own staging table.
# Does not change the legacy table or BIDPEROFFER_ARCHIVE_V2.
# =============================================================================

import io
import re
import gc
import csv
import zipfile
import requests
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

RUN_ID = "bidperiod-v2-2024-08"
CHUNK_SIZE = 250_000

# Get FILE01 metadata from the manifest.
file_sql = f"""
SELECT SOURCE_FILE_NAME, SOURCE_URL, STAGE_TABLE
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND STATUS = 'DISCOVERED'
  AND SOURCE_FILE_NAME LIKE '%BIDOFFERPERIOD%FILE01%'
"""

file_df = preflight_client.query(
    file_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

if len(file_df) != 1:
    raise RuntimeError(f"Expected one eligible FILE01; found {len(file_df)}.")

source_file = file_df.iloc[0]["SOURCE_FILE_NAME"]
source_url = file_df.iloc[0]["SOURCE_URL"]
stage_table = file_df.iloc[0]["STAGE_TABLE"]

file_token = re.search(r"FILE\d+", source_file.upper()).group(0)

# Never overwrite an existing real staging table.
try:
    preflight_client.get_table(stage_table)
    raise RuntimeError(
        f"Staging table already exists: {stage_table}. Stop; do not overwrite it."
    )
except NotFound:
    pass

# Mark this file as in progress in the small manifest table.
start_sql = f"""
UPDATE `{MANIFEST_TABLE}`
SET
  STATUS = 'STAGING',
  STARTED_AT = CURRENT_TIMESTAMP(),
  ERROR_MESSAGE = NULL
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME = @source_file
"""

preflight_client.query(
    start_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "source_file", "STRING", source_file
            ),
        ]
    ),
).result()

# Use the V2 schema so every stage table is structurally compatible.
v2_schema = preflight_client.get_table(V2_TABLE).schema

numeric_columns = [
    "PERIODID", "MAXAVAIL", "FIXEDLOAD", "RAMPUPRATE", "RAMPDOWNRATE",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4", "BANDAVAIL5",
    "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8", "BANDAVAIL9", "BANDAVAIL10",
    "PASAAVAILABILITY", "ENERGYLIMIT", "PERIODIDTO",
]

def prepare_v2_stage_chunk(df, load_time):
    df = df.rename(
        columns={
            "TRADINGDATE": "SETTLEMENTDATE",
            "OFFERDATETIME": "OFFERDATE",
        }
    ).copy()

    for column in ["SETTLEMENTDATE", "OFFERDATE"]:
        df[column] = pd.to_datetime(
            df[column], errors="coerce", utc=True
        )

    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df["_LOAD_DATE"] = load_time
    df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    df["_PIPELINE"] = "BIDPEROFFER_V2_STAGED"
    df["_RUN_ID"] = RUN_ID
    df["_SOURCE_FILE"] = file_token
    df["_SOURCE_URL"] = None  # Full URL is held once in the manifest.

    return df

rows_loaded = 0
chunk_count = 0
load_time = pd.Timestamp.now(tz="UTC")

try:
    print("Downloading:", source_file)
    response = requests.get(source_url, timeout=300)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
            and "BIDOFFERPERIOD" in name.upper()
        )

        with zip_file.open(csv_name) as csv_file:
            header_wrapper = io.TextIOWrapper(
                csv_file, encoding="utf-8", newline=""
            )
            header_reader = csv.reader(header_wrapper)

            columns = None
            for row in header_reader:
                if (
                    len(row) >= 5
                    and row[0].strip().upper() == "I"
                    and row[1].strip().upper() == "BIDS"
                    and row[2].strip().upper() == "BIDOFFERPERIOD"
                ):
                    columns = [column.strip() for column in row[4:]]
                    break

            header_wrapper.detach()

            if columns is None:
                raise ValueError("BIDOFFERPERIOD columns were not found.")

            csv_file.seek(0)
            data_wrapper = io.TextIOWrapper(
                csv_file, encoding="utf-8", newline=""
            )
            reader = csv.reader(data_wrapper)
            batch = []

            for row in reader:
                if (
                    len(row) >= 4
                    and row[0].strip().upper() == "D"
                    and row[1].strip().upper() == "BIDS"
                    and row[2].strip().upper() == "BIDOFFERPERIOD"
                ):
                    values = row[4:]
                    if len(values) == len(columns):
                        batch.append(values)

                if len(batch) >= CHUNK_SIZE:
                    chunk_count += 1
                    chunk = prepare_v2_stage_chunk(
                        pd.DataFrame(batch, columns=columns),
                        load_time,
                    )

                    job = preflight_client.load_table_from_dataframe(
                        chunk,
                        stage_table,
                        job_config=bigquery.LoadJobConfig(
                            schema=v2_schema,
                            write_disposition=(
                                bigquery.WriteDisposition.WRITE_EMPTY
                                if chunk_count == 1
                                else bigquery.WriteDisposition.WRITE_APPEND
                            ),
                        ),
                    )
                    job.result()

                    rows_loaded += len(chunk)
                    if chunk_count % 10 == 0:
                        print(
                            f"Chunks loaded: {chunk_count:,} | "
                            f"Rows loaded: {rows_loaded:,}"
                        )

                    del chunk
                    batch.clear()
                    gc.collect()

            if batch:
                chunk_count += 1
                chunk = prepare_v2_stage_chunk(
                    pd.DataFrame(batch, columns=columns),
                    load_time,
                )

                job = preflight_client.load_table_from_dataframe(
                    chunk,
                    stage_table,
                    job_config=bigquery.LoadJobConfig(
                        schema=v2_schema,
                        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
                    ),
                )
                job.result()
                rows_loaded += len(chunk)

            data_wrapper.detach()

    complete_sql = f"""
    UPDATE `{MANIFEST_TABLE}`
    SET
      STATUS = 'STAGED',
      SOURCE_ROWS = @rows_loaded,
      LOADED_ROWS = @rows_loaded,
      CHUNK_COUNT = @chunk_count,
      COMPLETED_AT = CURRENT_TIMESTAMP(),
      ERROR_MESSAGE = NULL
    WHERE RUN_ID = @run_id
      AND SOURCE_FILE_NAME = @source_file
    """

    preflight_client.query(
        complete_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "rows_loaded", "INT64", rows_loaded
                ),
                bigquery.ScalarQueryParameter(
                    "chunk_count", "INT64", chunk_count
                ),
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
                bigquery.ScalarQueryParameter(
                    "source_file", "STRING", source_file
                ),
            ]
        ),
    ).result()

    staged = preflight_client.get_table(stage_table)

    print("\n✅ FILE01 STAGED SUCCESSFULLY")
    print("Stage table:", staged.full_table_id)
    print("Rows loaded:", f"{rows_loaded:,}")
    print("Chunks:", chunk_count)
    print("Stage-table rows:", f"{staged.num_rows:,}")

except Exception as error:
    failed_sql = f"""
    UPDATE `{MANIFEST_TABLE}`
    SET
      STATUS = 'FAILED_STAGE',
      LOADED_ROWS = @rows_loaded,
      CHUNK_COUNT = @chunk_count,
      ERROR_MESSAGE = @error_message,
      COMPLETED_AT = CURRENT_TIMESTAMP()
    WHERE RUN_ID = @run_id
      AND SOURCE_FILE_NAME = @source_file
    """

    preflight_client.query(
        failed_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "rows_loaded", "INT64", rows_loaded
                ),
                bigquery.ScalarQueryParameter(
                    "chunk_count", "INT64", chunk_count
                ),
                bigquery.ScalarQueryParameter(
                    "error_message", "STRING", str(error)[:10000]
                ),
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
                bigquery.ScalarQueryParameter(
                    "source_file", "STRING", source_file
                ),
            ]
        ),
    ).result()

    raise

In [ ]:
# =============================================================================
# SECTION 23J — VALIDATE AND PROMOTE FILE01
# Moves one fully staged archive into V2 exactly once.
# =============================================================================

from google.cloud import bigquery

FILE_TOKEN = "FILE01"

stage = preflight_client.get_table(stage_table)
v2 = preflight_client.get_table(V2_TABLE)

stage_schema = [(f.name, f.field_type) for f in stage.schema]
v2_schema_check = [(f.name, f.field_type) for f in v2.schema]

if stage_schema != v2_schema_check:
    raise RuntimeError("Stage/V2 schemas differ. No promotion performed.")

stage_rows = stage.num_rows
print("Stage rows:", f"{stage_rows:,}")

existing_sql = f"""
SELECT COUNT(*) AS rows_already_in_v2
FROM `{V2_TABLE}`
WHERE _RUN_ID = @run_id
  AND _SOURCE_FILE = @file_token
"""

existing_rows = list(
    preflight_client.query(
        existing_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
                bigquery.ScalarQueryParameter(
                    "file_token", "STRING", FILE_TOKEN
                ),
            ]
        ),
    ).result()
)[0].rows_already_in_v2

if existing_rows not in [0, stage_rows]:
    raise RuntimeError(
        f"V2 already contains {existing_rows:,} FILE01 rows, "
        f"but stage has {stage_rows:,}. No promotion performed."
    )

if existing_rows == 0:
    print("Promoting FILE01 into V2...")

    promote_job = preflight_client.query(f"""
    INSERT INTO `{V2_TABLE}`
    SELECT *
    FROM `{stage_table}`
    """)
    promote_job.result()

    print("Promotion query completed.")
else:
    print("ℹ️ FILE01 is already present in V2; no second insert performed.")

verify_rows = list(
    preflight_client.query(
        existing_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
                bigquery.ScalarQueryParameter(
                    "file_token", "STRING", FILE_TOKEN
                ),
            ]
        ),
    ).result()
)[0].rows_already_in_v2

if verify_rows != stage_rows:
    raise RuntimeError(
        f"Verification failed: V2 has {verify_rows:,}; "
        f"stage has {stage_rows:,}."
    )

complete_sql = f"""
UPDATE `{MANIFEST_TABLE}`
SET
  STATUS = 'PROMOTED',
  COMPLETED_AT = CURRENT_TIMESTAMP(),
  ERROR_MESSAGE = NULL
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME = @source_file
"""

preflight_client.query(
    complete_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "source_file", "STRING", source_file
            ),
        ]
    ),
).result()

print("\n✅ FILE01 PROMOTED SUCCESSFULLY")
print("V2 FILE01 rows:", f"{verify_rows:,}")

In [ ]:
# =============================================================================
# SECTION 23K — CLEAN UP VERIFIED FILE01 STAGE
# Deletes only the large FILE01 staging table after count verification.
# =============================================================================

from google.cloud import bigquery

FILE_TOKEN = "FILE01"

stage = preflight_client.get_table(stage_table)
stage_rows = stage.num_rows

v2_rows_sql = f"""
SELECT COUNT(*) AS row_count
FROM `{V2_TABLE}`
WHERE _RUN_ID = @run_id
  AND _SOURCE_FILE = @file_token
"""

v2_rows = list(
    preflight_client.query(
        v2_rows_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
                bigquery.ScalarQueryParameter(
                    "file_token", "STRING", FILE_TOKEN
                ),
            ]
        ),
    ).result()
)[0].row_count

if v2_rows != stage_rows:
    raise RuntimeError(
        f"Stop: stage has {stage_rows:,} rows but V2 has {v2_rows:,}."
    )

# This is the approved deletion: only the verified temporary FILE01 stage.
preflight_client.delete_table(stage_table)

cleanup_sql = f"""
UPDATE `{MANIFEST_TABLE}`
SET STATUS = 'PROMOTED_CLEANED'
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME = @source_file
"""

preflight_client.query(
    cleanup_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "source_file", "STRING", source_file
            ),
        ]
    ),
).result()

print("✅ FILE01 stage deleted after verification.")
print("V2 retained rows:", f"{v2_rows:,}")
print("Manifest status: PROMOTED_CLEANED")

In [ ]:
# =============================================================================
# SECTION 24A — REUSABLE V2 FILE RUNNER
# Defines a safe stage → promote → cleanup function.
# Does not run any archive until called.
# =============================================================================

def run_bidperiod_v2_file(file_token):
    file_token = file_token.upper()
    run_id = "bidperiod-v2-2024-08"

    record_sql = f"""
    SELECT SOURCE_FILE_NAME, SOURCE_URL, STAGE_TABLE
    FROM `{MANIFEST_TABLE}`
    WHERE RUN_ID = @run_id
      AND STATUS = 'DISCOVERED'
      AND SOURCE_FILE_NAME LIKE CONCAT('%BIDOFFERPERIOD%', @file_token, '%')
    """

    record_df = preflight_client.query(
        record_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                bigquery.ScalarQueryParameter(
                    "file_token", "STRING", file_token
                ),
            ]
        ),
    ).to_dataframe()

    if len(record_df) != 1:
        raise RuntimeError(
            f"Expected one DISCOVERED {file_token} record; found {len(record_df)}."
        )

    source_file = record_df.iloc[0]["SOURCE_FILE_NAME"]
    source_url = record_df.iloc[0]["SOURCE_URL"]
    stage_table_id = record_df.iloc[0]["STAGE_TABLE"]

    try:
        preflight_client.get_table(stage_table_id)
        raise RuntimeError(
            f"Stage table already exists: {stage_table_id}. No overwrite allowed."
        )
    except NotFound:
        pass

    def manifest_update(status, rows=None, chunks=None, error=None):
        update_sql = f"""
        UPDATE `{MANIFEST_TABLE}`
        SET
          STATUS = @status,
          LOADED_ROWS = COALESCE(@rows, LOADED_ROWS),
          SOURCE_ROWS = COALESCE(@rows, SOURCE_ROWS),
          CHUNK_COUNT = COALESCE(@chunks, CHUNK_COUNT),
          ERROR_MESSAGE = @error,
          STARTED_AT = IF(@status = 'STAGING', CURRENT_TIMESTAMP(), STARTED_AT),
          COMPLETED_AT = IF(
            @status IN ('STAGED', 'PROMOTED_CLEANED',
                        'FAILED_STAGE', 'FAILED_PROMOTION'),
            CURRENT_TIMESTAMP(),
            COMPLETED_AT
          )
        WHERE RUN_ID = @run_id
          AND SOURCE_FILE_NAME = @source_file
        """

        preflight_client.query(
            update_sql,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[
                    bigquery.ScalarQueryParameter("status", "STRING", status),
                    bigquery.ScalarQueryParameter("rows", "INT64", rows),
                    bigquery.ScalarQueryParameter("chunks", "INT64", chunks),
                    bigquery.ScalarQueryParameter("error", "STRING", error),
                    bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
                    bigquery.ScalarQueryParameter(
                        "source_file", "STRING", source_file
                    ),
                ]
            ),
        ).result()

    manifest_update("STAGING")

    rows_loaded = 0
    chunk_count = 0
    load_time = pd.Timestamp.now(tz="UTC")
    v2_schema = preflight_client.get_table(V2_TABLE).schema

    try:
        print(f"Downloading {file_token}: {source_file}")
        response = requests.get(source_url, timeout=300)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
            csv_name = next(
                name for name in zip_file.namelist()
                if name.upper().endswith(".CSV")
                and "BIDOFFERPERIOD" in name.upper()
            )

            with zip_file.open(csv_name) as csv_file:
                header = io.TextIOWrapper(
                    csv_file, encoding="utf-8", newline=""
                )
                columns = None

                for row in csv.reader(header):
                    if (
                        len(row) >= 5
                        and row[0].strip().upper() == "I"
                        and row[1].strip().upper() == "BIDS"
                        and row[2].strip().upper() == "BIDOFFERPERIOD"
                    ):
                        columns = [column.strip() for column in row[4:]]
                        break

                header.detach()

                if columns is None:
                    raise ValueError("BIDOFFERPERIOD columns were not found.")

                csv_file.seek(0)
                data = io.TextIOWrapper(
                    csv_file, encoding="utf-8", newline=""
                )
                batch = []

                for row in csv.reader(data):
                    if (
                        len(row) >= 4
                        and row[0].strip().upper() == "D"
                        and row[1].strip().upper() == "BIDS"
                        and row[2].strip().upper() == "BIDOFFERPERIOD"
                    ):
                        values = row[4:]
                        if len(values) == len(columns):
                            batch.append(values)

                    if len(batch) >= CHUNK_SIZE:
                        chunk_count += 1
                        chunk = prepare_v2_stage_chunk(
                            pd.DataFrame(batch, columns=columns),
                            load_time,
                        )
                        chunk["_RUN_ID"] = run_id
                        chunk["_SOURCE_FILE"] = file_token
                        chunk["_SOURCE_URL"] = None

                        preflight_client.load_table_from_dataframe(
                            chunk,
                            stage_table_id,
                            job_config=bigquery.LoadJobConfig(
                                schema=v2_schema,
                                write_disposition=(
                                    bigquery.WriteDisposition.WRITE_EMPTY
                                    if chunk_count == 1
                                    else bigquery.WriteDisposition.WRITE_APPEND
                                ),
                            ),
                        ).result()

                        rows_loaded += len(chunk)
                        if chunk_count % 10 == 0:
                            print(
                                f"{file_token}: {chunk_count} chunks | "
                                f"{rows_loaded:,} rows"
                            )

                        del chunk
                        batch.clear()
                        gc.collect()

                if batch:
                    chunk_count += 1
                    chunk = prepare_v2_stage_chunk(
                        pd.DataFrame(batch, columns=columns),
                        load_time,
                    )
                    chunk["_RUN_ID"] = run_id
                    chunk["_SOURCE_FILE"] = file_token
                    chunk["_SOURCE_URL"] = None

                    preflight_client.load_table_from_dataframe(
                        chunk,
                        stage_table_id,
                        job_config=bigquery.LoadJobConfig(
                            schema=v2_schema,
                            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
                        ),
                    ).result()
                    rows_loaded += len(chunk)

                data.detach()

        manifest_update("STAGED", rows_loaded, chunk_count)
        print(f"✅ {file_token} staged: {rows_loaded:,} rows")

    except Exception as error:
        manifest_update(
            "FAILED_STAGE", rows_loaded, chunk_count, str(error)[:10000]
        )
        raise

    try:
        preflight_client.query(f"""
        INSERT INTO `{V2_TABLE}`
        SELECT *
        FROM `{stage_table_id}`
        """).result()

        check_sql = f"""
        SELECT COUNT(*) AS row_count
        FROM `{V2_TABLE}`
        WHERE _RUN_ID = @run_id
          AND _SOURCE_FILE = @file_token
        """

        v2_rows = list(
            preflight_client.query(
                check_sql,
                job_config=bigquery.QueryJobConfig(
                    query_parameters=[
                        bigquery.ScalarQueryParameter(
                            "run_id", "STRING", run_id
                        ),
                        bigquery.ScalarQueryParameter(
                            "file_token", "STRING", file_token
                        ),
                    ]
                ),
            ).result()
        )[0].row_count

        if v2_rows != rows_loaded:
            raise RuntimeError(
                f"V2 verification failed: {v2_rows:,} vs {rows_loaded:,}."
            )

    except Exception as error:
        manifest_update(
            "FAILED_PROMOTION", rows_loaded, chunk_count, str(error)[:10000]
        )
        raise

    preflight_client.delete_table(stage_table_id)
    manifest_update("PROMOTED_CLEANED", rows_loaded, chunk_count)

    print(f"\n✅ {file_token} COMPLETE")
    print(f"V2 rows retained: {v2_rows:,}")
    print("Temporary stage deleted.")

In [ ]:
run_bidperiod_v2_file("FILE02")

In [ ]:
run_bidperiod_v2_file("FILE03")

In [ ]:
run_bidperiod_v2_file("FILE04")



In [ ]:
# Show the actual columns in the manifest table
manifest_table = client.get_table(MANIFEST_TABLE)

print("\nMANIFEST TABLE COLUMNS:\n")

for field in manifest_table.schema:
    print(f"{field.name:30} {field.field_type}")

In [ ]:
# READ-ONLY: identify the manifest record(s) blocking the batch

query = f"""
SELECT
    SOURCE_FILE_NAME,
    STATUS,
    SOURCE_ROWS,
    LOADED_ROWS,
    CHUNK_COUNT,
    STARTED_AT,
    COMPLETED_AT,
    ERROR_MESSAGE,
    RUN_ID
FROM `{MANIFEST_TABLE}`
WHERE STATUS NOT IN ('PROMOTED_CLEANED', 'EXCLUDED_NON_TARGET')
ORDER BY CREATED_AT
"""

df_manifest_problem = client.query(query).to_dataframe()

print("\nFILES REQUIRING ATTENTION:\n")
display(df_manifest_problem)

In [ ]:
# Reset ONLY FILE08 after its failed download.
# FILE08 had 0 rows loaded, so there is no partial stage to clean up.

query = f"""
UPDATE `{MANIFEST_TABLE}`
SET
    STATUS = 'DISCOVERED',
    ERROR_MESSAGE = NULL,
    STARTED_AT = NULL,
    COMPLETED_AT = NULL,
    LOADED_ROWS = 0,
    CHUNK_COUNT = 0
WHERE SOURCE_FILE_NAME LIKE '%BIDOFFERPERIOD%FILE08%'
  AND STATUS = 'FAILED_STAGE'
"""

job = client.query(query)
job.result()

print("✅ FILE08 reset to DISCOVERED.")

In [ ]:
run_bidperiod_v2_file("FILE08")

In [ ]:
# =============================================================================
# SECTION 24B — RUN ALL REMAINING AUGUST 2024 V2 ARCHIVES
# Runs FILE05–FILE34 sequentially and stops safely on any error.
# =============================================================================

import re
import gc
import time

required_items = [
    "run_bidperiod_v2_file",
    "prepare_v2_stage_chunk",
    "CHUNK_SIZE",
]

missing_items = [
    item for item in required_items
    if item not in globals()
]

if missing_items:
    raise RuntimeError(
        f"Missing required V2 functions/variables: {missing_items}. "
        "Do not run this after restarting the kernel without restoring them."
    )

RUN_ID = "bidperiod-v2-2024-08"

status_sql = f"""
SELECT STATUS, COUNT(*) AS files
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
GROUP BY STATUS
ORDER BY STATUS
"""

status_df = preflight_client.query(
    status_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

print("CURRENT MANIFEST STATUS:")
display(status_df)

blocking_statuses = {
    "STAGING",
    "STAGED",
    "FAILED_STAGE",
    "FAILED_PROMOTION",
}

if status_df["STATUS"].isin(blocking_statuses).any():
    raise RuntimeError(
        "A file is still running, staged, or needs attention. "
        "Do not start the batch until FILE04 has completed cleanly."
    )

pending_sql = f"""
SELECT SOURCE_FILE_NAME
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND STATUS = 'DISCOVERED'
ORDER BY SAFE_CAST(
    REGEXP_EXTRACT(SOURCE_FILE_NAME, r'FILE(\\d+)') AS INT64
)
"""

pending_df = preflight_client.query(
    pending_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
        ]
    ),
).to_dataframe()

if pending_df.empty:
    print("✅ No remaining DISCOVERED files. August 2024 is complete.")
else:
    pending_tokens = [
        re.search(r"FILE\d+", filename.upper()).group(0)
        for filename in pending_df["SOURCE_FILE_NAME"]
    ]

    print(
        f"Starting {len(pending_tokens)} remaining files: "
        f"{pending_tokens[0]} through {pending_tokens[-1]}"
    )

    for file_token in pending_tokens:
        print("\n" + "=" * 100)
        print(f"STARTING {file_token}")
        print("=" * 100)

        try:
            run_bidperiod_v2_file(file_token)

        except Exception as error:
            print("\n❌ BATCH STOPPED SAFELY")
            print(f"File requiring attention: {file_token}")
            print(f"Error: {error}")
            print(
                "The manifest records the status. "
                "No later files were started."
            )
            break

        gc.collect()
        time.sleep(5)

    else:
        print("\n🎉 ALL REMAINING AUGUST 2024 FILES COMPLETED.")

In [ ]:
# =============================================================================
# SECTION 24C — RETRYING AEMO DOWNLOADS
# Patches downloads for this live notebook session only.
# BigQuery loading logic is unchanged.
# =============================================================================

import time
import requests

MAX_DOWNLOAD_ATTEMPTS = 5

if not hasattr(requests, "_aemo_original_get"):
    requests._aemo_original_get = requests.get

_original_get = requests._aemo_original_get

def retrying_aemo_get(*args, **kwargs):
    last_error = None

    for attempt in range(1, MAX_DOWNLOAD_ATTEMPTS + 1):
        response = None

        try:
            response = _original_get(*args, **kwargs)
            response.raise_for_status()

            # Force the complete ZIP download now, so incomplete transfers
            # are caught and retried before the pipeline starts parsing.
            _ = response.content
            return response

        except requests.exceptions.HTTPError as error:
            status = (
                error.response.status_code
                if error.response is not None
                else None
            )

            # Do not retry permanent client errors such as 404.
            if status is not None and status < 500 and status != 429:
                raise

            last_error = error

        except Exception as error:
            last_error = error

        finally:
            if response is not None:
                response.close()

        if attempt < MAX_DOWNLOAD_ATTEMPTS:
            wait_seconds = min(120, 5 * (2 ** (attempt - 1)))
            print(
                f"⚠️ Download attempt {attempt}/{MAX_DOWNLOAD_ATTEMPTS} failed. "
                f"Retrying in {wait_seconds}s..."
            )
            print(f"   {last_error}")
            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Download failed after {MAX_DOWNLOAD_ATTEMPTS} attempts: {last_error}"
    )

requests.get = retrying_aemo_get

print(
    f"✅ Retrying AEMO downloader installed "
    f"({MAX_DOWNLOAD_ATTEMPTS} attempts per ZIP)."
)

In [ ]:
# =============================================================================
# SECTION 24D — VERIFY, RESET, AND RETRY FILE10
# Resets FILE10 only if it has no staged rows or stage table.
# =============================================================================

FILE_TOKEN = "FILE10"
RUN_ID = "bidperiod-v2-2024-08"

file10_sql = f"""
SELECT SOURCE_FILE_NAME, STAGE_TABLE, STATUS, LOADED_ROWS, CHUNK_COUNT
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME LIKE CONCAT('%BIDOFFERPERIOD%', @file_token, '%')
"""

file10_df = preflight_client.query(
    file10_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "file_token", "STRING", FILE_TOKEN
            ),
        ]
    ),
).to_dataframe()

if len(file10_df) != 1:
    raise RuntimeError(f"Expected one FILE10 record; found {len(file10_df)}.")

file10 = file10_df.iloc[0]
print(file10)

if file10["STATUS"] != "FAILED_STAGE":
    raise RuntimeError(
        f"FILE10 status is {file10['STATUS']}, not FAILED_STAGE. No reset made."
    )

if pd.notna(file10["LOADED_ROWS"]) and file10["LOADED_ROWS"] != 0:
    raise RuntimeError("FILE10 has loaded rows. No reset made.")

try:
    preflight_client.get_table(file10["STAGE_TABLE"])
    raise RuntimeError(
        "FILE10 staging table exists. No reset made."
    )
except NotFound:
    pass

reset_sql = f"""
UPDATE `{MANIFEST_TABLE}`
SET
  STATUS = 'DISCOVERED',
  LOADED_ROWS = NULL,
  SOURCE_ROWS = NULL,
  CHUNK_COUNT = NULL,
  STARTED_AT = NULL,
  COMPLETED_AT = NULL,
  ERROR_MESSAGE = NULL
WHERE RUN_ID = @run_id
  AND SOURCE_FILE_NAME = @source_file
"""

preflight_client.query(
    reset_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID),
            bigquery.ScalarQueryParameter(
                "source_file", "STRING", file10["SOURCE_FILE_NAME"]
            ),
        ]
    ),
).result()

print("\n✅ FILE10 reset safely. Starting retry now...\n")
run_bidperiod_v2_file(FILE_TOKEN)

In [ ]:
# =============================================================================
# SECTION 25 — FINAL AUGUST 2024 V2 AUDIT
# Read-only: validates manifest, V2 row totals, and stage cleanup.
# =============================================================================

audit_sql = f"""
SELECT
  STATUS,
  COUNT(*) AS files,
  SUM(LOADED_ROWS) AS manifest_rows
FROM `{MANIFEST_TABLE}`
WHERE RUN_ID = 'bidperiod-v2-2024-08'
GROUP BY STATUS
ORDER BY STATUS
"""

audit_df = preflight_client.query(audit_sql).to_dataframe()
display(audit_df)

v2_table = preflight_client.get_table(V2_TABLE)
v2_rows = v2_table.num_rows

promoted_rows = int(
    audit_df.loc[
        audit_df["STATUS"] == "PROMOTED_CLEANED",
        "manifest_rows"
    ].iloc[0]
)

remaining_stages = [
    table.table_id
    for table in preflight_client.list_tables(DATASET_ID)
    if table.table_id.startswith("BIDPEROFFER_STAGE_202408_")
]

print("V2 metadata rows:", f"{v2_rows:,}")
print("Manifest promoted rows:", f"{promoted_rows:,}")
print("Remaining full staging tables:", len(remaining_stages))

if v2_rows == promoted_rows and len(remaining_stages) == 0:
    print("\n✅ AUGUST 2024 V2 AUDIT PASSED")
else:
    print("\n⚠️ Audit needs review before continuing.")

In [ ]:
# =============================================================================
# SECTION 26A — BIDOFFERPERIOD AUGUST 2024 SAMPLE PROFILE
# Read-only sample inspection of BIDPEROFFER_ARCHIVE_V2.
# =============================================================================

SAMPLE_PERCENT = 0.1

sample_from = (
    f"`{V2_TABLE}` "
    f"TABLESAMPLE SYSTEM ({SAMPLE_PERCENT} PERCENT)"
)

profile_sql = f"""
SELECT
  COUNT(*) AS sample_rows,
  MIN(SETTLEMENTDATE) AS first_settlementdate,
  MAX(SETTLEMENTDATE) AS last_settlementdate,
  COUNT(DISTINCT DUID) AS distinct_duids,
  COUNT(DISTINCT BIDTYPE) AS distinct_bidtypes,
  COUNT(DISTINCT DIRECTION) AS distinct_directions,
  COUNTIF(BANDAVAIL1 IS NULL) AS bandavail1_nulls,
  COUNTIF(BANDAVAIL10 IS NULL) AS bandavail10_nulls,
  MIN(LOWBREAKPOINT) AS lowbreakpoint_min,
  MAX(HIGHBREAKPOINT) AS highbreakpoint_max
FROM {sample_from}
WHERE _RUN_ID = 'bidperiod-v2-2024-08'
"""

mix_sql = f"""
SELECT
  BIDTYPE,
  DIRECTION,
  COUNT(*) AS sample_rows,
  AVG(MAXAVAIL) AS avg_maxavail,
  AVG(BANDAVAIL1) AS avg_bandavail1,
  AVG(BANDAVAIL10) AS avg_bandavail10
FROM {sample_from}
WHERE _RUN_ID = 'bidperiod-v2-2024-08'
GROUP BY BIDTYPE, DIRECTION
ORDER BY sample_rows DESC
"""

records_sql = f"""
SELECT
  DUID,
  BIDTYPE,
  SETTLEMENTDATE,
  OFFERDATE,
  DIRECTION,
  PERIODID,
  MAXAVAIL,
  LOWBREAKPOINT,
  HIGHBREAKPOINT,
  BANDAVAIL1,
  BANDAVAIL10
FROM {sample_from}
WHERE _RUN_ID = 'bidperiod-v2-2024-08'
LIMIT 20
"""

print("SAMPLE PROFILE:")
display(preflight_client.query(profile_sql).to_dataframe())

print("\nBID TYPE / DIRECTION MIX:")
display(preflight_client.query(mix_sql).to_dataframe())

print("\nSAMPLE RECORDS:")
display(preflight_client.query(records_sql).to_dataframe())

In [ ]:
# =============================================================================
# SECTION 26B — INSPECT BIDPEROFFER_D AND BIDDAYOFFER_D SCHEMAS
# Read-only: downloads source ZIPs for header inspection only.
# =============================================================================

import io
import csv
import zipfile
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin

YEAR = 2024
MONTH = 8

archive_url = (
    f"https://www.nemweb.com.au/"
    f"Data_Archive/Wholesale_Electricity/MMSDM/{YEAR}/"
    f"MMSDM_{YEAR}_{MONTH:02d}/"
    f"MMSDM_Historical_Data_SQLLoader/DATA/"
)

targets = ["BIDPEROFFER_D", "BIDDAYOFFER_D"]

response = requests.get(archive_url, timeout=120)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

found = {target: [] for target in targets}

for link in soup.find_all("a"):
    href = link.get("href")
    if not href or not href.upper().endswith(".ZIP"):
        continue

    href_upper = href.upper()

    for target in targets:
        if target in href_upper:
            found[target].append(urljoin(archive_url, href))

for target in targets:
    found[target] = sorted(set(found[target]))
    print(f"\n{target}: {len(found[target])} ZIP file(s)")
    for url in found[target][:5]:
        print(" ", url)

def inspect_header(target, source_url):
    print("\n" + "=" * 90)
    print(f"INSPECTING {target}")
    print("=" * 90)

    response = requests.get(source_url, timeout=300)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
            and target in name.upper()
        )

        print("ZIP:", source_url.split("/")[-1])
        print("CSV:", csv_name)

        with zip_file.open(csv_name) as csv_file:
            wrapper = io.TextIOWrapper(
                csv_file, encoding="utf-8", newline=""
            )

            columns = None
            for row in csv.reader(wrapper):
                if (
                    len(row) >= 5
                    and row[0].strip().upper() == "I"
                    and row[1].strip().upper() == "BIDS"
                    and row[2].strip().upper() == target
                ):
                    columns = [column.strip() for column in row[4:]]
                    break

            wrapper.detach()

    if columns is None:
        print("⚠️ Column-definition record not found.")
    else:
        print("Columns:", len(columns))
        print(columns)

for target in targets:
    if found[target]:
        inspect_header(target, found[target][0])
    else:
        print(f"\n⚠️ No {target} archive ZIP found for {YEAR}-{MONTH:02d}.")

In [ ]:
# =============================================================================
# SECTION 26C — RAW HEADER PREVIEW FOR _D BID TABLES
# Read-only: prints the first 25 CSV records from each source.
# =============================================================================

def preview_raw_rows(target, source_url, row_limit=25):
    print("\n" + "=" * 90)
    print(f"RAW HEADER PREVIEW: {target}")
    print("=" * 90)

    response = requests.get(source_url, timeout=300)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
            and target in name.upper()
        )

        with zip_file.open(csv_name) as csv_file:
            wrapper = io.TextIOWrapper(
                csv_file, encoding="utf-8", newline=""
            )

            for row_number, row in enumerate(csv.reader(wrapper), start=1):
                print(f"{row_number:02d}: {row}")
                if row_number >= row_limit:
                    break

            wrapper.detach()

for target in targets:
    if found[target]:
        preview_raw_rows(target, found[target][0])

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project="master-aemo")

print("BigQuery client ready for project:", bq_client.project)

In [ ]:
# SECTION 26D — Inspect the existing bid-price archive (READ ONLY)

BIDDAY_TABLE_ID = (
    "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"
)

try:
    bidday_table = bq_client.get_table(BIDDAY_TABLE_ID)

    print("TABLE:", BIDDAY_TABLE_ID)
    print("Rows:", f"{bidday_table.num_rows:,}")
    print("Size (GB):", round(bidday_table.num_bytes / 1024**3, 2))
    print("\nSCHEMA:")

    for field in bidday_table.schema:
        print(f"{field.name:<30} {field.field_type:<12} {field.mode}")

except Exception as exc:
    print("Could not inspect BIDDAYOFFER_ARCHIVE:")
    print(type(exc).__name__, str(exc))

In [ ]:
# SECTION 26E — BIDDAYOFFER_ARCHIVE metadata audit (READ ONLY)

bidday_table = bq_client.get_table(
    "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"
)

print("Rows:", f"{bidday_table.num_rows:,}")
print("Size (GB):", round(bidday_table.num_bytes / 1024**3, 2))
print("Created:", bidday_table.created)
print("Last modified:", bidday_table.modified)
print("Partitioning:", bidday_table.time_partitioning)
print("Clustering:", bidday_table.clustering_fields)

if bidday_table.time_partitioning:
    print("\nThis table is partitioned, so we can next audit its monthly coverage cheaply.")
else:
    print("\nThis table is not partitioned. We will use a carefully scoped coverage check next.")

In [ ]:
# SECTION 26F — Preview cost of the BIDDAYOFFER coverage audit (READ ONLY)

coverage_sql = """
SELECT
  DATE_TRUNC(DATE(SETTLEMENTDATE), MONTH) AS settlement_month,
  COUNT(*) AS rows_in_month
FROM `master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE`
GROUP BY settlement_month
ORDER BY settlement_month
"""

dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run = bq_client.query(coverage_sql, job_config=dry_run_config)

estimated_gb = dry_run.total_bytes_processed / 1024**3

print("Estimated data scanned:", round(estimated_gb, 3), "GB")
print("This is a preview only; no query was executed.")

In [ ]:
# SECTION 26G — BIDDAYOFFER_ARCHIVE monthly coverage audit (READ ONLY)

coverage_job = bq_client.query(coverage_sql)
bidday_coverage = coverage_job.to_dataframe()

print("Months found:", len(bidday_coverage))
print("First month:", bidday_coverage["settlement_month"].min())
print("Last month:", bidday_coverage["settlement_month"].max())
print("Total rows:", f"{bidday_coverage['rows_in_month'].sum():,}")

display(bidday_coverage)

In [ ]:
# SECTION 26H — Find the missing price-data month (no BigQuery query)

import pandas as pd

actual_months = pd.to_datetime(bidday_coverage["settlement_month"]).dt.to_period("M")
expected_months = pd.period_range(
    actual_months.min(),
    actual_months.max(),
    freq="M"
)

missing_months = expected_months.difference(actual_months)

print("Missing calendar month(s):")
for month in missing_months:
    print("-", month)

print("\nTen smallest monthly row counts:")
display(
    bidday_coverage
    .sort_values("rows_in_month")
    .head(10)
)

In [ ]:
# SECTION 26I — Examine the two suspect periods (no BigQuery query)

bidday_coverage["settlement_month"] = pd.to_datetime(
    bidday_coverage["settlement_month"]
)

suspect_windows = bidday_coverage[
    (
        (bidday_coverage["settlement_month"] >= "2016-07-01")
        & (bidday_coverage["settlement_month"] <= "2016-11-01")
    )
    |
    (
        (bidday_coverage["settlement_month"] >= "2023-02-01")
        & (bidday_coverage["settlement_month"] <= "2023-06-01")
    )
].copy()

display(suspect_windows.sort_values("settlement_month"))

In [ ]:
# SECTION 26J — Diagnose September 2016 daily coverage (READ ONLY)

september_2016_sql = """
SELECT
  DATE(SETTLEMENTDATE) AS settlement_day,
  COUNT(*) AS rows_on_day,
  COUNT(DISTINCT DUID) AS duids_on_day
FROM `master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE`
WHERE SETTLEMENTDATE >= TIMESTAMP("2016-09-01")
  AND SETTLEMENTDATE <  TIMESTAMP("2016-10-01")
GROUP BY settlement_day
ORDER BY settlement_day
"""

safe_query_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=5 * 1024**3
)

september_2016_daily = (
    bq_client.query(
        september_2016_sql,
        job_config=safe_query_config
    )
    .to_dataframe()
)

print("Days present:", len(september_2016_daily))
print("Rows present:", f"{september_2016_daily['rows_on_day'].sum():,}")
display(september_2016_daily)

In [ ]:
# SECTION 26K — Find source ZIPs for the two BIDDAYOFFER repairs (READ ONLY)

from bs4 import BeautifulSoup
from urllib.parse import urljoin
import requests

def list_biddayoffer_archives(year, month):
    archive_page = (
        f"https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    response = requests.get(archive_page, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    urls = sorted({
        urljoin(archive_page, link.get("href"))
        for link in soup.find_all("a", href=True)
        if "BIDDAYOFFER" in link.get("href").upper()
        and link.get("href").lower().endswith(".zip")
    })

    print(f"\n{year}-{month:02d}: {len(urls)} BIDDAYOFFER-related ZIP file(s)")
    for url in urls:
        print(url)

    return urls

bidday_2016_09_urls = list_biddayoffer_archives(2016, 9)
bidday_2023_04_urls = list_biddayoffer_archives(2023, 4)

In [ ]:
# SECTION 26L — Inspect repair-source layouts (READ ONLY)

import csv
import io
import zipfile

repair_sources = {
    "2016-09 standard BIDDAYOFFER":
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        "MMSDM/2016/MMSDM_2016_09/MMSDM_Historical_Data_SQLLoader/DATA/"
        "PUBLIC_DVD_BIDDAYOFFER_201609010000.zip",

    "2023-04 standard BIDDAYOFFER":
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        "MMSDM/2023/MMSDM_2023_04/MMSDM_Historical_Data_SQLLoader/DATA/"
        "PUBLIC_DVD_BIDDAYOFFER_202304010000.zip",
}

for label, source_url in repair_sources.items():
    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    response = requests.get(source_url, timeout=300)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as raw_file:
            text_file = io.TextIOWrapper(raw_file, encoding="utf-8-sig")
            reader = csv.reader(text_file)

            for record_number in range(6):
                row = next(reader)
                print(f"Record {record_number + 1}:", row)

In [ ]:
# SECTION 26M — Count the exact repair records (READ ONLY)

from datetime import datetime

repair_plan = [
    {
        "label": "2016-09 missing days only",
        "url": repair_sources["2016-09 standard BIDDAYOFFER"],
        "start": pd.Timestamp("2016-09-08"),
        "end": pd.Timestamp("2016-10-01"),
    },
    {
        "label": "2023-04 full missing month",
        "url": repair_sources["2023-04 standard BIDDAYOFFER"],
        "start": pd.Timestamp("2023-04-01"),
        "end": pd.Timestamp("2023-05-01"),
    },
]

def inspect_biddayoffer_repair_source(item):
    response = requests.get(item["url"], timeout=300)
    response.raise_for_status()

    total_data_rows = 0
    repair_rows = 0
    first_repair_date = None
    last_repair_date = None
    source_columns = None

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as raw_file:
            reader = csv.reader(
                io.TextIOWrapper(raw_file, encoding="utf-8-sig")
            )

            for row in reader:
                if len(row) < 5:
                    continue

                # Column-definition record
                if row[0] == "I" and row[2] == "BIDDAYOFFER":
                    source_columns = row[4:]
                    continue

                # Data record
                if row[0] != "D" or row[2] != "BIDDAYOFFER":
                    continue

                total_data_rows += 1

                if source_columns is None:
                    raise RuntimeError("Column-definition record was not found.")

                values = dict(zip(source_columns, row[4:]))
                settlement_date = pd.to_datetime(
                    values["SETTLEMENTDATE"],
                    format="%Y/%m/%d %H:%M:%S"
                )

                if item["start"] <= settlement_date < item["end"]:
                    repair_rows += 1
                    first_repair_date = (
                        settlement_date
                        if first_repair_date is None
                        else min(first_repair_date, settlement_date)
                    )
                    last_repair_date = (
                        settlement_date
                        if last_repair_date is None
                        else max(last_repair_date, settlement_date)
                    )

    print("\n" + item["label"])
    print("Source CSV:", csv_name)
    print("All source records:", f"{total_data_rows:,}")
    print("Records in repair scope:", f"{repair_rows:,}")
    print("First repair date:", first_repair_date)
    print("Last repair date:", last_repair_date)
    print("Source columns:", len(source_columns))

for item in repair_plan:
    inspect_biddayoffer_repair_source(item)

In [ ]:
# SECTION 26N — Stage validated BIDDAYOFFER repairs only
# No production-table rows are changed by this cell.

import csv
import io
import zipfile
import requests
import pandas as pd
from google.cloud import bigquery

TARGET_TABLE = "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"

repair_plan = [
    {
        "label": "2016-09 missing days",
        "source_url": repair_sources["2016-09 standard BIDDAYOFFER"],
        "stage_table": "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_201609",
        "start": pd.Timestamp("2016-09-08"),
        "end": pd.Timestamp("2016-10-01"),
        "expected_existing_rows": 0,
        "expected_stage_rows": 119_521,
        "pipeline_label": "BIDDAYOFFER_TARGETED_REPAIR_2016_09",
    },
    {
        "label": "2023-04 full month",
        "source_url": repair_sources["2023-04 standard BIDDAYOFFER"],
        "stage_table": "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_202304",
        "start": pd.Timestamp("2023-04-01"),
        "end": pd.Timestamp("2023-05-01"),
        "expected_existing_rows": 0,
        "expected_stage_rows": 3_019_285,
        "pipeline_label": "BIDDAYOFFER_TARGETED_REPAIR_2023_04",
    },
]

target_table = bq_client.get_table(TARGET_TABLE)
target_columns = [field.name for field in target_table.schema]
target_types = {field.name: field.field_type for field in target_table.schema}

# Safety check: both repair scopes must still be empty.
for item in repair_plan:
    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM `{TARGET_TABLE}`
    WHERE SETTLEMENTDATE >= TIMESTAMP("{item["start"].strftime("%Y-%m-%d")}")
      AND SETTLEMENTDATE <  TIMESTAMP("{item["end"].strftime("%Y-%m-%d")}")
    """

    existing_rows = int(
        bq_client.query(
            count_sql,
            job_config=bigquery.QueryJobConfig(
                maximum_bytes_billed=5 * 1024**3
            )
        ).to_dataframe().iloc[0]["row_count"]
    )

    print(item["label"], "| existing production rows:", f"{existing_rows:,}")

    if existing_rows != item["expected_existing_rows"]:
        raise RuntimeError(
            f"Safety stop for {item['label']}: expected "
            f"{item['expected_existing_rows']:,} production rows, found "
            f"{existing_rows:,}. Nothing has been staged."
        )

def prepare_biddayoffer_chunk(rows, source_columns, item):
    frame = pd.DataFrame(rows, columns=source_columns)

    frame["SETTLEMENTDATE"] = pd.to_datetime(
        frame["SETTLEMENTDATE"],
        format="%Y/%m/%d %H:%M:%S",
        errors="coerce",
    )

    frame = frame[
        (frame["SETTLEMENTDATE"] >= item["start"])
        & (frame["SETTLEMENTDATE"] < item["end"])
    ].copy()

    if frame.empty:
        return frame

    for column in target_columns:
        if column not in frame.columns:
            frame[column] = pd.NA

    frame["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
    frame["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    frame["_PIPELINE"] = item["pipeline_label"]

    for column, field_type in target_types.items():
        if field_type == "FLOAT":
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        elif field_type in ("TIMESTAMP", "DATETIME"):
            frame[column] = pd.to_datetime(frame[column], errors="coerce")
        elif field_type == "STRING":
            frame[column] = frame[column].replace("", pd.NA).astype("string")

    return frame[target_columns]

def stage_repair(item, chunk_size=100_000):
    print("\n" + "=" * 100)
    print("STAGING:", item["label"])

    response = requests.get(item["source_url"], timeout=300)
    response.raise_for_status()

    loaded_rows = 0
    chunk_number = 0
    source_columns = None
    buffered_rows = []

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as raw_file:
            reader = csv.reader(
                io.TextIOWrapper(raw_file, encoding="utf-8-sig")
            )

            for row in reader:
                if len(row) < 5:
                    continue

                if row[0] == "I" and row[2] == "BIDDAYOFFER":
                    source_columns = row[4:]
                    continue

                if row[0] != "D" or row[2] != "BIDDAYOFFER":
                    continue

                buffered_rows.append(row[4:])

                if len(buffered_rows) >= chunk_size:
                    chunk = prepare_biddayoffer_chunk(
                        buffered_rows, source_columns, item
                    )
                    buffered_rows = []

                    if not chunk.empty:
                        disposition = (
                            bigquery.WriteDisposition.WRITE_TRUNCATE
                            if chunk_number == 0
                            else bigquery.WriteDisposition.WRITE_APPEND
                        )

                        job = bq_client.load_table_from_dataframe(
                            chunk,
                            item["stage_table"],
                            job_config=bigquery.LoadJobConfig(
                                schema=target_table.schema,
                                write_disposition=disposition,
                            ),
                        )
                        job.result()

                        loaded_rows += len(chunk)
                        chunk_number += 1

                        print(
                            f"{item['label']}: {chunk_number} chunks | "
                            f"{loaded_rows:,} rows staged"
                        )

            if buffered_rows:
                chunk = prepare_biddayoffer_chunk(
                    buffered_rows, source_columns, item
                )

                if not chunk.empty:
                    disposition = (
                        bigquery.WriteDisposition.WRITE_TRUNCATE
                        if chunk_number == 0
                        else bigquery.WriteDisposition.WRITE_APPEND
                    )

                    job = bq_client.load_table_from_dataframe(
                        chunk,
                        item["stage_table"],
                        job_config=bigquery.LoadJobConfig(
                            schema=target_table.schema,
                            write_disposition=disposition,
                        ),
                    )
                    job.result()

                    loaded_rows += len(chunk)

    stage_rows = bq_client.get_table(item["stage_table"]).num_rows

    print("\nStage table:", item["stage_table"])
    print("Rows staged:", f"{stage_rows:,}")

    if stage_rows != item["expected_stage_rows"]:
        raise RuntimeError(
            f"Stage verification failed for {item['label']}: expected "
            f"{item['expected_stage_rows']:,}, found {stage_rows:,}."
        )

    print("✅ Stage verification passed:", item["label"])

for item in repair_plan:
    stage_repair(item)

In [ ]:
# SECTION 26O — Promote verified BIDDAYOFFER repairs
# Production change: append exactly 119,521 + 3,019,285 verified rows.
# Stage tables are retained until the final audit passes.

TARGET_TABLE = "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"

promotion_plan = [
    {
        "label": "2016-09 missing days",
        "stage_table": "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_201609",
        "start": "2016-09-08",
        "end": "2016-10-01",
        "expected_rows": 119_521,
    },
    {
        "label": "2023-04 full month",
        "stage_table": "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_202304",
        "start": "2023-04-01",
        "end": "2023-05-01",
        "expected_rows": 3_019_285,
    },
]

target_schema = bq_client.get_table(TARGET_TABLE).schema
column_list = ", ".join(f"`{field.name}`" for field in target_schema)

def count_scope(table_id, start, end):
    sql = f"""
    SELECT COUNT(*) AS row_count
    FROM `{table_id}`
    WHERE SETTLEMENTDATE >= TIMESTAMP("{start}")
      AND SETTLEMENTDATE <  TIMESTAMP("{end}")
    """

    return int(
        bq_client.query(
            sql,
            job_config=bigquery.QueryJobConfig(
                maximum_bytes_billed=5 * 1024**3
            ),
        ).to_dataframe().iloc[0]["row_count"]
    )

for item in promotion_plan:
    print("\n" + "=" * 100)
    print("PROMOTING:", item["label"])

    stage_rows = bq_client.get_table(item["stage_table"]).num_rows
    target_rows_before = count_scope(
        TARGET_TABLE, item["start"], item["end"]
    )

    print("Stage rows:", f"{stage_rows:,}")
    print("Production rows before:", f"{target_rows_before:,}")

    if stage_rows != item["expected_rows"]:
        raise RuntimeError(
            f"Safety stop: {item['stage_table']} has {stage_rows:,} rows; "
            f"expected {item['expected_rows']:,}."
        )

    # Makes a rerun safe if this repair was already promoted successfully.
    if target_rows_before == item["expected_rows"]:
        print("Already promoted previously; skipping duplicate insert.")
        continue

    if target_rows_before != 0:
        raise RuntimeError(
            f"Safety stop: production already has {target_rows_before:,} "
            f"rows in this repair scope."
        )

    promotion_sql = f"""
    INSERT INTO `{TARGET_TABLE}` ({column_list})
    SELECT {column_list}
    FROM `{item["stage_table"]}`
    """

    bq_client.query(promotion_sql).result()

    target_rows_after = count_scope(
        TARGET_TABLE, item["start"], item["end"]
    )

    print("Production rows after:", f"{target_rows_after:,}")

    if target_rows_after != item["expected_rows"]:
        raise RuntimeError(
            f"Promotion verification failed for {item['label']}: "
            f"expected {item['expected_rows']:,}, found "
            f"{target_rows_after:,}."
        )

    print("✅ PROMOTED AND VERIFIED:", item["label"])

print("\nBoth stage tables remain in place for the final audit.")

In [ ]:
# SECTION 26P — Final BIDDAYOFFER repair audit (READ ONLY)

final_coverage_sql = """
SELECT
  DATE_TRUNC(DATE(SETTLEMENTDATE), MONTH) AS settlement_month,
  COUNT(*) AS rows_in_month
FROM `master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE`
GROUP BY settlement_month
ORDER BY settlement_month
"""

final_coverage = (
    bq_client.query(
        final_coverage_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=5 * 1024**3
        ),
    )
    .to_dataframe()
)

final_coverage["settlement_month"] = pd.to_datetime(
    final_coverage["settlement_month"]
)

actual_months = final_coverage["settlement_month"].dt.to_period("M")
expected_months = pd.period_range(
    actual_months.min(),
    actual_months.max(),
    freq="M",
)
missing_months = expected_months.difference(actual_months)

sep_2016_rows = int(
    final_coverage.loc[
        final_coverage["settlement_month"] == pd.Timestamp("2016-09-01"),
        "rows_in_month"
    ].iloc[0]
)

apr_2023_rows = int(
    final_coverage.loc[
        final_coverage["settlement_month"] == pd.Timestamp("2023-04-01"),
        "rows_in_month"
    ].iloc[0]
)

print("Months found:", len(final_coverage))
print("First month:", final_coverage["settlement_month"].min())
print("Last month:", final_coverage["settlement_month"].max())
print("Missing months:", list(missing_months))
print("September 2016 rows:", f"{sep_2016_rows:,}")
print("April 2023 rows:", f"{apr_2023_rows:,}")
print("Total rows:", f"{final_coverage['rows_in_month'].sum():,}")

assert len(final_coverage) == 139
assert len(missing_months) == 0
assert sep_2016_rows == 154_039
assert apr_2023_rows == 3_019_285
assert final_coverage["rows_in_month"].sum() == 289_615_207

print("\n✅ BIDDAYOFFER ARCHIVE REPAIR AUDIT PASSED")

In [ ]:
# SECTION 26Q — Delete verified temporary repair stages only

stage_tables_to_delete = [
    "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_201609",
    "master-aemo.aemo_master_schema.BIDDAYOFFER_REPAIR_STAGE_202304",
]

for stage_table_id in stage_tables_to_delete:
    stage_table = bq_client.get_table(stage_table_id)
    print("Deleting:", stage_table_id, "| Rows:", f"{stage_table.num_rows:,}")

    bq_client.delete_table(stage_table_id, not_found_ok=False)

print("\n✅ Both temporary repair stages were deleted.")
print("✅ BIDDAYOFFER_ARCHIVE remains intact with 289,615,207 verified rows.")

In [ ]:
# SECTION 26R — Audit direction coverage in the bid-price archive (READ ONLY)

direction_sql = """
SELECT
  COALESCE(NULLIF(TRIM(DIRECTION), ""), "<NULL>") AS direction,
  COUNT(*) AS row_count,
  MIN(DATE(SETTLEMENTDATE)) AS first_date,
  MAX(DATE(SETTLEMENTDATE)) AS last_date
FROM `master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE`
GROUP BY direction
ORDER BY row_count DESC
"""

bidday_direction_audit = (
    bq_client.query(
        direction_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=5 * 1024**3
        ),
    )
    .to_dataframe()
)

print("Direction values found:")
display(bidday_direction_audit)

In [ ]:
# SECTION 27A — Map legacy BIDOFFERPERIOD coverage (READ ONLY)

legacy_period_sql = """
SELECT
  DATE_TRUNC(DATE(SETTLEMENTDATE), MONTH) AS settlement_month,
  COUNT(*) AS row_count
FROM `master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE`
GROUP BY settlement_month
ORDER BY settlement_month
"""

legacy_period_coverage = (
    bq_client.query(
        legacy_period_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=5 * 1024**3
        ),
    )
    .to_dataframe()
)

legacy_period_coverage["settlement_month"] = pd.to_datetime(
    legacy_period_coverage["settlement_month"]
)

print("Months found:", len(legacy_period_coverage))
print("First month:", legacy_period_coverage["settlement_month"].min())
print("Last month:", legacy_period_coverage["settlement_month"].max())
print("Total rows:", f"{legacy_period_coverage['row_count'].sum():,}")

display(legacy_period_coverage)

In [ ]:
# SECTION 27B — Find existing BESS / asset-master tables (READ ONLY)

dataset_ref = bigquery.DatasetReference(
    "master-aemo",
    "aemo_master_schema"
)

table_inventory = []

for table_item in bq_client.list_tables(dataset_ref):
    table_name = table_item.table_id
    upper_name = table_name.upper()

    if any(term in upper_name for term in [
        "BESS",
        "BATTERY",
        "ASSET",
        "GENERATOR",
        "DUID",
        "UNIT",
    ]):
        table_inventory.append(table_name)

print("Possible BESS / asset-master tables:")
for table_name in sorted(table_inventory):
    print("-", table_name)

if not table_inventory:
    print("No matching table names found.")

In [ ]:
# SECTION 27C — Inspect AEMO's current NEM Generation Information workbook
# No BigQuery data is changed.

GEN_INFO_URL = (
    "https://www.aemo.com.au/-/media/files/electricity/nem/"
    "planning_and_forecasting/generation_information/2026/"
    "nem-generation-information-july-2026.xlsx"
    "?rev=3455851f2bc945b7ab61c5ceed272992&sc_lang=en"
)

generation_session = requests.Session()

response = generation_session.get(
    GEN_INFO_URL,
    timeout=120,
    headers={
        "User-Agent": "Mozilla/5.0",
        "Referer": (
            "https://www.aemo.com.au/energy-systems/electricity/"
            "national-electricity-market-nem/nem-forecasting-and-planning/"
            "forecasting-and-planning-data/generation-information"
        ),
    },
)
response.raise_for_status()

print(
    "Downloaded workbook:",
    round(len(response.content) / 1024, 1),
    "KB"
)

generation_workbook = pd.ExcelFile(io.BytesIO(response.content))

print("\nWorkbook sheets:")
for sheet_name in generation_workbook.sheet_names:
    sample = pd.read_excel(
        generation_workbook,
        sheet_name=sheet_name,
        nrows=3,
    )

    print(f"\n- {sheet_name}")
    print("  Columns:", list(sample.columns))
    display(sample)

In [ ]:
# SECTION 27D — Identify AEMO battery candidates (READ ONLY)

generator_info = pd.read_excel(
    generation_workbook,
    sheet_name="Generator Information",
    header=3,
)

generator_info.columns = (
    generator_info.columns
    .astype(str)
    .str.strip()
)

print("Key columns found:")
print([
    column for column in generator_info.columns
    if any(word in column.lower() for word in [
        "technology", "duid", "commitment", "dispatch", "storage"
    ])
])

battery_candidates = generator_info[
    generator_info["Technology Type"]
    .astype("string")
    .str.contains("battery", case=False, na=False)
].copy()

print("\nBattery candidate rows:", len(battery_candidates))
print("\nBattery candidates by commitment status:")

display(
    battery_candidates
    .groupby("Commitment Status", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

bess_candidate_columns = [
    "Site Name",
    "Unit Name",
    "DUID",
    "Region",
    "Technology Type",
    "Technology Detail",
    "Dispatch Type",
    "Commitment Status",
    "Full Commercial Use Date",
    "Aggregated Nameplate Capacity (MW AC)",
    "Agg Nameplate Storage Capacity (MWh)",
]

display(
    battery_candidates[
        [column for column in bess_candidate_columns
         if column in battery_candidates.columns]
    ]
    .sort_values(["Commitment Status", "Site Name"])
)

In [ ]:
# SECTION 27E — Review operational BESS DUID candidates (READ ONLY)

backfill_statuses = ["In Service", "In Commissioning"]

operational_bess = battery_candidates[
    battery_candidates["Commitment Status"].isin(backfill_statuses)
].copy()

operational_bess["DUID"] = (
    operational_bess["DUID"]
    .astype("string")
    .str.strip()
)

print("Operational / commissioning BESS rows:", len(operational_bess))
print("Missing DUID values:", operational_bess["DUID"].isna().sum())

print("\nDUID values containing separators or multiple values:")
display(
    operational_bess[
        operational_bess["DUID"].str.contains(r"[,;/]", na=False)
    ][["Site Name", "Unit Name", "DUID", "Commitment Status"]]
)

registry_preview_columns = [
    "Site Name",
    "Unit Name",
    "DUID",
    "Region",
    "Dispatch Type",
    "Commitment Status",
    "Full Commercial Use Date",
    "Aggregated Nameplate Capacity (MW AC)",
    "Agg Nameplate Storage Capacity (MWh)",
]

display(
    operational_bess[
        [column for column in registry_preview_columns
         if column in operational_bess.columns]
    ]
    .sort_values(["Full Commercial Use Date", "Site Name"])
)

In [ ]:
# SECTION 27E — Review operational BESS DUID candidates (READ ONLY)

backfill_statuses = ["In Service", "In Commissioning"]

operational_bess = battery_candidates[
    battery_candidates["Commitment Status"].isin(backfill_statuses)
].copy()

operational_bess["DUID"] = (
    operational_bess["DUID"]
    .astype("string")
    .str.strip()
)

print("Operational / commissioning BESS rows:", len(operational_bess))
print("Missing DUID values:", operational_bess["DUID"].isna().sum())

print("\nDUID values containing separators or multiple values:")
display(
    operational_bess[
        operational_bess["DUID"].str.contains(r"[,;/]", na=False)
    ][["Site Name", "Unit Name", "DUID", "Commitment Status"]]
)

registry_preview_columns = [
    "Site Name",
    "Unit Name",
    "DUID",
    "Region",
    "Dispatch Type",
    "Commitment Status",
    "Full Commercial Use Date",
    "Aggregated Nameplate Capacity (MW AC)",
    "Agg Nameplate Storage Capacity (MWh)",
]

display(
    operational_bess[
        [column for column in registry_preview_columns
         if column in operational_bess.columns]
    ]
    .sort_values(["Full Commercial Use Date", "Site Name"])
)

In [ ]:
# SECTION 27F — Build a unique BESS DUID backfill list locally
# No BigQuery data is changed.

bess_registry_local = operational_bess.copy()

bess_registry_local["DUID"] = (
    bess_registry_local["DUID"]
    .astype("string")
    .str.strip()
    .str.upper()
)

duplicate_duids = (
    bess_registry_local[
        bess_registry_local.duplicated("DUID", keep=False)
    ]
    .sort_values(["DUID", "Site Name"])
)

print("Source candidate rows:", len(bess_registry_local))
print("Unique BESS DUIDs:", bess_registry_local["DUID"].nunique())
print("Duplicated DUID source rows:", len(duplicate_duids))

print("\nDuplicate DUIDs — retained once for backfill:")
display(
    duplicate_duids[
        ["DUID", "Site Name", "Unit Name", "Commitment Status",
         "Aggregated Nameplate Capacity (MW AC)",
         "Agg Nameplate Storage Capacity (MWh)"]
    ]
)

# Prefer an In Service row where the same DUID appears more than once.
status_rank = {"In Service": 1, "In Commissioning": 2}

bess_registry_unique = (
    bess_registry_local
    .assign(
        _status_rank=bess_registry_local["Commitment Status"]
        .map(status_rank)
        .fillna(99)
    )
    .sort_values(["DUID", "_status_rank"])
    .drop_duplicates("DUID", keep="first")
    .drop(columns="_status_rank")
    .sort_values("DUID")
    .reset_index(drop=True)
)

print("\nFinal BESS DUIDs for backfill:")
display(
    bess_registry_unique[
        ["DUID", "Site Name", "Region", "Dispatch Type",
         "Commitment Status", "Full Commercial Use Date"]
    ]
)

In [ ]:
# SECTION 27G — Create the BESS DUID backfill scope registry
# Creates/refreshes one small 55-row registry table.
# Does NOT load any BIDPEROFFER data.

BESS_SCOPE_TABLE = (
    "master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1"
)

source_publication_date = pd.Timestamp("2026-07-31").date()

bess_scope = (
    bess_registry_local
    .groupby("DUID", as_index=False)
    .agg(
        SITE_NAME=("Site Name", "first"),
        REGION=("Region", "first"),
        DISPATCH_TYPE=("Dispatch Type", "first"),
        COMMITMENT_STATUS=("Commitment Status", "first"),
        FULL_COMMERCIAL_USE_DATE=("Full Commercial Use Date", "first"),
        SOURCE_ROW_COUNT=("DUID", "size"),
    )
    .sort_values("DUID")
    .reset_index(drop=True)
)

bess_scope["FULL_COMMERCIAL_USE_DATE"] = pd.to_datetime(
    bess_scope["FULL_COMMERCIAL_USE_DATE"],
    errors="coerce",
).dt.date

bess_scope["SOURCE_PUBLICATION_DATE"] = source_publication_date
bess_scope["SOURCE_URL"] = GEN_INFO_URL
bess_scope["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
bess_scope["_PIPELINE"] = "BESS_DUID_SCOPE_REGISTRY_V1"

scope_schema = [
    bigquery.SchemaField("DUID", "STRING"),
    bigquery.SchemaField("SITE_NAME", "STRING"),
    bigquery.SchemaField("REGION", "STRING"),
    bigquery.SchemaField("DISPATCH_TYPE", "STRING"),
    bigquery.SchemaField("COMMITMENT_STATUS", "STRING"),
    bigquery.SchemaField("FULL_COMMERCIAL_USE_DATE", "DATE"),
    bigquery.SchemaField("SOURCE_ROW_COUNT", "INTEGER"),
    bigquery.SchemaField("SOURCE_PUBLICATION_DATE", "DATE"),
    bigquery.SchemaField("SOURCE_URL", "STRING"),
    bigquery.SchemaField("_LOAD_DATE", "TIMESTAMP"),
    bigquery.SchemaField("_PIPELINE", "STRING"),
]

load_job = bq_client.load_table_from_dataframe(
    bess_scope,
    BESS_SCOPE_TABLE,
    job_config=bigquery.LoadJobConfig(
        schema=scope_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
)
load_job.result()

saved_scope = bq_client.get_table(BESS_SCOPE_TABLE)

print("✅ BESS backfill scope registry created")
print("Table:", BESS_SCOPE_TABLE)
print("Unique BESS DUIDs:", f"{saved_scope.num_rows:,}")

display(
    bess_scope[
        ["DUID", "SITE_NAME", "REGION", "COMMITMENT_STATUS",
         "SOURCE_ROW_COUNT"]
    ]
)

In [ ]:
# SECTION 27H — Preview BESS-only August V2 measurement cost (READ ONLY)

bess_august_sql = """
SELECT
  b.DUID,
  b.BIDTYPE,
  b.DIRECTION,
  COUNT(*) AS row_count,
  MIN(b.SETTLEMENTDATE) AS first_settlementdate,
  MAX(b.SETTLEMENTDATE) AS last_settlementdate
FROM `master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE_V2` AS b
INNER JOIN `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1` AS s
  ON b.DUID = s.DUID
WHERE b.SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
  AND b.SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
GROUP BY b.DUID, b.BIDTYPE, b.DIRECTION
ORDER BY b.DUID, b.BIDTYPE, b.DIRECTION
"""

dry_run = bq_client.query(
    bess_august_sql,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
    ),
)

print(
    "Estimated data scanned:",
    round(dry_run.total_bytes_processed / 1024**3, 3),
    "GB"
)
print("Preview only; no query was executed.")

In [ ]:
# SECTION 27I — Measure BESS-only August 2024 V2 coverage (READ ONLY)

bess_august_detail = (
    bq_client.query(
        bess_august_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=50 * 1024**3
        ),
    )
    .to_dataframe()
)

print("BESS DUIDs with August 2024 bid-period records:",
      bess_august_detail["DUID"].nunique())

print("BESS bid-period rows in August 2024:",
      f"{bess_august_detail['row_count'].sum():,}")

display(bess_august_detail)

In [ ]:
# SECTION 27J — Correlate BESS registry, DUDETAIL, and August 2024 bids
# READ ONLY

bess_dudetail_sql = """
WITH scope AS (
  SELECT
    DUID,
    SITE_NAME,
    REGION,
    DISPATCH_TYPE AS REGISTRY_DISPATCH_TYPE,
    COMMITMENT_STATUS,
    FULL_COMMERCIAL_USE_DATE
  FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
),

dudetail_as_of_august_2024 AS (
  SELECT
    d.DUID,
    d.EFFECTIVEDATE,
    d.VERSIONNO,
    d.CONNECTIONPOINTID,
    d.REGISTEREDCAPACITY,
    d.AGCCAPABILITY,
    d.DISPATCHTYPE AS DUDETAIL_DISPATCH_TYPE,
    d.MAXCAPACITY,
    d.STARTTYPE,
    d.NORMALLYONFLAG
  FROM `master-aemo.aemo_master_schema.DUDETAIL` AS d
  INNER JOIN scope AS s
    ON d.DUID = s.DUID
  WHERE CAST(d.EFFECTIVEDATE AS DATE) <= DATE "2024-08-31"
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY d.DUID
    ORDER BY
      CAST(d.EFFECTIVEDATE AS DATE) DESC,
      d.VERSIONNO DESC
  ) = 1
)

SELECT
  s.*,
  d.EFFECTIVEDATE AS DUDETAIL_EFFECTIVEDATE,
  d.VERSIONNO AS DUDETAIL_VERSIONNO,
  d.CONNECTIONPOINTID,
  d.REGISTEREDCAPACITY,
  d.AGCCAPABILITY,
  d.DUDETAIL_DISPATCH_TYPE,
  d.MAXCAPACITY,
  d.STARTTYPE,
  d.NORMALLYONFLAG,
  CASE
    WHEN d.DUID IS NULL THEN FALSE
    ELSE TRUE
  END AS FOUND_IN_DUDETAIL_BY_AUG_2024
FROM scope AS s
LEFT JOIN dudetail_as_of_august_2024 AS d
  ON s.DUID = d.DUID
ORDER BY s.DUID
"""

bess_dudetail_audit = (
    bq_client.query(
        bess_dudetail_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=5 * 1024**3
        ),
    )
    .to_dataframe()
)

# Uses the August result already held in the notebook.
august_active_duids = set(bess_august_detail["DUID"].unique())

bess_dudetail_audit["HAS_AUGUST_2024_BID_RECORD"] = (
    bess_dudetail_audit["DUID"].isin(august_active_duids)
)

print("BESS registry DUIDs:", len(bess_dudetail_audit))
print(
    "Found in DUDETAIL by August 2024:",
    int(bess_dudetail_audit["FOUND_IN_DUDETAIL_BY_AUG_2024"].sum())
)
print(
    "Observed in August 2024 bid-period data:",
    int(bess_dudetail_audit["HAS_AUGUST_2024_BID_RECORD"].sum())
)

display(
    bess_dudetail_audit[
        [
            "DUID",
            "SITE_NAME",
            "COMMITMENT_STATUS",
            "FOUND_IN_DUDETAIL_BY_AUG_2024",
            "HAS_AUGUST_2024_BID_RECORD",
            "DUDETAIL_EFFECTIVEDATE",
            "CONNECTIONPOINTID",
            "DUDETAIL_DISPATCH_TYPE",
            "REGISTEREDCAPACITY",
            "MAXCAPACITY",
        ]
    ]
)

In [ ]:
# SECTION 27K — Audit DUDETAIL coverage (READ ONLY)

dudetail_coverage_sql = """
SELECT
  DATE_TRUNC(CAST(EFFECTIVEDATE AS DATE), MONTH) AS effective_month,
  COUNT(*) AS row_count,
  COUNT(DISTINCT DUID) AS distinct_duids
FROM `master-aemo.aemo_master_schema.DUDETAIL`
GROUP BY effective_month
ORDER BY effective_month
"""

dudetail_coverage = (
    bq_client.query(
        dudetail_coverage_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=2 * 1024**3
        ),
    )
    .to_dataframe()
)

print("Months found:", len(dudetail_coverage))
print("First month:", dudetail_coverage["effective_month"].min())
print("Last month:", dudetail_coverage["effective_month"].max())
print("Total DUDETAIL rows:", f"{dudetail_coverage['row_count'].sum():,}")

display(dudetail_coverage)

In [ ]:
# SECTION 27L — Preview raw-versus-final BESS bid reduction
# READ ONLY: cost preview only

bess_final_offer_sample_sql = """
WITH balb1_august AS (
  SELECT
    BIDTYPE,
    DIRECTION,
    SETTLEMENTDATE,
    PERIODID
  FROM `master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE_V2`
  WHERE DUID = "BALB1"
    AND SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
    AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
)

SELECT
  COUNT(*) AS raw_rows,
  COUNT(
    DISTINCT TO_JSON_STRING(
      STRUCT(
        BIDTYPE,
        DIRECTION,
        SETTLEMENTDATE,
        PERIODID
      )
    )
  ) AS final_effective_keys
FROM balb1_august
"""

dry_run = bq_client.query(
    bess_final_offer_sample_sql,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
    ),
)

print(
    "Estimated data scanned:",
    round(dry_run.total_bytes_processed / 1024**3, 3),
    "GB"
)
print("Preview only; no query was executed.")

In [ ]:
# SECTION 27M — Measure raw-versus-final reduction for BALB1 (READ ONLY)

balb1_reduction = (
    bq_client.query(
        bess_final_offer_sample_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=3 * 1024**3
        ),
    )
    .to_dataframe()
)

raw_rows = int(balb1_reduction.iloc[0]["raw_rows"])
final_keys = int(balb1_reduction.iloc[0]["final_effective_keys"])

print("BALB1 raw August rows:", f"{raw_rows:,}")
print("BALB1 final-effective interval keys:", f"{final_keys:,}")
print("Reduction factor:", round(raw_rows / final_keys, 1), "x")
print(
    "Rows retained by compact model:",
    f"{(100 * final_keys / raw_rows):.3f}%"
)

In [ ]:
# SECTION 27N — Validate offer timing before final-effective compaction
# READ ONLY

balb1_offer_timing_sql = """
SELECT
  COUNT(*) AS raw_rows,
  COUNTIF(OFFERDATE IS NULL) AS null_offerdate_rows,
  COUNTIF(OFFERDATE <= SETTLEMENTDATE) AS offer_on_or_before_settlement_rows,
  COUNTIF(OFFERDATE > SETTLEMENTDATE) AS offer_after_settlement_rows,
  MIN(TIMESTAMP_DIFF(OFFERDATE, SETTLEMENTDATE, HOUR)) AS min_hours_difference,
  MAX(TIMESTAMP_DIFF(OFFERDATE, SETTLEMENTDATE, HOUR)) AS max_hours_difference
FROM `master-aemo.aemo_master_schema.BIDPEROFFER_ARCHIVE_V2`
WHERE DUID = "BALB1"
  AND SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
  AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
"""

balb1_offer_timing = (
    bq_client.query(
        balb1_offer_timing_sql,
        job_config=bigquery.QueryJobConfig(
            maximum_bytes_billed=3 * 1024**3
        ),
    )
    .to_dataframe()
)

display(balb1_offer_timing)

In [ ]:
# SECTION 27O — Check BIDPEROFFER_D archive availability (READ ONLY)

def list_bidperoffer_d_archives(year, month):
    archive_page = (
        f"https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    response = generation_session.get(
        archive_page,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    urls = sorted({
        urljoin(archive_page, link.get("href"))
        for link in soup.find_all("a", href=True)
        if "BIDPEROFFER_D" in link.get("href").upper()
        and link.get("href").lower().endswith(".zip")
    })

    print(f"\n{year}-{month:02d}: {len(urls)} BIDPEROFFER_D ZIP file(s)")
    for url in urls:
        print(url)

    return urls

bidperoffer_d_2016_09 = list_bidperoffer_d_archives(2016, 9)
bidperoffer_d_2023_04 = list_bidperoffer_d_archives(2023, 4)
bidperoffer_d_2024_08 = list_bidperoffer_d_archives(2024, 8)

In [ ]:
# SECTION 27P — Check DISPATCHOFFERTRK archive availability (READ ONLY)

def list_dispatchoffertrk_archives(year, month):
    archive_page = (
        f"https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month:02d}/"
        f"MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    response = generation_session.get(
        archive_page,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    urls = sorted({
        urljoin(archive_page, link.get("href"))
        for link in soup.find_all("a", href=True)
        if "DISPATCHOFFERTRK" in link.get("href").upper()
        and link.get("href").lower().endswith(".zip")
    })

    print(f"\n{year}-{month:02d}: {len(urls)} DISPATCHOFFERTRK ZIP file(s)")
    for url in urls:
        print(url)

    return urls

dispatchoffertrk_2016_09 = list_dispatchoffertrk_archives(2016, 9)
dispatchoffertrk_2023_04 = list_dispatchoffertrk_archives(2023, 4)
dispatchoffertrk_2024_08 = list_dispatchoffertrk_archives(2024, 8)

In [ ]:
# SECTION 27Q — Inspect DISPATCHOFFERTRK source schemas (READ ONLY)

dispatchoffertrk_sources = {
    "2016-09 DISPATCHOFFERTRK":
        next(
            url for url in dispatchoffertrk_2016_09
            if "PREDISPATCHOFFERTRK" not in url.upper()
        ),

    "2024-08 DISPATCHOFFERTRK":
        next(
            url for url in dispatchoffertrk_2024_08
            if "PREDISPATCHOFFERTRK" not in url.upper()
        ),
}

for label, source_url in dispatchoffertrk_sources.items():
    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    response = generation_session.get(
        source_url,
        timeout=300,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as raw_file:
            reader = csv.reader(
                io.TextIOWrapper(raw_file, encoding="utf-8-sig")
            )

            for record_number in range(6):
                row = next(reader)
                print(f"Record {record_number + 1}:", row)

In [ ]:
# SECTION 27R — Find the BIDPEROFFER_D availability transition (READ ONLY)

transition_months = [
    (2024, 4),
    (2024, 5),
    (2024, 6),
    (2024, 7),
    (2024, 8),
    (2025, 1),
    (2025, 7),
    (2026, 1),
    (2026, 7),
]

bidperoffer_d_transition = {}

for year, month in transition_months:
    urls = list_bidperoffer_d_archives(year, month)
    bidperoffer_d_transition[f"{year}-{month:02d}"] = len(urls)

print("\nAvailability summary:")
for month_label, file_count in bidperoffer_d_transition.items():
    print(f"{month_label}: {file_count} file(s)")

In [ ]:
# SECTION 27S — Profile August 2024 compact BESS BIDPEROFFER_D source
# READ ONLY

from collections import Counter

bess_scope_df = bq_client.query("""
    SELECT DUID
    FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
""").to_dataframe()

bess_duid_set = set(bess_scope_df["DUID"].astype(str).str.upper())

source_url = bidperoffer_d_2024_08[0]

response = generation_session.get(
    source_url,
    timeout=300,
    headers={"User-Agent": "Mozilla/5.0"},
)
response.raise_for_status()

all_source_rows = 0
bess_source_rows = 0
bess_duid_counts = Counter()
source_columns = None
first_bess_date = None
last_bess_date = None

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    csv_name = next(
        name for name in zip_file.namelist()
        if name.upper().endswith(".CSV")
    )

    with zip_file.open(csv_name) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig")
        )

        for row in reader:
            if len(row) < 5:
                continue

            if row[0] == "I" and row[2] == "BIDPEROFFER_D":
                source_columns = row[4:]
                continue

            if row[0] != "D" or row[2] != "BIDPEROFFER_D":
                continue

            all_source_rows += 1
            values = dict(zip(source_columns, row[4:]))
            duid = values["DUID"].strip().upper()

            if duid in bess_duid_set:
                bess_source_rows += 1
                bess_duid_counts[duid] += 1

                settlement_date = values["SETTLEMENTDATE"]
                first_bess_date = (
                    settlement_date
                    if first_bess_date is None
                    else min(first_bess_date, settlement_date)
                )
                last_bess_date = (
                    settlement_date
                    if last_bess_date is None
                    else max(last_bess_date, settlement_date)
                )

print("CSV:", csv_name)
print("Source columns:", source_columns)
print("All BIDPEROFFER_D rows:", f"{all_source_rows:,}")
print("BESS-only rows:", f"{bess_source_rows:,}")
print("BESS DUIDs found:", len(bess_duid_counts))
print("First BESS settlement date:", first_bess_date)
print("Last BESS settlement date:", last_bess_date)

display(
    pd.DataFrame(
        sorted(bess_duid_counts.items()),
        columns=["DUID", "row_count"],
    )
)

In [ ]:
# SECTION 27T — Inventory historical AEMO Generation Information workbooks
# READ ONLY

GEN_INFO_PAGE = (
    "https://www.aemo.com.au/energy-systems/electricity/"
    "national-electricity-market-nem/nem-forecasting-and-planning/"
    "forecasting-and-planning-data/generation-information"
)

response = generation_session.get(
    GEN_INFO_PAGE,
    timeout=120,
    headers={"User-Agent": "Mozilla/5.0"},
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

generation_workbook_links = []

for link in soup.find_all("a", href=True):
    href = link.get("href")
    label = link.get_text(" ", strip=True)

    if ".xlsx" not in href.lower():
        continue

    full_url = urljoin(GEN_INFO_PAGE, href)

    if (
        "generation_information" in full_url.lower()
        or "generation information" in label.lower()
    ):
        generation_workbook_links.append(
            {"label": label, "url": full_url}
        )

generation_workbook_inventory = (
    pd.DataFrame(generation_workbook_links)
    .drop_duplicates()
)

print(
    "Generation Information workbook links found:",
    len(generation_workbook_inventory)
)

display(generation_workbook_inventory)

In [ ]:
# SECTION 27U — Create compact BESS effective-offer warehouse

EFFECTIVE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

EFFECTIVE_MANIFEST = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

effective_schema = [
    bigquery.SchemaField("SETTLEMENTDATE", "TIMESTAMP"),
    bigquery.SchemaField("DUID", "STRING"),
    bigquery.SchemaField("BIDTYPE", "STRING"),
    bigquery.SchemaField("DIRECTION", "STRING"),
    bigquery.SchemaField("INTERVAL_DATETIME", "TIMESTAMP"),
    bigquery.SchemaField("BIDSETTLEMENTDATE", "TIMESTAMP"),
    bigquery.SchemaField("OFFERDATE", "TIMESTAMP"),
    bigquery.SchemaField("PERIODID", "INTEGER"),
    bigquery.SchemaField("VERSIONNO", "FLOAT"),
    bigquery.SchemaField("MAXAVAIL", "FLOAT"),
    bigquery.SchemaField("FIXEDLOAD", "FLOAT"),
    bigquery.SchemaField("ROCUP", "FLOAT"),
    bigquery.SchemaField("ROCDOWN", "FLOAT"),
    bigquery.SchemaField("ENABLEMENTMIN", "FLOAT"),
    bigquery.SchemaField("ENABLEMENTMAX", "FLOAT"),
    bigquery.SchemaField("LOWBREAKPOINT", "FLOAT"),
    bigquery.SchemaField("HIGHBREAKPOINT", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL1", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL2", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL3", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL4", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL5", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL6", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL7", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL8", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL9", "FLOAT"),
    bigquery.SchemaField("BANDAVAIL10", "FLOAT"),
    bigquery.SchemaField("LASTCHANGED", "TIMESTAMP"),
    bigquery.SchemaField("PASAAVAILABILITY", "FLOAT"),
    bigquery.SchemaField("MR_CAPACITY", "FLOAT"),
    bigquery.SchemaField("ENERGYLIMIT", "FLOAT"),
    bigquery.SchemaField("_LOAD_DATE", "TIMESTAMP"),
    bigquery.SchemaField("_SOURCE", "STRING"),
    bigquery.SchemaField("_PIPELINE", "STRING"),
    bigquery.SchemaField("_SOURCE_FILE", "STRING"),
    bigquery.SchemaField("_SOURCE_URL", "STRING"),
    bigquery.SchemaField("_RUN_ID", "STRING"),
]

manifest_schema = [
    bigquery.SchemaField("RUN_ID", "STRING"),
    bigquery.SchemaField("ARCHIVE_YEAR", "INTEGER"),
    bigquery.SchemaField("ARCHIVE_MONTH", "INTEGER"),
    bigquery.SchemaField("SOURCE_FILE_NAME", "STRING"),
    bigquery.SchemaField("SOURCE_URL", "STRING"),
    bigquery.SchemaField("SOURCE_METHOD", "STRING"),
    bigquery.SchemaField("STATUS", "STRING"),
    bigquery.SchemaField("SOURCE_ROWS", "INTEGER"),
    bigquery.SchemaField("BESS_ROWS", "INTEGER"),
    bigquery.SchemaField("LOADED_ROWS", "INTEGER"),
    bigquery.SchemaField("STARTED_AT", "TIMESTAMP"),
    bigquery.SchemaField("COMPLETED_AT", "TIMESTAMP"),
    bigquery.SchemaField("ERROR_MESSAGE", "STRING"),
    bigquery.SchemaField("CREATED_AT", "TIMESTAMP"),
]

effective_table = bigquery.Table(EFFECTIVE_TABLE, schema=effective_schema)
effective_table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="SETTLEMENTDATE",
)
effective_table.clustering_fields = ["DUID", "BIDTYPE", "DIRECTION"]

bq_client.create_table(effective_table, exists_ok=True)

manifest_table = bigquery.Table(EFFECTIVE_MANIFEST, schema=manifest_schema)
bq_client.create_table(manifest_table, exists_ok=True)

print("✅ Effective BESS table ready:", EFFECTIVE_TABLE)
print("✅ Effective BESS manifest ready:", EFFECTIVE_MANIFEST)

In [ ]:
# SECTION 27V — Stage August 2024 compact BESS effective offers
# Creates/replaces ONLY the temporary stage table.
# Does NOT append to BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1.

EFFECTIVE_STAGE_202408 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202408"
)

EXPECTED_SOURCE_ROWS = 16_021_126
EXPECTED_BESS_ROWS = 1_060_726

SOURCE_URL_202408 = bidperoffer_d_2024_08[0]
RUN_ID_202408 = "bess-effective-direct-2024-08"

bess_duid_set = set(
    bq_client.query("""
        SELECT DUID
        FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
    """).to_dataframe()["DUID"].astype(str).str.upper()
)

# Production safety check: August must still be empty.
existing_target_rows = int(
    bq_client.query("""
        SELECT COUNT(*) AS row_count
        FROM `master-aemo.aemo_master_schema.BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1`
        WHERE SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
          AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
    """).to_dataframe().iloc[0]["row_count"]
)

print("Existing production August rows:", f"{existing_target_rows:,}")

if existing_target_rows != 0:
    raise RuntimeError(
        "Safety stop: August 2024 already has production rows. "
        "Nothing was staged."
    )

effective_field_names = [field.name for field in effective_schema]
effective_field_types = {
    field.name: field.field_type
    for field in effective_schema
}

def prepare_effective_bess_chunk(rows, source_columns):
    frame = pd.DataFrame(rows, columns=source_columns)

    for column in effective_field_names:
        if column not in frame.columns:
            frame[column] = pd.NA

    frame["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
    frame["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    frame["_PIPELINE"] = "BIDPEROFFER_D_BESS_EFFECTIVE_V1"
    frame["_SOURCE_FILE"] = source_csv_name
    frame["_SOURCE_URL"] = SOURCE_URL_202408
    frame["_RUN_ID"] = RUN_ID_202408

    for column, field_type in effective_field_types.items():
        if field_type == "FLOAT":
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce",
            )
        elif field_type == "INTEGER":
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce",
            ).astype("Int64")
        elif field_type == "TIMESTAMP":
            frame[column] = pd.to_datetime(
                frame[column],
                errors="coerce",
            )
        elif field_type == "STRING":
            frame[column] = (
                frame[column]
                .replace("", pd.NA)
                .astype("string")
            )

    return frame[effective_field_names]

response = generation_session.get(
    SOURCE_URL_202408,
    timeout=300,
    headers={"User-Agent": "Mozilla/5.0"},
)
response.raise_for_status()

all_source_rows = 0
bess_rows_seen = 0
chunk_number = 0
source_columns = None
buffered_rows = []

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
    source_csv_name = next(
        name for name in zip_file.namelist()
        if name.upper().endswith(".CSV")
    )

    print("Downloading:", source_csv_name)

    with zip_file.open(source_csv_name) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig")
        )

        for row in reader:
            if len(row) < 5:
                continue

            if row[0] == "I" and row[2] == "BIDPEROFFER_D":
                source_columns = row[4:]
                continue

            if row[0] != "D" or row[2] != "BIDPEROFFER_D":
                continue

            all_source_rows += 1
            duid = row[5].strip().upper()  # DUID is first source column

            if duid not in bess_duid_set:
                continue

            buffered_rows.append(row[4:])
            bess_rows_seen += 1

            if len(buffered_rows) >= 100_000:
                chunk = prepare_effective_bess_chunk(
                    buffered_rows,
                    source_columns,
                )
                buffered_rows = []

                disposition = (
                    bigquery.WriteDisposition.WRITE_TRUNCATE
                    if chunk_number == 0
                    else bigquery.WriteDisposition.WRITE_APPEND
                )

                bq_client.load_table_from_dataframe(
                    chunk,
                    EFFECTIVE_STAGE_202408,
                    job_config=bigquery.LoadJobConfig(
                        schema=effective_schema,
                        write_disposition=disposition,
                    ),
                ).result()

                chunk_number += 1
                print(
                    f"Chunks staged: {chunk_number} | "
                    f"BESS rows staged: {bess_rows_seen:,}"
                )

        if buffered_rows:
            chunk = prepare_effective_bess_chunk(
                buffered_rows,
                source_columns,
            )

            disposition = (
                bigquery.WriteDisposition.WRITE_TRUNCATE
                if chunk_number == 0
                else bigquery.WriteDisposition.WRITE_APPEND
            )

            bq_client.load_table_from_dataframe(
                chunk,
                EFFECTIVE_STAGE_202408,
                job_config=bigquery.LoadJobConfig(
                    schema=effective_schema,
                    write_disposition=disposition,
                ),
            ).result()

stage_rows = bq_client.get_table(EFFECTIVE_STAGE_202408).num_rows

print("\nSource rows read:", f"{all_source_rows:,}")
print("BESS rows identified:", f"{bess_rows_seen:,}")
print("Stage rows:", f"{stage_rows:,}")

if all_source_rows != EXPECTED_SOURCE_ROWS:
    raise RuntimeError(
        f"Unexpected source count: {all_source_rows:,}"
    )

if bess_rows_seen != EXPECTED_BESS_ROWS or stage_rows != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Stage verification failed; do not promote this stage."
    )

print("\n✅ AUGUST 2024 BESS EFFECTIVE-OFFER STAGE VERIFIED")

In [ ]:
# SECTION 27W — Promote August 2024 compact BESS effective offers
# Production change: appends exactly 1,060,726 verified rows.

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202408"
)

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

EXPECTED_BESS_ROWS = 1_060_726
RUN_ID_202408 = "bess-effective-direct-2024-08"

column_list = ", ".join(
    f"`{field.name}`"
    for field in effective_schema
)

def august_effective_row_count(table_id):
    return int(
        bq_client.query(f"""
            SELECT COUNT(*) AS row_count
            FROM `{table_id}`
            WHERE SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
              AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
        """).to_dataframe().iloc[0]["row_count"]
    )

stage_rows = bq_client.get_table(STAGE_TABLE).num_rows
target_rows_before = august_effective_row_count(TARGET_TABLE)

print("Stage rows:", f"{stage_rows:,}")
print("Production rows before:", f"{target_rows_before:,}")

if stage_rows != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        f"Safety stop: stage has {stage_rows:,} rows; "
        f"expected {EXPECTED_BESS_ROWS:,}."
    )

if target_rows_before == 0:
    bq_client.query(f"""
        INSERT INTO `{TARGET_TABLE}` ({column_list})
        SELECT {column_list}
        FROM `{STAGE_TABLE}`
    """).result()

elif target_rows_before == EXPECTED_BESS_ROWS:
    print("Already promoted previously; skipping duplicate insert.")

else:
    raise RuntimeError(
        "Safety stop: unexpected production row count for August 2024."
    )

target_rows_after = august_effective_row_count(TARGET_TABLE)

print("Production rows after:", f"{target_rows_after:,}")

if target_rows_after != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Promotion verification failed. Keep the stage table for review."
    )

manifest_exists = int(
    bq_client.query(f"""
        SELECT COUNT(*) AS row_count
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = "{RUN_ID_202408}"
    """).to_dataframe().iloc[0]["row_count"]
)

if manifest_exists == 0:
    manifest_row = pd.DataFrame([{
        "RUN_ID": RUN_ID_202408,
        "ARCHIVE_YEAR": 2024,
        "ARCHIVE_MONTH": 8,
        "SOURCE_FILE_NAME":
            "PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#202408010000.CSV",
        "SOURCE_URL": SOURCE_URL_202408,
        "SOURCE_METHOD": "BIDPEROFFER_D_DIRECT_APPLIED",
        "STATUS": "PROMOTED",
        "SOURCE_ROWS": 16_021_126,
        "BESS_ROWS": EXPECTED_BESS_ROWS,
        "LOADED_ROWS": EXPECTED_BESS_ROWS,
        "STARTED_AT": pd.Timestamp.now(tz="UTC"),
        "COMPLETED_AT": pd.Timestamp.now(tz="UTC"),
        "ERROR_MESSAGE": None,
        "CREATED_AT": pd.Timestamp.now(tz="UTC"),
    }])

    bq_client.load_table_from_dataframe(
        manifest_row,
        MANIFEST_TABLE,
        job_config=bigquery.LoadJobConfig(
            schema=manifest_schema,
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        ),
    ).result()

    print("Manifest entry created.")
else:
    print("Manifest entry already exists.")

print("\n✅ AUGUST 2024 BESS EFFECTIVE OFFERS PROMOTED")
print("Stage table retained for final audit:", STAGE_TABLE)

In [ ]:
# SECTION 27W — Promote August 2024 compact BESS effective offers
# Production change: appends exactly 1,060,726 verified rows.

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202408"
)

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

EXPECTED_BESS_ROWS = 1_060_726
RUN_ID_202408 = "bess-effective-direct-2024-08"

column_list = ", ".join(
    f"`{field.name}`"
    for field in effective_schema
)

def august_effective_row_count(table_id):
    return int(
        bq_client.query(f"""
            SELECT COUNT(*) AS row_count
            FROM `{table_id}`
            WHERE SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
              AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
        """).to_dataframe().iloc[0]["row_count"]
    )

stage_rows = bq_client.get_table(STAGE_TABLE).num_rows
target_rows_before = august_effective_row_count(TARGET_TABLE)

print("Stage rows:", f"{stage_rows:,}")
print("Production rows before:", f"{target_rows_before:,}")

if stage_rows != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        f"Safety stop: stage has {stage_rows:,} rows; "
        f"expected {EXPECTED_BESS_ROWS:,}."
    )

if target_rows_before == 0:
    bq_client.query(f"""
        INSERT INTO `{TARGET_TABLE}` ({column_list})
        SELECT {column_list}
        FROM `{STAGE_TABLE}`
    """).result()

elif target_rows_before == EXPECTED_BESS_ROWS:
    print("Already promoted previously; skipping duplicate insert.")

else:
    raise RuntimeError(
        "Safety stop: unexpected production row count for August 2024."
    )

target_rows_after = august_effective_row_count(TARGET_TABLE)

print("Production rows after:", f"{target_rows_after:,}")

if target_rows_after != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Promotion verification failed. Keep the stage table for review."
    )

manifest_exists = int(
    bq_client.query(f"""
        SELECT COUNT(*) AS row_count
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = "{RUN_ID_202408}"
    """).to_dataframe().iloc[0]["row_count"]
)

if manifest_exists == 0:
    manifest_row = pd.DataFrame([{
        "RUN_ID": RUN_ID_202408,
        "ARCHIVE_YEAR": 2024,
        "ARCHIVE_MONTH": 8,
        "SOURCE_FILE_NAME":
            "PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#202408010000.CSV",
        "SOURCE_URL": SOURCE_URL_202408,
        "SOURCE_METHOD": "BIDPEROFFER_D_DIRECT_APPLIED",
        "STATUS": "PROMOTED",
        "SOURCE_ROWS": 16_021_126,
        "BESS_ROWS": EXPECTED_BESS_ROWS,
        "LOADED_ROWS": EXPECTED_BESS_ROWS,
        "STARTED_AT": pd.Timestamp.now(tz="UTC"),
        "COMPLETED_AT": pd.Timestamp.now(tz="UTC"),
        "ERROR_MESSAGE": None,
        "CREATED_AT": pd.Timestamp.now(tz="UTC"),
    }])

    bq_client.load_table_from_dataframe(
        manifest_row,
        MANIFEST_TABLE,
        job_config=bigquery.LoadJobConfig(
            schema=manifest_schema,
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        ),
    ).result()

    print("Manifest entry created.")
else:
    print("Manifest entry already exists.")

print("\n✅ AUGUST 2024 BESS EFFECTIVE OFFERS PROMOTED")
print("Stage table retained for final audit:", STAGE_TABLE)

In [ ]:
# SECTION 27X — Audit August 2024 compact BESS promotion (READ ONLY)

effective_audit_sql = """
WITH stage_counts AS (
  SELECT
    DUID,
    BIDTYPE,
    DIRECTION,
    COUNT(*) AS row_count
  FROM `master-aemo.aemo_master_schema.BIDPEROFFER_BESS_EFFECTIVE_STAGE_202408`
  GROUP BY DUID, BIDTYPE, DIRECTION
),

target_counts AS (
  SELECT
    DUID,
    BIDTYPE,
    DIRECTION,
    COUNT(*) AS row_count
  FROM `master-aemo.aemo_master_schema.BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1`
  WHERE SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
    AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
  GROUP BY DUID, BIDTYPE, DIRECTION
)

SELECT
  COALESCE(s.DUID, t.DUID) AS DUID,
  COALESCE(s.BIDTYPE, t.BIDTYPE) AS BIDTYPE,
  COALESCE(s.DIRECTION, t.DIRECTION) AS DIRECTION,
  COALESCE(s.row_count, 0) AS stage_rows,
  COALESCE(t.row_count, 0) AS target_rows
FROM stage_counts AS s
FULL OUTER JOIN target_counts AS t
  USING (DUID, BIDTYPE, DIRECTION)
WHERE COALESCE(s.row_count, 0) != COALESCE(t.row_count, 0)
ORDER BY DUID, BIDTYPE, DIRECTION
"""

effective_audit_differences = (
    bq_client.query(effective_audit_sql)
    .to_dataframe()
)

target_august_rows = int(
    bq_client.query("""
        SELECT COUNT(*) AS row_count
        FROM `master-aemo.aemo_master_schema.BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1`
        WHERE SETTLEMENTDATE >= TIMESTAMP("2024-08-01")
          AND SETTLEMENTDATE <  TIMESTAMP("2024-09-01")
    """).to_dataframe().iloc[0]["row_count"]
)

print("August target rows:", f"{target_august_rows:,}")
print("Stage-versus-target differences:", len(effective_audit_differences))

assert target_august_rows == 1_060_726
assert effective_audit_differences.empty

print("\n✅ AUGUST 2024 BESS EFFECTIVE-OFFER AUDIT PASSED")

In [ ]:
# SECTION 27Y — Delete verified August 2024 effective-offer stage

stage_table_id = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202408"
)

stage_table = bq_client.get_table(stage_table_id)

print("Deleting:", stage_table_id)
print("Verified stage rows:", f"{stage_table.num_rows:,}")

bq_client.delete_table(stage_table_id, not_found_ok=False)

print("\n✅ Temporary stage deleted.")
print("✅ 1,060,726 August 2024 compact BESS offer rows remain in production.")

In [ ]:
# SECTION 27Z — Build historical BESS candidate inventory from AEMO snapshots
# READ ONLY: no BigQuery data is changed.

historical_generation_sources = {
    "2019-11": (
        "https://www.aemo.com.au/-/media/files/electricity/nem/"
        "planning_and_forecasting/generation_information/nov-2019/"
        "generationinformationpage_20191114.xlsx"
        "?rev=b9e12a74659a4fa0be40eb20ff56e553"
    ),
    "2021-11": (
        "https://www.aemo.com.au/-/media/files/electricity/nem/"
        "planning_and_forecasting/generation_information/2021/"
        "nem-generation-information-nov-2021.xlsx"
        "?rev=991ce4d7e0aa4391a3321b437537fc16&sc_lang=en"
    ),
    "2023-10": (
        "https://www.aemo.com.au/-/media/files/electricity/nem/"
        "planning_and_forecasting/generation_information/2023/"
        "nem-generation-information-oct-2023.xlsx"
        "?rev=7993bbfb03d74581a8623f22013df0a3&sc_lang=en"
    ),
    "2024-07": (
        "https://www.aemo.com.au/-/media/files/electricity/nem/"
        "planning_and_forecasting/generation_information/2024/"
        "nem-generation-information-july-2024.xlsx"
        "?rev=649a85903ca147ed8cf4f9ec48c23429&sc_lang=en"
    ),
}

def find_generator_sheet_and_header(workbook):
    for sheet_name in workbook.sheet_names:
        preview = pd.read_excel(
            workbook,
            sheet_name=sheet_name,
            header=None,
            nrows=20,
        )

        for row_number, row in preview.iterrows():
            values = {
                str(value).strip().lower()
                for value in row.tolist()
                if pd.notna(value)
            }

            if "duid" in values and "technology type" in values:
                return sheet_name, row_number

    return None, None

historical_bess_frames = []

for snapshot_label, source_url in historical_generation_sources.items():
    print("\n" + "=" * 100)
    print("Inspecting snapshot:", snapshot_label)

    response = generation_session.get(
        source_url,
        timeout=180,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": GEN_INFO_PAGE,
        },
    )
    response.raise_for_status()

    workbook = pd.ExcelFile(io.BytesIO(response.content))
    sheet_name, header_row = find_generator_sheet_and_header(workbook)

    if sheet_name is None:
        print("⚠️ Could not automatically find DUID / Technology Type headers.")
        print("Sheets:", workbook.sheet_names)
        continue

    frame = pd.read_excel(
        workbook,
        sheet_name=sheet_name,
        header=header_row,
    )

    frame.columns = frame.columns.astype(str).str.strip()

    battery_rows = frame[
        frame["Technology Type"]
        .astype("string")
        .str.contains("battery", case=False, na=False)
    ].copy()

    battery_rows["SNAPSHOT"] = snapshot_label
    battery_rows["SOURCE_URL"] = source_url

    print("Sheet:", sheet_name)
    print("Header row:", header_row)
    print("Battery rows:", len(battery_rows))
    print("Unique battery DUIDs:", battery_rows["DUID"].nunique())

    historical_bess_frames.append(battery_rows)

historical_bess_candidates = pd.concat(
    historical_bess_frames,
    ignore_index=True,
)

historical_bess_candidates["DUID"] = (
    historical_bess_candidates["DUID"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("\n" + "=" * 100)
print("Historical snapshot battery rows:", len(historical_bess_candidates))
print(
    "Unique historical battery DUIDs:",
    historical_bess_candidates["DUID"].nunique()
)

display(
    historical_bess_candidates[
        [
            column for column in [
                "SNAPSHOT",
                "DUID",
                "Site Name",
                "Unit Name",
                "Region",
                "Technology Type",
                "Commitment Status",
            ]
            if column in historical_bess_candidates.columns
        ]
    ]
    .sort_values(["DUID", "SNAPSHOT"])
)

In [ ]:
# SECTION 28A — Audit historical BESS DUID candidates
# READ ONLY

historical_bess_with_duid = historical_bess_candidates[
    historical_bess_candidates["DUID"].notna()
    & historical_bess_candidates["DUID"].ne("")
].copy()

print("Historical battery rows with a DUID:", len(historical_bess_with_duid))
print(
    "Unique historical battery DUIDs:",
    historical_bess_with_duid["DUID"].nunique()
)

display(
    historical_bess_with_duid
    .groupby(["SNAPSHOT", "Commitment Status"], dropna=False)
    .agg(
        rows=("DUID", "size"),
        unique_duids=("DUID", "nunique"),
    )
    .reset_index()
    .sort_values(["SNAPSHOT", "Commitment Status"])
)

current_bess_duids = set(
    bq_client.query("""
        SELECT DUID
        FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
    """).to_dataframe()["DUID"].astype(str).str.upper()
)

historical_bess_duids = set(
    historical_bess_with_duid["DUID"].astype(str).str.upper()
)

historical_only = sorted(historical_bess_duids - current_bess_duids)
current_only = sorted(current_bess_duids - historical_bess_duids)

print("Historical DUIDs not in the 2026 scope:", len(historical_only))
print("2026-scope DUIDs not in historical snapshots:", len(current_only))

display(pd.DataFrame({"HISTORICAL_ONLY_DUID": historical_only}))
display(pd.DataFrame({"CURRENT_2026_ONLY_DUID": current_only}))

In [ ]:
# SECTION 28A — Continue with flexible historical-status audit
# READ ONLY

status_columns = [
    column for column in historical_bess_with_duid.columns
    if (
        "commitment" in column.lower()
        or "status" in column.lower()
    )
]

print("Possible status columns:", status_columns)

if status_columns:
    historical_bess_with_duid["HISTORICAL_STATUS"] = (
        historical_bess_with_duid[status_columns[0]]
        .astype("string")
        .fillna("<NULL>")
    )
else:
    historical_bess_with_duid["HISTORICAL_STATUS"] = (
        "<NOT AVAILABLE IN THIS SNAPSHOT FORMAT>"
    )

display(
    historical_bess_with_duid
    .groupby(["SNAPSHOT", "HISTORICAL_STATUS"], dropna=False)
    .agg(
        rows=("DUID", "size"),
        unique_duids=("DUID", "nunique"),
    )
    .reset_index()
    .sort_values(["SNAPSHOT", "HISTORICAL_STATUS"])
)

current_bess_duids = set(
    bq_client.query("""
        SELECT DUID
        FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
    """).to_dataframe()["DUID"].astype(str).str.upper()
)

historical_bess_duids = set(
    historical_bess_with_duid["DUID"].astype(str).str.upper()
)

historical_only = sorted(historical_bess_duids - current_bess_duids)
current_only = sorted(current_bess_duids - historical_bess_duids)

print("Historical DUIDs not in the 2026 scope:", len(historical_only))
print("2026-scope DUIDs not in historical snapshots:", len(current_only))

display(pd.DataFrame({"HISTORICAL_ONLY_DUID": historical_only}))
display(pd.DataFrame({"CURRENT_2026_ONLY_DUID": current_only}))

In [ ]:
# SECTION 28B — Normalize historical operating BESS DUID candidates
# READ ONLY

historical_operational = historical_bess_with_duid[
    historical_bess_with_duid["HISTORICAL_STATUS"].isin(
        ["In Service", "In Commissioning"]
    )
].copy()

historical_operational["RAW_DUID_CELL"] = (
    historical_operational["DUID"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Split paired legacy DUID cells, for example:
# "BHBG1 & BHBL1" → BHBG1 and BHBL1
historical_operational["DUID"] = (
    historical_operational["RAW_DUID_CELL"]
    .str.replace(r"\s*(?:&|/|,|;)\s*", "|", regex=True)
    .str.split("|")
)

historical_operational = (
    historical_operational
    .explode("DUID")
    .assign(
        DUID=lambda frame: (
            frame["DUID"]
            .astype("string")
            .str.strip()
            .str.upper()
        )
    )
)

historical_operational = historical_operational[
    historical_operational["DUID"].notna()
    & historical_operational["DUID"].ne("")
].copy()

historical_operating_duids = (
    historical_operational
    .groupby("DUID", as_index=False)
    .agg(
        SITE_NAME=("Site Name", "first"),
        REGION=("Region", "first"),
        FIRST_SNAPSHOT=("SNAPSHOT", "min"),
        LAST_SNAPSHOT=("SNAPSHOT", "max"),
        STATUS_HISTORY=(
            "HISTORICAL_STATUS",
            lambda values: " | ".join(sorted(set(values))),
        ),
        SOURCE_ROW_COUNT=("DUID", "size"),
    )
    .sort_values("DUID")
    .reset_index(drop=True)
)

print(
    "Historical operating/commissioning source rows:",
    len(historical_operational)
)
print(
    "Unique historical operating BESS DUIDs:",
    len(historical_operating_duids)
)

display(historical_operating_duids)

In [ ]:
# SECTION 28C — Create review-only historical-to-current BESS DUID candidates
# READ ONLY

from difflib import SequenceMatcher
import re

current_bess_scope = bq_client.query("""
    SELECT
      DUID,
      SITE_NAME,
      REGION,
      COMMITMENT_STATUS
    FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`
""").to_dataframe()

def normalise_site_name(value):
    text = str(value).lower()

    # Remove punctuation and broad descriptive words.
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(
        r"\b(bess|battery|energy|storage|system|project|unit|"
        r"phase|reserve|power|plant|facility)\b",
        " ",
        text,
    )

    return " ".join(text.split())

def site_similarity(left, right):
    return round(
        100 * SequenceMatcher(
            None,
            normalise_site_name(left),
            normalise_site_name(right),
        ).ratio(),
        1,
    )

candidate_rows = []

for _, legacy in historical_operating_duids.iterrows():
    same_region_current = current_bess_scope[
        current_bess_scope["REGION"] == legacy["REGION"]
    ].copy()

    same_region_current["MATCH_SCORE"] = same_region_current[
        "SITE_NAME"
    ].apply(
        lambda current_site: site_similarity(
            legacy["SITE_NAME"],
            current_site,
        )
    )

    top_matches = (
        same_region_current
        .sort_values("MATCH_SCORE", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    candidate_row = {
        "LEGACY_DUID": legacy["DUID"],
        "LEGACY_SITE_NAME": legacy["SITE_NAME"],
        "REGION": legacy["REGION"],
        "FIRST_SNAPSHOT": legacy["FIRST_SNAPSHOT"],
        "LAST_SNAPSHOT": legacy["LAST_SNAPSHOT"],
    }

    for position in range(3):
        if position < len(top_matches):
            match = top_matches.iloc[position]
            candidate_row[f"CANDIDATE_{position + 1}_DUID"] = match["DUID"]
            candidate_row[f"CANDIDATE_{position + 1}_SITE"] = match["SITE_NAME"]
            candidate_row[f"CANDIDATE_{position + 1}_SCORE"] = match["MATCH_SCORE"]
        else:
            candidate_row[f"CANDIDATE_{position + 1}_DUID"] = None
            candidate_row[f"CANDIDATE_{position + 1}_SITE"] = None
            candidate_row[f"CANDIDATE_{position + 1}_SCORE"] = None

    candidate_rows.append(candidate_row)

bess_crosswalk_candidates = pd.DataFrame(candidate_rows)

display(
    bess_crosswalk_candidates.sort_values(
        ["REGION", "LEGACY_SITE_NAME"]
    )
)

In [ ]:
# SECTION 28 — create reviewed historical-to-current BESS DUID crosswalk

from google.cloud import bigquery
import pandas as pd

CROSSWALK_TABLE = (
    "master-aemo.aemo_master_schema.BESS_DUID_CROSSWALK_V1"
)

# Explicit mappings: never rely on fuzzy name matching during production loading.
reviewed_mappings = {
    "ADPBA1G": ("ADPBA1",  "SITE_REGION_EXACT", "HIGH"),
    "BALBG1":  ("BALB1",   "SITE_REGION_EXACT", "HIGH"),
    "BBATTERY":("BBATTERY1","SITE_REGION_EXACT", "HIGH"),
    "BHBG1":   ("BHB1",    "BDU_GEN_LOAD_TO_SINGLE_BDU", "HIGH"),
    "BHBL1":   ("BHB1",    "BDU_GEN_LOAD_TO_SINGLE_BDU", "HIGH"),
    "BOWWBA1G":("BOWWBA1", "SITE_REGION_EXACT", "HIGH"),
    "BULBESG1":("BULBES1", "SITE_REGION_EXACT", "HIGH"),
    "CBWWBA1G":("CBWWBA1", "SITE_REGION_EXACT", "HIGH"),
    "CHBESSG1":("CHBESS1", "SITE_REGION_EXACT", "HIGH"),
    "DALNTH01":("DALNTH1", "SITE_REGION_EXACT", "HIGH"),
    "DPNTBG1": ("DPNTB1",  "SITE_REGION_EXACT", "HIGH"),
    "GANNBG1": ("GANNB1",  "SITE_REGION_EXACT", "HIGH"),
    "HBESSG1": ("HBESS1",  "SITE_REGION_EXACT", "HIGH"),
    "HPRG1":   ("HPR1",    "SITE_REGION_EXACT", "HIGH"),
    "HVWABA1G":("HVWWBA1", "SITE_REGION_EXACT", "HIGH"),
    "KEPBG1":  ("KEPBG1",  "SITE_REGION_NAME_VARIANT", "HIGH"),
    "LBBG1":   ("LBB1",    "SITE_REGION_NAME_VARIANT", "HIGH"),
    "PIBESSG1":("PIBESS1", "SITE_REGION_EXACT", "HIGH"),
    "QBYNBG1": ("QBYNB1",  "REVIEWED_SITE_AND_BDU_TRANSITION", "REVIEWED"),
    "RESS1G":  ("RESS1",   "SITE_REGION_EXACT", "HIGH"),
    "RIVNBG2": ("RIVNB2",  "SITE_REGION_EXACT", "HIGH"),
    "TB2BG1":  ("TB2B1",   "SITE_REGION_EXACT", "HIGH"),
    "TIBG1":   ("TIB1",    "BDU_GEN_LOAD_TO_SINGLE_BDU", "HIGH"),
    "TIPL1":   ("TIB1",    "BDU_GEN_LOAD_TO_SINGLE_BDU", "HIGH"),
    "VBBG1":   ("VBB1",    "SITE_REGION_EXACT", "HIGH"),
    "WALGRVG1":("WALGRV1", "SITE_REGION_EXACT", "HIGH"),
    "WANDBG1": ("WANDB1",  "SITE_REGION_EXACT", "HIGH"),
}

crosswalk = historical_operating_duids.copy()

missing_mappings = sorted(
    set(crosswalk["DUID"]) - set(reviewed_mappings)
)
if missing_mappings:
    raise RuntimeError(
        f"Safety stop: historical DUIDs without reviewed mappings: {missing_mappings}"
    )

crosswalk["CANONICAL_BESS_DUID"] = crosswalk["DUID"].map(
    lambda duid: reviewed_mappings[duid][0]
)
crosswalk["MAPPING_METHOD"] = crosswalk["DUID"].map(
    lambda duid: reviewed_mappings[duid][1]
)
crosswalk["MAPPING_CONFIDENCE"] = crosswalk["DUID"].map(
    lambda duid: reviewed_mappings[duid][2]
)
crosswalk = crosswalk.rename(columns={"DUID": "LEGACY_DUID"})

crosswalk["SOURCE_NOTE"] = (
    "Historical AEMO Generation Information snapshots; "
    "mapped to current operational/commissioning BESS registry."
)
crosswalk["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
crosswalk["_PIPELINE"] = "BESS_DUID_CROSSWALK_V1"

crosswalk = crosswalk[
    [
        "LEGACY_DUID",
        "CANONICAL_BESS_DUID",
        "SITE_NAME",
        "REGION",
        "FIRST_SNAPSHOT",
        "LAST_SNAPSHOT",
        "STATUS_HISTORY",
        "SOURCE_ROW_COUNT",
        "MAPPING_METHOD",
        "MAPPING_CONFIDENCE",
        "SOURCE_NOTE",
        "_LOAD_DATE",
        "_PIPELINE",
    ]
].sort_values(["REGION", "SITE_NAME", "LEGACY_DUID"]).reset_index(drop=True)

if len(crosswalk) != 27:
    raise RuntimeError(f"Expected 27 historical mappings; found {len(crosswalk)}.")

job = bq_client.load_table_from_dataframe(
    crosswalk,
    CROSSWALK_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)
job.result()

print(f"✅ Crosswalk saved: {CROSSWALK_TABLE}")
print(f"Rows: {len(crosswalk):,}")
display(
    crosswalk.groupby(
        ["MAPPING_CONFIDENCE", "MAPPING_METHOD"],
        dropna=False
    ).size().reset_index(name="rows")
)
display(crosswalk)

In [ ]:
# SECTION 29A — discover historic BESS-like DUIDs already present in BIDDAYOFFER
# Read-only query. This helps identify historic generation/load companion DUIDs.

from google.cloud import bigquery
import pandas as pd

BIDDAYOFFER_TABLE = (
    "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"
)

sql = f"""
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `{BIDDAYOFFER_TABLE}`
WHERE DUID IS NOT NULL
  AND TRIM(DUID) != ''
"""

# Inspect cost first. Stop before executing if unexpectedly large.
dry_run = bq_client.query(
    sql,
    job_config=bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
    ),
)

estimated_gb = dry_run.total_bytes_processed / (1024 ** 3)
print(f"Estimated data scanned: {estimated_gb:,.3f} GB")

MAX_ALLOWED_GB = 5

if estimated_gb > MAX_ALLOWED_GB:
    raise RuntimeError(
        f"Safety stop: estimate is {estimated_gb:,.2f} GB, above the "
        f"{MAX_ALLOWED_GB} GB limit. Nothing was executed."
    )

duid_job = bq_client.query(
    sql,
    job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_ALLOWED_GB * (1024 ** 3)
    ),
)
all_biddayoffer_duids = duid_job.result().to_dataframe()

print(f"Distinct DUIDs found: {len(all_biddayoffer_duids):,}")

# Broad prefixes only for discovery and human review — not yet a production filter.
bess_prefixes = (
    "ADPBA", "BALB", "BBATTERY", "BHB", "BOWWBA", "BULBES",
    "CBWWBA", "CHBESS", "DALNTH", "DPNTB", "GANNB", "HBESS",
    "HPR", "HVW", "KEPB", "LBB", "PIBESS", "QBYN", "RESS",
    "RIVNB", "TB2B", "TIB", "TIPL", "VBB", "WALGRV", "WANDB",
)

historic_bess_duid_candidates = (
    all_biddayoffer_duids[
        all_biddayoffer_duids["DUID"].str.startswith(
            bess_prefixes,
            na=False,
        )
    ]
    .sort_values("DUID")
    .reset_index(drop=True)
)

print(
    "Possible historic BESS / companion DUIDs found:",
    f"{len(historic_bess_duid_candidates):,}",
)

display(historic_bess_duid_candidates)

In [ ]:
# SECTION 29B — classify discovered historic BESS DUID candidates
# Read-only local analysis; no BigQuery data is changed.

import pandas as pd

asset_rules = [
    ("ADPBA",    "ADPBA1"),
    ("BALB",     "BALB1"),
    ("BBATTERY", "BBATTERY1"),
    ("BHB",      "BHB1"),
    ("BOWWBA",   "BOWWBA1"),
    ("BULBES",   "BULBES1"),
    ("CBWWBA",   "CBWWBA1"),
    ("CHBESS",   "CHBESS1"),
    ("DALNTH",   "DALNTH1"),
    ("DPNTB",    "DPNTB1"),
    ("GANNB",    "GANNB1"),
    ("HBESS",    "HBESS1"),
    ("HPR",      "HPR1"),
    ("HVW",      "HVWWBA1"),
    ("KEPB",     "KEPBG1"),
    ("LBB",      "LBB1"),
    ("PIBESS",   "PIBESS1"),
    ("QBYN",     "QBYNB1"),
    ("RESS",     "RESS1"),
    ("RIVNB",    "RIVNB2"),
    ("TB2B",     "TB2B1"),
    ("TIB",      "TIB1"),
    ("TIPL",     "TIB1"),
    ("VBB",      "VBB1"),
    ("WALGRV",   "WALGRV1"),
    ("WANDB",    "WANDB1"),
]

known_legacy_duids = set(crosswalk["LEGACY_DUID"])
known_canonical_duids = set(crosswalk["CANONICAL_BESS_DUID"])

def classify_duid(duid):
    duid = str(duid).strip().upper()

    canonical = None
    matched_prefix = None

    for prefix, target in asset_rules:
        if duid.startswith(prefix):
            canonical = target
            matched_prefix = prefix
            break

    if duid in known_legacy_duids:
        role = "KNOWN_HISTORICAL_DUID"
    elif duid in known_canonical_duids:
        role = "CURRENT_CANONICAL_BDU"
    elif duid.endswith("L") or duid.endswith("L1"):
        role = "POTENTIAL_HISTORICAL_LOAD_ALIAS"
    elif duid.endswith("G") or duid.endswith("G1"):
        role = "POTENTIAL_HISTORICAL_GENERATION_ALIAS"
    else:
        role = "RELATED_IDENTIFIER_REVIEW"

    return pd.Series(
        {
            "DUID": duid,
            "MATCHED_PREFIX": matched_prefix,
            "PROPOSED_CANONICAL_BESS_DUID": canonical,
            "CLASSIFICATION": role,
        }
    )

bess_duid_alias_review = (
    historic_bess_duid_candidates["DUID"]
    .apply(classify_duid)
    .sort_values(
        ["PROPOSED_CANONICAL_BESS_DUID", "CLASSIFICATION", "DUID"]
    )
    .reset_index(drop=True)
)

print("Classification summary:")
display(
    bess_duid_alias_review
    .groupby("CLASSIFICATION", dropna=False)
    .size()
    .reset_index(name="DUID_COUNT")
    .sort_values("CLASSIFICATION")
)

print("\nFull identifier review:")
pd.set_option("display.max_rows", 200)
display(bess_duid_alias_review)

In [ ]:
# SECTION 29C — save complete historic BESS DUID alias crosswalk
# Creates a small reference table only. No bid data is changed.

from google.cloud import bigquery
import pandas as pd

ALIAS_CROSSWALK_TABLE = (
    "master-aemo.aemo_master_schema.BESS_DUID_ALIAS_CROSSWALK_V1"
)

# 1. Historic identifiers directly observed in AEMO Generation Information snapshots.
snapshot_aliases = crosswalk.copy()

snapshot_aliases = snapshot_aliases.rename(
    columns={"LEGACY_DUID": "SOURCE_DUID"}
)

snapshot_aliases["ROW_ROLE"] = "HISTORICAL_SNAPSHOT_DUID"

# 2. Extra historic generation/load aliases discovered in BIDDAYOFFER.
# Exclude HVWWPV1: it is photovoltaic, not a BESS identifier.
extra_aliases = bess_duid_alias_review[
    (
        bess_duid_alias_review["CLASSIFICATION"].isin(
            [
                "POTENTIAL_HISTORICAL_LOAD_ALIAS",
                "POTENTIAL_HISTORICAL_GENERATION_ALIAS",
            ]
        )
        |
        (bess_duid_alias_review["DUID"] == "RIVNBL2")
    )
    &
    (bess_duid_alias_review["DUID"] != "HVWWPV1")
].copy()

expected_extra_aliases = 24
if len(extra_aliases) != expected_extra_aliases:
    raise RuntimeError(
        f"Expected {expected_extra_aliases} discovered aliases, "
        f"found {len(extra_aliases)}. Nothing was saved."
    )

# Attach site/region details from the already reviewed crosswalk.
canonical_details = (
    crosswalk.sort_values(["CANONICAL_BESS_DUID", "LAST_SNAPSHOT"])
    .drop_duplicates("CANONICAL_BESS_DUID", keep="last")
    [
        [
            "CANONICAL_BESS_DUID",
            "SITE_NAME",
            "REGION",
            "FIRST_SNAPSHOT",
            "LAST_SNAPSHOT",
            "STATUS_HISTORY",
        ]
    ]
)

extra_aliases = extra_aliases.rename(
    columns={
        "DUID": "SOURCE_DUID",
        "PROPOSED_CANONICAL_BESS_DUID": "CANONICAL_BESS_DUID",
    }
).merge(
    canonical_details,
    on="CANONICAL_BESS_DUID",
    how="left",
    validate="many_to_one",
)

if extra_aliases["CANONICAL_BESS_DUID"].isna().any():
    raise RuntimeError("A discovered alias has no canonical BESS mapping.")

extra_aliases["SOURCE_ROW_COUNT"] = pd.NA
extra_aliases["MAPPING_METHOD"] = "PAIRED_DUID_PATTERN_IN_BIDDAYOFFER"
extra_aliases["MAPPING_CONFIDENCE"] = "HIGH"
extra_aliases["SOURCE_NOTE"] = (
    "Historic generation/load alias discovered in BIDDAYOFFER_ARCHIVE; "
    "mapped using the reviewed BESS asset prefix and paired-DUID structure."
)
extra_aliases["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
extra_aliases["_PIPELINE"] = "BESS_DUID_ALIAS_CROSSWALK_V1"

extra_aliases["ROW_ROLE"] = extra_aliases["CLASSIFICATION"].map(
    {
        "POTENTIAL_HISTORICAL_LOAD_ALIAS": "DISCOVERED_HISTORICAL_LOAD_ALIAS",
        "POTENTIAL_HISTORICAL_GENERATION_ALIAS": (
            "DISCOVERED_HISTORICAL_GENERATION_ALIAS"
        ),
    }
).fillna("DISCOVERED_HISTORICAL_LOAD_ALIAS")

# Keep exactly the same schema for both groups.
output_columns = [
    "SOURCE_DUID",
    "CANONICAL_BESS_DUID",
    "SITE_NAME",
    "REGION",
    "FIRST_SNAPSHOT",
    "LAST_SNAPSHOT",
    "STATUS_HISTORY",
    "SOURCE_ROW_COUNT",
    "ROW_ROLE",
    "MAPPING_METHOD",
    "MAPPING_CONFIDENCE",
    "SOURCE_NOTE",
    "_LOAD_DATE",
    "_PIPELINE",
]

snapshot_aliases = snapshot_aliases[output_columns]
extra_aliases = extra_aliases[output_columns]

bess_duid_alias_crosswalk = (
    pd.concat([snapshot_aliases, extra_aliases], ignore_index=True)
    .drop_duplicates("SOURCE_DUID", keep="first")
    .sort_values(["CANONICAL_BESS_DUID", "SOURCE_DUID"])
    .reset_index(drop=True)
)

expected_total_aliases = 51
if len(bess_duid_alias_crosswalk) != expected_total_aliases:
    raise RuntimeError(
        f"Expected {expected_total_aliases} total aliases, "
        f"found {len(bess_duid_alias_crosswalk)}. Nothing was saved."
    )

load_job = bq_client.load_table_from_dataframe(
    bess_duid_alias_crosswalk,
    ALIAS_CROSSWALK_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)
load_job.result()

print(f"✅ Historic BESS alias crosswalk saved: {ALIAS_CROSSWALK_TABLE}")
print(f"Total historic aliases: {len(bess_duid_alias_crosswalk):,}")

display(
    bess_duid_alias_crosswalk
    .groupby(["ROW_ROLE", "MAPPING_CONFIDENCE"], dropna=False)
    .size()
    .reset_index(name="rows")
)

display(bess_duid_alias_crosswalk)

In [ ]:
# SECTION 30A — inspect September 2016 BESS bid source files
# Read-only source test. No BigQuery tables are created, changed, or deleted.

import csv
import io
import time
import zipfile
import requests
import pandas as pd
from google.cloud import bigquery

MONTH = "2016-09"

SOURCE_BASE = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2016/MMSDM_2016_09/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

source_urls = {
    "BIDPEROFFER": (
        SOURCE_BASE + "PUBLIC_DVD_BIDPEROFFER_201609010000.zip"
    ),
    "DISPATCHOFFERTRK": (
        SOURCE_BASE + "PUBLIC_DVD_DISPATCHOFFERTRK_201609010000.zip"
    ),
}

# Use both modern BDU DUIDs and all historic aliases as the BESS source scope.
scope_sql = """
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `master-aemo.aemo_master_schema.BESS_DUID_ALIAS_CROSSWALK_V1`
"""

bess_source_duids = set(
    bq_client.query(scope_sql).result().to_dataframe()["DUID"]
)

print(f"BESS source DUIDs in inspection scope: {len(bess_source_duids):,}")

def download_zip_bytes(url, label, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Downloading {label} — attempt {attempt}/{attempts}...")
            response = requests.get(
                url,
                timeout=600,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            return response.content

        except Exception as error:
            last_error = error
            if attempt < attempts:
                wait_seconds = min(60, 5 * (2 ** (attempt - 1)))
                print(f"⚠️ Download failed: {error}")
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"Could not download {label} after {attempts} attempts: {last_error}"
    )

def inspect_bess_rows(zip_bytes, source_name, max_samples=10):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"{source_name}: expected one CSV; found {csv_names}"
            )

        csv_name = csv_names[0]
        print(f"\n{source_name}")
        print(f"CSV: {csv_name}")

        with zip_file.open(csv_name) as raw_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline="",
                )
            )

            source_columns = None
            duid_position = None
            total_data_rows = 0
            bess_rows = 0
            bess_duids = set()
            sample_rows = []

            for record in reader:
                if not record:
                    continue

                # AEMO column-definition record.
                if record[0] == "I" and "DUID" in record:
                    source_columns = record[4:]
                    duid_position = source_columns.index("DUID")
                    continue

                # AEMO data record.
                if record[0] != "D":
                    continue

                if source_columns is None:
                    raise RuntimeError(
                        f"{source_name}: data found before column definition."
                    )

                values = record[4:]
                if len(values) != len(source_columns):
                    continue

                total_data_rows += 1
                duid = values[duid_position].strip().upper()

                if duid not in bess_source_duids:
                    continue

                bess_rows += 1
                bess_duids.add(duid)

                if len(sample_rows) < max_samples:
                    sample_rows.append(
                        {
                            column: value
                            for column, value in zip(source_columns, values)
                        }
                    )

    print("Source columns:", source_columns)
    print(f"All source rows scanned: {total_data_rows:,}")
    print(f"BESS rows found: {bess_rows:,}")
    print(f"BESS DUIDs found: {len(bess_duids):,}")
    print("BESS DUID list:", sorted(bess_duids))

    return {
        "SOURCE_NAME": source_name,
        "CSV_NAME": csv_name,
        "SOURCE_COLUMNS": source_columns,
        "TOTAL_SOURCE_ROWS": total_data_rows,
        "BESS_ROWS": bess_rows,
        "BESS_DUIDS": sorted(bess_duids),
        "SAMPLE_ROWS": pd.DataFrame(sample_rows),
    }

inspection_results = {}

for source_name, source_url in source_urls.items():
    zip_bytes = download_zip_bytes(source_url, source_name)

    inspection_results[source_name] = inspect_bess_rows(
        zip_bytes=zip_bytes,
        source_name=source_name,
    )

    print("\nBESS sample records:")
    display(inspection_results[source_name]["SAMPLE_ROWS"])

print("\n" + "=" * 100)
print(f"✅ {MONTH} HISTORICAL BESS SOURCE INSPECTION COMPLETE")

summary = pd.DataFrame(
    [
        {
            "SOURCE": name,
            "ALL_SOURCE_ROWS": result["TOTAL_SOURCE_ROWS"],
            "BESS_ROWS": result["BESS_ROWS"],
            "BESS_DUID_COUNT": len(result["BESS_DUIDS"]),
        }
        for name, result in inspection_results.items()
    ]
)

display(summary)

In [ ]:
# SECTION 30B — inspect November 2019 BESS bid source files
# Read-only source test. No BigQuery data is changed.

MONTH = "2019-11"

SOURCE_BASE = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2019/MMSDM_2019_11/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

source_urls_201911 = {
    "BIDPEROFFER": (
        SOURCE_BASE + "PUBLIC_DVD_BIDPEROFFER_201911010000.zip"
    ),
    "DISPATCHOFFERTRK": (
        SOURCE_BASE + "PUBLIC_DVD_DISPATCHOFFERTRK_201911010000.zip"
    ),
}

inspection_results_201911 = {}

for source_name, source_url in source_urls_201911.items():
    zip_bytes = download_zip_bytes(source_url, source_name)

    inspection_results_201911[source_name] = inspect_bess_rows(
        zip_bytes=zip_bytes,
        source_name=source_name,
    )

    print("\nBESS sample records:")
    display(inspection_results_201911[source_name]["SAMPLE_ROWS"])

summary_201911 = pd.DataFrame(
    [
        {
            "SOURCE": name,
            "ALL_SOURCE_ROWS": result["TOTAL_SOURCE_ROWS"],
            "BESS_ROWS": result["BESS_ROWS"],
            "BESS_DUID_COUNT": len(result["BESS_DUIDS"]),
            "BESS_DUIDS": ", ".join(result["BESS_DUIDS"]),
        }
        for name, result in inspection_results_201911.items()
    ]
)

if (summary_201911["BESS_ROWS"] == 0).any():
    raise RuntimeError(
        "Safety stop: expected BESS records in November 2019, "
        "but at least one source returned zero."
    )

print("\n" + "=" * 100)
print("✅ NOVEMBER 2019 HISTORICAL BESS SOURCE INSPECTION COMPLETE")
display(summary_201911)

In [ ]:
# SECTION 30C — prove the historical effective-offer join for November 2019
# Read-only local validation. No BigQuery data is changed.

from datetime import datetime
import csv
import io
import zipfile
import pandas as pd

def read_aemo_records(zip_bytes, source_name):
    """Yield AEMO CSV column names and D-record values from a one-CSV ZIP."""
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"{source_name}: expected one CSV; found {csv_names}"
            )

        with zip_file.open(csv_names[0]) as raw_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline="",
                )
            )

            columns = None

            for record in reader:
                if not record:
                    continue

                # AEMO column-definition record.
                if record[0] == "I" and "DUID" in record:
                    columns = record[4:]
                    continue

                # AEMO data record.
                if record[0] != "D":
                    continue

                if columns is None:
                    raise RuntimeError(
                        f"{source_name}: data found before its header."
                    )

                values = record[4:]

                if len(values) == len(columns):
                    yield columns, values

def tracker_period_id(settlementdate_text):
    """
    Convert AEMO dispatch settlement time to BIDPEROFFER PERIODID.

    A 00:00 dispatch record closes the previous trading day, so it belongs
    to period 288 rather than period zero.
    """
    settlement_ts = datetime.strptime(
        settlementdate_text.strip(),
        "%Y/%m/%d %H:%M:%S",
    )

    if settlement_ts.hour == 0 and settlement_ts.minute == 0:
        return "288"

    period_id = (
        (settlement_ts.hour * 60 + settlement_ts.minute) // 5
    )

    if not 1 <= period_id <= 288:
        raise RuntimeError(
            f"Unexpected tracker settlement time {settlementdate_text}; "
            f"derived PERIODID={period_id}."
        )

    return str(period_id)

# Download the two November 2019 sources again for the exact key-level test.
bid_zip = download_zip_bytes(
    source_urls_201911["BIDPEROFFER"],
    "BIDPEROFFER join test",
)

tracker_zip = download_zip_bytes(
    source_urls_201911["DISPATCHOFFERTRK"],
    "DISPATCHOFFERTRK join test",
)

# Build one expected raw-offer key for every applied tracker record.
tracker_target_keys = {}
tracker_bess_rows = 0

for columns, values in read_aemo_records(
    tracker_zip,
    "DISPATCHOFFERTRK",
):
    row = dict(zip(columns, values))
    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    tracker_bess_rows += 1

    key = (
        duid,
        row["BIDTYPE"].strip().upper(),
        row["BIDSETTLEMENTDATE"].strip(),
        row["BIDOFFERDATE"].strip(),
        tracker_period_id(row["SETTLEMENTDATE"]),
    )

    if key in tracker_target_keys:
        raise RuntimeError(
            "Duplicate tracker target key found. "
            f"Example: {key}"
        )

    tracker_target_keys[key] = {
        "APPLIED_SETTLEMENTDATE": row["SETTLEMENTDATE"].strip(),
        "SOURCE_DUID": duid,
        "BIDTYPE": row["BIDTYPE"].strip().upper(),
    }

print(f"BESS tracker rows: {tracker_bess_rows:,}")
print(f"Unique applied-offer keys: {len(tracker_target_keys):,}")

# Stream the larger raw offer file and retain only exact applied-offer rows.
matched_keys = set()
duplicate_raw_keys = set()
effective_samples = []

raw_bess_rows = 0
raw_rows_scanned = 0

for columns, values in read_aemo_records(
    bid_zip,
    "BIDPEROFFER",
):
    raw_rows_scanned += 1
    row = dict(zip(columns, values))

    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    raw_bess_rows += 1

    key = (
        duid,
        row["BIDTYPE"].strip().upper(),
        row["SETTLEMENTDATE"].strip(),
        row["OFFERDATE"].strip(),
        str(int(float(row["PERIODID"]))),
    )

    if key not in tracker_target_keys:
        continue

    if key in matched_keys:
        duplicate_raw_keys.add(key)
        continue

    matched_keys.add(key)

    if len(effective_samples) < 20:
        effective_samples.append(
            {
                **tracker_target_keys[key],
                "BIDSETTLEMENTDATE": row["SETTLEMENTDATE"].strip(),
                "BIDOFFERDATE": row["OFFERDATE"].strip(),
                "PERIODID": int(float(row["PERIODID"])),
                "VERSIONNO": row["VERSIONNO"].strip(),
                "MAXAVAIL": row["MAXAVAIL"].strip(),
                "BANDAVAIL1": row["BANDAVAIL1"].strip(),
                "BANDAVAIL10": row["BANDAVAIL10"].strip(),
            }
        )

unmatched_keys = set(tracker_target_keys) - matched_keys

print("\nJOIN AUDIT")
print(f"Raw BIDPEROFFER rows scanned: {raw_rows_scanned:,}")
print(f"Raw BESS BIDPEROFFER rows: {raw_bess_rows:,}")
print(f"Tracker keys expected: {len(tracker_target_keys):,}")
print(f"Raw offer keys matched: {len(matched_keys):,}")
print(f"Duplicate raw matches: {len(duplicate_raw_keys):,}")
print(f"Tracker keys unmatched: {len(unmatched_keys):,}")

if duplicate_raw_keys:
    raise RuntimeError(
        "Safety stop: more than one raw offer row matched at least one "
        "applied tracker key. No data has been written."
    )

if unmatched_keys:
    print("\nExample unmatched tracker keys:")
    display(
        pd.DataFrame(
            list(unmatched_keys)[:20],
            columns=[
                "DUID",
                "BIDTYPE",
                "BIDSETTLEMENTDATE",
                "BIDOFFERDATE",
                "PERIODID",
            ],
        )
    )

    raise RuntimeError(
        "Safety stop: not every applied tracker record found its raw offer row. "
        "No data has been written."
    )

if len(matched_keys) != len(tracker_target_keys):
    raise RuntimeError("Join count mismatch. No data has been written.")

print("\n✅ NOVEMBER 2019 HISTORICAL EFFECTIVE-OFFER JOIN PASSED")
display(pd.DataFrame(effective_samples))

In [ ]:
# SECTION 30E — corrected historical effective-offer join with 48-period support
# Read-only local validation. No BigQuery data is changed.

from datetime import datetime
import csv
import io
import zipfile
import pandas as pd

def read_aemo_records(zip_bytes, source_name):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"{source_name}: expected one CSV; found {csv_names}"
            )

        with zip_file.open(csv_names[0]) as raw_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline="",
                )
            )

            columns = None

            for record in reader:
                if not record:
                    continue

                if record[0] == "I" and "DUID" in record:
                    columns = record[4:]
                    continue

                if record[0] != "D":
                    continue

                if columns is None:
                    raise RuntimeError(
                        f"{source_name}: data found before its header."
                    )

                values = record[4:]

                if len(values) == len(columns):
                    yield columns, values

def tracker_to_bid_period_id(settlementdate_text, periods_per_day):
    """
    Map a five-minute tracker timestamp to a BIDPEROFFER bid period.

    Historical source: 48 half-hour bid periods.
    Modern source: 288 five-minute bid periods.
    """
    settlement_ts = datetime.strptime(
        settlementdate_text.strip(),
        "%Y/%m/%d %H:%M:%S",
    )

    if settlement_ts.hour == 0 and settlement_ts.minute == 0:
        return str(periods_per_day)

    minute_of_day = settlement_ts.hour * 60 + settlement_ts.minute

    if periods_per_day == 48:
        return str(((minute_of_day - 1) // 30) + 1)

    if periods_per_day == 288:
        return str(minute_of_day // 5)

    raise RuntimeError(f"Unsupported period count: {periods_per_day}")

# Download the two November 2019 source files.
bid_zip = download_zip_bytes(
    source_urls_201911["BIDPEROFFER"],
    "BIDPEROFFER corrected join test",
)

tracker_zip = download_zip_bytes(
    source_urls_201911["DISPATCHOFFERTRK"],
    "DISPATCHOFFERTRK corrected join test",
)

# First raw-offer pass: determine the real bid-period resolution.
raw_period_ids = set()
raw_bess_rows_first_pass = 0

for columns, values in read_aemo_records(bid_zip, "BIDPEROFFER"):
    row = dict(zip(columns, values))
    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    raw_bess_rows_first_pass += 1
    raw_period_ids.add(int(float(row["PERIODID"])))

raw_periods_per_day = max(raw_period_ids)

print(f"Raw BESS BIDPEROFFER rows: {raw_bess_rows_first_pass:,}")
print(f"Raw PERIODID range: {min(raw_period_ids)} to {max(raw_period_ids)}")
print(f"Detected bid periods per day: {raw_periods_per_day}")

if min(raw_period_ids) != 1 or raw_periods_per_day not in {48, 288}:
    raise RuntimeError(
        "Unexpected BIDPEROFFER period structure. No data has been written."
    )

# A tracker row is unique at five-minute level.
# A raw-offer key can correctly serve multiple tracker rows.
tracker_rows_by_raw_key = {}
tracker_record_ids = set()
tracker_bess_rows = 0

for columns, values in read_aemo_records(
    tracker_zip,
    "DISPATCHOFFERTRK",
):
    row = dict(zip(columns, values))
    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    tracker_bess_rows += 1

    tracker_record_id = (
        row["SETTLEMENTDATE"].strip(),
        duid,
        row["BIDTYPE"].strip().upper(),
    )

    if tracker_record_id in tracker_record_ids:
        raise RuntimeError(
            "Duplicate five-minute tracker record found. "
            f"Example: {tracker_record_id}"
        )

    tracker_record_ids.add(tracker_record_id)

    raw_offer_key = (
        duid,
        row["BIDTYPE"].strip().upper(),
        row["BIDSETTLEMENTDATE"].strip(),
        row["BIDOFFERDATE"].strip(),
        tracker_to_bid_period_id(
            row["SETTLEMENTDATE"],
            raw_periods_per_day,
        ),
    )

    tracker_rows_by_raw_key.setdefault(
        raw_offer_key,
        [],
    ).append(
        {
            "APPLIED_SETTLEMENTDATE": row["SETTLEMENTDATE"].strip(),
            "SOURCE_DUID": duid,
            "BIDTYPE": row["BIDTYPE"].strip().upper(),
        }
    )

print(f"BESS tracker rows: {tracker_bess_rows:,}")
print(f"Unique raw offer keys required: {len(tracker_rows_by_raw_key):,}")

# Second raw-offer pass: one raw offer may supply several tracker intervals.
matched_raw_keys = set()
duplicate_raw_keys = set()
matched_tracker_rows = 0
effective_samples = []

raw_rows_scanned = 0
raw_bess_rows_second_pass = 0

for columns, values in read_aemo_records(bid_zip, "BIDPEROFFER"):
    raw_rows_scanned += 1
    row = dict(zip(columns, values))

    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    raw_bess_rows_second_pass += 1

    raw_offer_key = (
        duid,
        row["BIDTYPE"].strip().upper(),
        row["SETTLEMENTDATE"].strip(),
        row["OFFERDATE"].strip(),
        str(int(float(row["PERIODID"]))),
    )

    if raw_offer_key not in tracker_rows_by_raw_key:
        continue

    if raw_offer_key in matched_raw_keys:
        duplicate_raw_keys.add(raw_offer_key)
        continue

    matched_raw_keys.add(raw_offer_key)

    for tracker_row in tracker_rows_by_raw_key[raw_offer_key]:
        matched_tracker_rows += 1

        if len(effective_samples) < 20:
            effective_samples.append(
                {
                    **tracker_row,
                    "BIDSETTLEMENTDATE": row["SETTLEMENTDATE"].strip(),
                    "BIDOFFERDATE": row["OFFERDATE"].strip(),
                    "PERIODID": int(float(row["PERIODID"])),
                    "VERSIONNO": row["VERSIONNO"].strip(),
                    "MAXAVAIL": row["MAXAVAIL"].strip(),
                    "BANDAVAIL1": row["BANDAVAIL1"].strip(),
                    "BANDAVAIL10": row["BANDAVAIL10"].strip(),
                }
            )

unmatched_raw_keys = (
    set(tracker_rows_by_raw_key) - matched_raw_keys
)

unmatched_tracker_rows = sum(
    len(tracker_rows_by_raw_key[key])
    for key in unmatched_raw_keys
)

print("\nJOIN AUDIT")
print(f"Raw BIDPEROFFER rows scanned: {raw_rows_scanned:,}")
print(f"Raw BESS rows in second pass: {raw_bess_rows_second_pass:,}")
print(f"Five-minute tracker rows expected: {tracker_bess_rows:,}")
print(f"Unique raw offer keys required: {len(tracker_rows_by_raw_key):,}")
print(f"Raw offer keys matched: {len(matched_raw_keys):,}")
print(f"Matched five-minute tracker rows: {matched_tracker_rows:,}")
print(f"Duplicate raw matches: {len(duplicate_raw_keys):,}")
print(f"Unmatched raw offer keys: {len(unmatched_raw_keys):,}")
print(f"Unmatched five-minute tracker rows: {unmatched_tracker_rows:,}")

if duplicate_raw_keys:
    raise RuntimeError(
        "Safety stop: more than one raw offer row matched the same "
        "required offer key. No data has been written."
    )

if unmatched_raw_keys:
    display(
        pd.DataFrame(
            list(unmatched_raw_keys)[:20],
            columns=[
                "DUID",
                "BIDTYPE",
                "BIDSETTLEMENTDATE",
                "BIDOFFERDATE",
                "PERIODID",
            ],
        )
    )

    raise RuntimeError(
        "Safety stop: some tracker rows still have no raw offer match. "
        "No data has been written."
    )

if matched_tracker_rows != tracker_bess_rows:
    raise RuntimeError(
        "Safety stop: matched tracker-row count does not equal the "
        "source tracker count. No data has been written."
    )

print("\n✅ NOVEMBER 2019 HISTORICAL EFFECTIVE-OFFER JOIN PASSED")
display(pd.DataFrame(effective_samples))

In [ ]:
# SECTION 30F — diagnose unmatched November 2019 tracker-to-offer keys
# Read-only local analysis. No BigQuery data is changed.

# The failed Section 30E left these variables available:
# - unmatched_raw_keys
# - tracker_rows_by_raw_key
# - bid_zip
# - bess_source_duids
# - read_aemo_records

unmatched_by_offer_identity = {}

for raw_key in unmatched_raw_keys:
    duid, bidtype, expected_bidsettlementdate, bidofferdate, periodid = raw_key

    # Ignore only BIDSETTLEMENTDATE for this diagnosis.
    identity_without_bidsettlementdate = (
        duid,
        bidtype,
        bidofferdate,
        periodid,
    )

    unmatched_by_offer_identity.setdefault(
        identity_without_bidsettlementdate,
        [],
    ).append(raw_key)

print(f"Unmatched raw-offer keys: {len(unmatched_raw_keys):,}")
print(
    "Unmatched offer identities excluding BIDSETTLEMENTDATE:",
    f"{len(unmatched_by_offer_identity):,}",
)

diagnostic_matches = []
still_missing_offer_identities = set(unmatched_by_offer_identity)

for columns, values in read_aemo_records(bid_zip, "BIDPEROFFER"):
    row = dict(zip(columns, values))
    duid = row["DUID"].strip().upper()

    if duid not in bess_source_duids:
        continue

    offer_identity = (
        duid,
        row["BIDTYPE"].strip().upper(),
        row["OFFERDATE"].strip(),
        str(int(float(row["PERIODID"]))),
    )

    if offer_identity not in unmatched_by_offer_identity:
        continue

    still_missing_offer_identities.discard(offer_identity)

    for expected_key in unmatched_by_offer_identity[offer_identity]:
        tracker_rows = tracker_rows_by_raw_key[expected_key]

        diagnostic_matches.append(
            {
                "SOURCE_DUID": duid,
                "BIDTYPE": row["BIDTYPE"].strip().upper(),
                "EXPECTED_BIDSETTLEMENTDATE": expected_key[2],
                "RAW_BIDPEROFFER_SETTLEMENTDATE": (
                    row["SETTLEMENTDATE"].strip()
                ),
                "BIDOFFERDATE": row["OFFERDATE"].strip(),
                "PERIODID": int(float(row["PERIODID"])),
                "TRACKER_ROW_COUNT": len(tracker_rows),
                "FIRST_APPLIED_SETTLEMENTDATE": min(
                    item["APPLIED_SETTLEMENTDATE"]
                    for item in tracker_rows
                ),
                "LAST_APPLIED_SETTLEMENTDATE": max(
                    item["APPLIED_SETTLEMENTDATE"]
                    for item in tracker_rows
                ),
            }
        )

diagnostic_df = pd.DataFrame(diagnostic_matches)

print(
    "Unmatched offer identities found in raw BIDPEROFFER "
    "when ignoring BIDSETTLEMENTDATE:",
    f"{diagnostic_df[['SOURCE_DUID', 'BIDTYPE', 'BIDOFFERDATE', 'PERIODID']].drop_duplicates().shape[0]:,}",
)
print(
    "Offer identities still absent from raw BIDPEROFFER:",
    f"{len(still_missing_offer_identities):,}",
)

if diagnostic_df.empty:
    raise RuntimeError(
        "No unmatched identities were found under another settlement date. "
        "Stop here; no data has been written."
    )

date_difference_summary = (
    diagnostic_df
    .groupby(
        [
            "EXPECTED_BIDSETTLEMENTDATE",
            "RAW_BIDPEROFFER_SETTLEMENTDATE",
        ],
        dropna=False,
    )
    .agg(
        raw_offer_keys=(
            "PERIODID",
            "size",
        ),
        tracker_rows=(
            "TRACKER_ROW_COUNT",
            "sum",
        ),
        first_applied=(
            "FIRST_APPLIED_SETTLEMENTDATE",
            "min",
        ),
        last_applied=(
            "LAST_APPLIED_SETTLEMENTDATE",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "EXPECTED_BIDSETTLEMENTDATE",
            "RAW_BIDPEROFFER_SETTLEMENTDATE",
        ]
    )
)

print("\nSettlement-date diagnostic summary:")
display(date_difference_summary.head(100))

print("\nExample matched identities with differing settlement-date values:")
display(
    diagnostic_df[
        diagnostic_df["EXPECTED_BIDSETTLEMENTDATE"]
        != diagnostic_df["RAW_BIDPEROFFER_SETTLEMENTDATE"]
    ]
    .sort_values(
        [
            "SOURCE_DUID",
            "BIDTYPE",
            "BIDOFFERDATE",
            "PERIODID",
        ]
    )
    .head(50)
)

In [ ]:
# SECTION 30G — self-contained inspection of November 2019 BIDPEROFFER_D
# Read-only. No BigQuery tables are created, changed, or deleted.

import csv
import io
import time
import zipfile
import requests
import pandas as pd

# Section 6.5 created the authenticated local BigQuery client named `client`.
scope_sql = """
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `master-aemo.aemo_master_schema.BESS_DUID_BACKFILL_SCOPE_V1`

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `master-aemo.aemo_master_schema.BESS_DUID_ALIAS_CROSSWALK_V1`
"""

bess_source_duids = set(
    client.query(scope_sql).result().to_dataframe()["DUID"]
)

print(f"BESS source DUIDs in scope: {len(bess_source_duids):,}")

if len(bess_source_duids) != 105:
    raise RuntimeError(
        f"Expected 105 BESS source DUIDs; found {len(bess_source_duids)}."
    )

BIDPEROFFER_D_URL_201911 = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2019/MMSDM_2019_11/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    "PUBLIC_DVD_BIDPEROFFER_D_201911010000.zip"
)

def download_zip(url, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Downloading — attempt {attempt}/{attempts}...")

            response = requests.get(
                url,
                timeout=600,
                headers={"User-Agent": "Mozilla/5.0"},
            )

            response.raise_for_status()

            return response.content

        except Exception as error:
            last_error = error

            if attempt < attempts:
                wait_seconds = min(60, 5 * (2 ** (attempt - 1)))
                print(f"⚠️ Download failed: {error}")
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"Download failed after {attempts} attempts: {last_error}"
    )

zip_bytes = download_zip(BIDPEROFFER_D_URL_201911)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
    csv_names = [
        name for name in zip_file.namelist()
        if name.upper().endswith(".CSV")
    ]

    if len(csv_names) != 1:
        raise RuntimeError(
            f"Expected one CSV in ZIP; found: {csv_names}"
        )

    csv_name = csv_names[0]
    print(f"CSV: {csv_name}")

    with zip_file.open(csv_name) as raw_csv:
        reader = csv.reader(
            io.TextIOWrapper(
                raw_csv,
                encoding="utf-8-sig",
                errors="replace",
                newline="",
            )
        )

        source_columns = None
        duid_position = None
        total_rows = 0
        bess_rows = 0
        bess_duids = set()
        sample_rows = []

        for record in reader:
            if not record:
                continue

            if record[0] == "I" and "DUID" in record:
                source_columns = record[4:]
                duid_position = source_columns.index("DUID")
                continue

            if record[0] != "D":
                continue

            if source_columns is None:
                continue

            values = record[4:]

            if len(values) != len(source_columns):
                continue

            total_rows += 1
            duid = values[duid_position].strip().upper()

            if duid not in bess_source_duids:
                continue

            bess_rows += 1
            bess_duids.add(duid)

            if len(sample_rows) < 20:
                sample_rows.append(dict(zip(source_columns, values)))

if source_columns is None:
    raise RuntimeError(
        "No AEMO column-definition record was found in BIDPEROFFER_D."
    )

print("\n✅ NOVEMBER 2019 BIDPEROFFER_D INSPECTION COMPLETE")
print("Source columns:", source_columns)
print(f"All source rows: {total_rows:,}")
print(f"BESS rows: {bess_rows:,}")
print(f"BESS DUIDs found: {len(bess_duids):,}")
print("BESS DUID list:", sorted(bess_duids))

display(pd.DataFrame(sample_rows))

In [ ]:
# SECTION 30H — audit November 2019 direct effective BESS offers
# Read-only local audit. No BigQuery data is changed.

# `zip_bytes`, `bess_source_duids`, and `client` were created in Section 30G.

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
    csv_name = next(
        name for name in zip_file.namelist()
        if name.upper().endswith(".CSV")
    )

    with zip_file.open(csv_name) as raw_csv:
        reader = csv.reader(
            io.TextIOWrapper(
                raw_csv,
                encoding="utf-8-sig",
                errors="replace",
                newline="",
            )
        )

        source_columns = None
        bess_records = []

        for record in reader:
            if not record:
                continue

            if record[0] == "I" and "DUID" in record:
                source_columns = record[4:]
                continue

            if record[0] != "D" or source_columns is None:
                continue

            values = record[4:]

            if len(values) != len(source_columns):
                continue

            row = dict(zip(source_columns, values))
            row["DUID"] = row["DUID"].strip().upper()

            if row["DUID"] in bess_source_duids:
                bess_records.append(row)

bess_effective_201911_probe = pd.DataFrame(bess_records)

print(f"BESS effective-offer rows read: {len(bess_effective_201911_probe):,}")

if len(bess_effective_201911_probe) != 432_000:
    raise RuntimeError(
        "Expected 432,000 November 2019 BESS effective-offer rows; "
        f"found {len(bess_effective_201911_probe):,}."
    )

# Retrieve historic-to-canonical BESS DUID mapping.
alias_sql = """
SELECT
    UPPER(TRIM(SOURCE_DUID)) AS SOURCE_DUID,
    CANONICAL_BESS_DUID
FROM `master-aemo.aemo_master_schema.BESS_DUID_ALIAS_CROSSWALK_V1`
"""

bess_alias_map = client.query(alias_sql).result().to_dataframe()

if bess_alias_map["SOURCE_DUID"].duplicated().any():
    raise RuntimeError(
        "Duplicate SOURCE_DUID values found in alias crosswalk."
    )

bess_effective_201911_probe = bess_effective_201911_probe.merge(
    bess_alias_map,
    left_on="DUID",
    right_on="SOURCE_DUID",
    how="left",
    validate="many_to_one",
)

if bess_effective_201911_probe["CANONICAL_BESS_DUID"].isna().any():
    missing_duids = sorted(
        bess_effective_201911_probe.loc[
            bess_effective_201911_probe["CANONICAL_BESS_DUID"].isna(),
            "DUID",
        ].unique()
    )

    raise RuntimeError(
        f"Historic BESS DUID(s) missing from alias crosswalk: {missing_duids}"
    )

# Explicit direction mapping for the ten DUIDs observed in November 2019.
# Old NEM units used separate generation and load identifiers.
direction_map_201911 = {
    "BALBG1": "GEN",
    "BALBL1": "LOAD",
    "DALNTH01": "GEN",
    "DALNTHL1": "LOAD",
    "GANNBG1": "GEN",
    "GANNBL1": "LOAD",
    "HPRG1": "GEN",
    "HPRL1": "LOAD",
    "LBBG1": "GEN",
    "LBBL1": "LOAD",
}

bess_effective_201911_probe["DIRECTION"] = (
    bess_effective_201911_probe["DUID"].map(direction_map_201911)
)

if bess_effective_201911_probe["DIRECTION"].isna().any():
    unmapped_direction_duids = sorted(
        bess_effective_201911_probe.loc[
            bess_effective_201911_probe["DIRECTION"].isna(),
            "DUID",
        ].unique()
    )

    raise RuntimeError(
        "Direction mapping missing for: "
        f"{unmapped_direction_duids}"
    )

bess_effective_201911_probe["INTERVAL_DATETIME"] = pd.to_datetime(
    bess_effective_201911_probe["INTERVAL_DATETIME"],
    errors="coerce",
)

bess_effective_201911_probe["SETTLEMENTDATE"] = pd.to_datetime(
    bess_effective_201911_probe["SETTLEMENTDATE"],
    errors="coerce",
)

if bess_effective_201911_probe[
    ["INTERVAL_DATETIME", "SETTLEMENTDATE"]
].isna().any().any():
    raise RuntimeError(
        "Timestamp parsing failed for at least one effective-offer row."
    )

effective_key = [
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
]

duplicate_effective_rows = int(
    bess_effective_201911_probe.duplicated(effective_key).sum()
)

print("\nAUDIT")
print(
    "First interval:",
    bess_effective_201911_probe["INTERVAL_DATETIME"].min(),
)
print(
    "Last interval:",
    bess_effective_201911_probe["INTERVAL_DATETIME"].max(),
)
print(
    "Canonical BESS assets:",
    bess_effective_201911_probe["CANONICAL_BESS_DUID"].nunique(),
)
print(
    "Source DUIDs:",
    bess_effective_201911_probe["DUID"].nunique(),
)
print(f"Duplicate effective keys: {duplicate_effective_rows:,}")

if duplicate_effective_rows != 0:
    raise RuntimeError(
        "Duplicate effective-offer keys found. No data has been written."
    )

print("\nRows by canonical BESS asset and direction:")
display(
    bess_effective_201911_probe
    .groupby(
        [
            "CANONICAL_BESS_DUID",
            "DIRECTION",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="ROWS")
    .sort_values(
        [
            "CANONICAL_BESS_DUID",
            "DIRECTION",
        ]
    )
)

print("\n✅ NOVEMBER 2019 DIRECT BESS EFFECTIVE-OFFER AUDIT PASSED")
display(
    bess_effective_201911_probe[
        [
            "INTERVAL_DATETIME",
            "SETTLEMENTDATE",
            "DUID",
            "CANONICAL_BESS_DUID",
            "DIRECTION",
            "BIDTYPE",
            "BIDSETTLEMENTDATE",
            "OFFERDATE",
            "PERIODID",
            "VERSIONNO",
            "MAXAVAIL",
            "BANDAVAIL1",
            "BANDAVAIL10",
        ]
    ].head(30)
)

In [ ]:
# SECTION 31A — inspect the effective BESS archive target schema
# Read-only. No BigQuery data is changed.

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

target_table = client.get_table(TARGET_TABLE)

print(f"Table: {TARGET_TABLE}")
print(f"Current rows: {target_table.num_rows:,}")
print(f"Partitioning: {target_table.time_partitioning.field}")
print(
    "Clustering:",
    target_table.clustering_fields
)

schema_df = pd.DataFrame(
    [
        {
            "FIELD_NAME": field.name,
            "FIELD_TYPE": field.field_type,
            "MODE": field.mode,
        }
        for field in target_table.schema
    ]
)

print("\nTarget schema:")
display(schema_df)

In [ ]:
# SECTION 31B — prepare November 2019 BESS effective offers for staging
# Local preparation and validation only. No BigQuery data is changed.

STAGE_TABLE_201911 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_201911"
)

SOURCE_FILE_201911 = "PUBLIC_DVD_BIDPEROFFER_D_201911010000.CSV"

SOURCE_URL_201911 = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2019/MMSDM_2019_11/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    "PUBLIC_DVD_BIDPEROFFER_D_201911010000.zip"
)

RUN_ID_201911 = "bess-effective-direct-2019-11"

target_columns = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
    "_LOAD_DATE",
    "_SOURCE",
    "_PIPELINE",
    "_SOURCE_FILE",
    "_SOURCE_URL",
    "_RUN_ID",
]

stage_201911_df = bess_effective_201911_probe.copy()

# The 2019 format has no ENERGYLIMIT field; preserve that fact as null.
stage_201911_df["ENERGYLIMIT"] = pd.NA

timestamp_columns = [
    "SETTLEMENTDATE",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "LASTCHANGED",
]

for column in timestamp_columns:
    stage_201911_df[column] = pd.to_datetime(
        stage_201911_df[column],
        errors="coerce",
    )

numeric_columns = [
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
]

for column in numeric_columns:
    stage_201911_df[column] = pd.to_numeric(
        stage_201911_df[column],
        errors="coerce",
    )

stage_201911_df["PERIODID"] = pd.to_numeric(
    stage_201911_df["PERIODID"],
    errors="coerce",
).astype("Int64")

stage_201911_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
stage_201911_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
stage_201911_df["_PIPELINE"] = "BIDPEROFFER_BESS_EFFECTIVE_HISTORICAL_DIRECT"
stage_201911_df["_SOURCE_FILE"] = SOURCE_FILE_201911
stage_201911_df["_SOURCE_URL"] = SOURCE_URL_201911
stage_201911_df["_RUN_ID"] = RUN_ID_201911

stage_201911_df = stage_201911_df[target_columns].copy()

critical_columns = [
    "SETTLEMENTDATE",
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "PERIODID",
]

null_critical_columns = {
    column: int(stage_201911_df[column].isna().sum())
    for column in critical_columns
    if stage_201911_df[column].isna().any()
}

if null_critical_columns:
    raise RuntimeError(
        "Critical fields contain null values: "
        f"{null_critical_columns}"
    )

duplicate_rows = int(
    stage_201911_df.duplicated(
        ["INTERVAL_DATETIME", "DUID", "BIDTYPE"]
    ).sum()
)

if duplicate_rows != 0:
    raise RuntimeError(
        f"Found {duplicate_rows:,} duplicate effective-offer keys."
    )

if len(stage_201911_df) != 432_000:
    raise RuntimeError(
        "Expected 432,000 prepared rows; "
        f"found {len(stage_201911_df):,}."
    )

print("✅ NOVEMBER 2019 STAGING PREFLIGHT PASSED")
print(f"Rows prepared: {len(stage_201911_df):,}")
print(f"Columns prepared: {len(stage_201911_df.columns):,}")
print(
    "Interval range:",
    stage_201911_df["INTERVAL_DATETIME"].min(),
    "to",
    stage_201911_df["INTERVAL_DATETIME"].max(),
)
print(
    "Settlement-date range:",
    stage_201911_df["SETTLEMENTDATE"].min(),
    "to",
    stage_201911_df["SETTLEMENTDATE"].max(),
)

display(stage_201911_df.head(20))

In [ ]:
# SECTION 31C — stage November 2019 BESS effective offers
# Writes only a temporary stage table. Production archive remains unchanged.

from google.api_core.exceptions import NotFound
from google.cloud import bigquery

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

STAGE_TABLE_201911 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_201911"
)

EXPECTED_STAGE_ROWS = 432_000

# Safety check 1: production must not already contain November 2019.
production_check_sql = f"""
SELECT COUNT(*) AS existing_rows
FROM `{TARGET_TABLE}`
WHERE DATE(SETTLEMENTDATE) >= DATE("2019-11-01")
  AND DATE(SETTLEMENTDATE) < DATE("2019-12-01")
"""

existing_production_rows = int(
    client.query(production_check_sql)
    .result()
    .to_dataframe()
    .iloc[0]["existing_rows"]
)

print(
    "Existing production November 2019 rows:",
    f"{existing_production_rows:,}",
)

if existing_production_rows != 0:
    raise RuntimeError(
        "Safety stop: November 2019 already exists in the production archive. "
        "Nothing was staged."
    )

# Safety check 2: never overwrite an existing non-empty stage automatically.
try:
    existing_stage = client.get_table(STAGE_TABLE_201911)

    if existing_stage.num_rows > 0:
        raise RuntimeError(
            f"Safety stop: stage table already exists with "
            f"{existing_stage.num_rows:,} rows:\n{STAGE_TABLE_201911}\n"
            "Nothing was overwritten."
        )

except NotFound:
    print("✅ No existing November 2019 stage table found.")

# Reuse the production schema exactly.
target_schema = client.get_table(TARGET_TABLE).schema

stage_job = client.load_table_from_dataframe(
    stage_201911_df,
    STAGE_TABLE_201911,
    job_config=bigquery.LoadJobConfig(
        schema=target_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        time_partitioning=bigquery.TimePartitioning(
            field="SETTLEMENTDATE"
        ),
        clustering_fields=[
            "DUID",
            "BIDTYPE",
            "DIRECTION",
        ],
    ),
)

stage_job.result()

stage_table = client.get_table(STAGE_TABLE_201911)

print("\nSTAGE VERIFICATION")
print(f"Stage table: {STAGE_TABLE_201911}")
print(f"Rows staged: {stage_table.num_rows:,}")

if stage_table.num_rows != EXPECTED_STAGE_ROWS:
    raise RuntimeError(
        f"Stage verification failed: expected {EXPECTED_STAGE_ROWS:,} rows, "
        f"found {stage_table.num_rows:,}. Production was not changed."
    )

stage_audit_sql = f"""
SELECT
    COUNT(*) AS rows_staged,
    COUNT(DISTINCT DUID) AS source_duids,
    COUNT(DISTINCT CONCAT(
        CAST(INTERVAL_DATETIME AS STRING),
        "|",
        DUID,
        "|",
        BIDTYPE
    )) AS distinct_effective_keys,
    MIN(INTERVAL_DATETIME) AS first_interval,
    MAX(INTERVAL_DATETIME) AS last_interval
FROM `{STAGE_TABLE_201911}`
"""

stage_audit = client.query(stage_audit_sql).result().to_dataframe()

display(stage_audit)

if int(stage_audit.iloc[0]["distinct_effective_keys"]) != EXPECTED_STAGE_ROWS:
    raise RuntimeError(
        "Stage contains duplicate effective-offer keys. "
        "Production was not changed."
    )

print("\n✅ NOVEMBER 2019 BESS EFFECTIVE-OFFER STAGE VERIFIED")

In [ ]:
# SECTION 31D — inspect effective-offer manifest before promotion
# Read-only. No BigQuery data is changed.

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

manifest_table = client.get_table(MANIFEST_TABLE)

print(f"Manifest table: {MANIFEST_TABLE}")
print(f"Existing manifest rows: {manifest_table.num_rows:,}")

manifest_schema_df = pd.DataFrame(
    [
        {
            "FIELD_NAME": field.name,
            "FIELD_TYPE": field.field_type,
            "MODE": field.mode,
        }
        for field in manifest_table.schema
    ]
)

display(manifest_schema_df)

manifest_preview = client.query(
    f"SELECT * FROM `{MANIFEST_TABLE}` LIMIT 10"
).result().to_dataframe()

print("\nExisting manifest entries:")
display(manifest_preview)

In [ ]:
# SECTION 31E — promote verified November 2019 BESS effective offers
# Production is changed only after every safety check passes.

from google.cloud import bigquery

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

STAGE_TABLE_201911 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_201911"
)

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

RUN_ID_201911 = "bess-effective-direct-2019-11"
EXPECTED_SOURCE_ROWS = 9_264_480
EXPECTED_BESS_ROWS = 432_000

SOURCE_FILE_201911 = "PUBLIC_DVD_BIDPEROFFER_D_201911010000.CSV"

SOURCE_URL_201911 = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2019/MMSDM_2019_11/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    "PUBLIC_DVD_BIDPEROFFER_D_201911010000.zip"
)

# Safety check 1 — stage must still be complete.
stage_row_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{STAGE_TABLE_201911}`
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"Stage rows: {stage_row_count:,}")

if stage_row_count != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        f"Safety stop: expected {EXPECTED_BESS_ROWS:,} stage rows, "
        f"found {stage_row_count:,}. Production was not changed."
    )

# Safety check 2 — this run cannot already be in production.
existing_run_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    RUN_ID_201911,
                )
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

if existing_run_count != 0:
    raise RuntimeError(
        f"Safety stop: production already contains {existing_run_count:,} "
        f"rows for run {RUN_ID_201911}. Nothing was promoted."
    )

# Safety check 3 — manifest cannot already contain this run.
existing_manifest_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    RUN_ID_201911,
                )
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

if existing_manifest_count != 0:
    raise RuntimeError(
        f"Safety stop: manifest already contains run {RUN_ID_201911}. "
        "Nothing was promoted."
    )

production_before = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"Production rows before promotion: {production_before:,}")
print("Promoting verified November 2019 stage...")

# Stage and production tables use the same approved 37-field schema.
promotion_job = client.query(
    f"""
    INSERT INTO `{TARGET_TABLE}`
    SELECT *
    FROM `{STAGE_TABLE_201911}`
    """
)

promotion_job.result()

# Verify the exact promoted run and November 2019 coverage.
production_run_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    RUN_ID_201911,
                )
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

november_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        WHERE DATE(SETTLEMENTDATE) >= DATE '2019-11-01'
          AND DATE(SETTLEMENTDATE) < DATE '2019-12-01'
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

production_after = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"Production rows after promotion: {production_after:,}")
print(f"Rows retained for this run: {production_run_count:,}")
print(f"Total November 2019 production rows: {november_count:,}")

if production_run_count != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Promotion verification failed: run-level row count mismatch."
    )

if november_count != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Promotion verification failed: November coverage mismatch."
    )

if production_after - production_before != EXPECTED_BESS_ROWS:
    raise RuntimeError(
        "Promotion verification failed: production growth mismatch."
    )

# Record successful promotion in the persistent manifest.
manifest_sql = f"""
INSERT INTO `{MANIFEST_TABLE}` (
    RUN_ID,
    ARCHIVE_YEAR,
    ARCHIVE_MONTH,
    SOURCE_FILE_NAME,
    SOURCE_URL,
    SOURCE_METHOD,
    STATUS,
    SOURCE_ROWS,
    BESS_ROWS,
    LOADED_ROWS,
    STARTED_AT,
    COMPLETED_AT,
    ERROR_MESSAGE,
    CREATED_AT
)
VALUES (
    @run_id,
    2019,
    11,
    @source_file,
    @source_url,
    "BIDPEROFFER_D_DIRECT_APPLIED",
    "PROMOTED",
    @source_rows,
    @bess_rows,
    @loaded_rows,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    NULL,
    CURRENT_TIMESTAMP()
)
"""

manifest_job = client.query(
    manifest_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "run_id",
                "STRING",
                RUN_ID_201911,
            ),
            bigquery.ScalarQueryParameter(
                "source_file",
                "STRING",
                SOURCE_FILE_201911,
            ),
            bigquery.ScalarQueryParameter(
                "source_url",
                "STRING",
                SOURCE_URL_201911,
            ),
            bigquery.ScalarQueryParameter(
                "source_rows",
                "INT64",
                EXPECTED_SOURCE_ROWS,
            ),
            bigquery.ScalarQueryParameter(
                "bess_rows",
                "INT64",
                EXPECTED_BESS_ROWS,
            ),
            bigquery.ScalarQueryParameter(
                "loaded_rows",
                "INT64",
                EXPECTED_BESS_ROWS,
            ),
        ]
    ),
)

manifest_job.result()

print("\n✅ NOVEMBER 2019 BESS EFFECTIVE OFFERS PROMOTED")
print("Stage table retained for final audit:")
print(STAGE_TABLE_201911)

In [ ]:
# SECTION 31F — final audit of November 2019 promotion
# Read-only. No BigQuery data is changed.

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

STAGE_TABLE_201911 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_201911"
)

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

RUN_ID_201911 = "bess-effective-direct-2019-11"
EXPECTED_ROWS = 432_000

audit_sql = f"""
WITH
stage_data AS (
    SELECT
        SETTLEMENTDATE,
        INTERVAL_DATETIME,
        DUID,
        BIDTYPE,
        DIRECTION,
        BIDSETTLEMENTDATE,
        OFFERDATE,
        PERIODID,
        VERSIONNO,
        MAXAVAIL,
        BANDAVAIL1,
        BANDAVAIL10
    FROM `{STAGE_TABLE_201911}`
),
production_data AS (
    SELECT
        SETTLEMENTDATE,
        INTERVAL_DATETIME,
        DUID,
        BIDTYPE,
        DIRECTION,
        BIDSETTLEMENTDATE,
        OFFERDATE,
        PERIODID,
        VERSIONNO,
        MAXAVAIL,
        BANDAVAIL1,
        BANDAVAIL10
    FROM `{TARGET_TABLE}`
    WHERE _RUN_ID = @run_id
),
stage_missing_from_production AS (
    SELECT * FROM stage_data
    EXCEPT DISTINCT
    SELECT * FROM production_data
),
production_missing_from_stage AS (
    SELECT * FROM production_data
    EXCEPT DISTINCT
    SELECT * FROM stage_data
)
SELECT
    (SELECT COUNT(*) FROM stage_data) AS stage_row_count,
    (SELECT COUNT(*) FROM production_data) AS production_run_row_count,
    (SELECT COUNT(*) FROM stage_missing_from_production)
        AS stage_missing_from_production_count,
    (SELECT COUNT(*) FROM production_missing_from_stage)
        AS production_missing_from_stage_count
"""

audit_result = client.query(
    audit_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "run_id",
                "STRING",
                RUN_ID_201911,
            )
        ]
    ),
).result().to_dataframe()

display(audit_result)

audit_row = audit_result.iloc[0]

if int(audit_row["stage_row_count"]) != EXPECTED_ROWS:
    raise RuntimeError("Stage row-count audit failed.")

if int(audit_row["production_run_row_count"]) != EXPECTED_ROWS:
    raise RuntimeError("Production row-count audit failed.")

if int(audit_row["stage_missing_from_production_count"]) != 0:
    raise RuntimeError("Some stage records are missing from production.")

if int(audit_row["production_missing_from_stage_count"]) != 0:
    raise RuntimeError("Some production records are absent from stage.")

manifest_result = client.query(
    f"""
    SELECT
        RUN_ID,
        ARCHIVE_YEAR,
        ARCHIVE_MONTH,
        STATUS,
        SOURCE_ROWS,
        BESS_ROWS,
        LOADED_ROWS,
        SOURCE_METHOD,
        COMPLETED_AT
    FROM `{MANIFEST_TABLE}`
    WHERE RUN_ID = @run_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "run_id",
                "STRING",
                RUN_ID_201911,
            )
        ]
    ),
).result().to_dataframe()

print("\nManifest entry:")
display(manifest_result)

if len(manifest_result) != 1:
    raise RuntimeError("Expected exactly one November 2019 manifest entry.")

if manifest_result.iloc[0]["STATUS"] != "PROMOTED":
    raise RuntimeError("Manifest status is not PROMOTED.")

if int(manifest_result.iloc[0]["LOADED_ROWS"]) != EXPECTED_ROWS:
    raise RuntimeError("Manifest loaded-row count is incorrect.")

print("\n✅ NOVEMBER 2019 PROMOTION AUDIT PASSED")

In [ ]:
# SECTION 31G — delete verified November 2019 temporary stage
# Deletes only the named temporary stage table.

STAGE_TABLE_201911 = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_201911"
)

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

RUN_ID_201911 = "bess-effective-direct-2019-11"
EXPECTED_ROWS = 432_000

# Final safety confirmation before deletion.
production_run_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    RUN_ID_201911,
                )
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

stage_row_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{STAGE_TABLE_201911}`
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"Production rows retained: {production_run_count:,}")
print(f"Temporary stage rows: {stage_row_count:,}")

if production_run_count != EXPECTED_ROWS:
    raise RuntimeError("Safety stop: production verification failed.")

if stage_row_count != EXPECTED_ROWS:
    raise RuntimeError("Safety stop: stage verification failed.")

client.delete_table(STAGE_TABLE_201911)

print("\n✅ November 2019 temporary stage deleted.")
print("✅ Production archive and manifest remain intact.")

In [ ]:
# SECTION 32A — map historical BESS bid coverage before backfill
# Read-only BigQuery analysis. No data is changed.

from google.cloud import bigquery
import pandas as pd

BIDDAYOFFER_TABLE = (
    "master-aemo.aemo_master_schema.BIDDAYOFFER_ARCHIVE"
)

ALIAS_TABLE = (
    "master-aemo.aemo_master_schema.BESS_DUID_ALIAS_CROSSWALK_V1"
)

# Read the approved historic source-DUID list.
alias_df = client.query(
    f"""
    SELECT
        UPPER(TRIM(SOURCE_DUID)) AS SOURCE_DUID,
        CANONICAL_BESS_DUID,
        SITE_NAME,
        REGION,
        ROW_ROLE
    FROM `{ALIAS_TABLE}`
    """
).result().to_dataframe()

historic_source_duids = alias_df["SOURCE_DUID"].dropna().unique().tolist()

print(f"Historic BESS source DUIDs to check: {len(historic_source_duids):,}")

coverage_sql = f"""
SELECT
    UPPER(TRIM(DUID)) AS SOURCE_DUID,
    DATE_TRUNC(DATE(SETTLEMENTDATE), MONTH) AS SETTLEMENT_MONTH,
    COUNT(*) AS BIDDAYOFFER_ROWS
FROM `{BIDDAYOFFER_TABLE}`
WHERE UPPER(TRIM(DUID)) IN UNNEST(@source_duids)
  AND DATE(SETTLEMENTDATE) < DATE '2024-08-01'
GROUP BY
    SOURCE_DUID,
    SETTLEMENT_MONTH
ORDER BY
    SOURCE_DUID,
    SETTLEMENT_MONTH
"""

coverage_job_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False,
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "source_duids",
            "STRING",
            historic_source_duids,
        )
    ],
)

dry_run = client.query(
    coverage_sql,
    job_config=coverage_job_config,
)

estimated_gb = dry_run.total_bytes_processed / (1024 ** 3)

print(f"Estimated data scanned: {estimated_gb:,.3f} GB")

MAX_ALLOWED_GB = 5

if estimated_gb > MAX_ALLOWED_GB:
    raise RuntimeError(
        f"Safety stop: estimate is {estimated_gb:,.2f} GB, above the "
        f"{MAX_ALLOWED_GB} GB limit. No query was executed."
    )

coverage_job_config.dry_run = False
coverage_job_config.maximum_bytes_billed = (
    MAX_ALLOWED_GB * (1024 ** 3)
)

bess_historical_coverage = client.query(
    coverage_sql,
    job_config=coverage_job_config,
).result().to_dataframe()

print(
    "Historic DUIDs with bid coverage:",
    f"{bess_historical_coverage['SOURCE_DUID'].nunique():,}",
)
print(
    "Historic months with BESS bid coverage:",
    f"{bess_historical_coverage['SETTLEMENT_MONTH'].nunique():,}",
)

coverage_summary = (
    bess_historical_coverage
    .groupby("SOURCE_DUID", dropna=False)
    .agg(
        first_bid_month=("SETTLEMENT_MONTH", "min"),
        last_bid_month=("SETTLEMENT_MONTH", "max"),
        months_with_bids=("SETTLEMENT_MONTH", "nunique"),
        biddayoffer_rows=("BIDDAYOFFER_ROWS", "sum"),
    )
    .reset_index()
    .merge(
        alias_df,
        on="SOURCE_DUID",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "first_bid_month",
            "SOURCE_DUID",
        ]
    )
    .reset_index(drop=True)
)

print("\nHistoric BESS source-DUID coverage:")
display(coverage_summary)

print("\nMonthly BESS bid coverage:")
display(
    bess_historical_coverage
    .sort_values(
        [
            "SETTLEMENT_MONTH",
            "SOURCE_DUID",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# SECTION 32B — continue: build and validate historic BESS direct-source plan
# Read-only. No BigQuery data is changed.

import time
import requests
import pandas as pd

month_scope = (
    bess_historical_coverage
    .merge(
        alias_df[
            [
                "SOURCE_DUID",
                "CANONICAL_BESS_DUID",
                "SITE_NAME",
                "REGION",
            ]
        ],
        on="SOURCE_DUID",
        how="left",
        validate="many_to_one",
    )
)

bess_historical_month_plan = (
    month_scope
    .groupby("SETTLEMENT_MONTH", as_index=False)
    .agg(
        active_source_duids=("SOURCE_DUID", "nunique"),
        active_canonical_assets=("CANONICAL_BESS_DUID", "nunique"),
        biddayoffer_rows=("BIDDAYOFFER_ROWS", "sum"),
    )
    .sort_values("SETTLEMENT_MONTH")
    .reset_index(drop=True)
)

# BigQuery returned this field as an object/date value; convert it explicitly.
bess_historical_month_plan["SETTLEMENT_MONTH"] = pd.to_datetime(
    bess_historical_month_plan["SETTLEMENT_MONTH"],
    errors="raise",
)

bess_historical_month_plan["ARCHIVE_YEAR"] = (
    bess_historical_month_plan["SETTLEMENT_MONTH"].dt.year
)

bess_historical_month_plan["ARCHIVE_MONTH"] = (
    bess_historical_month_plan["SETTLEMENT_MONTH"].dt.month
)

bess_historical_month_plan["SOURCE_FILE_NAME"] = (
    "PUBLIC_DVD_BIDPEROFFER_D_"
    + bess_historical_month_plan["SETTLEMENT_MONTH"].dt.strftime("%Y%m")
    + "010000.zip"
)

bess_historical_month_plan["SOURCE_URL"] = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    + "MMSDM/"
    + bess_historical_month_plan["ARCHIVE_YEAR"].astype(str)
    + "/MMSDM_"
    + bess_historical_month_plan["ARCHIVE_YEAR"].astype(str)
    + "_"
    + bess_historical_month_plan["ARCHIVE_MONTH"].astype(str).str.zfill(2)
    + "/MMSDM_Historical_Data_SQLLoader/DATA/"
    + bess_historical_month_plan["SOURCE_FILE_NAME"]
)

expected_months = 81

if len(bess_historical_month_plan) != expected_months:
    raise RuntimeError(
        f"Expected {expected_months} active historic BESS months; "
        f"found {len(bess_historical_month_plan)}."
    )

print(
    f"Historic direct-source plan: "
    f"{len(bess_historical_month_plan):,} monthly archives"
)
print(
    "First month:",
    bess_historical_month_plan["SETTLEMENT_MONTH"].min().date(),
)
print(
    "Last month:",
    bess_historical_month_plan["SETTLEMENT_MONTH"].max().date(),
)

def check_archive_url(url):
    """Request only response headers; do not download the ZIP."""
    response = None

    try:
        response = requests.get(
            url,
            timeout=90,
            stream=True,
            headers={
                "User-Agent": "Mozilla/5.0",
                "Range": "bytes=0-0",
            },
        )

        return {
            "HTTP_STATUS": response.status_code,
            "CONTENT_LENGTH": response.headers.get("Content-Length"),
            "ERROR": None,
        }

    except Exception as error:
        return {
            "HTTP_STATUS": None,
            "CONTENT_LENGTH": None,
            "ERROR": str(error),
        }

    finally:
        if response is not None:
            response.close()

availability_results = []

for index, plan_row in bess_historical_month_plan.iterrows():
    result = check_archive_url(plan_row["SOURCE_URL"])

    availability_results.append(
        {
            "SETTLEMENT_MONTH": plan_row["SETTLEMENT_MONTH"],
            "SOURCE_FILE_NAME": plan_row["SOURCE_FILE_NAME"],
            "HTTP_STATUS": result["HTTP_STATUS"],
            "CONTENT_LENGTH": result["CONTENT_LENGTH"],
            "ERROR": result["ERROR"],
        }
    )

    if (index + 1) % 10 == 0 or index + 1 == len(bess_historical_month_plan):
        print(
            f"Checked {index + 1:,} / "
            f"{len(bess_historical_month_plan):,} archive URLs"
        )

    time.sleep(0.15)

bess_historical_archive_availability = pd.DataFrame(
    availability_results
)

bess_historical_month_plan = bess_historical_month_plan.merge(
    bess_historical_archive_availability,
    on=["SETTLEMENT_MONTH", "SOURCE_FILE_NAME"],
    how="left",
    validate="one_to_one",
)

unavailable_archives = bess_historical_month_plan[
    bess_historical_month_plan["HTTP_STATUS"] != 200
].copy()

print("\nArchive availability summary:")
display(
    bess_historical_month_plan
    .groupby("HTTP_STATUS", dropna=False)
    .size()
    .reset_index(name="MONTH_COUNT")
)

if not unavailable_archives.empty:
    print("\nUnavailable archive URLs:")
    display(
        unavailable_archives[
            [
                "SETTLEMENT_MONTH",
                "SOURCE_FILE_NAME",
                "HTTP_STATUS",
                "ERROR",
                "SOURCE_URL",
            ]
        ]
    )

    raise RuntimeError(
        "Safety stop: at least one required historic source archive "
        "is unavailable. No backfill has started."
    )

print("\n✅ ALL 81 HISTORIC BESS BIDPEROFFER_D ARCHIVES ARE AVAILABLE")

display(
    bess_historical_month_plan[
        [
            "SETTLEMENT_MONTH",
            "active_source_duids",
            "active_canonical_assets",
            "SOURCE_FILE_NAME",
            "HTTP_STATUS",
        ]
    ]
)

In [ ]:
# SECTION 32C — classify available and unavailable historic direct-source eras
# Local analysis only. No data is changed.

bess_historical_month_plan["ARCHIVE_AVAILABLE"] = (
    bess_historical_month_plan["HTTP_STATUS"].isin([200, 206])
)

available_direct_months = bess_historical_month_plan[
    bess_historical_month_plan["ARCHIVE_AVAILABLE"]
].copy()

unavailable_direct_months = bess_historical_month_plan[
    ~bess_historical_month_plan["ARCHIVE_AVAILABLE"]
].copy()

print("DIRECT BIDPEROFFER_D AVAILABILITY")
print(
    f"Available months: {len(available_direct_months):,}"
)
print(
    f"Unavailable months: {len(unavailable_direct_months):,}"
)

availability_era_summary = pd.DataFrame(
    [
        {
            "SOURCE_ERA": "DIRECT_BIDPEROFFER_D_AVAILABLE",
            "MONTH_COUNT": len(available_direct_months),
            "FIRST_MONTH": available_direct_months[
                "SETTLEMENT_MONTH"
            ].min(),
            "LAST_MONTH": available_direct_months[
                "SETTLEMENT_MONTH"
            ].max(),
        },
        {
            "SOURCE_ERA": "DIRECT_BIDPEROFFER_D_UNAVAILABLE",
            "MONTH_COUNT": len(unavailable_direct_months),
            "FIRST_MONTH": unavailable_direct_months[
                "SETTLEMENT_MONTH"
            ].min(),
            "LAST_MONTH": unavailable_direct_months[
                "SETTLEMENT_MONTH"
            ].max(),
        },
    ]
)

display(availability_era_summary)

print("\nAvailable direct-source months:")
display(
    available_direct_months[
        [
            "SETTLEMENT_MONTH",
            "active_source_duids",
            "active_canonical_assets",
            "HTTP_STATUS",
            "SOURCE_FILE_NAME",
        ]
    ]
)

print("\nFirst 10 unavailable months:")
display(
    unavailable_direct_months[
        [
            "SETTLEMENT_MONTH",
            "active_source_duids",
            "active_canonical_assets",
            "HTTP_STATUS",
            "SOURCE_FILE_NAME",
        ]
    ].head(10)
)

print("\nLast 10 unavailable months:")
display(
    unavailable_direct_months[
        [
            "SETTLEMENT_MONTH",
            "active_source_duids",
            "active_canonical_assets",
            "HTTP_STATUS",
            "SOURCE_FILE_NAME",
        ]
    ].tail(10)
)

if available_direct_months.empty:
    raise RuntimeError(
        "No direct-source months were found. Do not start a backfill."
    )

print("\n✅ Direct-source availability classified.")
print(
    "We will batch only the available months after one final "
    "multi-month validation."
)

In [ ]:
# SECTION 32D — boundary validation for direct-loadable historic era
# Read-only. No BigQuery data is changed.

import csv
import io
import time
import zipfile
import requests
import pandas as pd

DIRECT_ERA_DIRECTION_MAP = {
    "HPRG1": "GEN",
    "HPRL1": "LOAD",
    "DALNTH01": "GEN",
    "DALNTHL1": "LOAD",
    "BALBG1": "GEN",
    "BALBL1": "LOAD",
    "GANNBG1": "GEN",
    "GANNBL1": "LOAD",
    "LBBG1": "GEN",
    "LBBL1": "LOAD",
}

EXPECTED_DIRECT_COLUMNS = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "INTERVAL_DATETIME",
    "MR_CAPACITY",
]

boundary_months = pd.concat(
    [
        available_direct_months.head(1),
        available_direct_months.tail(1),
    ],
    ignore_index=True,
)

def download_boundary_zip(url, label, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Downloading {label} — attempt {attempt}/{attempts}...")

            response = requests.get(
                url,
                timeout=600,
                headers={"User-Agent": "Mozilla/5.0"},
            )

            response.raise_for_status()
            return response.content

        except Exception as error:
            last_error = error

            if attempt < attempts:
                wait_seconds = min(60, 5 * (2 ** (attempt - 1)))
                print(f"⚠️ Download failed: {error}")
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

    raise RuntimeError(f"Could not download {label}: {last_error}")

boundary_results = []

for _, plan_row in boundary_months.iterrows():
    month_label = plan_row["SETTLEMENT_MONTH"].strftime("%Y-%m")

    zip_bytes = download_boundary_zip(
        plan_row["SOURCE_URL"],
        f"BIDPEROFFER_D {month_label}",
    )

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"{month_label}: expected one CSV; found {csv_names}"
            )

        with zip_file.open(csv_names[0]) as raw_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline="",
                )
            )

            columns = None
            total_rows = 0
            bess_rows = 0
            bess_duids = set()
            first_interval = None
            last_interval = None

            for record in reader:
                if not record:
                    continue

                if record[0] == "I" and "DUID" in record:
                    columns = record[4:]

                    if columns != EXPECTED_DIRECT_COLUMNS:
                        raise RuntimeError(
                            f"{month_label}: unexpected BIDPEROFFER_D schema.\n"
                            f"Found: {columns}"
                        )

                    continue

                if record[0] != "D" or columns is None:
                    continue

                values = record[4:]

                if len(values) != len(columns):
                    continue

                total_rows += 1
                row = dict(zip(columns, values))
                duid = row["DUID"].strip().upper()

                if duid not in DIRECT_ERA_DIRECTION_MAP:
                    continue

                bess_rows += 1
                bess_duids.add(duid)

                interval_text = row["INTERVAL_DATETIME"].strip()

                if first_interval is None or interval_text < first_interval:
                    first_interval = interval_text

                if last_interval is None or interval_text > last_interval:
                    last_interval = interval_text

    missing_direction_mappings = sorted(
        bess_duids - set(DIRECT_ERA_DIRECTION_MAP)
    )

    if missing_direction_mappings:
        raise RuntimeError(
            f"{month_label}: DUIDs need direction mapping: "
            f"{missing_direction_mappings}"
        )

    if bess_rows == 0:
        raise RuntimeError(
            f"{month_label}: no BESS records found in direct source."
        )

    boundary_results.append(
        {
            "SETTLEMENT_MONTH": plan_row["SETTLEMENT_MONTH"],
            "ALL_SOURCE_ROWS": total_rows,
            "BESS_ROWS": bess_rows,
            "BESS_DUID_COUNT": len(bess_duids),
            "BESS_DUIDS": ", ".join(sorted(bess_duids)),
            "FIRST_INTERVAL": first_interval,
            "LAST_INTERVAL": last_interval,
            "SCHEMA_MATCH": True,
        }
    )

boundary_validation = pd.DataFrame(boundary_results)

print("\n✅ DIRECT-ERA BOUNDARY VALIDATION COMPLETE")
display(boundary_validation)

In [ ]:
# SECTION 33 — batch backfill the proven direct-source BESS era
# Scope: Nov 2017 through Feb 2021.
# November 2019 is skipped because it is already promoted.
# Does NOT touch Mar 2021 through Jul 2024.

from google.cloud import bigquery
from google.api_core.exceptions import NotFound
import csv
import io
import time
import zipfile
import requests
import pandas as pd

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

DIRECT_ERA_DIRECTION_MAP = {
    "HPRG1": "GEN",
    "HPRL1": "LOAD",
    "DALNTH01": "GEN",
    "DALNTHL1": "LOAD",
    "BALBG1": "GEN",
    "BALBL1": "LOAD",
    "GANNBG1": "GEN",
    "GANNBL1": "LOAD",
    "LBBG1": "GEN",
    "LBBL1": "LOAD",
}

EXPECTED_DIRECT_COLUMNS = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "INTERVAL_DATETIME",
    "MR_CAPACITY",
]

TARGET_COLUMNS = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
    "_LOAD_DATE",
    "_SOURCE",
    "_PIPELINE",
    "_SOURCE_FILE",
    "_SOURCE_URL",
    "_RUN_ID",
]

TIMESTAMP_COLUMNS = [
    "SETTLEMENTDATE",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "LASTCHANGED",
]

NUMERIC_COLUMNS = [
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
]

# Use only the 40 archive URLs already verified as available.
direct_batch_plan = (
    available_direct_months
    .sort_values("SETTLEMENT_MONTH")
    .reset_index(drop=True)
    .copy()
)

if len(direct_batch_plan) != 40:
    raise RuntimeError(
        f"Expected 40 direct-era months; found {len(direct_batch_plan)}."
    )

target_schema = client.get_table(TARGET_TABLE).schema

def download_source_zip(url, month_label, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(
                f"Downloading {month_label} "
                f"— attempt {attempt}/{attempts}..."
            )

            response = requests.get(
                url,
                timeout=600,
                headers={"User-Agent": "Mozilla/5.0"},
            )

            response.raise_for_status()

            return response.content

        except Exception as error:
            last_error = error

            if attempt < attempts:
                wait_seconds = min(60, 5 * (2 ** (attempt - 1)))
                print(f"⚠️ Download failed: {error}")
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"{month_label}: download failed after {attempts} attempts: "
        f"{last_error}"
    )

def parse_direct_bess_source(zip_bytes, month_label):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"{month_label}: expected one CSV; found {csv_names}"
            )

        csv_name = csv_names[0]

        with zip_file.open(csv_name) as raw_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    errors="replace",
                    newline="",
                )
            )

            columns = None
            all_source_rows = 0
            bess_records = []

            for record in reader:
                if not record:
                    continue

                if record[0] == "I" and "DUID" in record:
                    columns = record[4:]

                    if columns != EXPECTED_DIRECT_COLUMNS:
                        raise RuntimeError(
                            f"{month_label}: unexpected source schema.\n"
                            f"Found columns: {columns}"
                        )

                    continue

                if record[0] != "D" or columns is None:
                    continue

                values = record[4:]

                if len(values) != len(columns):
                    continue

                all_source_rows += 1

                row = dict(zip(columns, values))
                row["DUID"] = row["DUID"].strip().upper()

                if row["DUID"] not in DIRECT_ERA_DIRECTION_MAP:
                    continue

                row["DIRECTION"] = DIRECT_ERA_DIRECTION_MAP[row["DUID"]]
                bess_records.append(row)

    if columns is None:
        raise RuntimeError(
            f"{month_label}: no AEMO column-definition record found."
        )

    if not bess_records:
        raise RuntimeError(
            f"{month_label}: direct source contains zero BESS rows."
        )

    return pd.DataFrame(bess_records), all_source_rows, csv_name

def prepare_direct_stage(
    source_df,
    month_label,
    source_file_name,
    source_url,
):
    prepared = source_df.copy()

    prepared["ENERGYLIMIT"] = pd.NA

    for column in TIMESTAMP_COLUMNS:
        prepared[column] = pd.to_datetime(
            prepared[column],
            errors="coerce",
        )

    for column in NUMERIC_COLUMNS:
        prepared[column] = pd.to_numeric(
            prepared[column],
            errors="coerce",
        )

    prepared["PERIODID"] = pd.to_numeric(
        prepared["PERIODID"],
        errors="coerce",
    ).astype("Int64")

    critical_columns = [
        "SETTLEMENTDATE",
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "PERIODID",
    ]

    null_counts = {
        column: int(prepared[column].isna().sum())
        for column in critical_columns
        if prepared[column].isna().any()
    }

    if null_counts:
        raise RuntimeError(
            f"{month_label}: critical nulls found: {null_counts}"
        )

    duplicate_count = int(
        prepared.duplicated(
            ["INTERVAL_DATETIME", "DUID", "BIDTYPE"]
        ).sum()
    )

    if duplicate_count != 0:
        raise RuntimeError(
            f"{month_label}: found {duplicate_count:,} duplicate "
            "effective-offer keys."
        )

    prepared["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
    prepared["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    prepared["_PIPELINE"] = (
        "BIDPEROFFER_BESS_EFFECTIVE_HISTORICAL_DIRECT"
    )
    prepared["_SOURCE_FILE"] = source_file_name
    prepared["_SOURCE_URL"] = source_url
    prepared["_RUN_ID"] = (
        f"bess-effective-direct-{month_label}"
    )

    return prepared[TARGET_COLUMNS].copy()

batch_results = []

for _, plan_row in direct_batch_plan.iterrows():
    month_ts = pd.Timestamp(plan_row["SETTLEMENT_MONTH"])
    month_label = month_ts.strftime("%Y-%m")
    month_compact = month_ts.strftime("%Y%m")

    run_id = f"bess-effective-direct-{month_label}"

    stage_table = (
        "master-aemo.aemo_master_schema."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_compact}"
    )

    print("\n" + "=" * 100)
    print(f"STARTING {month_label}")
    print("=" * 100)

    # Resume-safe: skip only a verified promoted manifest entry.
    manifest_check = client.query(
        f"""
        SELECT
            STATUS,
            LOADED_ROWS
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    run_id,
                )
            ]
        ),
    ).result().to_dataframe()

    if not manifest_check.empty:
        status = manifest_check.iloc[0]["STATUS"]
        loaded_rows = int(manifest_check.iloc[0]["LOADED_ROWS"])

        if status == "PROMOTED" and loaded_rows > 0:
            print(
                f"⏭️ Skipping {month_label}: already promoted "
                f"({loaded_rows:,} rows)."
            )

            batch_results.append(
                {
                    "MONTH": month_label,
                    "STATUS": "SKIPPED_ALREADY_PROMOTED",
                    "SOURCE_ROWS": pd.NA,
                    "BESS_ROWS": loaded_rows,
                    "LOADED_ROWS": loaded_rows,
                }
            )
            continue

        raise RuntimeError(
            f"{month_label}: manifest already exists with status {status}. "
            "Stopping safely."
        )

    # Do not overwrite a stage table left by an interrupted earlier run.
    try:
        old_stage = client.get_table(stage_table)

        if old_stage.num_rows > 0:
            raise RuntimeError(
                f"{month_label}: non-empty stage already exists "
                f"({old_stage.num_rows:,} rows). Stopping safely."
            )

    except NotFound:
        pass

    zip_bytes = download_source_zip(
        plan_row["SOURCE_URL"],
        month_label,
    )

    source_df, source_rows, detected_source_file = parse_direct_bess_source(
        zip_bytes,
        month_label,
    )

    stage_df = prepare_direct_stage(
        source_df=source_df,
        month_label=month_label,
        source_file_name=detected_source_file,
        source_url=plan_row["SOURCE_URL"],
    )

    expected_bess_rows = len(stage_df)

    print(
        f"{month_label}: source rows {source_rows:,} | "
        f"BESS rows {expected_bess_rows:,}"
    )

    # Create and verify the month-specific temporary stage.
    load_job = client.load_table_from_dataframe(
        stage_df,
        stage_table,
        job_config=bigquery.LoadJobConfig(
            schema=target_schema,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            time_partitioning=bigquery.TimePartitioning(
                field="SETTLEMENTDATE"
            ),
            clustering_fields=[
                "DUID",
                "BIDTYPE",
                "DIRECTION",
            ],
        ),
    )

    load_job.result()

    stage_count = int(
        client.query(
            f"""
            SELECT COUNT(*) AS row_count
            FROM `{stage_table}`
            """
        ).result().to_dataframe().iloc[0]["row_count"]
    )

    if stage_count != expected_bess_rows:
        raise RuntimeError(
            f"{month_label}: stage mismatch; expected "
            f"{expected_bess_rows:,}, found {stage_count:,}."
        )

    # Atomic promotion and manifest registration.
    promotion_sql = f"""
    BEGIN TRANSACTION;

    INSERT INTO `{TARGET_TABLE}`
    SELECT *
    FROM `{stage_table}`;

    INSERT INTO `{MANIFEST_TABLE}` (
        RUN_ID,
        ARCHIVE_YEAR,
        ARCHIVE_MONTH,
        SOURCE_FILE_NAME,
        SOURCE_URL,
        SOURCE_METHOD,
        STATUS,
        SOURCE_ROWS,
        BESS_ROWS,
        LOADED_ROWS,
        STARTED_AT,
        COMPLETED_AT,
        ERROR_MESSAGE,
        CREATED_AT
    )
    VALUES (
        @run_id,
        @archive_year,
        @archive_month,
        @source_file_name,
        @source_url,
        "BIDPEROFFER_D_DIRECT_APPLIED",
        "PROMOTED",
        @source_rows,
        @bess_rows,
        @loaded_rows,
        CURRENT_TIMESTAMP(),
        CURRENT_TIMESTAMP(),
        NULL,
        CURRENT_TIMESTAMP()
    );

    COMMIT TRANSACTION;
    """

    client.query(
        promotion_sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    run_id,
                ),
                bigquery.ScalarQueryParameter(
                    "archive_year",
                    "INT64",
                    int(month_ts.year),
                ),
                bigquery.ScalarQueryParameter(
                    "archive_month",
                    "INT64",
                    int(month_ts.month),
                ),
                bigquery.ScalarQueryParameter(
                    "source_file_name",
                    "STRING",
                    detected_source_file,
                ),
                bigquery.ScalarQueryParameter(
                    "source_url",
                    "STRING",
                    plan_row["SOURCE_URL"],
                ),
                bigquery.ScalarQueryParameter(
                    "source_rows",
                    "INT64",
                    int(source_rows),
                ),
                bigquery.ScalarQueryParameter(
                    "bess_rows",
                    "INT64",
                    int(expected_bess_rows),
                ),
                bigquery.ScalarQueryParameter(
                    "loaded_rows",
                    "INT64",
                    int(expected_bess_rows),
                ),
            ]
        ),
    ).result()

    # Post-transaction verification.
    promoted_count = int(
        client.query(
            f"""
            SELECT COUNT(*) AS row_count
            FROM `{TARGET_TABLE}`
            WHERE _RUN_ID = @run_id
            """,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[
                    bigquery.ScalarQueryParameter(
                        "run_id",
                        "STRING",
                        run_id,
                    )
                ]
            ),
        ).result().to_dataframe().iloc[0]["row_count"]
    )

    if promoted_count != expected_bess_rows:
        raise RuntimeError(
            f"{month_label}: promotion mismatch; expected "
            f"{expected_bess_rows:,}, found {promoted_count:,}."
        )

    client.delete_table(stage_table)

    print(
        f"✅ {month_label} promoted: {promoted_count:,} rows. "
        "Temporary stage deleted."
    )

    batch_results.append(
        {
            "MONTH": month_label,
            "STATUS": "PROMOTED",
            "SOURCE_ROWS": source_rows,
            "BESS_ROWS": expected_bess_rows,
            "LOADED_ROWS": promoted_count,
        }
    )

    del zip_bytes, source_df, stage_df
    time.sleep(1)

direct_era_batch_results = pd.DataFrame(batch_results)

print("\n" + "=" * 100)
print("DIRECT-ERA BATCH COMPLETE")
print("=" * 100)

display(
    direct_era_batch_results
    .groupby("STATUS", dropna=False)
    .agg(
        months=("MONTH", "count"),
        loaded_rows=("LOADED_ROWS", "sum"),
    )
    .reset_index()
)

display(direct_era_batch_results)

In [ ]:
# =============================================================================
# SECTION 33C — FINAL AUDIT: HISTORIC DIRECT BIDPEROFFER_D ERA
# Read-only. Confirms November 2017 through February 2021 is complete.
# =============================================================================

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

MANIFEST_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

EXPECTED_MONTHS = 40
EXPECTED_ROWS = 12_676_128

direct_month_filter = """
(
    (ARCHIVE_YEAR = 2017 AND ARCHIVE_MONTH >= 11)
    OR ARCHIVE_YEAR IN (2018, 2019, 2020)
    OR (ARCHIVE_YEAR = 2021 AND ARCHIVE_MONTH <= 2)
)
"""

manifest_sql = f"""
SELECT
    ARCHIVE_YEAR,
    ARCHIVE_MONTH,
    RUN_ID,
    STATUS,
    SOURCE_ROWS,
    BESS_ROWS,
    LOADED_ROWS,
    SOURCE_METHOD,
    COMPLETED_AT
FROM `{MANIFEST_TABLE}`
WHERE {direct_month_filter}
  AND SOURCE_METHOD = 'BIDPEROFFER_D_DIRECT_APPLIED'
ORDER BY ARCHIVE_YEAR, ARCHIVE_MONTH
"""

manifest_df = client.query(manifest_sql).result().to_dataframe()

print("DIRECT-ERA MANIFEST AUDIT")
print("Manifest months found:", len(manifest_df))
print("Manifest rows loaded:", f"{int(manifest_df['LOADED_ROWS'].sum()):,}")
print("Manifest statuses:", manifest_df["STATUS"].value_counts().to_dict())

display(manifest_df)

if len(manifest_df) != EXPECTED_MONTHS:
    raise RuntimeError(
        f"Expected {EXPECTED_MONTHS} direct-era manifest months, "
        f"found {len(manifest_df)}."
    )

if not (manifest_df["STATUS"] == "PROMOTED").all():
    raise RuntimeError(
        "One or more direct-era manifest entries are not PROMOTED."
    )

manifest_rows = int(manifest_df["LOADED_ROWS"].sum())

if manifest_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_ROWS:,} manifest-loaded rows, "
        f"found {manifest_rows:,}."
    )

production_sql = f"""
WITH direct_manifest AS (
    SELECT RUN_ID
    FROM `{MANIFEST_TABLE}`
    WHERE {direct_month_filter}
      AND SOURCE_METHOD = 'BIDPEROFFER_D_DIRECT_APPLIED'
      AND STATUS = 'PROMOTED'
)
SELECT
    COUNT(*) AS production_row_count,
    COUNT(DISTINCT p._RUN_ID) AS production_run_count,
    COUNT(DISTINCT p.DUID) AS source_duid_count,
    MIN(p.INTERVAL_DATETIME) AS first_interval,
    MAX(p.INTERVAL_DATETIME) AS last_interval
FROM `{TARGET_TABLE}` AS p
INNER JOIN direct_manifest AS m
    ON p._RUN_ID = m.RUN_ID
"""

production_df = client.query(production_sql).result().to_dataframe()
display(production_df)

production_rows = int(production_df.iloc[0]["production_row_count"])
production_runs = int(production_df.iloc[0]["production_run_count"])

if production_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_ROWS:,} production rows for the direct era, "
        f"found {production_rows:,}."
    )

if production_runs != EXPECTED_MONTHS:
    raise RuntimeError(
        f"Expected {EXPECTED_MONTHS} production run IDs, "
        f"found {production_runs}."
    )

stage_sql = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
WHERE table_name LIKE 'BIDPEROFFER_BESS_EFFECTIVE_STAGE_%'
ORDER BY table_name
"""

stage_df = client.query(stage_sql).result().to_dataframe()

print("Remaining effective-offer stage tables:", len(stage_df))
if len(stage_df) > 0:
    display(stage_df)
    raise RuntimeError(
        "Temporary effective-offer stage table(s) remain. "
        "Do not continue until they are reviewed."
    )

print()
print("✅ HISTORIC DIRECT BIDPEROFFER_D ERA AUDIT PASSED")
print(f"✅ Months promoted: {EXPECTED_MONTHS}")
print(f"✅ BESS effective-offer rows retained: {EXPECTED_ROWS:,}")
print("✅ Coverage: November 2017 through February 2021")
print("✅ No temporary effective-offer stage tables remain")

In [ ]:
# =============================================================================
# SECTION 34A — MARCH 2021 BID SOURCE AVAILABILITY CHECK
# Read-only: tests AEMO file availability only.
# =============================================================================

import pandas as pd
import requests

YEAR = 2021
MONTH = 3
MONTH_TOKEN = f"{YEAR}{MONTH:02d}010000"

AEMO_MONTH_BASE = (
    f"https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    f"MMSDM/{YEAR}/MMSDM_{YEAR}_{MONTH:02d}/"
    f"MMSDM_Historical_Data_SQLLoader/DATA/"
)

candidate_files = [
    f"PUBLIC_DVD_BIDPEROFFER_{MONTH_TOKEN}.zip",
    f"PUBLIC_DVD_BIDDAYOFFER_{MONTH_TOKEN}.zip",
    f"PUBLIC_DVD_BIDPEROFFER_D_{MONTH_TOKEN}.zip",
    f"PUBLIC_DVD_BIDDAYOFFER_D_{MONTH_TOKEN}.zip",
]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "*/*",
})

results = []

for source_file in candidate_files:
    source_url = AEMO_MONTH_BASE + source_file

    try:
        # stream=True gets the HTTP result without downloading the large ZIP.
        response = session.get(
            source_url,
            stream=True,
            timeout=120,
            allow_redirects=True,
        )

        results.append({
            "SOURCE_FILE": source_file,
            "HTTP_STATUS": response.status_code,
            "AVAILABLE": response.status_code in (200, 206),
            "CONTENT_LENGTH": response.headers.get("Content-Length"),
            "SOURCE_URL": source_url,
        })

        response.close()

    except Exception as error:
        results.append({
            "SOURCE_FILE": source_file,
            "HTTP_STATUS": "REQUEST_ERROR",
            "AVAILABLE": False,
            "CONTENT_LENGTH": None,
            "SOURCE_URL": source_url,
            "ERROR": str(error),
        })

availability_df = pd.DataFrame(results)

print("MARCH 2021 BID-SOURCE AVAILABILITY")
display(availability_df)

print()
print("Interpretation:")
print("• BIDPEROFFER / BIDDAYOFFER = ordinary offer and rebid history.")
print("• BIDPEROFFER_D / BIDDAYOFFER_D = effective/applied offer history.")
print("• This cell has not downloaded, staged, or written any data.")

In [ ]:
# =============================================================================
# SECTION 34B — INSPECT MARCH 2021 ORDINARY BIDPEROFFER FOR BESS
# Read-only. No BigQuery tables are created or changed.
# =============================================================================

import csv
import io
import os
import time
import zipfile
import tempfile
from collections import Counter

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

BESS_SCOPE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_BACKFILL_SCOPE_V1"
)

BESS_ALIAS_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_ALIAS_CROSSWALK_V1"
)

SOURCE_FILE = "PUBLIC_DVD_BIDPEROFFER_202103010000.zip"
SOURCE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + SOURCE_FILE
)

# Get every current and historical BESS source DUID we have identified.
bess_scope_sql = f"""
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `{BESS_SCOPE_TABLE}`
WHERE DUID IS NOT NULL

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `{BESS_ALIAS_TABLE}`
WHERE SOURCE_DUID IS NOT NULL
"""

bess_scope_df = client.query(bess_scope_sql).result().to_dataframe()
BESS_DUIDS = set(bess_scope_df["DUID"].dropna().astype(str).str.upper().str.strip())

print("BESS source DUIDs in inspection scope:", len(BESS_DUIDS))
print("Source:", SOURCE_FILE)
print("This is an inspection only — no BigQuery writes will occur.")

temp_zip_path = None

try:
    # Download safely to a temporary local ZIP, with retries.
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_bpo_202103_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    download_complete = False

    for attempt in range(1, 6):
        try:
            print(f"Downloading BIDPEROFFER — attempt {attempt}/5...")

            response = requests.get(
                SOURCE_URL,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            expected_bytes = int(response.headers.get("Content-Length", 0))
            downloaded_bytes = 0
            next_progress_mark = 25

            with open(temp_zip_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if not chunk:
                        continue

                    output_file.write(chunk)
                    downloaded_bytes += len(chunk)

                    if expected_bytes > 0:
                        progress = downloaded_bytes / expected_bytes * 100

                        if progress >= next_progress_mark:
                            print(
                                f"Download progress: {progress:.0f}% "
                                f"({downloaded_bytes / 1024 / 1024:.1f} MB)"
                            )
                            next_progress_mark += 25

            response.close()

            if expected_bytes > 0 and downloaded_bytes != expected_bytes:
                raise RuntimeError(
                    f"Incomplete download: got {downloaded_bytes:,} bytes; "
                    f"expected {expected_bytes:,}."
                )

            download_complete = True
            break

        except Exception as error:
            print(f"⚠️ Download attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                raise

            wait_seconds = 5 * attempt
            print(f"Retrying in {wait_seconds} seconds...")
            time.sleep(wait_seconds)

    if not download_complete:
        raise RuntimeError("March 2021 BIDPEROFFER download did not complete.")

    all_source_rows = 0
    bess_rows = 0
    source_columns = None
    duid_index = None
    periodid_index = None
    settlementdate_index = None
    bidofferdate_index = None

    period_ids = set()
    bess_duid_counts = Counter()
    bess_samples = []

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_names = [
            name
            for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"Expected exactly one CSV in {SOURCE_FILE}; found {csv_names}"
            )

        csv_name = csv_names[0]
        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            text_csv = io.TextIOWrapper(
                binary_csv,
                encoding="utf-8-sig",
                newline="",
            )
            reader = csv.reader(text_csv)

            for row in reader:
                if not row:
                    continue

                # AEMO's I record defines the CSV columns after the first 4 fields.
                if (
                    row[0] == "I"
                    and len(row) >= 5
                    and len(row) > 2
                    and row[2].strip().upper() == "BIDPEROFFER"
                ):
                    source_columns = [value.strip() for value in row[4:]]

                    if "DUID" not in source_columns:
                        raise RuntimeError("BIDPEROFFER header has no DUID column.")

                    duid_index = source_columns.index("DUID")
                    periodid_index = (
                        source_columns.index("PERIODID")
                        if "PERIODID" in source_columns
                        else None
                    )
                    settlementdate_index = (
                        source_columns.index("SETTLEMENTDATE")
                        if "SETTLEMENTDATE" in source_columns
                        else None
                    )
                    bidofferdate_index = (
                        source_columns.index("OFFERDATE")
                        if "OFFERDATE" in source_columns
                        else None
                    )

                    continue

                # D records contain data after the first 4 AEMO metadata fields.
                if (
                    row[0] != "D"
                    or len(row) < 5
                    or len(row) < 3
                    or row[2].strip().upper() != "BIDPEROFFER"
                ):
                    continue

                if source_columns is None:
                    raise RuntimeError(
                        "Encountered BIDPEROFFER data before its column-definition record."
                    )

                values = row[4:]

                if len(values) != len(source_columns):
                    raise RuntimeError(
                        f"Unexpected column count: got {len(values)}, "
                        f"expected {len(source_columns)}."
                    )

                all_source_rows += 1

                duid = values[duid_index].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                bess_rows += 1
                bess_duid_counts[duid] += 1

                if periodid_index is not None:
                    period_ids.add(values[periodid_index].strip())

                if len(bess_samples) < 20:
                    sample = {
                        column: values[index]
                        for index, column in enumerate(source_columns)
                    }
                    bess_samples.append(sample)

    print()
    print("✅ MARCH 2021 ORDINARY BIDPEROFFER INSPECTION COMPLETE")
    print("Source columns:", source_columns)
    print("All source rows scanned:", f"{all_source_rows:,}")
    print("BESS rows found:", f"{bess_rows:,}")
    print("BESS DUIDs found:", len(bess_duid_counts))
    print("BESS DUID list:", sorted(bess_duid_counts))

    if period_ids:
        numeric_period_ids = sorted(
            int(value)
            for value in period_ids
            if value.strip().isdigit()
        )
        print(
            "PERIODID range:",
            min(numeric_period_ids),
            "to",
            max(numeric_period_ids),
        )

    duid_counts_df = (
        pd.DataFrame(
            [
                {"DUID": duid, "ROW_COUNT": row_count}
                for duid, row_count in bess_duid_counts.items()
            ]
        )
        .sort_values(["DUID"])
        .reset_index(drop=True)
    )

    print()
    print("BESS rows by DUID:")
    display(duid_counts_df)

    print()
    print("BESS sample records:")
    display(pd.DataFrame(bess_samples))

    if bess_rows == 0:
        raise RuntimeError(
            "No BESS rows were found. Do not design a Group 2 pipeline from "
            "this source until the BESS scope is reviewed."
        )

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print()
        print("✅ Temporary March 2021 ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34B2 — CORRECTED MARCH 2021 BIDPEROFFER BESS INSPECTION
# Read-only. No BigQuery tables are created or changed.
# =============================================================================

import csv
import io
import os
import time
import zipfile
import tempfile
from collections import Counter

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

BESS_SCOPE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_BACKFILL_SCOPE_V1"
)

BESS_ALIAS_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_ALIAS_CROSSWALK_V1"
)

SOURCE_FILE = "PUBLIC_DVD_BIDPEROFFER_202103010000.zip"
SOURCE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + SOURCE_FILE
)

bess_scope_sql = f"""
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `{BESS_SCOPE_TABLE}`
WHERE DUID IS NOT NULL

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `{BESS_ALIAS_TABLE}`
WHERE SOURCE_DUID IS NOT NULL
"""

bess_scope_df = client.query(bess_scope_sql).result().to_dataframe()
BESS_DUIDS = set(
    bess_scope_df["DUID"].dropna().astype(str).str.upper().str.strip()
)

print("BESS source DUIDs in inspection scope:", len(BESS_DUIDS))
print("Source:", SOURCE_FILE)
print("This is an inspection only — no BigQuery writes will occur.")

temp_zip_path = None

try:
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_bpo_202103_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading BIDPEROFFER — attempt {attempt}/5...")

            response = requests.get(
                SOURCE_URL,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            expected_bytes = int(response.headers.get("Content-Length", 0))
            downloaded_bytes = 0
            next_progress_mark = 25

            with open(temp_zip_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if not chunk:
                        continue

                    output_file.write(chunk)
                    downloaded_bytes += len(chunk)

                    if expected_bytes > 0:
                        progress = downloaded_bytes / expected_bytes * 100

                        if progress >= next_progress_mark:
                            print(
                                f"Download progress: {progress:.0f}% "
                                f"({downloaded_bytes / 1024 / 1024:.1f} MB)"
                            )
                            next_progress_mark += 25

            response.close()

            if expected_bytes > 0 and downloaded_bytes != expected_bytes:
                raise RuntimeError(
                    f"Incomplete download: got {downloaded_bytes:,} bytes; "
                    f"expected {expected_bytes:,}."
                )

            break

        except Exception as error:
            print(f"⚠️ Download attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                raise

            time.sleep(5 * attempt)

    all_source_rows = 0
    bess_rows = 0
    source_columns = None
    header_start_index = None
    duid_index = None
    periodid_index = None

    opening_records = []
    bess_duid_counts = Counter()
    bess_samples = []
    period_ids = set()

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"Expected one CSV in ZIP; found: {csv_names}"
            )

        csv_name = csv_names[0]
        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            text_csv = io.TextIOWrapper(
                binary_csv,
                encoding="utf-8-sig",
                newline="",
            )
            reader = csv.reader(text_csv)

            for record_number, row in enumerate(reader, start=1):
                if not row:
                    continue

                if len(opening_records) < 8:
                    opening_records.append({
                        "RECORD_NUMBER": record_number,
                        "RAW_RECORD": row,
                    })

                # Find the header dynamically: the header is the I-record
                # containing the DUID column. This works across AEMO layouts.
                if row[0].strip().upper() == "I" and "DUID" in row:
                    header_start_index = row.index("DUID")
                    source_columns = [
                        value.strip()
                        for value in row[header_start_index:]
                    ]

                    duid_index = source_columns.index("DUID")

                    if "PERIODID" in source_columns:
                        periodid_index = source_columns.index("PERIODID")

                    continue

                if row[0].strip().upper() != "D":
                    continue

                if source_columns is None:
                    continue

                values = row[header_start_index:]

                if len(values) != len(source_columns):
                    continue

                all_source_rows += 1

                duid = values[duid_index].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                bess_rows += 1
                bess_duid_counts[duid] += 1

                if periodid_index is not None:
                    period_value = values[periodid_index].strip()

                    if period_value.isdigit():
                        period_ids.add(int(period_value))

                if len(bess_samples) < 20:
                    bess_samples.append({
                        column: values[index]
                        for index, column in enumerate(source_columns)
                    })

    print()
    print("First eight raw CSV records:")
    display(pd.DataFrame(opening_records))

    if source_columns is None:
        raise RuntimeError(
            "Could not locate a BIDPEROFFER header containing DUID. "
            "The opening records above show the exact source layout."
        )

    print()
    print("✅ MARCH 2021 ORDINARY BIDPEROFFER INSPECTION COMPLETE")
    print("Source columns:", source_columns)
    print("All source rows scanned:", f"{all_source_rows:,}")
    print("BESS rows found:", f"{bess_rows:,}")
    print("BESS DUIDs found:", len(bess_duid_counts))
    print("BESS DUID list:", sorted(bess_duid_counts))

    if period_ids:
        print("PERIODID range:", min(period_ids), "to", max(period_ids))
        print("Distinct PERIODID values:", len(period_ids))

    if bess_duid_counts:
        duid_counts_df = (
            pd.DataFrame([
                {"DUID": duid, "ROW_COUNT": row_count}
                for duid, row_count in bess_duid_counts.items()
            ])
            .sort_values("DUID")
            .reset_index(drop=True)
        )

        print()
        print("BESS rows by DUID:")
        display(duid_counts_df)

    print()
    print("BESS sample records:")
    display(pd.DataFrame(bess_samples))

    if bess_rows == 0:
        raise RuntimeError(
            "The file parsed successfully but contains no BESS rows from our "
            "current historical scope. No loading decision has been made."
        )

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print()
        print("✅ Temporary March 2021 ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34C — INSPECT MARCH 2021 DISPATCHOFFERTRK FOR BESS
# Read-only. No BigQuery changes.
# =============================================================================

import csv
import io
import os
import time
import zipfile
import tempfile
from collections import Counter

SOURCE_FILE = "PUBLIC_DVD_DISPATCHOFFERTRK_202103010000.zip"
SOURCE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + SOURCE_FILE
)

print("Source:", SOURCE_FILE)
print("This is an inspection only — no BigQuery writes will occur.")

temp_zip_path = None

try:
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_tracker_202103_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading DISPATCHOFFERTRK — attempt {attempt}/5...")

            response = requests.get(
                SOURCE_URL,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_zip_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            break

        except Exception as error:
            print(f"⚠️ Download attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                raise

            time.sleep(5 * attempt)

    source_columns = None
    header_start_index = None
    duid_index = None

    all_source_rows = 0
    bess_rows = 0
    bess_duid_counts = Counter()
    opening_records = []
    bess_samples = []

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            for record_number, row in enumerate(reader, start=1):
                if not row:
                    continue

                if len(opening_records) < 8:
                    opening_records.append({
                        "RECORD_NUMBER": record_number,
                        "RAW_RECORD": row,
                    })

                if row[0].strip().upper() == "I" and "DUID" in row:
                    header_start_index = row.index("DUID")
                    source_columns = [
                        value.strip()
                        for value in row[header_start_index:]
                    ]
                    duid_index = source_columns.index("DUID")
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or source_columns is None
                ):
                    continue

                values = row[header_start_index:]

                if len(values) != len(source_columns):
                    continue

                all_source_rows += 1
                duid = values[duid_index].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                bess_rows += 1
                bess_duid_counts[duid] += 1

                if len(bess_samples) < 20:
                    bess_samples.append({
                        column: values[index]
                        for index, column in enumerate(source_columns)
                    })

    print()
    print("First eight raw CSV records:")
    display(pd.DataFrame(opening_records))

    print()
    print("✅ MARCH 2021 DISPATCHOFFERTRK INSPECTION COMPLETE")
    print("Source columns:", source_columns)
    print("All source rows scanned:", f"{all_source_rows:,}")
    print("BESS tracker rows:", f"{bess_rows:,}")
    print("BESS DUIDs found:", len(bess_duid_counts))
    print("BESS DUID list:", sorted(bess_duid_counts))

    display(
        pd.DataFrame([
            {"DUID": duid, "ROW_COUNT": row_count}
            for duid, row_count in bess_duid_counts.items()
        ]).sort_values("DUID").reset_index(drop=True)
    )

    print("BESS tracker samples:")
    display(pd.DataFrame(bess_samples))

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print()
        print("✅ Temporary March 2021 tracker ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34D — MARCH 2021 RAW-OFFER + TRACKER JOIN AUDIT
# Read-only. Tests whether raw BIDPEROFFER can reproduce applied offers.
# =============================================================================

import csv
import io
import os
import time
import zipfile
import tempfile
from collections import Counter, defaultdict
from datetime import datetime, timedelta

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

BESS_SCOPE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_BACKFILL_SCOPE_V1"
)
BESS_ALIAS_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_ALIAS_CROSSWALK_V1"
)

BASE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202103010000.zip"
TRACKER_FILE = "PUBLIC_DVD_DISPATCHOFFERTRK_202103010000.zip"

bess_scope_sql = f"""
SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `{BESS_SCOPE_TABLE}`
WHERE DUID IS NOT NULL

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `{BESS_ALIAS_TABLE}`
WHERE SOURCE_DUID IS NOT NULL
"""

BESS_DUIDS = set(
    client.query(bess_scope_sql)
    .result()
    .to_dataframe()["DUID"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
)

print("BESS source DUIDs in scope:", len(BESS_DUIDS))
print("Read-only join audit — no BigQuery writes.")

def download_aemo_zip(source_file):
    source_url = BASE_URL + source_file

    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_join_audit_",
        delete=False,
    ) as temp_file:
        temp_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading {source_file} — attempt {attempt}/5...")

            response = requests.get(
                source_url,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            return temp_path

        except Exception as error:
            print(f"⚠️ Attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                if os.path.exists(temp_path):
                    os.remove(temp_path)
                raise

            time.sleep(5 * attempt)


def market_day_and_period(settlement_text):
    """
    Convert a five-minute tracker timestamp into the BIDPEROFFER
    TRADINGDATE and PERIODID convention.

    NEM trading days run from 04:05 through 04:00 the following day.
    """
    settlement_ts = datetime.strptime(
        settlement_text.strip(),
        "%Y/%m/%d %H:%M:%S",
    )

    current_time = settlement_ts.time()

    if current_time <= datetime.strptime("04:00:00", "%H:%M:%S").time():
        market_day = settlement_ts.date() - timedelta(days=1)
    else:
        market_day = settlement_ts.date()

    market_day_start = datetime.combine(
        market_day,
        datetime.strptime("04:00:00", "%H:%M:%S").time(),
    )

    period_id = int(
        (settlement_ts - market_day_start).total_seconds() // 300
    )

    if not 1 <= period_id <= 288:
        raise RuntimeError(
            f"Unexpected tracker time {settlement_text}; "
            f"derived PERIODID={period_id}."
        )

    trading_date = market_day.strftime("%Y/%m/%d 00:00:00")

    return trading_date, str(period_id)


raw_zip_path = None
tracker_zip_path = None

try:
    tracker_zip_path = download_aemo_zip(TRACKER_FILE)

    # Key = raw BIDPEROFFER identity expected by each tracker interval.
    tracker_key_to_intervals = defaultdict(list)
    tracker_rows = 0
    tracker_duids = Counter()

    with zipfile.ZipFile(tracker_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "SETTLEMENTDATE" in row
                    and "DUID" in row
                ):
                    header_start = row.index("SETTLEMENTDATE")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                trading_date, period_id = market_day_and_period(
                    record["SETTLEMENTDATE"]
                )

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    trading_date,
                    record["BIDOFFERDATE"].strip(),
                    period_id,
                )

                tracker_key_to_intervals[raw_key].append(
                    record["SETTLEMENTDATE"].strip()
                )
                tracker_rows += 1
                tracker_duids[duid] += 1

    print()
    print("Tracker BESS rows:", f"{tracker_rows:,}")
    print("Unique raw-offer identities required:",
          f"{len(tracker_key_to_intervals):,}")

    raw_zip_path = download_aemo_zip(RAW_FILE)

    matched_raw_keys = set()
    duplicate_raw_keys = set()
    raw_bess_rows_scanned = 0

    with zipfile.ZipFile(raw_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "DUID" in row
                    and "TRADINGDATE" in row
                    and "OFFERDATETIME" in row
                ):
                    header_start = row.index("DUID")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                raw_bess_rows_scanned += 1

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    record["TRADINGDATE"].strip(),
                    record["OFFERDATETIME"].strip(),
                    record["PERIODID"].strip(),
                )

                if raw_key not in tracker_key_to_intervals:
                    continue

                if raw_key in matched_raw_keys:
                    duplicate_raw_keys.add(raw_key)

                matched_raw_keys.add(raw_key)

    unmatched_raw_keys = (
        set(tracker_key_to_intervals)
        - matched_raw_keys
    )

    matched_tracker_rows = sum(
        len(tracker_key_to_intervals[key])
        for key in matched_raw_keys
    )

    unmatched_samples = []

    for key in sorted(unmatched_raw_keys)[:20]:
        duid, bidtype, trading_date, offer_datetime, period_id = key

        unmatched_samples.append({
            "DUID": duid,
            "BIDTYPE": bidtype,
            "TRADINGDATE": trading_date,
            "OFFERDATETIME": offer_datetime,
            "PERIODID": period_id,
            "TRACKER_INTERVALS_AFFECTED": len(
                tracker_key_to_intervals[key]
            ),
        })

    print()
    print("=" * 90)
    print("MARCH 2021 RAW-OFFER + TRACKER JOIN AUDIT")
    print("=" * 90)
    print("Raw BESS BIDPEROFFER rows scanned:",
          f"{raw_bess_rows_scanned:,}")
    print("Tracker BESS five-minute rows expected:",
          f"{tracker_rows:,}")
    print("Unique raw-offer identities required:",
          f"{len(tracker_key_to_intervals):,}")
    print("Raw-offer identities matched:",
          f"{len(matched_raw_keys):,}")
    print("Matched five-minute tracker rows:",
          f"{matched_tracker_rows:,}")
    print("Unmatched raw-offer identities:",
          f"{len(unmatched_raw_keys):,}")
    print("Unmatched five-minute tracker rows:",
          f"{tracker_rows - matched_tracker_rows:,}")
    print("Duplicate matching raw identities:",
          f"{len(duplicate_raw_keys):,}")

    if unmatched_samples:
        print()
        print("Example unmatched tracker identities:")
        display(pd.DataFrame(unmatched_samples))

    if duplicate_raw_keys:
        raise RuntimeError(
            "Safety stop: duplicate raw offer identities were found. "
            "No reconstruction decision has been made."
        )

    if unmatched_raw_keys:
        raise RuntimeError(
            "Safety stop: not every tracker-selected offer matched a raw "
            "BIDPEROFFER row. No data has been written."
        )

    print()
    print("✅ JOIN AUDIT PASSED")
    print("✅ March 2021 can be reconstructed from raw offers + tracker.")
    print("✅ No data has been written.")

finally:
    for temp_path in [raw_zip_path, tracker_zip_path]:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

    print()
    print("✅ Temporary join-audit ZIP files deleted.")

In [ ]:
# =============================================================================
# SECTION 34D1 — EXPLAIN THE MARCH 2021 UNMATCHED TRACKER RECORDS
# Read-only. Uses results already produced by Section 34D.
# =============================================================================

unmatched_diagnostic_rows = []

for raw_key in unmatched_raw_keys:
    duid, bidtype, trading_date, offer_datetime, period_id = raw_key
    intervals = tracker_key_to_intervals[raw_key]

    unmatched_diagnostic_rows.append({
        "DUID": duid,
        "BIDTYPE": bidtype,
        "TRADINGDATE": trading_date,
        "OFFERDATETIME": offer_datetime,
        "PERIODID": int(period_id),
        "TRACKER_INTERVAL_COUNT": len(intervals),
        "FIRST_TRACKER_INTERVAL": min(intervals),
        "LAST_TRACKER_INTERVAL": max(intervals),
    })

unmatched_diagnostic_df = pd.DataFrame(unmatched_diagnostic_rows)

print("Unmatched raw-offer identities:", f"{len(unmatched_diagnostic_df):,}")
print(
    "Unmatched tracker intervals:",
    f"{int(unmatched_diagnostic_df['TRACKER_INTERVAL_COUNT'].sum()):,}",
)

print()
print("Summary by raw trading date:")
display(
    unmatched_diagnostic_df
    .groupby("TRADINGDATE", as_index=False)
    .agg(
        raw_offer_identities=("DUID", "size"),
        tracker_intervals=("TRACKER_INTERVAL_COUNT", "sum"),
        minimum_period=("PERIODID", "min"),
        maximum_period=("PERIODID", "max"),
        first_tracker_interval=("FIRST_TRACKER_INTERVAL", "min"),
        last_tracker_interval=("LAST_TRACKER_INTERVAL", "max"),
    )
)

print()
print("Summary by BESS DUID:")
display(
    unmatched_diagnostic_df
    .groupby("DUID", as_index=False)
    .agg(
        raw_offer_identities=("BIDTYPE", "size"),
        tracker_intervals=("TRACKER_INTERVAL_COUNT", "sum"),
        minimum_period=("PERIODID", "min"),
        maximum_period=("PERIODID", "max"),
    )
    .sort_values("DUID")
    .reset_index(drop=True)
)

print()
print("Sample unmatched identities:")
display(
    unmatched_diagnostic_df
    .sort_values(["DUID", "BIDTYPE", "PERIODID"])
    .head(30)
    .reset_index(drop=True)
)

In [ ]:
# =============================================================================
# SECTION 34D2 — PROVE MARCH 2021 CARRY-IN MATCHES FEBRUARY RAW OFFERS
# Read-only. Uses unmatched_raw_keys created by Section 34D.
# =============================================================================

FEB_RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202102010000.zip"
FEB_RAW_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_02/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + FEB_RAW_FILE
)

if "unmatched_raw_keys" not in globals():
    raise RuntimeError(
        "Run Section 34D first. Its unmatched_raw_keys result is required."
    )

print("Carry-in identities required:", f"{len(unmatched_raw_keys):,}")
print("Source:", FEB_RAW_FILE)
print("Read-only proof — no BigQuery writes.")

temp_zip_path = None

try:
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_carryin_202102_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading February BIDPEROFFER — attempt {attempt}/5...")

            response = requests.get(
                FEB_RAW_URL,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_zip_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            break

        except Exception as error:
            print(f"⚠️ Attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                raise

            time.sleep(5 * attempt)

    matched_carryin_keys = set()
    duplicate_carryin_keys = set()
    feb_bess_rows_scanned = 0

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "DUID" in row
                    and "TRADINGDATE" in row
                    and "OFFERDATETIME" in row
                ):
                    header_start = row.index("DUID")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                feb_bess_rows_scanned += 1

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    record["TRADINGDATE"].strip(),
                    record["OFFERDATETIME"].strip(),
                    record["PERIODID"].strip(),
                )

                if raw_key not in unmatched_raw_keys:
                    continue

                if raw_key in matched_carryin_keys:
                    duplicate_carryin_keys.add(raw_key)

                matched_carryin_keys.add(raw_key)

    still_unmatched = unmatched_raw_keys - matched_carryin_keys

    print()
    print("=" * 90)
    print("MARCH 2021 CARRY-IN AUDIT")
    print("=" * 90)
    print("February BESS rows scanned:", f"{feb_bess_rows_scanned:,}")
    print("Carry-in identities required:", f"{len(unmatched_raw_keys):,}")
    print("Carry-in identities matched:", f"{len(matched_carryin_keys):,}")
    print("Still unmatched:", f"{len(still_unmatched):,}")
    print("Duplicate carry-in identities:", f"{len(duplicate_carryin_keys):,}")

    if still_unmatched:
        display(
            pd.DataFrame(
                sorted(still_unmatched),
                columns=[
                    "DUID",
                    "BIDTYPE",
                    "TRADINGDATE",
                    "OFFERDATETIME",
                    "PERIODID",
                ],
            ).head(30)
        )
        raise RuntimeError(
            "Safety stop: February did not supply every March carry-in offer. "
            "No loading decision has been made."
        )

    if duplicate_carryin_keys:
        raise RuntimeError(
            "Safety stop: duplicate February carry-in offer identities found. "
            "No loading decision has been made."
        )

    print()
    print("✅ CARRY-IN AUDIT PASSED")
    print("✅ March 2021 can be reconstructed exactly from:")
    print("   • March raw BIDPEROFFER")
    print("   • February carry-in raw BIDPEROFFER")
    print("   • March DISPATCHOFFERTRK")
    print("✅ No data has been written.")

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)

    print()
    print("✅ Temporary February ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34D3 — INSPECT FEBRUARY 2021 RAW BIDPEROFFER FORMAT
# Read-only. Establishes the pre-March source schema.
# =============================================================================

FEB_RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202102010000.zip"
FEB_RAW_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_02/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + FEB_RAW_FILE
)

temp_zip_path = None

try:
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_feb_header_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    print("Downloading February source for schema inspection...")

    response = requests.get(
        FEB_RAW_URL,
        stream=True,
        timeout=300,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    with open(temp_zip_path, "wb") as output_file:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                output_file.write(chunk)

    response.close()

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            opening_records = []

            for record_number, row in enumerate(reader, start=1):
                opening_records.append({
                    "RECORD_NUMBER": record_number,
                    "RAW_RECORD": row,
                })

                if len(opening_records) == 6:
                    break

    print()
    print("February 2021 first six raw records:")
    display(pd.DataFrame(opening_records))

    header_record = next(
        item["RAW_RECORD"]
        for item in opening_records
        if item["RAW_RECORD"][0].strip().upper() == "I"
    )

    print()
    print("February source header record:")
    print(header_record)

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)

    print()
    print("✅ Temporary February schema-inspection ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34D4 — TEST FEBRUARY 30-MINUTE → MARCH 5-MINUTE CARRY-IN MAPPING
# Read-only. No BigQuery tables are changed.
# =============================================================================

FEB_RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202102010000.zip"
FEB_RAW_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_02/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + FEB_RAW_FILE
)

if "unmatched_raw_keys" not in globals():
    raise RuntimeError(
        "Run Section 34D first. Its unmatched_raw_keys result is required."
    )

# Convert each missing March five-minute key into the expected February
# half-hour key. Six five-minute periods map to one half-hour period.
february_key_to_march_keys = defaultdict(list)

for march_key in unmatched_raw_keys:
    duid, bidtype, trading_date, offer_datetime, five_minute_period = march_key

    half_hour_period = (
        (int(five_minute_period) - 1) // 6
    ) + 1

    february_key = (
        duid,
        bidtype,
        trading_date,
        offer_datetime,
        str(half_hour_period),
    )

    february_key_to_march_keys[february_key].append(march_key)

print("March carry-in five-minute identities:", f"{len(unmatched_raw_keys):,}")
print(
    "February half-hour identities required:",
    f"{len(february_key_to_march_keys):,}",
)
print("Read-only conversion audit — no BigQuery writes.")

temp_zip_path = None

try:
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_feb_carryin_",
        delete=False,
    ) as temp_file:
        temp_zip_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading February BIDPEROFFER — attempt {attempt}/5...")

            response = requests.get(
                FEB_RAW_URL,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_zip_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            break

        except Exception as error:
            print(f"⚠️ Attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                raise

            time.sleep(5 * attempt)

    matched_february_keys = set()
    duplicate_february_keys = set()
    february_bess_rows_scanned = 0

    with zipfile.ZipFile(temp_zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "DUID" in row
                    and "SETTLEMENTDATE" in row
                    and "OFFERDATE" in row
                ):
                    header_start = row.index("DUID")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                february_bess_rows_scanned += 1

                february_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    record["SETTLEMENTDATE"].strip(),
                    record["OFFERDATE"].strip(),
                    record["PERIODID"].strip(),
                )

                if february_key not in february_key_to_march_keys:
                    continue

                if february_key in matched_february_keys:
                    duplicate_february_keys.add(february_key)

                matched_february_keys.add(february_key)

    missing_february_keys = (
        set(february_key_to_march_keys)
        - matched_february_keys
    )

    matched_march_keys = sum(
        len(february_key_to_march_keys[key])
        for key in matched_february_keys
    )

    print()
    print("=" * 90)
    print("FEBRUARY 30-MINUTE → MARCH 5-MINUTE CARRY-IN AUDIT")
    print("=" * 90)
    print("February BESS rows scanned:", f"{february_bess_rows_scanned:,}")
    print(
        "February half-hour identities required:",
        f"{len(february_key_to_march_keys):,}",
    )
    print(
        "February half-hour identities matched:",
        f"{len(matched_february_keys):,}",
    )
    print(
        "March five-minute intervals restored:",
        f"{matched_march_keys:,}",
    )
    print(
        "March five-minute intervals still unmatched:",
        f"{len(unmatched_raw_keys) - matched_march_keys:,}",
    )
    print(
        "Duplicate February identities:",
        f"{len(duplicate_february_keys):,}",
    )

    if missing_february_keys:
        display(
            pd.DataFrame(
                sorted(missing_february_keys),
                columns=[
                    "DUID",
                    "BIDTYPE",
                    "SETTLEMENTDATE",
                    "OFFERDATE",
                    "HALF_HOUR_PERIODID",
                ],
            ).head(30)
        )

        raise RuntimeError(
            "Safety stop: the February half-hour source did not supply every "
            "March carry-in offer. No loading decision has been made."
        )

    if duplicate_february_keys:
        raise RuntimeError(
            "Safety stop: duplicate February carry-in identities found. "
            "No loading decision has been made."
        )

    print()
    print("✅ CARRY-IN CONVERSION AUDIT PASSED")
    print("✅ The February half-hour offers correctly cover March's")
    print("   first four hours at five-minute resolution.")
    print("✅ No data has been written.")

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)

    print()
    print("✅ Temporary February ZIP deleted.")

In [ ]:
# =============================================================================
# SECTION 34E — INSPECT EXISTING EFFECTIVE-OFFER DATE MAPPING
# Read-only. No BigQuery changes.
# =============================================================================

TARGET_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

mapping_summary_sql = f"""
SELECT
    _RUN_ID,
    COUNT(*) AS row_count,
    MIN(SETTLEMENTDATE) AS first_settlementdate,
    MAX(SETTLEMENTDATE) AS last_settlementdate,
    MIN(INTERVAL_DATETIME) AS first_interval,
    MAX(INTERVAL_DATETIME) AS last_interval,
    COUNTIF(
        DATE(SETTLEMENTDATE) = DATE(BIDSETTLEMENTDATE)
    ) AS same_market_day_rows,
    COUNTIF(
        DATE(SETTLEMENTDATE) != DATE(BIDSETTLEMENTDATE)
    ) AS different_market_day_rows
FROM `{TARGET_TABLE}`
WHERE _RUN_ID IN (
    "bess-effective-direct-2019-11",
    "bess-effective-direct-2024-08"
)
GROUP BY _RUN_ID
ORDER BY _RUN_ID
"""

mapping_sample_sql = f"""
SELECT
    _RUN_ID,
    DUID,
    BIDTYPE,
    DIRECTION,
    SETTLEMENTDATE,
    INTERVAL_DATETIME,
    BIDSETTLEMENTDATE,
    OFFERDATE,
    PERIODID,
    MAXAVAIL,
    BANDAVAIL1,
    BANDAVAIL10
FROM `{TARGET_TABLE}`
WHERE _RUN_ID IN (
    "bess-effective-direct-2019-11",
    "bess-effective-direct-2024-08"
)
  AND DUID IN ("BALBG1", "BALB1", "HPRG1")
ORDER BY _RUN_ID, INTERVAL_DATETIME, DUID, BIDTYPE
LIMIT 30
"""

print("Existing direct-source date mapping summary:")
display(client.query(mapping_summary_sql).result().to_dataframe())

print()
print("Existing direct-source sample rows:")
display(client.query(mapping_sample_sql).result().to_dataframe())

print()
print("✅ Read-only mapping inspection complete.")

In [ ]:
# =============================================================================
# SECTION 34E1 — DEFINE THE EXACT MARCH 2021 TARGET INTERVAL WINDOW
# Read-only. Uses tracker_key_to_intervals from Section 34D.
# =============================================================================

if "tracker_key_to_intervals" not in globals():
    raise RuntimeError(
        "Run Section 34D first. tracker_key_to_intervals is required."
    )

tracker_window_rows = []

for raw_key, interval_list in tracker_key_to_intervals.items():
    duid, bidtype, bidsettlementdate, offerdate, periodid = raw_key

    for interval_text in interval_list:
        interval_ts = pd.to_datetime(
            interval_text,
            format="%Y/%m/%d %H:%M:%S",
        )

        market_day, derived_periodid = market_day_and_period(interval_text)

        tracker_window_rows.append({
            "INTERVAL_DATETIME": interval_ts,
            "DUID": duid,
            "BIDTYPE": bidtype,
            "BIDSETTLEMENTDATE": bidsettlementdate,
            "OFFERDATE": offerdate,
            "PERIODID": int(periodid),
            "DERIVED_MARKET_DAY": market_day,
            "DERIVED_PERIODID": int(derived_periodid),
        })

tracker_window_df = pd.DataFrame(tracker_window_rows)

TARGET_START = pd.Timestamp("2021-03-01 04:05:00")
TARGET_END = pd.Timestamp("2021-04-01 04:00:00")

tracker_window_df["IN_MARCH_TRADING_MONTH"] = (
    tracker_window_df["INTERVAL_DATETIME"].between(
        TARGET_START,
        TARGET_END,
        inclusive="both",
    )
)

print("Tracker first interval:", tracker_window_df["INTERVAL_DATETIME"].min())
print("Tracker last interval:", tracker_window_df["INTERVAL_DATETIME"].max())
print("All tracker rows:", f"{len(tracker_window_df):,}")
print(
    "Rows in March trading-month target window:",
    f"{int(tracker_window_df['IN_MARCH_TRADING_MONTH'].sum()):,}",
)
print(
    "Rows outside target window:",
    f"{int((~tracker_window_df['IN_MARCH_TRADING_MONTH']).sum()):,}",
)

print()
print("Coverage by calendar date:")
display(
    tracker_window_df
    .assign(CALENDAR_DATE=tracker_window_df["INTERVAL_DATETIME"].dt.date)
    .groupby("CALENDAR_DATE", as_index=False)
    .agg(
        tracker_rows=("DUID", "size"),
        first_interval=("INTERVAL_DATETIME", "min"),
        last_interval=("INTERVAL_DATETIME", "max"),
        rows_in_target=(
            "IN_MARCH_TRADING_MONTH",
            "sum",
        ),
    )
)

print()
print("Check that tracker period mapping matches its raw-offer key:")
display(
    tracker_window_df
    .assign(
        PERIOD_MATCH=(
            tracker_window_df["PERIODID"]
            == tracker_window_df["DERIVED_PERIODID"]
        )
    )
    .groupby("PERIOD_MATCH", as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

if not (
    tracker_window_df["PERIODID"]
    == tracker_window_df["DERIVED_PERIODID"]
).all():
    raise RuntimeError(
        "Safety stop: tracker PERIODID does not match the derived "
        "five-minute market-day period."
    )

print()
print("✅ March target-window inspection complete.")

In [ ]:
# =============================================================================
# SECTION 34E2 — PROVE APRIL TRACKER COMPLETES MARCH TRADING MONTH
# Read-only. No BigQuery writes.
# =============================================================================

APRIL_TRACKER_FILE = "PUBLIC_DVD_DISPATCHOFFERTRK_202104010000.zip"

APRIL_TRACKER_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_04/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + APRIL_TRACKER_FILE
)

MARCH_RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202103010000.zip"

MARCH_RAW_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + MARCH_RAW_FILE
)

TARGET_START = pd.Timestamp("2021-04-01 00:05:00")
TARGET_END = pd.Timestamp("2021-04-01 04:00:00")

print("Read-only April-boundary audit — no BigQuery writes.")

def download_boundary_zip(source_url, label):
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_boundary_",
        delete=False,
    ) as temp_file:
        temp_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading {label} — attempt {attempt}/5...")

            response = requests.get(
                source_url,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            return temp_path

        except Exception as error:
            print(f"⚠️ Attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                if os.path.exists(temp_path):
                    os.remove(temp_path)
                raise

            time.sleep(5 * attempt)


april_tracker_zip = None
march_raw_zip = None

try:
    # -------------------------------------------------------------------------
    # 1. Read only the April tracker records for 1 April 00:05–04:00.
    # -------------------------------------------------------------------------
    april_tracker_zip = download_boundary_zip(
        APRIL_TRACKER_URL,
        APRIL_TRACKER_FILE,
    )

    april_tracker_keys = defaultdict(list)
    april_tracker_first = None
    april_tracker_last = None

    with zipfile.ZipFile(april_tracker_zip) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "SETTLEMENTDATE" in row
                    and "DUID" in row
                ):
                    header_start = row.index("SETTLEMENTDATE")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                interval_ts = pd.to_datetime(
                    record["SETTLEMENTDATE"],
                    format="%Y/%m/%d %H:%M:%S",
                )

                if april_tracker_first is None:
                    april_tracker_first = interval_ts

                april_tracker_last = interval_ts

                if not TARGET_START <= interval_ts <= TARGET_END:
                    continue

                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                trading_date, period_id = market_day_and_period(
                    record["SETTLEMENTDATE"]
                )

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    trading_date,
                    record["BIDOFFERDATE"].strip(),
                    period_id,
                )

                april_tracker_keys[raw_key].append(
                    record["SETTLEMENTDATE"].strip()
                )

    print()
    print("April tracker first interval:", april_tracker_first)
    print("April tracker last interval:", april_tracker_last)
    print(
        "BESS tracker rows required for March completion:",
        f"{sum(len(v) for v in april_tracker_keys.values()):,}",
    )
    print(
        "Unique March raw-offer identities required:",
        f"{len(april_tracker_keys):,}",
    )

    # -------------------------------------------------------------------------
    # 2. Find those April tracker-selected offers in March raw BIDPEROFFER.
    # -------------------------------------------------------------------------
    march_raw_zip = download_boundary_zip(
        MARCH_RAW_URL,
        MARCH_RAW_FILE,
    )

    matched_raw_keys = set()
    duplicate_raw_keys = set()
    march_bess_rows_scanned = 0

    with zipfile.ZipFile(march_raw_zip) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "DUID" in row
                    and "TRADINGDATE" in row
                    and "OFFERDATETIME" in row
                ):
                    header_start = row.index("DUID")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in BESS_DUIDS:
                    continue

                march_bess_rows_scanned += 1

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    record["TRADINGDATE"].strip(),
                    record["OFFERDATETIME"].strip(),
                    record["PERIODID"].strip(),
                )

                if raw_key not in april_tracker_keys:
                    continue

                if raw_key in matched_raw_keys:
                    duplicate_raw_keys.add(raw_key)

                matched_raw_keys.add(raw_key)

    missing_raw_keys = set(april_tracker_keys) - matched_raw_keys

    matched_tracker_rows = sum(
        len(april_tracker_keys[key])
        for key in matched_raw_keys
    )

    print()
    print("=" * 90)
    print("APRIL TRACKER → MARCH RAW-OFFER BOUNDARY AUDIT")
    print("=" * 90)
    print("March BESS raw rows scanned:", f"{march_bess_rows_scanned:,}")
    print(
        "April BESS tracker rows required:",
        f"{sum(len(v) for v in april_tracker_keys.values()):,}",
    )
    print("Raw identities matched:", f"{len(matched_raw_keys):,}")
    print("Tracker rows matched:", f"{matched_tracker_rows:,}")
    print("Tracker rows still unmatched:",
          f"{sum(len(v) for v in april_tracker_keys.values()) - matched_tracker_rows:,}")
    print("Duplicate raw identities:", f"{len(duplicate_raw_keys):,}")

    if missing_raw_keys:
        display(
            pd.DataFrame(
                sorted(missing_raw_keys),
                columns=[
                    "DUID",
                    "BIDTYPE",
                    "TRADINGDATE",
                    "OFFERDATETIME",
                    "PERIODID",
                ],
            ).head(30)
        )

        raise RuntimeError(
            "Safety stop: April boundary tracker records do not all match "
            "March raw offers. No data has been written."
        )

    if duplicate_raw_keys:
        raise RuntimeError(
            "Safety stop: duplicate March raw-offer identities found. "
            "No data has been written."
        )

    print()
    print("✅ APRIL BOUNDARY AUDIT PASSED")
    print("✅ March 2021 has a complete 04:05–04:00 trading-month window.")
    print("✅ Expected March stage size: 446,400 BESS effective-offer rows.")
    print("✅ No data has been written.")

finally:
    for temp_path in [april_tracker_zip, march_raw_zip]:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

    print()
    print("✅ Temporary boundary-audit ZIP files deleted.")

In [ ]:
# =============================================================================
# SECTION 34F — STAGE MARCH 2021 RECONSTRUCTED BESS EFFECTIVE OFFERS
# Creates a temporary stage only. No production or manifest changes.
# =============================================================================

from google.cloud import bigquery
import csv, io, os, time, zipfile, tempfile
from datetime import datetime, timedelta
from collections import defaultdict

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202103"
)

RUN_ID = "bess-effective-reconstructed-2021-03"
EXPECTED_STAGE_ROWS = 446_400

MARCH_RAW_FILE = "PUBLIC_DVD_BIDPEROFFER_202103010000.zip"
MARCH_TRACKER_FILE = "PUBLIC_DVD_DISPATCHOFFERTRK_202103010000.zip"
APRIL_TRACKER_FILE = "PUBLIC_DVD_DISPATCHOFFERTRK_202104010000.zip"

MARCH_BASE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_03/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)
APRIL_BASE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2021/MMSDM_2021_04/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

MARCH_RAW_URL = MARCH_BASE_URL + MARCH_RAW_FILE
MARCH_TRACKER_URL = MARCH_BASE_URL + MARCH_TRACKER_FILE
APRIL_TRACKER_URL = APRIL_BASE_URL + APRIL_TRACKER_FILE

TARGET_START = pd.Timestamp("2021-03-01 04:05:00")
TARGET_END = pd.Timestamp("2021-04-01 04:00:00")

# These ten historical units were the active BESS source DUIDs in March 2021.
DIRECTION_MAP = {
    "BALBG1": "GEN", "BALBL1": "LOAD",
    "DALNTH01": "GEN", "DALNTHL1": "LOAD",
    "GANNBG1": "GEN", "GANNBL1": "LOAD",
    "HPRG1": "GEN", "HPRL1": "LOAD",
    "LBBG1": "GEN", "LBBL1": "LOAD",
}

def market_day_and_period(settlement_text):
    settlement_ts = datetime.strptime(
        settlement_text.strip(),
        "%Y/%m/%d %H:%M:%S",
    )

    if settlement_ts.time() <= datetime.strptime(
        "04:00:00", "%H:%M:%S"
    ).time():
        market_day = settlement_ts.date() - timedelta(days=1)
    else:
        market_day = settlement_ts.date()

    market_day_start = datetime.combine(
        market_day,
        datetime.strptime("04:00:00", "%H:%M:%S").time(),
    )

    period_id = int(
        (settlement_ts - market_day_start).total_seconds() // 300
    )

    if not 1 <= period_id <= 288:
        raise RuntimeError(
            f"Unexpected tracker time {settlement_text}; "
            f"derived PERIODID={period_id}."
        )

    return market_day.strftime("%Y/%m/%d 00:00:00"), str(period_id)

def download_zip(source_url, label):
    with tempfile.NamedTemporaryFile(
        suffix=".zip",
        prefix="aemo_march_stage_",
        delete=False,
    ) as temp_file:
        temp_path = temp_file.name

    for attempt in range(1, 6):
        try:
            print(f"Downloading {label} — attempt {attempt}/5...")

            response = requests.get(
                source_url,
                stream=True,
                timeout=300,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()

            with open(temp_path, "wb") as output_file:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        output_file.write(chunk)

            response.close()
            return temp_path

        except Exception as error:
            print(f"⚠️ Attempt {attempt}/5 failed: {error}")

            if attempt == 5:
                if os.path.exists(temp_path):
                    os.remove(temp_path)
                raise

            time.sleep(5 * attempt)

def as_timestamp(value):
    value = str(value).strip()
    return pd.to_datetime(value, utc=True) if value else None

def as_float(value):
    value = str(value).strip()
    return float(value) if value else None

# -----------------------------------------------------------------------------
# Safety checks: production March must be empty and no old stage may exist.
# -----------------------------------------------------------------------------
existing_production_sql = f"""
SELECT COUNT(*) AS row_count
FROM `{TARGET_TABLE}`
WHERE SETTLEMENTDATE >= TIMESTAMP("2021-03-01 00:00:00+00")
  AND SETTLEMENTDATE < TIMESTAMP("2021-04-01 00:00:00+00")
"""

existing_production_rows = int(
    client.query(existing_production_sql)
    .result()
    .to_dataframe()
    .iloc[0]["row_count"]
)

print("Existing production March 2021 rows:", f"{existing_production_rows:,}")

if existing_production_rows != 0:
    raise RuntimeError(
        "Safety stop: production already contains March 2021 rows. "
        "Nothing has been staged."
    )

try:
    client.get_table(STAGE_TABLE)
    raise RuntimeError(
        "Safety stop: March 2021 stage table already exists. "
        "Inspect it before attempting another stage."
    )
except Exception as error:
    if "Not found" not in str(error):
        if "already exists" in str(error):
            raise

print("✅ Staging preflight passed.")

march_tracker_zip = None
april_tracker_zip = None
march_raw_zip = None

try:
    # -------------------------------------------------------------------------
    # Build the exact tracker-selected raw-offer keys for March trading month.
    # -------------------------------------------------------------------------
    required_targets = {}

    tracker_sources = [
        (MARCH_TRACKER_URL, MARCH_TRACKER_FILE),
        (APRIL_TRACKER_URL, APRIL_TRACKER_FILE),
    ]

    for tracker_url, tracker_file in tracker_sources:
        tracker_zip_path = download_zip(tracker_url, tracker_file)

        try:
            with zipfile.ZipFile(tracker_zip_path) as zip_file:
                csv_name = next(
                    name for name in zip_file.namelist()
                    if name.upper().endswith(".CSV")
                )

                with zip_file.open(csv_name) as binary_csv:
                    reader = csv.reader(
                        io.TextIOWrapper(
                            binary_csv,
                            encoding="utf-8-sig",
                            newline="",
                        )
                    )

                    header = None
                    header_start = None

                    for row in reader:
                        if not row:
                            continue

                        if (
                            row[0].strip().upper() == "I"
                            and "SETTLEMENTDATE" in row
                            and "DUID" in row
                        ):
                            header_start = row.index("SETTLEMENTDATE")
                            header = [
                                value.strip()
                                for value in row[header_start:]
                            ]
                            continue

                        if (
                            row[0].strip().upper() != "D"
                            or header is None
                        ):
                            continue

                        values = row[header_start:]

                        if len(values) != len(header):
                            continue

                        record = dict(zip(header, values))
                        interval_ts = pd.to_datetime(
                            record["SETTLEMENTDATE"],
                            format="%Y/%m/%d %H:%M:%S",
                        )

                        if not TARGET_START <= interval_ts <= TARGET_END:
                            continue

                        duid = record["DUID"].strip().upper()

                        if duid not in DIRECTION_MAP:
                            continue

                        market_day, period_id = market_day_and_period(
                            record["SETTLEMENTDATE"]
                        )

                        raw_key = (
                            duid,
                            record["BIDTYPE"].strip().upper(),
                            record["BIDSETTLEMENTDATE"].strip(),
                            record["BIDOFFERDATE"].strip(),
                            period_id,
                        )

                        if raw_key in required_targets:
                            raise RuntimeError(
                                f"Duplicate tracker raw key found: {raw_key}"
                            )

                        required_targets[raw_key] = {
                            "SETTLEMENTDATE": pd.to_datetime(
                                market_day,
                                format="%Y/%m/%d %H:%M:%S",
                                utc=True,
                            ),
                            "INTERVAL_DATETIME": pd.to_datetime(
                                record["SETTLEMENTDATE"],
                                format="%Y/%m/%d %H:%M:%S",
                                utc=True,
                            ),
                            "BIDSETTLEMENTDATE": as_timestamp(
                                record["BIDSETTLEMENTDATE"]
                            ),
                            "OFFERDATE": as_timestamp(
                                record["BIDOFFERDATE"]
                            ),
                        }
        finally:
            if os.path.exists(tracker_zip_path):
                os.remove(tracker_zip_path)

    print("Tracker-selected March target rows:",
          f"{len(required_targets):,}")

    if len(required_targets) != EXPECTED_STAGE_ROWS:
        raise RuntimeError(
            f"Expected {EXPECTED_STAGE_ROWS:,} tracker rows, "
            f"found {len(required_targets):,}."
        )

    # -------------------------------------------------------------------------
    # Create the temporary stage with the same schema/layout as production.
    # -------------------------------------------------------------------------
    target_definition = client.get_table(TARGET_TABLE)

    stage_definition = bigquery.Table(
        STAGE_TABLE,
        schema=target_definition.schema,
    )
    stage_definition.time_partitioning = bigquery.TimePartitioning(
        field="SETTLEMENTDATE"
    )
    stage_definition.clustering_fields = [
        "DUID",
        "BIDTYPE",
        "DIRECTION",
    ]

    client.create_table(stage_definition)
    print("✅ Empty March stage table created.")

    # -------------------------------------------------------------------------
    # Scan March raw offers once. Keep only the tracker-selected BESS records.
    # -------------------------------------------------------------------------
    march_raw_zip = download_zip(MARCH_RAW_URL, MARCH_RAW_FILE)

    target_columns = [field.name for field in target_definition.schema]
    load_date = pd.Timestamp.now(tz="UTC")

    staged_rows = 0
    matched_raw_keys = set()
    chunk_rows = []
    chunk_number = 0

    with zipfile.ZipFile(march_raw_zip) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        print("Processing raw source CSV:", csv_name)

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            header = None
            header_start = None

            for row in reader:
                if not row:
                    continue

                if (
                    row[0].strip().upper() == "I"
                    and "DUID" in row
                    and "TRADINGDATE" in row
                    and "OFFERDATETIME" in row
                ):
                    header_start = row.index("DUID")
                    header = [
                        value.strip()
                        for value in row[header_start:]
                    ]
                    continue

                if (
                    row[0].strip().upper() != "D"
                    or header is None
                ):
                    continue

                values = row[header_start:]

                if len(values) != len(header):
                    continue

                record = dict(zip(header, values))
                duid = record["DUID"].strip().upper()

                if duid not in DIRECTION_MAP:
                    continue

                raw_key = (
                    duid,
                    record["BIDTYPE"].strip().upper(),
                    record["TRADINGDATE"].strip(),
                    record["OFFERDATETIME"].strip(),
                    record["PERIODID"].strip(),
                )

                if raw_key not in required_targets:
                    continue

                if raw_key in matched_raw_keys:
                    raise RuntimeError(
                        f"Duplicate matching raw offer found: {raw_key}"
                    )

                matched_raw_keys.add(raw_key)
                tracker_record = required_targets[raw_key]

                stage_row = {
                    "SETTLEMENTDATE": tracker_record["SETTLEMENTDATE"],
                    "DUID": duid,
                    "BIDTYPE": record["BIDTYPE"].strip().upper(),
                    "DIRECTION": DIRECTION_MAP[duid],
                    "INTERVAL_DATETIME": tracker_record["INTERVAL_DATETIME"],
                    "BIDSETTLEMENTDATE": tracker_record["BIDSETTLEMENTDATE"],
                    "OFFERDATE": tracker_record["OFFERDATE"],
                    "PERIODID": int(record["PERIODID"]),
                    "VERSIONNO": None,
                    "MAXAVAIL": as_float(record.get("MAXAVAIL")),
                    "FIXEDLOAD": as_float(record.get("FIXEDLOAD")),
                    "ROCUP": as_float(record.get("RAMPUPRATE")),
                    "ROCDOWN": as_float(record.get("RAMPDOWNRATE")),
                    "ENABLEMENTMIN": as_float(record.get("ENABLEMENTMIN")),
                    "ENABLEMENTMAX": as_float(record.get("ENABLEMENTMAX")),
                    "LOWBREAKPOINT": as_float(record.get("LOWBREAKPOINT")),
                    "HIGHBREAKPOINT": as_float(record.get("HIGHBREAKPOINT")),
                    "BANDAVAIL1": as_float(record.get("BANDAVAIL1")),
                    "BANDAVAIL2": as_float(record.get("BANDAVAIL2")),
                    "BANDAVAIL3": as_float(record.get("BANDAVAIL3")),
                    "BANDAVAIL4": as_float(record.get("BANDAVAIL4")),
                    "BANDAVAIL5": as_float(record.get("BANDAVAIL5")),
                    "BANDAVAIL6": as_float(record.get("BANDAVAIL6")),
                    "BANDAVAIL7": as_float(record.get("BANDAVAIL7")),
                    "BANDAVAIL8": as_float(record.get("BANDAVAIL8")),
                    "BANDAVAIL9": as_float(record.get("BANDAVAIL9")),
                    "BANDAVAIL10": as_float(record.get("BANDAVAIL10")),
                    "LASTCHANGED": None,
                    "PASAAVAILABILITY": as_float(
                        record.get("PASAAVAILABILITY")
                    ),
                    "MR_CAPACITY": None,
                    "ENERGYLIMIT": None,
                    "_LOAD_DATE": load_date,
                    "_SOURCE": "AEMO_NEMWEB_ARCHIVE",
                    "_PIPELINE": "BIDPEROFFER_TRACKER_RECONSTRUCTED",
                    "_SOURCE_FILE": MARCH_RAW_FILE,
                    "_SOURCE_URL": MARCH_RAW_URL,
                    "_RUN_ID": RUN_ID,
                }

                chunk_rows.append(stage_row)

                if len(chunk_rows) >= 50_000:
                    chunk_number += 1

                    chunk_df = pd.DataFrame(
                        chunk_rows,
                        columns=target_columns,
                    )

                    client.load_table_from_dataframe(
                        chunk_df,
                        STAGE_TABLE,
                        job_config=bigquery.LoadJobConfig(
                            write_disposition=(
                                bigquery.WriteDisposition.WRITE_APPEND
                            )
                        ),
                    ).result()

                    staged_rows += len(chunk_rows)
                    print(
                        f"Chunks staged: {chunk_number} | "
                        f"Rows staged: {staged_rows:,}"
                    )
                    chunk_rows = []

    if chunk_rows:
        chunk_number += 1

        chunk_df = pd.DataFrame(
            chunk_rows,
            columns=target_columns,
        )

        client.load_table_from_dataframe(
            chunk_df,
            STAGE_TABLE,
            job_config=bigquery.LoadJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_APPEND
            ),
        ).result()

        staged_rows += len(chunk_rows)
        print(
            f"Chunks staged: {chunk_number} | "
            f"Rows staged: {staged_rows:,}"
        )

    # -------------------------------------------------------------------------
    # Stage audit. Stop here: promotion is a separate explicit step.
    # -------------------------------------------------------------------------
    missing_raw_keys = set(required_targets) - matched_raw_keys

    if missing_raw_keys:
        raise RuntimeError(
            f"Safety stop: {len(missing_raw_keys):,} tracker-selected "
            "offers were absent from March raw data."
        )

    stage_audit_sql = f"""
    SELECT
        COUNT(*) AS stage_row_count,
        COUNT(DISTINCT DUID) AS source_duid_count,
        COUNT(DISTINCT STRUCT(INTERVAL_DATETIME, DUID, BIDTYPE))
            AS distinct_effective_key_count,
        MIN(INTERVAL_DATETIME) AS first_interval,
        MAX(INTERVAL_DATETIME) AS last_interval,
        MIN(SETTLEMENTDATE) AS first_settlementdate,
        MAX(SETTLEMENTDATE) AS last_settlementdate
    FROM `{STAGE_TABLE}`
    """

    stage_audit_df = client.query(stage_audit_sql).result().to_dataframe()
    display(stage_audit_df)

    verified_stage_rows = int(
        stage_audit_df.iloc[0]["stage_row_count"]
    )
    verified_effective_keys = int(
        stage_audit_df.iloc[0]["distinct_effective_key_count"]
    )

    if verified_stage_rows != EXPECTED_STAGE_ROWS:
        raise RuntimeError(
            f"Safety stop: expected {EXPECTED_STAGE_ROWS:,} staged rows, "
            f"found {verified_stage_rows:,}."
        )

    if verified_effective_keys != EXPECTED_STAGE_ROWS:
        raise RuntimeError(
            "Safety stop: stage contains duplicate effective-offer keys."
        )

    print()
    print("✅ MARCH 2021 RECONSTRUCTED STAGE VERIFIED")
    print("Stage table:", STAGE_TABLE)
    print("Rows staged:", f"{verified_stage_rows:,}")
    print("Production has not been changed.")

finally:
    if march_raw_zip and os.path.exists(march_raw_zip):
        os.remove(march_raw_zip)

    print()
    print("✅ Temporary local source ZIPs cleaned up.")

In [ ]:
# =============================================================================
# SECTION 34F1 — AUDIT THE ALREADY-STAGED MARCH 2021 TABLE
# Read-only. Production and manifest remain unchanged.
# =============================================================================

EXPECTED_STAGE_ROWS = 446_400

stage_audit_sql = f"""
WITH stage_rows AS (
    SELECT
        SETTLEMENTDATE,
        INTERVAL_DATETIME,
        DUID,
        BIDTYPE,
        BIDSETTLEMENTDATE,
        OFFERDATE,
        DIRECTION
    FROM `{STAGE_TABLE}`
),
effective_keys AS (
    SELECT
        INTERVAL_DATETIME,
        DUID,
        BIDTYPE
    FROM stage_rows
    GROUP BY
        INTERVAL_DATETIME,
        DUID,
        BIDTYPE
)
SELECT
    (SELECT COUNT(*) FROM stage_rows) AS stage_row_count,
    (SELECT COUNT(*) FROM effective_keys) AS distinct_effective_key_count,
    (
        SELECT COUNT(DISTINCT DUID)
        FROM stage_rows
    ) AS source_duid_count,
    (
        SELECT COUNT(*)
        FROM stage_rows
        WHERE SETTLEMENTDATE IS NULL
           OR INTERVAL_DATETIME IS NULL
           OR BIDSETTLEMENTDATE IS NULL
           OR OFFERDATE IS NULL
           OR DIRECTION IS NULL
    ) AS null_required_field_count,
    (
        SELECT MIN(INTERVAL_DATETIME)
        FROM stage_rows
    ) AS first_interval,
    (
        SELECT MAX(INTERVAL_DATETIME)
        FROM stage_rows
    ) AS last_interval,
    (
        SELECT MIN(SETTLEMENTDATE)
        FROM stage_rows
    ) AS first_settlementdate,
    (
        SELECT MAX(SETTLEMENTDATE)
        FROM stage_rows
    ) AS last_settlementdate
"""

stage_audit_df = client.query(stage_audit_sql).result().to_dataframe()
display(stage_audit_df)

stage_row_count = int(stage_audit_df.iloc[0]["stage_row_count"])
distinct_key_count = int(
    stage_audit_df.iloc[0]["distinct_effective_key_count"]
)
null_required_count = int(
    stage_audit_df.iloc[0]["null_required_field_count"]
)

if stage_row_count != EXPECTED_STAGE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_STAGE_ROWS:,} stage rows, "
        f"found {stage_row_count:,}."
    )

if distinct_key_count != EXPECTED_STAGE_ROWS:
    raise RuntimeError(
        "Safety stop: the stage has duplicate "
        "(INTERVAL_DATETIME, DUID, BIDTYPE) keys."
    )

if null_required_count != 0:
    raise RuntimeError(
        f"Safety stop: {null_required_count:,} staged rows have null "
        "required fields."
    )

print()
print("✅ MARCH 2021 RECONSTRUCTED STAGE AUDIT PASSED")
print("Stage table:", STAGE_TABLE)
print("Rows staged:", f"{stage_row_count:,}")
print("Production has not been changed.")

In [ ]:
# =============================================================================
# SECTION 34G — PROMOTE VERIFIED MARCH 2021 RECONSTRUCTED STAGE
# Production + manifest change only after all safety checks pass.
# =============================================================================

MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

RUN_ID = "bess-effective-reconstructed-2021-03"
EXPECTED_ROWS = 446_400

# Final safety checks before any production write.
preflight_sql = f"""
SELECT
    (
        SELECT COUNT(*)
        FROM `{STAGE_TABLE}`
    ) AS stage_row_count,
    (
        SELECT COUNT(*)
        FROM `{TARGET_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP("2021-03-01 00:00:00+00")
          AND SETTLEMENTDATE < TIMESTAMP("2021-04-01 00:00:00+00")
    ) AS existing_production_march_rows,
    (
        SELECT COUNT(*)
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = "{RUN_ID}"
    ) AS existing_production_run_rows,
    (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = "{RUN_ID}"
    ) AS existing_manifest_run_rows
"""

preflight_df = client.query(preflight_sql).result().to_dataframe()
display(preflight_df)

stage_rows = int(preflight_df.iloc[0]["stage_row_count"])
existing_march_rows = int(
    preflight_df.iloc[0]["existing_production_march_rows"]
)
existing_run_rows = int(
    preflight_df.iloc[0]["existing_production_run_rows"]
)
existing_manifest_rows = int(
    preflight_df.iloc[0]["existing_manifest_run_rows"]
)

if stage_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Safety stop: stage has {stage_rows:,} rows; "
        f"expected {EXPECTED_ROWS:,}."
    )

if existing_march_rows != 0:
    raise RuntimeError(
        f"Safety stop: production already has "
        f"{existing_march_rows:,} March 2021 rows."
    )

if existing_run_rows != 0 or existing_manifest_rows != 0:
    raise RuntimeError(
        "Safety stop: this March reconstruction run already exists."
    )

print("✅ Promotion preflight passed.")
print("Promoting verified March 2021 stage...")

promotion_sql = f"""
BEGIN TRANSACTION;

INSERT INTO `{TARGET_TABLE}`
SELECT *
FROM `{STAGE_TABLE}`;

INSERT INTO `{MANIFEST_TABLE}` (
    RUN_ID,
    ARCHIVE_YEAR,
    ARCHIVE_MONTH,
    SOURCE_FILE_NAME,
    SOURCE_URL,
    SOURCE_METHOD,
    STATUS,
    SOURCE_ROWS,
    BESS_ROWS,
    LOADED_ROWS,
    STARTED_AT,
    COMPLETED_AT,
    ERROR_MESSAGE,
    CREATED_AT
)
VALUES (
    "{RUN_ID}",
    2021,
    3,
    "PUBLIC_DVD_BIDPEROFFER_202103010000.zip + "
    "PUBLIC_DVD_DISPATCHOFFERTRK_202103010000.zip + "
    "PUBLIC_DVD_DISPATCHOFFERTRK_202104010000.zip",
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2021/",
    "BIDPEROFFER_PLUS_DISPATCHOFFERTRK_RECONSTRUCTED",
    "PROMOTED",
    180361440,
    66681216,
    {EXPECTED_ROWS},
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    NULL,
    CURRENT_TIMESTAMP()
);

COMMIT TRANSACTION;
"""

client.query(promotion_sql).result()

post_promotion_sql = f"""
SELECT
    (
        SELECT COUNT(*)
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = "{RUN_ID}"
    ) AS production_run_row_count,
    (
        SELECT COUNT(*)
        FROM `{TARGET_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP("2021-03-01 00:00:00+00")
          AND SETTLEMENTDATE < TIMESTAMP("2021-04-01 00:00:00+00")
    ) AS production_march_row_count,
    (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = "{RUN_ID}"
          AND STATUS = "PROMOTED"
    ) AS promoted_manifest_row_count
"""

post_promotion_df = (
    client.query(post_promotion_sql)
    .result()
    .to_dataframe()
)

display(post_promotion_df)

production_run_rows = int(
    post_promotion_df.iloc[0]["production_run_row_count"]
)
production_march_rows = int(
    post_promotion_df.iloc[0]["production_march_row_count"]
)
manifest_rows = int(
    post_promotion_df.iloc[0]["promoted_manifest_row_count"]
)

if (
    production_run_rows != EXPECTED_ROWS
    or production_march_rows != EXPECTED_ROWS
    or manifest_rows != 1
):
    raise RuntimeError(
        "Promotion completed but final counts did not match expectations. "
        "Leave the stage table in place for review."
    )

print()
print("✅ MARCH 2021 RECONSTRUCTED EFFECTIVE OFFERS PROMOTED")
print("Production rows retained:", f"{production_run_rows:,}")
print("Manifest entry created.")
print("Stage table retained for final audit:", STAGE_TABLE)

In [ ]:
# SECTION 34H — Final March 2021 stage-versus-production audit
# Read-only: no BigQuery writes or deletes.

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202103"
)
RUN_ID = "bess-effective-reconstructed-2021-03"

audit_sql = f"""
WITH
stage_rows AS (
  SELECT
    SETTLEMENTDATE, DUID, BIDTYPE, DIRECTION, INTERVAL_DATETIME,
    BIDSETTLEMENTDATE, OFFERDATE, PERIODID, VERSIONNO,
    MAXAVAIL, FIXEDLOAD, ROCUP, ROCDOWN,
    ENABLEMENTMIN, ENABLEMENTMAX, LOWBREAKPOINT, HIGHBREAKPOINT,
    BANDAVAIL1, BANDAVAIL2, BANDAVAIL3, BANDAVAIL4, BANDAVAIL5,
    BANDAVAIL6, BANDAVAIL7, BANDAVAIL8, BANDAVAIL9, BANDAVAIL10,
    LASTCHANGED, PASAAVAILABILITY, MR_CAPACITY, ENERGYLIMIT,
    _SOURCE, _PIPELINE, _SOURCE_FILE, _SOURCE_URL, _RUN_ID
  FROM `{STAGE_TABLE}`
),
production_rows AS (
  SELECT
    SETTLEMENTDATE, DUID, BIDTYPE, DIRECTION, INTERVAL_DATETIME,
    BIDSETTLEMENTDATE, OFFERDATE, PERIODID, VERSIONNO,
    MAXAVAIL, FIXEDLOAD, ROCUP, ROCDOWN,
    ENABLEMENTMIN, ENABLEMENTMAX, LOWBREAKPOINT, HIGHBREAKPOINT,
    BANDAVAIL1, BANDAVAIL2, BANDAVAIL3, BANDAVAIL4, BANDAVAIL5,
    BANDAVAIL6, BANDAVAIL7, BANDAVAIL8, BANDAVAIL9, BANDAVAIL10,
    LASTCHANGED, PASAAVAILABILITY, MR_CAPACITY, ENERGYLIMIT,
    _SOURCE, _PIPELINE, _SOURCE_FILE, _SOURCE_URL, _RUN_ID
  FROM `{TARGET_TABLE}`
  WHERE _RUN_ID = @run_id
),
stage_minus_production AS (
  SELECT * FROM stage_rows
  EXCEPT DISTINCT
  SELECT * FROM production_rows
),
production_minus_stage AS (
  SELECT * FROM production_rows
  EXCEPT DISTINCT
  SELECT * FROM stage_rows
)
SELECT
  (SELECT COUNT(*) FROM stage_rows) AS stage_row_count,
  (SELECT COUNT(*) FROM production_rows) AS production_run_row_count,
  (SELECT COUNT(*) FROM stage_minus_production)
    AS stage_missing_from_production_count,
  (SELECT COUNT(*) FROM production_minus_stage)
    AS production_missing_from_stage_count
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
    ]
)

audit_df = client.query(audit_sql, job_config=job_config).result().to_dataframe()
display(audit_df)

result = audit_df.iloc[0]

if (
    int(result["stage_row_count"]) != 446_400
    or int(result["production_run_row_count"]) != 446_400
    or int(result["stage_missing_from_production_count"]) != 0
    or int(result["production_missing_from_stage_count"]) != 0
):
    raise RuntimeError(
        "March promotion audit did not pass. "
        "The stage table has been retained and nothing was deleted."
    )

print("✅ MARCH 2021 FINAL PROMOTION AUDIT PASSED")
print("✅ Stage and production are an exact match: 446,400 rows.")
print("✅ Safe to delete only the temporary March stage table next.")

In [ ]:
# SECTION 34I — Delete verified temporary March 2021 stage only

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202103"
)

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

RUN_ID = "bess-effective-reconstructed-2021-03"

# Final safety check before deleting the temporary table.
production_rows = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{TARGET_TABLE}`
        WHERE _RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

if production_rows != 446_400:
    raise RuntimeError(
        f"Safety stop: expected 446,400 promoted March rows, "
        f"found {production_rows:,}. Stage was not deleted."
    )

client.delete_table(STAGE_TABLE, not_found_ok=False)

print("✅ March 2021 temporary stage deleted.")
print(f"✅ Production archive retains {production_rows:,} reconstructed March rows.")
print("✅ Manifest entry remains intact.")

In [ ]:
# SECTION 35A — Historical reconstruction batch preflight
# Read-only: checks BigQuery metadata and AEMO source availability.
# No production, manifest, or staging tables are changed.

import time
import requests
import pandas as pd

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

CROSSWALK_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BESS_DUID_ALIAS_CROSSWALK_V1"
)
MANIFEST_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}.BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

# Remaining reconstruction gap after the successful March 2021 pilot.
month_plan = pd.DataFrame(
    {"SETTLEMENT_MONTH": pd.date_range("2021-04-01", "2024-07-01", freq="MS")}
)
month_plan["ARCHIVE_YEAR"] = month_plan["SETTLEMENT_MONTH"].dt.year
month_plan["ARCHIVE_MONTH"] = month_plan["SETTLEMENT_MONTH"].dt.month

def aemo_monthly_url(table_name, month_start):
    year = month_start.year
    month = month_start.month
    yyyymm = month_start.strftime("%Y%m")
    file_name = f"PUBLIC_DVD_{table_name}_{yyyymm}010000.zip"
    return (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
        f"{file_name}"
    )

def archive_status(url, attempts=3):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Range": "bytes=0-0",
    }

    last_error = None

    for attempt in range(1, attempts + 1):
        response = None
        try:
            response = requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=60,
            )
            return response.status_code, None
        except requests.RequestException as error:
            last_error = str(error)
            if attempt < attempts:
                time.sleep(3 * attempt)
        finally:
            if response is not None:
                response.close()

    return None, last_error

# 1. Inspect the stored alias crosswalk rather than assuming its structure.
crosswalk = client.get_table(CROSSWALK_TABLE)

print("BESS alias crosswalk:", CROSSWALK_TABLE)
print("Crosswalk columns:")
print([field.name for field in crosswalk.schema])

crosswalk_sample = client.query(
    f"""
    SELECT *
    FROM `{CROSSWALK_TABLE}`
    ORDER BY SOURCE_DUID
    LIMIT 20
    """
).result().to_dataframe()

display(crosswalk_sample)

# 2. Show existing reconstructed runs, so the later batch can resume safely.
existing_runs = client.query(
    f"""
    SELECT
      ARCHIVE_YEAR,
      ARCHIVE_MONTH,
      RUN_ID,
      STATUS,
      LOADED_ROWS
    FROM `{MANIFEST_TABLE}`
    WHERE SOURCE_METHOD = 'BIDPEROFFER_PLUS_DISPATCHOFFERTRK_RECONSTRUCTED'
    ORDER BY ARCHIVE_YEAR, ARCHIVE_MONTH
    """
).result().to_dataframe()

print("\nExisting reconstructed-month manifest entries:")
display(existing_runs)

# 3. Every target month needs:
#    - that month's ordinary BIDPEROFFER archive;
#    - that month's tracker archive;
#    - next month's tracker archive for the final 00:05–04:00 boundary.
checks = []

for _, item in month_plan.iterrows():
    target_month = item["SETTLEMENT_MONTH"].to_pydatetime()
    next_month = (item["SETTLEMENT_MONTH"] + pd.offsets.MonthBegin(1)).to_pydatetime()

    checks.extend([
        {
            "SETTLEMENT_MONTH": item["SETTLEMENT_MONTH"],
            "SOURCE_ROLE": "RAW_BIDPEROFFER",
            "SOURCE_MONTH": item["SETTLEMENT_MONTH"],
            "SOURCE_URL": aemo_monthly_url("BIDPEROFFER", target_month),
        },
        {
            "SETTLEMENT_MONTH": item["SETTLEMENT_MONTH"],
            "SOURCE_ROLE": "TRACKER_CURRENT_MONTH",
            "SOURCE_MONTH": item["SETTLEMENT_MONTH"],
            "SOURCE_URL": aemo_monthly_url("DISPATCHOFFERTRK", target_month),
        },
        {
            "SETTLEMENT_MONTH": item["SETTLEMENT_MONTH"],
            "SOURCE_ROLE": "TRACKER_NEXT_MONTH_BOUNDARY",
            "SOURCE_MONTH": pd.Timestamp(next_month),
            "SOURCE_URL": aemo_monthly_url("DISPATCHOFFERTRK", next_month),
        },
    ])

availability = pd.DataFrame(checks)
results = []

for index, row in availability.iterrows():
    status_code, error = archive_status(row["SOURCE_URL"])

    results.append({
        **row.to_dict(),
        "HTTP_STATUS": status_code,
        "AVAILABLE": status_code in (200, 206),
        "ERROR": error,
    })

    if (index + 1) % 15 == 0 or index + 1 == len(availability):
        print(f"Checked {index + 1:,} / {len(availability):,} required archives")

availability_df = pd.DataFrame(results)

summary_df = (
    availability_df
    .groupby(["SOURCE_ROLE", "AVAILABLE"], dropna=False)
    .size()
    .reset_index(name="archive_count")
    .sort_values(["SOURCE_ROLE", "AVAILABLE"])
)

missing_df = availability_df[
    ~availability_df["AVAILABLE"]
].sort_values(["SETTLEMENT_MONTH", "SOURCE_ROLE"])

print("\nHISTORICAL RECONSTRUCTION ARCHIVE AVAILABILITY")
display(summary_df)

if len(missing_df) > 0:
    print("\n⚠️ Unavailable or failed archive checks:")
    display(
        missing_df[
            [
                "SETTLEMENT_MONTH",
                "SOURCE_ROLE",
                "SOURCE_MONTH",
                "HTTP_STATUS",
                "ERROR",
                "SOURCE_URL",
            ]
        ]
    )
    print(
        "\nPreflight finished. No historical batch has started and no "
        "BigQuery data was changed."
    )
else:
    print("\n✅ All required April 2021–July 2024 archives are reachable.")
    print("✅ No historical batch has started yet; this was preflight only.")

In [ ]:
# SECTION 35B — Discover newer BIDOFFERPERIOD source archives
# Read-only: no BigQuery writes and no data downloads.

import re
import time
from html import unescape
from urllib.parse import urljoin

import pandas as pd
import requests

def aemo_data_directory(month_start):
    year = month_start.year
    month = month_start.month

    return (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

def get_aemo_listing(directory_url, attempts=3):
    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, attempts + 1):
        response = None
        try:
            response = requests.get(
                directory_url,
                headers=headers,
                timeout=60,
            )
            response.raise_for_status()
            return response.text
        except requests.RequestException as error:
            if attempt == attempts:
                raise RuntimeError(
                    f"Could not read AEMO directory after {attempts} attempts:\n"
                    f"{directory_url}\n{error}"
                )
            print(f"⚠️ Listing attempt {attempt}/{attempts} failed; retrying...")
            time.sleep(5 * attempt)
        finally:
            if response is not None:
                response.close()

def extract_zip_urls(directory_url, listing_html, required_text):
    # Extract ZIP links without requiring BeautifulSoup.
    hrefs = re.findall(
        r"""href\s*=\s*["']([^"']+\.zip(?:\?[^"']*)?)["']""",
        listing_html,
        flags=re.IGNORECASE,
    )

    urls = []
    for href in hrefs:
        full_url = urljoin(directory_url, unescape(href))
        if required_text.upper() in unescape(full_url).upper():
            urls.append(full_url)

    return sorted(set(urls))

new_format_months = pd.date_range("2022-06-01", "2024-07-01", freq="MS")

discovered = []

for number, settlement_month in enumerate(new_format_months, start=1):
    directory_url = aemo_data_directory(settlement_month.to_pydatetime())

    print(
        f"Discovering {settlement_month.strftime('%Y-%m')} "
        f"({number}/{len(new_format_months)})..."
    )

    listing_html = get_aemo_listing(directory_url)

    raw_urls = extract_zip_urls(
        directory_url=directory_url,
        listing_html=listing_html,
        required_text="BIDOFFERPERIOD",
    )

    if not raw_urls:
        print("   ⚠️ No BIDOFFERPERIOD ZIP files found.")
    else:
        print(f"   ✅ Found {len(raw_urls)} BIDOFFERPERIOD ZIP file(s).")

    for source_url in raw_urls:
        source_file_name = source_url.rsplit("/", 1)[-1]

        discovered.append({
            "SETTLEMENT_MONTH": settlement_month,
            "SOURCE_FILE_NAME": source_file_name,
            "SOURCE_URL": source_url,
        })

discovered_raw_df = pd.DataFrame(discovered)

month_summary = (
    discovered_raw_df
    .groupby("SETTLEMENT_MONTH")
    .size()
    .reset_index(name="bidofferperiod_archive_count")
)

expected_months = pd.DataFrame({"SETTLEMENT_MONTH": new_format_months})

coverage = (
    expected_months
    .merge(month_summary, on="SETTLEMENT_MONTH", how="left")
    .fillna({"bidofferperiod_archive_count": 0})
)

coverage["bidofferperiod_archive_count"] = (
    coverage["bidofferperiod_archive_count"].astype(int)
)

print("\nNEW-FORMAT RAW-ARCHIVE COVERAGE")
display(coverage)

missing_new_format_months = coverage[
    coverage["bidofferperiod_archive_count"] == 0
]

if len(missing_new_format_months) > 0:
    print("\n⚠️ Months with no discovered BIDOFFERPERIOD ZIP:")
    display(missing_new_format_months)
    print("\nDiscovery complete. Nothing has been loaded or changed.")
else:
    print("\n✅ Every June 2022–July 2024 month has raw BIDOFFERPERIOD source files.")
    print(
        f"✅ Total newer raw archives discovered: "
        f"{len(discovered_raw_df):,}"
    )

print("\nExample discovered sources:")
display(discovered_raw_df.head(20))

In [ ]:
# SECTION 35B1 — Inspect AEMO archive-directory naming at key boundaries
# Read-only. No BigQuery writes and no archive downloads.

import re
import requests
import pandas as pd

def aemo_data_directory(month_start_text):
    month_start = pd.Timestamp(month_start_text)
    return (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{month_start.year}/"
        f"MMSDM_{month_start.year}_{month_start.month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

def fetch_listing(url):
    response = None
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=60,
        )
        response.raise_for_status()
        return response.text
    finally:
        if response is not None:
            response.close()

# These dates bracket the apparent source-format change.
inspect_months = [
    "2022-05-01",
    "2022-06-01",
    "2023-04-01",
    "2024-07-01",
    "2024-08-01",
]

for month_text in inspect_months:
    directory_url = aemo_data_directory(month_text)
    listing_html = fetch_listing(directory_url)

    # Pull any text fragments that include BID or OFFER, whether or not
    # they occur inside a conventional HTML href.
    fragments = re.findall(
        r"[^\"'<>\s]{0,100}(?:BID|OFFER)[^\"'<>\s]{0,160}",
        listing_html,
        flags=re.IGNORECASE,
    )

    # Keep only archive-like filenames / URL fragments, deduplicated.
    likely_files = sorted({
        fragment
        for fragment in fragments
        if (
            ".zip" in fragment.lower()
            or "BIDPEROFFER" in fragment.upper()
            or "BIDOFFERPERIOD" in fragment.upper()
            or "DISPATCHOFFERTRK" in fragment.upper()
        )
    })

    print("\n" + "=" * 100)
    print(f"MONTH: {month_text}")
    print(f"DIRECTORY: {directory_url}")
    print(f"Listing characters received: {len(listing_html):,}")
    print(f"Relevant BID/OFFER fragments found: {len(likely_files):,}")

    if likely_files:
        for item in likely_files[:80]:
            print(item)
    else:
        print("No matching filenames were visible.")
        print("First 1,000 listing characters for diagnosis:")
        print(listing_html[:1000])

In [ ]:
# SECTION 35C — Inspect split BIDPEROFFER1 / BIDPEROFFER2 source schemas
# Read-only. No BigQuery writes.

import os
import io
import csv
import time
import zipfile
import tempfile
import requests
import pandas as pd

def aemo_data_directory(month_start_text):
    month_start = pd.Timestamp(month_start_text)
    return (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{month_start.year}/"
        f"MMSDM_{month_start.year}_{month_start.month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

def download_aemo_zip(source_url, label, attempts=5):
    temp_path = None
    last_error = None

    for attempt in range(1, attempts + 1):
        response = None

        try:
            print(f"Downloading {label} — attempt {attempt}/{attempts}...")

            response = requests.get(
                source_url,
                headers={"User-Agent": "Mozilla/5.0"},
                stream=True,
                timeout=300,
            )
            response.raise_for_status()

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".zip",
            ) as temp_file:
                temp_path = temp_file.name

                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        temp_file.write(chunk)

            return temp_path

        except requests.RequestException as error:
            last_error = error

            if temp_path and os.path.exists(temp_path):
                os.remove(temp_path)

            temp_path = None

            if attempt < attempts:
                wait_seconds = 5 * attempt
                print(f"⚠️ Download failed. Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

        finally:
            if response is not None:
                response.close()

    raise RuntimeError(
        f"Could not download {label} after {attempts} attempts:\n{last_error}"
    )

def inspect_aemo_zip(source_url, label):
    temp_zip_path = None

    try:
        temp_zip_path = download_aemo_zip(source_url, label)

        with zipfile.ZipFile(temp_zip_path) as zip_file:
            csv_names = [
                name for name in zip_file.namelist()
                if name.upper().endswith(".CSV")
            ]

            if len(csv_names) != 1:
                raise RuntimeError(
                    f"{label}: expected one CSV, found {csv_names}"
                )

            csv_name = csv_names[0]

            with zip_file.open(csv_name) as raw_csv:
                text_csv = io.TextIOWrapper(
                    raw_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
                reader = csv.reader(text_csv)

                header = None
                samples = []

                for record in reader:
                    if not record:
                        continue

                    record_type = record[0].strip().upper()

                    if record_type == "I":
                        header = [value.strip() for value in record[4:]]
                    elif record_type == "D":
                        samples.append(record[4:])

                        if len(samples) >= 3:
                            break

            print("\n" + "=" * 100)
            print(label)
            print("ZIP:", source_url.rsplit("/", 1)[-1])
            print("CSV:", csv_name)
            print("Source columns:", header)

            if header is None:
                raise RuntimeError(f"{label}: no AEMO column-definition record found.")

            sample_df = pd.DataFrame(samples, columns=header)
            display(sample_df)

            return header

    finally:
        if temp_zip_path and os.path.exists(temp_zip_path):
            os.remove(temp_zip_path)
            print(f"✅ Temporary ZIP deleted: {label}")

sources_to_inspect = [
    (
        "June 2022 — BIDPEROFFER1",
        aemo_data_directory("2022-06-01")
        + "PUBLIC_DVD_BIDPEROFFER1_202206010000.zip",
    ),
    (
        "June 2022 — BIDPEROFFER2",
        aemo_data_directory("2022-06-01")
        + "PUBLIC_DVD_BIDPEROFFER2_202206010000.zip",
    ),
    (
        "July 2024 — BIDPEROFFER1",
        aemo_data_directory("2024-07-01")
        + "PUBLIC_DVD_BIDPEROFFER1_202407010000.zip",
    ),
    (
        "July 2024 — BIDPEROFFER2",
        aemo_data_directory("2024-07-01")
        + "PUBLIC_DVD_BIDPEROFFER2_202407010000.zip",
    ),
]

schemas = {}

for label, source_url in sources_to_inspect:
    schemas[label] = inspect_aemo_zip(source_url, label)

print("\nSCHEMA COMPARISON")
for label, columns in schemas.items():
    print(f"{label}: {len(columns)} columns")

all_same_schema = len({tuple(columns) for columns in schemas.values()}) == 1

if all_same_schema:
    print("✅ All four split-archive sources have the same schema.")
else:
    print("⚠️ Source schemas differ. Do not start the batch yet.")

In [ ]:
# SECTION 35D — BULK STAGE: April 2021 to July 2024
# Creates and verifies one stage table per month.
# Does NOT promote rows into BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1.

import os
import io
import csv
import time
import zipfile
import tempfile
from datetime import timedelta

import pandas as pd
import requests
from google.cloud import bigquery

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
MANIFEST_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)
CROSSWALK_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BESS_DUID_ALIAS_CROSSWALK_V1"
)
SCOPE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BESS_DUID_BACKFILL_SCOPE_V1"
)

BATCH_START = pd.Timestamp("2021-04-01")
BATCH_END = pd.Timestamp("2024-07-01")

OUTPUT_CHUNK_ROWS = 50_000
STOP_ON_FIRST_FAILURE = True

# ------------------------------------------------------------
# Source locations
# ------------------------------------------------------------

def aemo_data_directory(month_start):
    return (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{month_start.year}/"
        f"MMSDM_{month_start.year}_{month_start.month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

def raw_source_urls(month_start):
    directory = aemo_data_directory(month_start)
    stamp = month_start.strftime("%Y%m") + "010000"

    # The ordinary raw-offer archive split into two files in June 2022.
    if month_start <= pd.Timestamp("2022-05-01"):
        file_names = [f"PUBLIC_DVD_BIDPEROFFER_{stamp}.zip"]
    else:
        file_names = [
            f"PUBLIC_DVD_BIDPEROFFER1_{stamp}.zip",
            f"PUBLIC_DVD_BIDPEROFFER2_{stamp}.zip",
        ]

    return [(file_name, directory + file_name) for file_name in file_names]

def tracker_source_url(month_start):
    directory = aemo_data_directory(month_start)
    stamp = month_start.strftime("%Y%m") + "010000"

    # August 2024 changed to the newer archive filename convention.
    if month_start >= pd.Timestamp("2024-08-01"):
        file_name = (
            "PUBLIC_ARCHIVE%23DISPATCHOFFERTRK%23"
            f"FILE01%23{stamp}.zip"
        )
    else:
        file_name = f"PUBLIC_DVD_DISPATCHOFFERTRK_{stamp}.zip"

    return file_name, directory + file_name

# ------------------------------------------------------------
# Download and CSV helpers
# ------------------------------------------------------------

def download_aemo_zip(source_url, label, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        temp_path = None
        response = None

        try:
            print(f"Downloading {label} — attempt {attempt}/{attempts}...")

            response = requests.get(
                source_url,
                headers={"User-Agent": "Mozilla/5.0"},
                stream=True,
                timeout=300,
            )
            response.raise_for_status()

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".zip",
            ) as temp_file:
                temp_path = temp_file.name

                for block in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if block:
                        temp_file.write(block)

            return temp_path

        except requests.RequestException as error:
            last_error = error

            if temp_path and os.path.exists(temp_path):
                os.remove(temp_path)

            if attempt < attempts:
                wait_seconds = attempt * 5
                print(f"⚠️ Download failed. Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

        finally:
            if response is not None:
                response.close()

    raise RuntimeError(
        f"Could not download {label} after {attempts} attempts: {last_error}"
    )

def read_aemo_records(zip_path):
    with zipfile.ZipFile(zip_path) as zip_file:
        csv_files = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_files) != 1:
            raise RuntimeError(
                f"Expected exactly one CSV in archive; found {csv_files}"
            )

        with zip_file.open(csv_files[0]) as binary_csv:
            text_csv = io.TextIOWrapper(
                binary_csv,
                encoding="utf-8-sig",
                newline="",
            )
            reader = csv.reader(text_csv)

            header = None

            for record in reader:
                if not record:
                    continue

                record_type = record[0].strip().upper()

                if record_type == "I":
                    header = [value.strip().upper() for value in record[4:]]
                    continue

                if record_type == "D":
                    if header is None:
                        raise RuntimeError(
                            "Found a data record before the AEMO header record."
                        )

                    yield header, record[4:]

def normalise_key_timestamp(value):
    parsed = pd.to_datetime(value, errors="raise")

    if pd.isna(parsed):
        raise RuntimeError(f"Could not parse required key timestamp: {value}")

    return parsed.strftime("%Y-%m-%d %H:%M:%S.%f")

def to_utc_timestamp(value):
    if value in (None, "", "NULL"):
        return None

    parsed = pd.to_datetime(value, errors="coerce", utc=True)

    if pd.isna(parsed):
        return None

    return parsed

def to_number(value):
    if value in (None, "", "NULL"):
        return None

    return pd.to_numeric(value, errors="coerce")

def trading_period_from_interval(interval_timestamp):
    # AEMO trading day is 04:05 through 04:00 next calendar day.
    minutes_after_midnight = (
        interval_timestamp.hour * 60 + interval_timestamp.minute
    )

    return ((minutes_after_midnight - 245) % 1440) // 5 + 1

def market_day_from_interval(interval_timestamp):
    # 04:00 belongs to the preceding trading day.
    if (
        interval_timestamp.hour < 4
        or (
            interval_timestamp.hour == 4
            and interval_timestamp.minute == 0
        )
    ):
        return (interval_timestamp - pd.Timedelta(days=1)).normalize()

    return interval_timestamp.normalize()

def legacy_direction(source_duid):
    # Earlier raw files do not contain DIRECTION.
    # Historic load aliases consistently end in L or L1.
    if source_duid.endswith("L") or source_duid.endswith("L1"):
        return "LOAD"

    # The remaining historic aliases are generation-side identifiers.
    return "GEN"

# ------------------------------------------------------------
# Read BESS source-DUID scope from the warehouse
# ------------------------------------------------------------

bess_scope_sql = f"""
SELECT DISTINCT UPPER(TRIM(SOURCE_DUID)) AS DUID
FROM `{CROSSWALK_TABLE}`
WHERE SOURCE_DUID IS NOT NULL

UNION DISTINCT

SELECT DISTINCT UPPER(TRIM(DUID)) AS DUID
FROM `{SCOPE_TABLE}`
WHERE DUID IS NOT NULL
"""

bess_scope_df = client.query(bess_scope_sql).result().to_dataframe()

BESS_SOURCE_DUIDS = set(bess_scope_df["DUID"].dropna().tolist())

if not BESS_SOURCE_DUIDS:
    raise RuntimeError("No BESS source DUIDs were returned from the registry.")

print(f"BESS source DUIDs in bulk scope: {len(BESS_SOURCE_DUIDS):,}")

target_metadata = client.get_table(TARGET_TABLE)
target_schema = target_metadata.schema
target_field_names = [field.name for field in target_schema]

# ------------------------------------------------------------
# Tracker extraction
# ------------------------------------------------------------

def read_tracker_targets(month_start):
    next_month = month_start + pd.offsets.MonthBegin(1)

    target_start = pd.Timestamp(
        f"{month_start.strftime('%Y-%m-%d')} 04:05:00",
        tz="UTC",
    )
    target_end = pd.Timestamp(
        f"{next_month.strftime('%Y-%m-%d')} 04:00:00",
        tz="UTC",
    )

    tracker_sources = [
        tracker_source_url(month_start),
        tracker_source_url(next_month),
    ]

    targets_by_raw_key = {}
    seen_intervals = set()
    tracker_rows_read = 0

    for source_file_name, source_url in tracker_sources:
        temp_zip_path = None

        try:
            temp_zip_path = download_aemo_zip(
                source_url,
                f"tracker {source_file_name}",
            )

            for header, values in read_aemo_records(temp_zip_path):
                row = dict(zip(header, values))

                duid = row["DUID"].strip().upper()

                if duid not in BESS_SOURCE_DUIDS:
                    continue

                interval_timestamp = to_utc_timestamp(
                    row["SETTLEMENTDATE"]
                )

                if (
                    interval_timestamp is None
                    or interval_timestamp < target_start
                    or interval_timestamp > target_end
                ):
                    continue

                bidtype = row["BIDTYPE"].strip().upper()
                bidsettlement_key = normalise_key_timestamp(
                    row["BIDSETTLEMENTDATE"]
                )
                bidoffer_key = normalise_key_timestamp(
                    row["BIDOFFERDATE"]
                )
                period_id = int(
                    trading_period_from_interval(interval_timestamp)
                )

                raw_key = (
                    duid,
                    bidtype,
                    bidsettlement_key,
                    bidoffer_key,
                    period_id,
                )

                interval_key = raw_key + (interval_timestamp.isoformat(),)

                if interval_key in seen_intervals:
                    raise RuntimeError(
                        "Duplicate tracker interval detected: "
                        f"{interval_key}"
                    )

                seen_intervals.add(interval_key)

                targets_by_raw_key.setdefault(raw_key, []).append(
                    {
                        "INTERVAL_DATETIME": interval_timestamp,
                        "SETTLEMENTDATE": market_day_from_interval(
                            interval_timestamp
                        ),
                        "BIDSETTLEMENTDATE": to_utc_timestamp(
                            row["BIDSETTLEMENTDATE"]
                        ),
                        "OFFERDATE": to_utc_timestamp(
                            row["BIDOFFERDATE"]
                        ),
                    }
                )

                tracker_rows_read += 1

        finally:
            if temp_zip_path and os.path.exists(temp_zip_path):
                os.remove(temp_zip_path)

    if not targets_by_raw_key:
        raise RuntimeError(
            f"No BESS tracker targets found for {month_start:%Y-%m}."
        )

    return (
        targets_by_raw_key,
        tracker_rows_read,
        target_start,
        target_end,
    )

# ------------------------------------------------------------
# Stage one month
# ------------------------------------------------------------

def append_stage_rows(stage_table, output_rows):
    if not output_rows:
        return

    output_df = pd.DataFrame(output_rows, columns=target_field_names)

    timestamp_columns = [
        "SETTLEMENTDATE",
        "INTERVAL_DATETIME",
        "BIDSETTLEMENTDATE",
        "OFFERDATE",
        "LASTCHANGED",
        "_LOAD_DATE",
    ]

    integer_columns = ["PERIODID"]

    float_columns = [
        "VERSIONNO",
        "MAXAVAIL",
        "FIXEDLOAD",
        "ROCUP",
        "ROCDOWN",
        "ENABLEMENTMIN",
        "ENABLEMENTMAX",
        "LOWBREAKPOINT",
        "HIGHBREAKPOINT",
        "BANDAVAIL1",
        "BANDAVAIL2",
        "BANDAVAIL3",
        "BANDAVAIL4",
        "BANDAVAIL5",
        "BANDAVAIL6",
        "BANDAVAIL7",
        "BANDAVAIL8",
        "BANDAVAIL9",
        "BANDAVAIL10",
        "PASAAVAILABILITY",
        "MR_CAPACITY",
        "ENERGYLIMIT",
    ]

    for column in timestamp_columns:
        output_df[column] = pd.to_datetime(
            output_df[column],
            errors="coerce",
            utc=True,
        )

    for column in integer_columns:
        output_df[column] = pd.to_numeric(
            output_df[column],
            errors="coerce",
        ).astype("Int64")

    for column in float_columns:
        output_df[column] = pd.to_numeric(
            output_df[column],
            errors="coerce",
        )

    load_job = client.load_table_from_dataframe(
        output_df,
        stage_table,
        job_config=bigquery.LoadJobConfig(
            schema=target_schema,
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        ),
    )

    load_job.result()

def stage_month(month_start):
    year = int(month_start.year)
    month = int(month_start.month)
    yyyymm = month_start.strftime("%Y%m")

    run_id = f"bess-effective-reconstructed-{month_start:%Y-%m}"
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{yyyymm}"
    )

    print("\n" + "=" * 100)
    print(f"STARTING STAGE: {month_start:%Y-%m}")
    print("=" * 100)

    manifest_check = client.query(
        f"""
        SELECT STATUS, LOADED_ROWS
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    run_id,
                )
            ]
        ),
    ).result().to_dataframe()

    if len(manifest_check) > 0:
        print(
            f"⏭️ Skipping {month_start:%Y-%m}: "
            f"manifest already has status "
            f"{manifest_check.iloc[0]['STATUS']}."
        )
        return {
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "SKIPPED_EXISTING_MANIFEST",
        }

    try:
        client.get_table(stage_table)

        raise RuntimeError(
            f"Stage table already exists without a manifest entry: "
            f"{stage_table}. Nothing was overwritten."
        )

    except bigquery.NotFound:
        pass

    production_count = int(
        client.query(
            f"""
            SELECT COUNT(*) AS row_count
            FROM `{TARGET_TABLE}`
            WHERE _RUN_ID = @run_id
            """,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[
                    bigquery.ScalarQueryParameter(
                        "run_id",
                        "STRING",
                        run_id,
                    )
                ]
            ),
        ).result().to_dataframe().iloc[0]["row_count"]
    )

    if production_count != 0:
        raise RuntimeError(
            f"Production already contains {production_count:,} rows "
            f"for {run_id}; staging stopped."
        )

    targets_by_raw_key, expected_rows, target_start, target_end = (
        read_tracker_targets(month_start)
    )

    print(
        f"Tracker target window: {target_start} to {target_end}"
    )
    print(f"Expected effective BESS rows: {expected_rows:,}")
    print(
        f"Unique raw offer identities required: "
        f"{len(targets_by_raw_key):,}"
    )

    stage = bigquery.Table(stage_table, schema=target_schema)
    stage.time_partitioning = bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY,
        field="SETTLEMENTDATE",
    )
    stage.clustering_fields = ["DUID", "BIDTYPE", "DIRECTION"]

    client.create_table(stage)

    unmatched_raw_keys = set(targets_by_raw_key.keys())
    matched_raw_keys = set()

    source_rows_read = 0
    bess_raw_rows_read = 0
    output_rows_written = 0
    output_buffer = []

    raw_sources = raw_source_urls(month_start)

    try:
        for source_file_name, source_url in raw_sources:
            temp_zip_path = None

            try:
                temp_zip_path = download_aemo_zip(
                    source_url,
                    f"raw source {source_file_name}",
                )

                for header, values in read_aemo_records(temp_zip_path):
                    source_rows_read += 1
                    row = dict(zip(header, values))

                    duid = row["DUID"].strip().upper()

                    if duid not in BESS_SOURCE_DUIDS:
                        continue

                    bess_raw_rows_read += 1

                    bidtype = row["BIDTYPE"].strip().upper()
                    tradingdate_key = normalise_key_timestamp(
                        row["TRADINGDATE"]
                    )
                    offerdatetime_key = normalise_key_timestamp(
                        row["OFFERDATETIME"]
                    )
                    period_id = int(row["PERIODID"])

                    raw_key = (
                        duid,
                        bidtype,
                        tradingdate_key,
                        offerdatetime_key,
                        period_id,
                    )

                    tracker_matches = targets_by_raw_key.get(raw_key)

                    if not tracker_matches:
                        continue

                    if raw_key in matched_raw_keys:
                        raise RuntimeError(
                            "Duplicate raw offer identity found across "
                            f"the source files: {raw_key}"
                        )

                    matched_raw_keys.add(raw_key)
                    unmatched_raw_keys.discard(raw_key)

                    source_direction = (
                        row.get("DIRECTION", "").strip().upper()
                    )

                    direction = (
                        source_direction
                        if source_direction
                        else legacy_direction(duid)
                    )

                    for tracker_target in tracker_matches:
                        output_buffer.append({
                            "SETTLEMENTDATE":
                                tracker_target["SETTLEMENTDATE"],
                            "DUID": duid,
                            "BIDTYPE": bidtype,
                            "DIRECTION": direction,
                            "INTERVAL_DATETIME":
                                tracker_target["INTERVAL_DATETIME"],
                            "BIDSETTLEMENTDATE":
                                tracker_target["BIDSETTLEMENTDATE"],
                            "OFFERDATE":
                                tracker_target["OFFERDATE"],
                            "PERIODID": period_id,
                            "VERSIONNO": None,
                            "MAXAVAIL": to_number(row.get("MAXAVAIL")),
                            "FIXEDLOAD": to_number(row.get("FIXEDLOAD")),
                            "ROCUP": to_number(
                                row.get("RAMPUPRATE")
                            ),
                            "ROCDOWN": to_number(
                                row.get("RAMPDOWNRATE")
                            ),
                            "ENABLEMENTMIN": to_number(
                                row.get("ENABLEMENTMIN")
                            ),
                            "ENABLEMENTMAX": to_number(
                                row.get("ENABLEMENTMAX")
                            ),
                            "LOWBREAKPOINT": to_number(
                                row.get("LOWBREAKPOINT")
                            ),
                            "HIGHBREAKPOINT": to_number(
                                row.get("HIGHBREAKPOINT")
                            ),
                            "BANDAVAIL1": to_number(
                                row.get("BANDAVAIL1")
                            ),
                            "BANDAVAIL2": to_number(
                                row.get("BANDAVAIL2")
                            ),
                            "BANDAVAIL3": to_number(
                                row.get("BANDAVAIL3")
                            ),
                            "BANDAVAIL4": to_number(
                                row.get("BANDAVAIL4")
                            ),
                            "BANDAVAIL5": to_number(
                                row.get("BANDAVAIL5")
                            ),
                            "BANDAVAIL6": to_number(
                                row.get("BANDAVAIL6")
                            ),
                            "BANDAVAIL7": to_number(
                                row.get("BANDAVAIL7")
                            ),
                            "BANDAVAIL8": to_number(
                                row.get("BANDAVAIL8")
                            ),
                            "BANDAVAIL9": to_number(
                                row.get("BANDAVAIL9")
                            ),
                            "BANDAVAIL10": to_number(
                                row.get("BANDAVAIL10")
                            ),
                            "LASTCHANGED": None,
                            "PASAAVAILABILITY": to_number(
                                row.get("PASAAVAILABILITY")
                            ),
                            "MR_CAPACITY": None,
                            "ENERGYLIMIT": to_number(
                                row.get("ENERGYLIMIT")
                            ),
                            "_LOAD_DATE": pd.Timestamp.now(tz="UTC"),
                            "_SOURCE": "AEMO_NEMWEB_ARCHIVE",
                            "_PIPELINE":
                                "BIDPEROFFER_PLUS_DISPATCHOFFERTRK_"
                                "RECONSTRUCTED_STAGE",
                            "_SOURCE_FILE": source_file_name,
                            "_SOURCE_URL": source_url,
                            "_RUN_ID": run_id,
                        })

                    if len(output_buffer) >= OUTPUT_CHUNK_ROWS:
                        append_stage_rows(stage_table, output_buffer)
                        output_rows_written += len(output_buffer)
                        output_buffer = []

                        print(
                            f"{month_start:%Y-%m}: "
                            f"{output_rows_written:,} rows staged"
                        )

            finally:
                if temp_zip_path and os.path.exists(temp_zip_path):
                    os.remove(temp_zip_path)

        if output_buffer:
            append_stage_rows(stage_table, output_buffer)
            output_rows_written += len(output_buffer)

        if unmatched_raw_keys:
            examples = list(unmatched_raw_keys)[:10]

            raise RuntimeError(
                f"{month_start:%Y-%m}: "
                f"{len(unmatched_raw_keys):,} tracker offer identities "
                "were not found in the raw source. "
                f"Examples: {examples}"
            )

        audit_df = client.query(
            f"""
            WITH distinct_effective_keys AS (
              SELECT
                DUID,
                BIDTYPE,
                DIRECTION,
                INTERVAL_DATETIME
              FROM `{stage_table}`
              GROUP BY
                DUID,
                BIDTYPE,
                DIRECTION,
                INTERVAL_DATETIME
            )
            SELECT
              COUNT(*) AS stage_row_count,
              (SELECT COUNT(*) FROM distinct_effective_keys)
                AS distinct_effective_key_count,
              COUNT(DISTINCT DUID) AS source_duid_count,
              COUNTIF(
                DUID IS NULL
                OR BIDTYPE IS NULL
                OR DIRECTION IS NULL
                OR INTERVAL_DATETIME IS NULL
                OR OFFERDATE IS NULL
                OR PERIODID IS NULL
              ) AS null_required_field_count
            FROM `{stage_table}`
            """
        ).result().to_dataframe()

        display(audit_df)

        audit = audit_df.iloc[0]

        if (
            int(audit["stage_row_count"]) != expected_rows
            or int(audit["distinct_effective_key_count"]) != expected_rows
            or int(audit["null_required_field_count"]) != 0
        ):
            raise RuntimeError(
                f"{month_start:%Y-%m}: stage audit failed. "
                "The stage was retained and no manifest was written."
            )

        manifest_row = {
            "RUN_ID": run_id,
            "ARCHIVE_YEAR": year,
            "ARCHIVE_MONTH": month,
            "SOURCE_FILE_NAME": " + ".join(
                file_name for file_name, _ in raw_sources
            ),
            "SOURCE_URL": " + ".join(
                source_url for _, source_url in raw_sources
            ),
            "SOURCE_METHOD":
                "BIDPEROFFER_PLUS_DISPATCHOFFERTRK_RECONSTRUCTED",
            "STATUS": "STAGED",
            "SOURCE_ROWS": int(source_rows_read),
            "BESS_ROWS": int(bess_raw_rows_read),
            "LOADED_ROWS": int(audit["stage_row_count"]),
            "STARTED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
            "COMPLETED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
            "ERROR_MESSAGE": None,
            "CREATED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
        }

        manifest_errors = client.insert_rows_json(
            MANIFEST_TABLE,
            [manifest_row],
        )

        if manifest_errors:
            raise RuntimeError(
                f"{month_start:%Y-%m}: stage passed but manifest write "
                f"failed: {manifest_errors}"
            )

        print(
            f"✅ {month_start:%Y-%m} STAGED AND AUDITED: "
            f"{int(audit['stage_row_count']):,} rows"
        )

        return {
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "STAGED",
            "STAGE_ROWS": int(audit["stage_row_count"]),
            "SOURCE_ROWS": source_rows_read,
            "BESS_RAW_ROWS": bess_raw_rows_read,
        }

    except Exception:
        print(
            f"⚠️ {month_start:%Y-%m} stopped. "
            f"The stage table, if created, was retained: {stage_table}"
        )
        raise

# ------------------------------------------------------------
# Launch the bulk staging run
# ------------------------------------------------------------

batch_months = pd.date_range(BATCH_START, BATCH_END, freq="MS")
batch_results = []

print(
    f"\nLaunching staging-only batch for {len(batch_months):,} months: "
    f"{BATCH_START:%Y-%m} to {BATCH_END:%Y-%m}"
)
print("Production table will not be modified.")

for month_start in batch_months:
    try:
        result = stage_month(month_start)
        batch_results.append(result)

    except Exception as error:
        batch_results.append({
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "FAILED",
            "ERROR": str(error),
        })

        print(f"\n❌ Batch stopped at {month_start:%Y-%m}: {error}")

        if STOP_ON_FIRST_FAILURE:
            break

batch_results_df = pd.DataFrame(batch_results)

print("\nBULK STAGING SUMMARY")
display(batch_results_df)

print("\n✅ Any successful month is staged, audited, and recorded as STAGED.")
print("✅ No staged month has been promoted into production by this cell.")

In [ ]:
# PATCH — make the BigQuery "table not found" exception available
# No BigQuery data is changed.

from google.api_core.exceptions import NotFound

bigquery.NotFound = NotFound

print("✅ BigQuery NotFound exception patched.")
print("✅ Safe to restart the staging launcher.")

In [ ]:
# RESTART SECTION 35D — launch the existing bulk staging engine

batch_results = []

print(
    f"\nRestarting staging-only batch for {len(batch_months):,} months: "
    f"{BATCH_START:%Y-%m} to {BATCH_END:%Y-%m}"
)
print("Production table will not be modified.")

for month_start in batch_months:
    try:
        result = stage_month(month_start)
        batch_results.append(result)

    except Exception as error:
        batch_results.append({
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "FAILED",
            "ERROR": str(error),
        })

        print(f"\n❌ Batch stopped at {month_start:%Y-%m}: {error}")

        if STOP_ON_FIRST_FAILURE:
            break

batch_results_df = pd.DataFrame(batch_results)

print("\nBULK STAGING SUMMARY")
display(batch_results_df)

In [ ]:
# SECTION 35E — May 2021 carry-in proof
# Read-only: no BigQuery writes, promotions, or deletions.

MAY_MONTH = pd.Timestamp("2021-05-01")
APRIL_MONTH = pd.Timestamp("2021-04-01")

MAY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202105"
)

# Rebuild May's tracker identities.
may_targets, expected_rows, _, _ = read_tracker_targets(MAY_MONTH)

# Identify identities already represented in the retained May stage.
stage_keys_df = client.query(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID
    FROM `{MAY_STAGE_TABLE}`
    """
).result().to_dataframe()

staged_raw_keys = {
    (
        row.DUID.strip().upper(),
        row.BIDTYPE.strip().upper(),
        normalise_key_timestamp(row.BIDSETTLEMENTDATE),
        normalise_key_timestamp(row.OFFERDATE),
        int(row.PERIODID),
    )
    for row in stage_keys_df.itertuples(index=False)
}

missing_keys = set(may_targets.keys()) - staged_raw_keys

print(f"May tracker target rows: {expected_rows:,}")
print(f"May stage rows retained: {len(stage_keys_df):,}")
print(f"Missing raw-offer identities to locate: {len(missing_keys):,}")

if len(missing_keys) != 576:
    raise RuntimeError(
        "Safety stop: expected 576 missing May identities, "
        f"found {len(missing_keys):,}. No source archive was scanned."
    )

april_matches = {}
temp_zip_path = None

try:
    april_file_name, april_source_url = raw_source_urls(APRIL_MONTH)[0]

    temp_zip_path = download_aemo_zip(
        april_source_url,
        f"April carry-in source {april_file_name}",
    )

    for header, values in read_aemo_records(temp_zip_path):
        row = dict(zip(header, values))

        duid = row["DUID"].strip().upper()

        if duid not in BESS_SOURCE_DUIDS:
            continue

        raw_key = (
            duid,
            row["BIDTYPE"].strip().upper(),
            normalise_key_timestamp(row["TRADINGDATE"]),
            normalise_key_timestamp(row["OFFERDATETIME"]),
            int(row["PERIODID"]),
        )

        if raw_key in missing_keys:
            if raw_key in april_matches:
                raise RuntimeError(
                    f"Duplicate April raw-offer identity found: {raw_key}"
                )

            april_matches[raw_key] = row

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print("✅ Temporary April carry-in ZIP deleted.")

still_missing = missing_keys - set(april_matches)

print("\nMAY 2021 CARRY-IN AUDIT")
print(f"Missing identities required: {len(missing_keys):,}")
print(f"Found in April raw source: {len(april_matches):,}")
print(f"Still unmatched: {len(still_missing):,}")

examples_df = pd.DataFrame(
    [
        {
            "DUID": key[0],
            "BIDTYPE": key[1],
            "BIDSETTLEMENTDATE": key[2],
            "OFFERDATE": key[3],
            "PERIODID": key[4],
        }
        for key in sorted(still_missing)[:20]
    ]
)

if len(still_missing) > 0:
    display(examples_df)
    raise RuntimeError(
        "Carry-in proof did not fully pass. "
        "No BigQuery data was changed."
    )

print("✅ MAY 2021 CARRY-IN AUDIT PASSED")
print("✅ April supplies every missing May effective-offer identity.")
print("✅ We can repair May and update the bulk loader with carry-in logic.")

In [ ]:
# SECTION 35F — Repair May 2021 stage using proven April carry-in offers
# Uses april_matches and may_targets from the completed carry-in audit.
# No production promotion or deletion occurs.

MAY_MONTH = pd.Timestamp("2021-05-01")
MAY_RUN_ID = "bess-effective-reconstructed-2021-05"

MAY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202105"
)

APRIL_FILE_NAME, APRIL_SOURCE_URL = raw_source_urls(
    pd.Timestamp("2021-04-01")
)[0]

required_variables = [
    "april_matches",
    "may_targets",
    "missing_keys",
    "append_stage_rows",
]

for variable_name in required_variables:
    if variable_name not in globals():
        raise RuntimeError(
            f"Required in-memory variable missing: {variable_name}. "
            "Do not rerun this repair cell after a kernel reset."
        )

if len(missing_keys) != 576 or len(april_matches) != 576:
    raise RuntimeError(
        "Safety stop: expected 576 proven carry-in identities."
    )

existing_manifest = client.query(
    f"""
    SELECT COUNT(*) AS row_count
    FROM `{MANIFEST_TABLE}`
    WHERE RUN_ID = @run_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "run_id",
                "STRING",
                MAY_RUN_ID,
            )
        ]
    ),
).result().to_dataframe().iloc[0]["row_count"]

if int(existing_manifest) != 0:
    raise RuntimeError(
        "Safety stop: May already has a manifest entry. "
        "Nothing was added."
    )

before_count = int(
    client.query(
        f"SELECT COUNT(*) AS row_count FROM `{MAY_STAGE_TABLE}`"
    ).result().to_dataframe().iloc[0]["row_count"]
)

if before_count != 459_936:
    raise RuntimeError(
        f"Safety stop: expected 459,936 retained May stage rows, "
        f"found {before_count:,}."
    )

repair_rows = []

for raw_key in missing_keys:
    raw_row = april_matches[raw_key]

    duid = raw_key[0]
    bidtype = raw_key[1]
    period_id = raw_key[4]

    for tracker_target in may_targets[raw_key]:
        repair_rows.append({
            "SETTLEMENTDATE": tracker_target["SETTLEMENTDATE"],
            "DUID": duid,
            "BIDTYPE": bidtype,
            "DIRECTION": legacy_direction(duid),
            "INTERVAL_DATETIME": tracker_target["INTERVAL_DATETIME"],
            "BIDSETTLEMENTDATE": tracker_target["BIDSETTLEMENTDATE"],
            "OFFERDATE": tracker_target["OFFERDATE"],
            "PERIODID": period_id,
            "VERSIONNO": None,
            "MAXAVAIL": to_number(raw_row.get("MAXAVAIL")),
            "FIXEDLOAD": to_number(raw_row.get("FIXEDLOAD")),
            "ROCUP": to_number(raw_row.get("RAMPUPRATE")),
            "ROCDOWN": to_number(raw_row.get("RAMPDOWNRATE")),
            "ENABLEMENTMIN": to_number(raw_row.get("ENABLEMENTMIN")),
            "ENABLEMENTMAX": to_number(raw_row.get("ENABLEMENTMAX")),
            "LOWBREAKPOINT": to_number(raw_row.get("LOWBREAKPOINT")),
            "HIGHBREAKPOINT": to_number(raw_row.get("HIGHBREAKPOINT")),
            "BANDAVAIL1": to_number(raw_row.get("BANDAVAIL1")),
            "BANDAVAIL2": to_number(raw_row.get("BANDAVAIL2")),
            "BANDAVAIL3": to_number(raw_row.get("BANDAVAIL3")),
            "BANDAVAIL4": to_number(raw_row.get("BANDAVAIL4")),
            "BANDAVAIL5": to_number(raw_row.get("BANDAVAIL5")),
            "BANDAVAIL6": to_number(raw_row.get("BANDAVAIL6")),
            "BANDAVAIL7": to_number(raw_row.get("BANDAVAIL7")),
            "BANDAVAIL8": to_number(raw_row.get("BANDAVAIL8")),
            "BANDAVAIL9": to_number(raw_row.get("BANDAVAIL9")),
            "BANDAVAIL10": to_number(raw_row.get("BANDAVAIL10")),
            "LASTCHANGED": None,
            "PASAAVAILABILITY": to_number(
                raw_row.get("PASAAVAILABILITY")
            ),
            "MR_CAPACITY": None,
            "ENERGYLIMIT": None,
            "_LOAD_DATE": pd.Timestamp.now(tz="UTC"),
            "_SOURCE": "AEMO_NEMWEB_ARCHIVE",
            "_PIPELINE": (
                "BIDPEROFFER_PLUS_DISPATCHOFFERTRK_"
                "RECONSTRUCTED_STAGE_CARRY_IN"
            ),
            "_SOURCE_FILE": APRIL_FILE_NAME,
            "_SOURCE_URL": APRIL_SOURCE_URL,
            "_RUN_ID": MAY_RUN_ID,
        })

print(f"Carry-in raw identities: {len(april_matches):,}")
print(f"May effective rows to append: {len(repair_rows):,}")

if len(repair_rows) != 4_320:
    raise RuntimeError(
        f"Safety stop: expected 4,320 repair rows, "
        f"prepared {len(repair_rows):,}."
    )

append_stage_rows(MAY_STAGE_TABLE, repair_rows)

audit_df = client.query(
    f"""
    WITH distinct_effective_keys AS (
      SELECT
        DUID,
        BIDTYPE,
        DIRECTION,
        INTERVAL_DATETIME
      FROM `{MAY_STAGE_TABLE}`
      GROUP BY
        DUID,
        BIDTYPE,
        DIRECTION,
        INTERVAL_DATETIME
    )
    SELECT
      COUNT(*) AS stage_row_count,
      (SELECT COUNT(*) FROM distinct_effective_keys)
        AS distinct_effective_key_count,
      COUNTIF(
        DUID IS NULL
        OR BIDTYPE IS NULL
        OR DIRECTION IS NULL
        OR INTERVAL_DATETIME IS NULL
        OR OFFERDATE IS NULL
        OR PERIODID IS NULL
      ) AS null_required_field_count
    FROM `{MAY_STAGE_TABLE}`
    """
).result().to_dataframe()

display(audit_df)

audit = audit_df.iloc[0]

if (
    int(audit["stage_row_count"]) != 464_256
    or int(audit["distinct_effective_key_count"]) != 464_256
    or int(audit["null_required_field_count"]) != 0
):
    raise RuntimeError(
        "May stage audit failed. The repaired stage was retained, "
        "but no manifest was written."
    )

manifest_errors = client.insert_rows_json(
    MANIFEST_TABLE,
    [{
        "RUN_ID": MAY_RUN_ID,
        "ARCHIVE_YEAR": 2021,
        "ARCHIVE_MONTH": 5,
        "SOURCE_FILE_NAME": (
            "PUBLIC_DVD_BIDPEROFFER_202105010000.zip + "
            "PUBLIC_DVD_BIDPEROFFER_202104010000.zip "
            "(carry-in)"
        ),
        "SOURCE_URL": (
            raw_source_urls(MAY_MONTH)[0][1] + " + " + APRIL_SOURCE_URL
        ),
        "SOURCE_METHOD": (
            "BIDPEROFFER_PLUS_DISPATCHOFFERTRK_RECONSTRUCTED_"
            "WITH_CARRY_IN"
        ),
        "STATUS": "STAGED",
        "SOURCE_ROWS": None,
        "BESS_ROWS": None,
        "LOADED_ROWS": 464_256,
        "STARTED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
        "COMPLETED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
        "ERROR_MESSAGE": None,
        "CREATED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
    }],
)

if manifest_errors:
    raise RuntimeError(
        f"May stage passed, but manifest write failed: {manifest_errors}"
    )

print("✅ MAY 2021 CARRY-IN REPAIR STAGED AND AUDITED")
print("✅ May stage now contains exactly 464,256 effective-offer rows.")
print("✅ May manifest recorded as STAGED.")
print("✅ Production remains unchanged.")

In [ ]:
# SECTION 36A — Create and seed compact BESS carry-in lookup
# Writes only the new lookup table. Does not alter production or stage tables.

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

TARGET_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
APRIL_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202104"
)
MAY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202105"
)
CARRY_LOOKUP_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_CARRY_LOOKUP_V1"
)

create_cache_sql = f"""
CREATE TABLE IF NOT EXISTS `{CARRY_LOOKUP_TABLE}` (
  DUID STRING NOT NULL,
  BIDTYPE STRING NOT NULL,
  BIDSETTLEMENTDATE TIMESTAMP NOT NULL,
  OFFERDATE TIMESTAMP NOT NULL,
  PERIODID INT64 NOT NULL,
  DIRECTION STRING NOT NULL,
  MAXAVAIL FLOAT64,
  FIXEDLOAD FLOAT64,
  ROCUP FLOAT64,
  ROCDOWN FLOAT64,
  ENABLEMENTMIN FLOAT64,
  ENABLEMENTMAX FLOAT64,
  LOWBREAKPOINT FLOAT64,
  HIGHBREAKPOINT FLOAT64,
  BANDAVAIL1 FLOAT64,
  BANDAVAIL2 FLOAT64,
  BANDAVAIL3 FLOAT64,
  BANDAVAIL4 FLOAT64,
  BANDAVAIL5 FLOAT64,
  BANDAVAIL6 FLOAT64,
  BANDAVAIL7 FLOAT64,
  BANDAVAIL8 FLOAT64,
  BANDAVAIL9 FLOAT64,
  BANDAVAIL10 FLOAT64,
  PASAAVAILABILITY FLOAT64,
  ENERGYLIMIT FLOAT64,
  SOURCE_FILE STRING,
  SOURCE_URL STRING,
  FIRST_SEEN_INTERVAL TIMESTAMP,
  ORIGIN_RUN_ID STRING,
  CREATED_AT TIMESTAMP NOT NULL
)
CLUSTER BY DUID, BIDTYPE, BIDSETTLEMENTDATE
"""

client.query(create_cache_sql).result()

seed_cache_sql = f"""
MERGE `{CARRY_LOOKUP_TABLE}` AS target
USING (
  WITH source_rows AS (
    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID,
      DIRECTION,
      MAXAVAIL,
      FIXEDLOAD,
      ROCUP,
      ROCDOWN,
      ENABLEMENTMIN,
      ENABLEMENTMAX,
      LOWBREAKPOINT,
      HIGHBREAKPOINT,
      BANDAVAIL1,
      BANDAVAIL2,
      BANDAVAIL3,
      BANDAVAIL4,
      BANDAVAIL5,
      BANDAVAIL6,
      BANDAVAIL7,
      BANDAVAIL8,
      BANDAVAIL9,
      BANDAVAIL10,
      PASAAVAILABILITY,
      ENERGYLIMIT,
      _SOURCE_FILE AS SOURCE_FILE,
      _SOURCE_URL AS SOURCE_URL,
      INTERVAL_DATETIME AS FIRST_SEEN_INTERVAL,
      _RUN_ID AS ORIGIN_RUN_ID
    FROM `{TARGET_TABLE}`
    WHERE _RUN_ID = 'bess-effective-reconstructed-2021-03'

    UNION ALL

    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID,
      DIRECTION,
      MAXAVAIL,
      FIXEDLOAD,
      ROCUP,
      ROCDOWN,
      ENABLEMENTMIN,
      ENABLEMENTMAX,
      LOWBREAKPOINT,
      HIGHBREAKPOINT,
      BANDAVAIL1,
      BANDAVAIL2,
      BANDAVAIL3,
      BANDAVAIL4,
      BANDAVAIL5,
      BANDAVAIL6,
      BANDAVAIL7,
      BANDAVAIL8,
      BANDAVAIL9,
      BANDAVAIL10,
      PASAAVAILABILITY,
      ENERGYLIMIT,
      _SOURCE_FILE AS SOURCE_FILE,
      _SOURCE_URL AS SOURCE_URL,
      INTERVAL_DATETIME AS FIRST_SEEN_INTERVAL,
      _RUN_ID AS ORIGIN_RUN_ID
    FROM `{APRIL_STAGE_TABLE}`

    UNION ALL

    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID,
      DIRECTION,
      MAXAVAIL,
      FIXEDLOAD,
      ROCUP,
      ROCDOWN,
      ENABLEMENTMIN,
      ENABLEMENTMAX,
      LOWBREAKPOINT,
      HIGHBREAKPOINT,
      BANDAVAIL1,
      BANDAVAIL2,
      BANDAVAIL3,
      BANDAVAIL4,
      BANDAVAIL5,
      BANDAVAIL6,
      BANDAVAIL7,
      BANDAVAIL8,
      BANDAVAIL9,
      BANDAVAIL10,
      PASAAVAILABILITY,
      ENERGYLIMIT,
      _SOURCE_FILE AS SOURCE_FILE,
      _SOURCE_URL AS SOURCE_URL,
      INTERVAL_DATETIME AS FIRST_SEEN_INTERVAL,
      _RUN_ID AS ORIGIN_RUN_ID
    FROM `{MAY_STAGE_TABLE}`
  ),

  deduplicated_source_rows AS (
    SELECT *
    FROM source_rows
    QUALIFY ROW_NUMBER() OVER (
      PARTITION BY
        DUID,
        BIDTYPE,
        BIDSETTLEMENTDATE,
        OFFERDATE,
        PERIODID
      ORDER BY FIRST_SEEN_INTERVAL
    ) = 1
  )

  SELECT
    *,
    CURRENT_TIMESTAMP() AS CREATED_AT
  FROM deduplicated_source_rows
) AS source
ON
  target.DUID = source.DUID
  AND target.BIDTYPE = source.BIDTYPE
  AND target.BIDSETTLEMENTDATE = source.BIDSETTLEMENTDATE
  AND target.OFFERDATE = source.OFFERDATE
  AND target.PERIODID = source.PERIODID

WHEN NOT MATCHED THEN
  INSERT (
    DUID,
    BIDTYPE,
    BIDSETTLEMENTDATE,
    OFFERDATE,
    PERIODID,
    DIRECTION,
    MAXAVAIL,
    FIXEDLOAD,
    ROCUP,
    ROCDOWN,
    ENABLEMENTMIN,
    ENABLEMENTMAX,
    LOWBREAKPOINT,
    HIGHBREAKPOINT,
    BANDAVAIL1,
    BANDAVAIL2,
    BANDAVAIL3,
    BANDAVAIL4,
    BANDAVAIL5,
    BANDAVAIL6,
    BANDAVAIL7,
    BANDAVAIL8,
    BANDAVAIL9,
    BANDAVAIL10,
    PASAAVAILABILITY,
    ENERGYLIMIT,
    SOURCE_FILE,
    SOURCE_URL,
    FIRST_SEEN_INTERVAL,
    ORIGIN_RUN_ID,
    CREATED_AT
  )
  VALUES (
    source.DUID,
    source.BIDTYPE,
    source.BIDSETTLEMENTDATE,
    source.OFFERDATE,
    source.PERIODID,
    source.DIRECTION,
    source.MAXAVAIL,
    source.FIXEDLOAD,
    source.ROCUP,
    source.ROCDOWN,
    source.ENABLEMENTMIN,
    source.ENABLEMENTMAX,
    source.LOWBREAKPOINT,
    source.HIGHBREAKPOINT,
    source.BANDAVAIL1,
    source.BANDAVAIL2,
    source.BANDAVAIL3,
    source.BANDAVAIL4,
    source.BANDAVAIL5,
    source.BANDAVAIL6,
    source.BANDAVAIL7,
    source.BANDAVAIL8,
    source.BANDAVAIL9,
    source.BANDAVAIL10,
    source.PASAAVAILABILITY,
    source.ENERGYLIMIT,
    source.SOURCE_FILE,
    source.SOURCE_URL,
    source.FIRST_SEEN_INTERVAL,
    source.ORIGIN_RUN_ID,
    source.CREATED_AT
  )
"""

client.query(seed_cache_sql).result()

audit_df = client.query(
    f"""
    SELECT
      COUNT(*) AS compact_offer_identity_count,
      COUNT(DISTINCT DUID) AS source_duid_count,
      MIN(FIRST_SEEN_INTERVAL) AS first_interval,
      MAX(FIRST_SEEN_INTERVAL) AS last_interval,
      COUNTIF(
        DUID IS NULL
        OR BIDTYPE IS NULL
        OR BIDSETTLEMENTDATE IS NULL
        OR OFFERDATE IS NULL
        OR PERIODID IS NULL
        OR DIRECTION IS NULL
      ) AS null_key_field_count
    FROM `{CARRY_LOOKUP_TABLE}`
    """
).result().to_dataframe()

display(audit_df)

audit = audit_df.iloc[0]

if int(audit["compact_offer_identity_count"]) == 0:
    raise RuntimeError("Carry-in lookup is empty.")

if int(audit["null_key_field_count"]) != 0:
    raise RuntimeError(
        "Carry-in lookup contains null identity fields. "
        "Do not start the optimized batch."
    )

print("✅ Compact BESS carry-in lookup is ready.")
print(
    f"✅ Reusable offer identities: "
    f"{int(audit['compact_offer_identity_count']):,}"
)
print("✅ Production and existing stage tables remain unchanged.")

In [ ]:
# SECTION 36B — Check fast local parsing options
# Read-only: no downloads and no BigQuery changes.

available_engines = []

try:
    import pyarrow
    available_engines.append(
        f"pyarrow {pyarrow.__version__}"
    )
except ModuleNotFoundError:
    pass

try:
    import polars
    available_engines.append(
        f"polars {polars.__version__}"
    )
except ModuleNotFoundError:
    pass

print("Fast parsing engines available:", available_engines)

if not available_engines:
    print(
        "No fast parser is currently installed. "
        "We will use the safest available fallback."
    )

In [ ]:
# SECTION 36C — Activate fast PyArrow BESS-only archive reader

import io
import csv
import zipfile

import pyarrow as pa
import pyarrow.csv as pa_csv
import pyarrow.compute as pc

def read_aemo_records_fast(zip_path):
    """
    Reads only BESS D-records from an AEMO ZIP CSV using PyArrow batches.
    This avoids converting every non-BESS row through Python.
    """

    with zipfile.ZipFile(zip_path) as zip_file:
        csv_files = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_files) != 1:
            raise RuntimeError(
                f"Expected one CSV in {zip_path}; found {csv_files}"
            )

        csv_name = csv_files[0]

        # First lightweight pass: find the AEMO I-record / schema row.
        with zip_file.open(csv_name) as binary_csv:
            text_csv = io.TextIOWrapper(
                binary_csv,
                encoding="utf-8-sig",
                newline="",
            )
            reader = csv.reader(text_csv)

            header = None
            data_start_row = None

            for row_number, record in enumerate(reader):
                if record and record[0].strip().upper() == "I":
                    header = [
                        value.strip().upper()
                        for value in record[4:]
                    ]
                    data_start_row = row_number + 1
                    break

        if header is None:
            raise RuntimeError(
                f"No AEMO I-record found in {csv_name}"
            )

        full_column_names = [
            "RECORD_TYPE",
            "_AEMO_GROUP",
            "_AEMO_TABLE",
            "_AEMO_VERSION",
        ] + header

        # Second pass: high-speed C++ CSV parsing from the first data row.
        with zip_file.open(csv_name) as binary_csv:
            csv_reader = pa_csv.open_csv(
                binary_csv,
                read_options=pa_csv.ReadOptions(
                    skip_rows=data_start_row,
                    column_names=full_column_names,
                    block_size=64 * 1024 * 1024,
                ),
                parse_options=pa_csv.ParseOptions(
                    delimiter=",",
                ),
                convert_options=pa_csv.ConvertOptions(
                    strings_can_be_null=False,
                ),
            )

            bess_values = pa.array(sorted(BESS_SOURCE_DUIDS))

            for record_batch in csv_reader:
                record_type_mask = pc.equal(
                    record_batch.column("RECORD_TYPE"),
                    pa.scalar("D"),
                )

                duid_mask = pc.is_in(
                    record_batch.column("DUID"),
                    value_set=bess_values,
                )

                bess_data = record_batch.filter(
                    pc.and_(record_type_mask, duid_mask)
                )

                if bess_data.num_rows == 0:
                    continue

                batch_values = bess_data.to_pydict()

                for index in range(bess_data.num_rows):
                    yield (
                        header,
                        [
                            batch_values[column_name][index]
                            for column_name in header
                        ],
                    )

# Replace the older slow reader used by the staging functions.
read_aemo_records = read_aemo_records_fast

print("✅ Fast PyArrow BESS-only archive reader is active.")
print("✅ Future archive scans will skip non-BESS rows before Python processing.")

In [ ]:
# SECTION 36D — Optimized carry-in bulk staging launcher
# Requires the earlier helpers already active in this notebook:
# read_tracker_targets, raw_source_urls, read_aemo_records,
# append_stage_rows, legacy_direction, normalise_key_timestamp,
# to_number, target_schema, target_field_names.

import uuid

CARRY_LOOKUP_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_CARRY_LOOKUP_V1"
)

RAW_TO_TARGET = {
    "MAXAVAIL": "MAXAVAIL",
    "FIXEDLOAD": "FIXEDLOAD",
    "ROCUP": "RAMPUPRATE",
    "ROCDOWN": "RAMPDOWNRATE",
    "ENABLEMENTMIN": "ENABLEMENTMIN",
    "ENABLEMENTMAX": "ENABLEMENTMAX",
    "LOWBREAKPOINT": "LOWBREAKPOINT",
    "HIGHBREAKPOINT": "HIGHBREAKPOINT",
    "BANDAVAIL1": "BANDAVAIL1",
    "BANDAVAIL2": "BANDAVAIL2",
    "BANDAVAIL3": "BANDAVAIL3",
    "BANDAVAIL4": "BANDAVAIL4",
    "BANDAVAIL5": "BANDAVAIL5",
    "BANDAVAIL6": "BANDAVAIL6",
    "BANDAVAIL7": "BANDAVAIL7",
    "BANDAVAIL8": "BANDAVAIL8",
    "BANDAVAIL9": "BANDAVAIL9",
    "BANDAVAIL10": "BANDAVAIL10",
    "PASAAVAILABILITY": "PASAAVAILABILITY",
    "ENERGYLIMIT": "ENERGYLIMIT",
}

def build_stage_record(
    values,
    tracker_target,
    raw_key,
    direction,
    source_file,
    source_url,
    run_id,
    pipeline_name,
    source_is_cache=False,
):
    record = {field: None for field in target_field_names}

    record.update({
        "SETTLEMENTDATE": tracker_target["SETTLEMENTDATE"],
        "DUID": raw_key[0],
        "BIDTYPE": raw_key[1],
        "DIRECTION": direction,
        "INTERVAL_DATETIME": tracker_target["INTERVAL_DATETIME"],
        "BIDSETTLEMENTDATE": tracker_target["BIDSETTLEMENTDATE"],
        "OFFERDATE": tracker_target["OFFERDATE"],
        "PERIODID": raw_key[4],
        "_LOAD_DATE": pd.Timestamp.now(tz="UTC"),
        "_SOURCE": "AEMO_NEMWEB_ARCHIVE",
        "_PIPELINE": pipeline_name,
        "_SOURCE_FILE": source_file,
        "_SOURCE_URL": source_url,
        "_RUN_ID": run_id,
    })

    for target_column, raw_column in RAW_TO_TARGET.items():
        source_column = (
            target_column if source_is_cache else raw_column
        )
        record[target_column] = to_number(values.get(source_column))

    return record

def lookup_carry_records(raw_keys):
    if not raw_keys:
        return {}

    temp_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BESS_CARRY_KEYS_TMP_{uuid.uuid4().hex[:12]}"
    )

    key_df = pd.DataFrame(
        [{
            "DUID": key[0],
            "BIDTYPE": key[1],
            "BIDSETTLEMENTDATE": pd.to_datetime(key[2], utc=True),
            "OFFERDATE": pd.to_datetime(key[3], utc=True),
            "PERIODID": key[4],
        } for key in raw_keys]
    )

    temp_schema = [
        bigquery.SchemaField("DUID", "STRING"),
        bigquery.SchemaField("BIDTYPE", "STRING"),
        bigquery.SchemaField("BIDSETTLEMENTDATE", "TIMESTAMP"),
        bigquery.SchemaField("OFFERDATE", "TIMESTAMP"),
        bigquery.SchemaField("PERIODID", "INTEGER"),
    ]

    try:
        client.load_table_from_dataframe(
            key_df,
            temp_table,
            job_config=bigquery.LoadJobConfig(
                schema=temp_schema,
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            ),
        ).result()

        carry_df = client.query(
            f"""
            SELECT carry.*
            FROM `{CARRY_LOOKUP_TABLE}` AS carry
            INNER JOIN `{temp_table}` AS needed
              ON carry.DUID = needed.DUID
              AND carry.BIDTYPE = needed.BIDTYPE
              AND carry.BIDSETTLEMENTDATE = needed.BIDSETTLEMENTDATE
              AND carry.OFFERDATE = needed.OFFERDATE
              AND carry.PERIODID = needed.PERIODID
            """
        ).result().to_dataframe()

    finally:
        client.delete_table(temp_table, not_found_ok=True)

    carry_records = {}

    for row in carry_df.to_dict("records"):
        key = (
            row["DUID"].strip().upper(),
            row["BIDTYPE"].strip().upper(),
            normalise_key_timestamp(row["BIDSETTLEMENTDATE"]),
            normalise_key_timestamp(row["OFFERDATE"]),
            int(row["PERIODID"]),
        )

        if key in carry_records:
            raise RuntimeError(f"Duplicate carry lookup identity: {key}")

        carry_records[key] = row

    return carry_records

def upsert_stage_into_carry_lookup(stage_table):
    client.query(
        f"""
        MERGE `{CARRY_LOOKUP_TABLE}` AS target
        USING (
          SELECT
            DUID, BIDTYPE, BIDSETTLEMENTDATE, OFFERDATE, PERIODID,
            DIRECTION,
            MAXAVAIL, FIXEDLOAD, ROCUP, ROCDOWN,
            ENABLEMENTMIN, ENABLEMENTMAX,
            LOWBREAKPOINT, HIGHBREAKPOINT,
            BANDAVAIL1, BANDAVAIL2, BANDAVAIL3, BANDAVAIL4,
            BANDAVAIL5, BANDAVAIL6, BANDAVAIL7, BANDAVAIL8,
            BANDAVAIL9, BANDAVAIL10,
            PASAAVAILABILITY, ENERGYLIMIT,
            _SOURCE_FILE AS SOURCE_FILE,
            _SOURCE_URL AS SOURCE_URL,
            INTERVAL_DATETIME AS FIRST_SEEN_INTERVAL,
            _RUN_ID AS ORIGIN_RUN_ID,
            CURRENT_TIMESTAMP() AS CREATED_AT
          FROM `{stage_table}`
          QUALIFY ROW_NUMBER() OVER (
            PARTITION BY
              DUID, BIDTYPE, BIDSETTLEMENTDATE, OFFERDATE, PERIODID
            ORDER BY INTERVAL_DATETIME
          ) = 1
        ) AS source
        ON target.DUID = source.DUID
          AND target.BIDTYPE = source.BIDTYPE
          AND target.BIDSETTLEMENTDATE = source.BIDSETTLEMENTDATE
          AND target.OFFERDATE = source.OFFERDATE
          AND target.PERIODID = source.PERIODID
        WHEN NOT MATCHED THEN
          INSERT ROW
        """
    ).result()

def stage_month_with_carry(month_start):
    yyyymm = month_start.strftime("%Y%m")
    run_id = f"bess-effective-reconstructed-{month_start:%Y-%m}"
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{yyyymm}"
    )

    print("\n" + "=" * 100)
    print(f"STARTING OPTIMIZED STAGE: {month_start:%Y-%m}")

    existing_manifest = client.query(
        f"""
        SELECT STATUS
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id", "STRING", run_id
                )
            ]
        ),
    ).result().to_dataframe()

    if len(existing_manifest) > 0:
        print(
            f"⏭️ Skipping {month_start:%Y-%m}: "
            f"{existing_manifest.iloc[0]['STATUS']}"
        )
        return {"SETTLEMENT_MONTH": month_start, "RESULT": "SKIPPED"}

    try:
        client.get_table(stage_table)
        raise RuntimeError(
            f"Existing unmanifested stage found: {stage_table}"
        )
    except NotFound:
        pass

    targets_by_key, expected_rows, _, _ = read_tracker_targets(month_start)

    print(f"Expected effective rows: {expected_rows:,}")
    print(f"Unique raw identities: {len(targets_by_key):,}")

    stage = bigquery.Table(stage_table, schema=target_schema)
    stage.time_partitioning = bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY,
        field="SETTLEMENTDATE",
    )
    stage.clustering_fields = ["DUID", "BIDTYPE", "DIRECTION"]

    client.create_table(stage)

    unmatched_keys = set(targets_by_key)
    matched_current_keys = set()
    output_buffer = []
    current_source_bess_rows = 0
    output_rows = 0
    raw_sources = raw_source_urls(month_start)

    try:
        for source_file, source_url in raw_sources:
            temp_zip_path = None

            try:
                temp_zip_path = download_aemo_zip(
                    source_url,
                    f"raw {source_file}",
                )

                for header, values in read_aemo_records(temp_zip_path):
                    row = dict(zip(header, values))
                    current_source_bess_rows += 1

                    duid = str(row["DUID"]).strip().upper()
                    bidtype = str(row["BIDTYPE"]).strip().upper()

                    raw_key = (
                        duid,
                        bidtype,
                        normalise_key_timestamp(row["TRADINGDATE"]),
                        normalise_key_timestamp(row["OFFERDATETIME"]),
                        int(row["PERIODID"]),
                    )

                    tracker_targets = targets_by_key.get(raw_key)

                    if not tracker_targets:
                        continue

                    if raw_key in matched_current_keys:
                        raise RuntimeError(
                            f"Duplicate raw identity: {raw_key}"
                        )

                    matched_current_keys.add(raw_key)
                    unmatched_keys.discard(raw_key)

                    supplied_direction = str(
                        row.get("DIRECTION") or ""
                    ).strip().upper()

                    direction = (
                        supplied_direction
                        if supplied_direction
                        else legacy_direction(duid)
                    )

                    for tracker_target in tracker_targets:
                        output_buffer.append(
                            build_stage_record(
                                row,
                                tracker_target,
                                raw_key,
                                direction,
                                source_file,
                                source_url,
                                run_id,
                                "BIDPEROFFER_RECONSTRUCTED_STAGE_FAST",
                            )
                        )

                    if len(output_buffer) >= OUTPUT_CHUNK_ROWS:
                        append_stage_rows(stage_table, output_buffer)
                        output_rows += len(output_buffer)
                        output_buffer = []
                        print(
                            f"{month_start:%Y-%m}: "
                            f"{output_rows:,} rows staged from current source"
                        )

            finally:
                if temp_zip_path and os.path.exists(temp_zip_path):
                    os.remove(temp_zip_path)

        if output_buffer:
            append_stage_rows(stage_table, output_buffer)
            output_rows += len(output_buffer)

        # Resolve only the remaining identities from the compact lookup.
        carry_records = lookup_carry_records(unmatched_keys)
        still_unmatched = unmatched_keys - set(carry_records)

        print(
            f"Current-source identities matched: "
            f"{len(matched_current_keys):,}"
        )
        print(
            f"Carry-in identities found in lookup: "
            f"{len(carry_records):,}"
        )

        if still_unmatched:
            raise RuntimeError(
                f"{len(still_unmatched):,} identities were absent from "
                "both the current raw archive and carry lookup. "
                f"Examples: {list(still_unmatched)[:10]}"
            )

        carry_buffer = []

        for raw_key, cached_row in carry_records.items():
            for tracker_target in targets_by_key[raw_key]:
                carry_buffer.append(
                    build_stage_record(
                        cached_row,
                        tracker_target,
                        raw_key,
                        cached_row["DIRECTION"],
                        f"CARRY_LOOKUP:{cached_row['SOURCE_FILE']}",
                        cached_row["SOURCE_URL"],
                        run_id,
                        "BIDPEROFFER_RECONSTRUCTED_STAGE_CARRY_LOOKUP",
                        source_is_cache=True,
                    )
                )

        if carry_buffer:
            append_stage_rows(stage_table, carry_buffer)
            output_rows += len(carry_buffer)

        audit_df = client.query(
            f"""
            WITH distinct_keys AS (
              SELECT DUID, BIDTYPE, DIRECTION, INTERVAL_DATETIME
              FROM `{stage_table}`
              GROUP BY DUID, BIDTYPE, DIRECTION, INTERVAL_DATETIME
            )
            SELECT
              COUNT(*) AS stage_row_count,
              (SELECT COUNT(*) FROM distinct_keys)
                AS distinct_effective_key_count,
              COUNTIF(
                DUID IS NULL OR BIDTYPE IS NULL OR DIRECTION IS NULL
                OR INTERVAL_DATETIME IS NULL OR OFFERDATE IS NULL
                OR PERIODID IS NULL
              ) AS null_required_field_count
            FROM `{stage_table}`
            """
        ).result().to_dataframe()

        display(audit_df)
        audit = audit_df.iloc[0]

        if (
            int(audit["stage_row_count"]) != expected_rows
            or int(audit["distinct_effective_key_count"]) != expected_rows
            or int(audit["null_required_field_count"]) != 0
        ):
            raise RuntimeError("Stage audit failed.")

        upsert_stage_into_carry_lookup(stage_table)

        manifest_errors = client.insert_rows_json(
            MANIFEST_TABLE,
            [{
                "RUN_ID": run_id,
                "ARCHIVE_YEAR": int(month_start.year),
                "ARCHIVE_MONTH": int(month_start.month),
                "SOURCE_FILE_NAME": " + ".join(
                    name for name, _ in raw_sources
                ),
                "SOURCE_URL": " + ".join(
                    url for _, url in raw_sources
                ),
                "SOURCE_METHOD": (
                    "BIDPEROFFER_RECONSTRUCTED_FAST_"
                    "WITH_CARRY_LOOKUP"
                ),
                "STATUS": "STAGED",
                "SOURCE_ROWS": None,
                "BESS_ROWS": int(current_source_bess_rows),
                "LOADED_ROWS": int(audit["stage_row_count"]),
                "STARTED_AT": pd.Timestamp.now(
                    tz="UTC"
                ).isoformat(),
                "COMPLETED_AT": pd.Timestamp.now(
                    tz="UTC"
                ).isoformat(),
                "ERROR_MESSAGE": None,
                "CREATED_AT": pd.Timestamp.now(
                    tz="UTC"
                ).isoformat(),
            }],
        )

        if manifest_errors:
            raise RuntimeError(manifest_errors)

        print(
            f"✅ {month_start:%Y-%m} STAGED: "
            f"{int(audit['stage_row_count']):,} rows"
        )

        return {
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "STAGED",
            "STAGE_ROWS": int(audit["stage_row_count"]),
            "CARRY_IDENTITIES": len(carry_records),
        }

    except Exception:
        print(
            f"⚠️ Stage retained for inspection: {stage_table}"
        )
        raise

# Launch only the remaining months.
optimized_months = pd.date_range(
    "2021-06-01",
    "2024-07-01",
    freq="MS",
)

optimized_results = []

print(
    f"Launching optimized staging batch: "
    f"{len(optimized_months):,} months"
)
print("Production will not be modified.")

for month_start in optimized_months:
    try:
        optimized_results.append(
            stage_month_with_carry(month_start)
        )
    except Exception as error:
        optimized_results.append({
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "FAILED",
            "ERROR": str(error),
        })
        print(f"\n❌ Batch stopped: {error}")
        break

optimized_results_df = pd.DataFrame(optimized_results)

print("\nOPTIMIZED BULK STAGING SUMMARY")
display(optimized_results_df)

In [ ]:
# PATCH — let PyArrow skip AEMO footer/control records

def skip_aemo_control_row(invalid_row):
    return "skip"

def read_aemo_records_fast(zip_path):
    with zipfile.ZipFile(zip_path) as zip_file:
        csv_files = [
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_files) != 1:
            raise RuntimeError(
                f"Expected one CSV in {zip_path}; found {csv_files}"
            )

        csv_name = csv_files[0]

        with zip_file.open(csv_name) as binary_csv:
            text_csv = io.TextIOWrapper(
                binary_csv,
                encoding="utf-8-sig",
                newline="",
            )
            reader = csv.reader(text_csv)

            header = None
            data_start_row = None

            for row_number, record in enumerate(reader):
                if record and record[0].strip().upper() == "I":
                    header = [
                        value.strip().upper()
                        for value in record[4:]
                    ]
                    data_start_row = row_number + 1
                    break

        if header is None:
            raise RuntimeError(f"No AEMO header found in {csv_name}")

        full_column_names = [
            "RECORD_TYPE",
            "_AEMO_GROUP",
            "_AEMO_TABLE",
            "_AEMO_VERSION",
        ] + header

        with zip_file.open(csv_name) as binary_csv:
            csv_reader = pa_csv.open_csv(
                binary_csv,
                read_options=pa_csv.ReadOptions(
                    skip_rows=data_start_row,
                    column_names=full_column_names,
                    block_size=64 * 1024 * 1024,
                ),
                parse_options=pa_csv.ParseOptions(
                    delimiter=",",
                    invalid_row_handler=skip_aemo_control_row,
                ),
                convert_options=pa_csv.ConvertOptions(
                    strings_can_be_null=False,
                ),
            )

            bess_values = pa.array(sorted(BESS_SOURCE_DUIDS))

            for record_batch in csv_reader:
                record_type_mask = pc.equal(
                    record_batch.column("RECORD_TYPE"),
                    pa.scalar("D"),
                )

                duid_mask = pc.is_in(
                    record_batch.column("DUID"),
                    value_set=bess_values,
                )

                bess_data = record_batch.filter(
                    pc.and_(record_type_mask, duid_mask)
                )

                if bess_data.num_rows == 0:
                    continue

                batch_values = bess_data.to_pydict()

                for index in range(bess_data.num_rows):
                    yield (
                        header,
                        [
                            batch_values[column_name][index]
                            for column_name in header
                        ],
                    )

read_aemo_records = read_aemo_records_fast

print("✅ PyArrow reader patched to skip AEMO footer records.")

In [ ]:
# PATCH — force AEMO CSV columns to strings and safely reset empty July stage

JULY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202107"
)
JULY_RUN_ID = "bess-effective-reconstructed-2021-07"

july_stage_rows = int(
    client.query(
        f"SELECT COUNT(*) AS row_count FROM `{JULY_STAGE_TABLE}`"
    ).result().to_dataframe().iloc[0]["row_count"]
)

july_manifest_rows = int(
    client.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = @run_id
        """,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "run_id",
                    "STRING",
                    JULY_RUN_ID,
                )
            ]
        ),
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"July retained stage rows: {july_stage_rows:,}")
print(f"July manifest rows: {july_manifest_rows:,}")

if july_stage_rows != 0 or july_manifest_rows != 0:
    raise RuntimeError(
        "Safety stop: July is not an empty failed stage. "
        "Nothing was deleted."
    )

client.delete_table(JULY_STAGE_TABLE, not_found_ok=False)

print("✅ Empty July stage deleted safely.")

def read_aemo_records_fast(zip_path):
    with zipfile.ZipFile(zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            for row_number, record in enumerate(reader):
                if record and record[0].strip().upper() == "I":
                    header = [
                        value.strip().upper()
                        for value in record[4:]
                    ]
                    data_start_row = row_number + 1
                    break
            else:
                raise RuntimeError(f"No AEMO header found in {csv_name}")

        full_column_names = [
            "RECORD_TYPE",
            "_AEMO_GROUP",
            "_AEMO_TABLE",
            "_AEMO_VERSION",
        ] + header

        string_column_types = {
            column_name: pa.string()
            for column_name in full_column_names
        }

        with zip_file.open(csv_name) as binary_csv:
            csv_reader = pa_csv.open_csv(
                binary_csv,
                read_options=pa_csv.ReadOptions(
                    skip_rows=data_start_row,
                    column_names=full_column_names,
                    block_size=64 * 1024 * 1024,
                ),
                parse_options=pa_csv.ParseOptions(
                    delimiter=",",
                    invalid_row_handler=skip_aemo_control_row,
                ),
                convert_options=pa_csv.ConvertOptions(
                    column_types=string_column_types,
                    strings_can_be_null=False,
                ),
            )

            bess_values = pa.array(sorted(BESS_SOURCE_DUIDS))

            for record_batch in csv_reader:
                mask = pc.and_(
                    pc.equal(
                        record_batch.column("RECORD_TYPE"),
                        pa.scalar("D"),
                    ),
                    pc.is_in(
                        record_batch.column("DUID"),
                        value_set=bess_values,
                    ),
                )

                bess_data = record_batch.filter(mask)

                if bess_data.num_rows == 0:
                    continue

                batch_values = bess_data.to_pydict()

                for index in range(bess_data.num_rows):
                    yield (
                        header,
                        [
                            batch_values[column_name][index]
                            for column_name in header
                        ],
                    )

read_aemo_records = read_aemo_records_fast

print("✅ Fast reader patched: all AEMO source fields are read safely as text.")

In [ ]:
optimized_months = pd.date_range(
    "2021-07-01",
    "2024-07-01",
    freq="MS",
)

optimized_results = []

for month_start in optimized_months:
    try:
        optimized_results.append(
            stage_month_with_carry(month_start)
        )
    except Exception as error:
        optimized_results.append({
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "FAILED",
            "ERROR": str(error),
        })
        print(f"\n❌ Batch stopped: {error}")
        break

display(pd.DataFrame(optimized_results))

In [ ]:
# SECTION 36E — Prove July's remaining carry-in identities are in June raw source
# Read-only: no BigQuery writes, deletes, or promotions.

JULY_MONTH = pd.Timestamp("2021-07-01")
JULY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202107"
)

# Rebuild July tracker requirements.
july_targets, july_expected_rows, _, _ = read_tracker_targets(JULY_MONTH)

# Identify offer identities already loaded into the retained July stage.
july_stage_key_df = client.query(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID
    FROM `{JULY_STAGE_TABLE}`
    """
).result().to_dataframe()

july_stage_keys = {
    (
        row.DUID.strip().upper(),
        row.BIDTYPE.strip().upper(),
        normalise_key_timestamp(row.BIDSETTLEMENTDATE),
        normalise_key_timestamp(row.OFFERDATE),
        int(row.PERIODID),
    )
    for row in july_stage_key_df.itertuples(index=False)
}

remaining_keys = set(july_targets) - july_stage_keys
cached_records = lookup_carry_records(remaining_keys)
june_required_keys = remaining_keys - set(cached_records)

print(f"July tracker target rows: {july_expected_rows:,}")
print(f"July stage rows retained: {len(july_stage_key_df):,}")
print(f"Already in carry lookup: {len(cached_records):,}")
print(f"Remaining identities to seek in June raw: {len(june_required_keys):,}")

if len(june_required_keys) != 526:
    raise RuntimeError(
        "Safety stop: expected 526 June-source identities, "
        f"found {len(june_required_keys):,}."
    )

june_matches = {}
june_file_name, june_source_url = raw_source_urls(
    pd.Timestamp("2021-06-01")
)[0]

temp_zip_path = None

try:
    temp_zip_path = download_aemo_zip(
        june_source_url,
        f"June boundary source {june_file_name}",
    )

    for header, values in read_aemo_records(temp_zip_path):
        row = dict(zip(header, values))

        raw_key = (
            str(row["DUID"]).strip().upper(),
            str(row["BIDTYPE"]).strip().upper(),
            normalise_key_timestamp(row["TRADINGDATE"]),
            normalise_key_timestamp(row["OFFERDATETIME"]),
            int(row["PERIODID"]),
        )

        if raw_key in june_required_keys:
            if raw_key in june_matches:
                raise RuntimeError(
                    f"Duplicate June raw identity: {raw_key}"
                )

            june_matches[raw_key] = row

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print("✅ Temporary June raw ZIP deleted.")

still_missing = june_required_keys - set(june_matches)

print("\nJULY 2021 MONTH-END CARRY-IN AUDIT")
print(f"Identities required from June: {len(june_required_keys):,}")
print(f"Found in June raw source: {len(june_matches):,}")
print(f"Still unmatched: {len(still_missing):,}")

if still_missing:
    examples = pd.DataFrame(
        [
            {
                "DUID": key[0],
                "BIDTYPE": key[1],
                "BIDSETTLEMENTDATE": key[2],
                "OFFERDATE": key[3],
                "PERIODID": key[4],
            }
            for key in sorted(still_missing)[:20]
        ]
    )
    display(examples)
    raise RuntimeError(
        "July carry-in proof failed. No data was changed."
    )

print("✅ JULY 2021 MONTH-END CARRY-IN AUDIT PASSED")
print("✅ June supplies every remaining July offer identity.")
print("✅ We can repair July and add the boundary-cache rule.")

In [ ]:
# SECTION 36F — Repair and finalize July 2021 stage
# Uses the completed read-only audit variables:
# july_targets, cached_records, june_matches, june_file_name, june_source_url.

JULY_RUN_ID = "bess-effective-reconstructed-2021-07"
JULY_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202107"
)

required_variables = [
    "july_targets",
    "cached_records",
    "june_matches",
    "build_stage_record",
    "append_stage_rows",
    "upsert_stage_into_carry_lookup",
]

for variable_name in required_variables:
    if variable_name not in globals():
        raise RuntimeError(
            f"Missing required variable: {variable_name}. "
            "Do not run after a kernel reset."
        )

before_rows = int(
    client.query(
        f"SELECT COUNT(*) AS row_count FROM `{JULY_STAGE_TABLE}`"
    ).result().to_dataframe().iloc[0]["row_count"]
)

if before_rows != 585_892:
    raise RuntimeError(
        f"Safety stop: expected 585,892 July rows, found {before_rows:,}."
    )

repair_rows = []

# Add identities already held in the compact carry lookup.
for raw_key, cached_row in cached_records.items():
    for tracker_target in july_targets[raw_key]:
        repair_rows.append(
            build_stage_record(
                cached_row,
                tracker_target,
                raw_key,
                cached_row["DIRECTION"],
                f"CARRY_LOOKUP:{cached_row['SOURCE_FILE']}",
                cached_row["SOURCE_URL"],
                JULY_RUN_ID,
                "BIDPEROFFER_RECONSTRUCTED_STAGE_CARRY_LOOKUP",
                source_is_cache=True,
            )
        )

# Add the proven June month-end offers.
for raw_key, june_row in june_matches.items():
    for tracker_target in july_targets[raw_key]:
        repair_rows.append(
            build_stage_record(
                june_row,
                tracker_target,
                raw_key,
                legacy_direction(raw_key[0]),
                june_file_name,
                june_source_url,
                JULY_RUN_ID,
                "BIDPEROFFER_RECONSTRUCTED_STAGE_MONTH_END_CARRY",
            )
        )

print(f"July repair rows prepared: {len(repair_rows):,}")

if len(repair_rows) != 3_548:
    raise RuntimeError(
        f"Safety stop: expected 3,548 repair rows, "
        f"prepared {len(repair_rows):,}."
    )

append_stage_rows(JULY_STAGE_TABLE, repair_rows)

audit_df = client.query(
    f"""
    WITH distinct_effective_keys AS (
      SELECT
        DUID,
        BIDTYPE,
        DIRECTION,
        INTERVAL_DATETIME
      FROM `{JULY_STAGE_TABLE}`
      GROUP BY
        DUID,
        BIDTYPE,
        DIRECTION,
        INTERVAL_DATETIME
    )
    SELECT
      COUNT(*) AS stage_row_count,
      (SELECT COUNT(*) FROM distinct_effective_keys)
        AS distinct_effective_key_count,
      COUNTIF(
        DUID IS NULL
        OR BIDTYPE IS NULL
        OR DIRECTION IS NULL
        OR INTERVAL_DATETIME IS NULL
        OR OFFERDATE IS NULL
        OR PERIODID IS NULL
      ) AS null_required_field_count
    FROM `{JULY_STAGE_TABLE}`
    """
).result().to_dataframe()

display(audit_df)

audit = audit_df.iloc[0]

if (
    int(audit["stage_row_count"]) != 589_440
    or int(audit["distinct_effective_key_count"]) != 589_440
    or int(audit["null_required_field_count"]) != 0
):
    raise RuntimeError(
        "July audit failed. Stage was retained; no manifest was written."
    )

upsert_stage_into_carry_lookup(JULY_STAGE_TABLE)

manifest_errors = client.insert_rows_json(
    MANIFEST_TABLE,
    [{
        "RUN_ID": JULY_RUN_ID,
        "ARCHIVE_YEAR": 2021,
        "ARCHIVE_MONTH": 7,
        "SOURCE_FILE_NAME": (
            "PUBLIC_DVD_BIDPEROFFER_202107010000.zip + "
            "PUBLIC_DVD_BIDPEROFFER_202106010000.zip "
            "(month-end carry-in)"
        ),
        "SOURCE_URL": (
            raw_source_urls(JULY_MONTH)[0][1]
            + " + "
            + june_source_url
        ),
        "SOURCE_METHOD": (
            "BIDPEROFFER_RECONSTRUCTED_FAST_"
            "WITH_CARRY_AND_MONTH_END_LOOKBACK"
        ),
        "STATUS": "STAGED",
        "SOURCE_ROWS": None,
        "BESS_ROWS": None,
        "LOADED_ROWS": 589_440,
        "STARTED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
        "COMPLETED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
        "ERROR_MESSAGE": None,
        "CREATED_AT": pd.Timestamp.now(tz="UTC").isoformat(),
    }],
)

if manifest_errors:
    raise RuntimeError(manifest_errors)

print("✅ JULY 2021 REPAIRED, STAGED, AND AUDITED")
print("✅ July now has exactly 589,440 effective-offer rows.")
print("✅ July carry state is available for later months.")
print("✅ Production remains unchanged.")

In [ ]:
# SECTION 36G — Seed July-to-August boundary offer cache
# Captures July raw offers that first become effective during August.

BOUNDARY_LOOKUP_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

JULY_MONTH = pd.Timestamp("2021-07-01")
AUGUST_MONTH = pd.Timestamp("2021-08-01")

# Create an empty table with the same structure as the carry lookup.
client.query(
    f"""
    CREATE TABLE IF NOT EXISTS `{BOUNDARY_LOOKUP_TABLE}` AS
    SELECT *
    FROM `{CARRY_LOOKUP_TABLE}`
    WHERE FALSE
    """
).result()

existing_boundary_rows = int(
    client.query(
        f"SELECT COUNT(*) AS row_count FROM `{BOUNDARY_LOOKUP_TABLE}`"
    ).result().to_dataframe().iloc[0]["row_count"]
)

if existing_boundary_rows != 0:
    raise RuntimeError(
        f"Safety stop: boundary lookup already contains "
        f"{existing_boundary_rows:,} rows. Nothing was overwritten."
    )

# August tracker identifies which raw offer identities may be needed in August.
august_targets, august_target_rows, _, _ = read_tracker_targets(
    AUGUST_MONTH
)
august_required_keys = set(august_targets)

print(f"August effective tracker rows: {august_target_rows:,}")
print(
    f"Unique August raw offer identities required: "
    f"{len(august_required_keys):,}"
)

july_file_name, july_source_url = raw_source_urls(JULY_MONTH)[0]

boundary_records = {}
temp_zip_path = None

try:
    temp_zip_path = download_aemo_zip(
        july_source_url,
        f"July boundary source {july_file_name}",
    )

    for header, values in read_aemo_records(temp_zip_path):
        raw_row = dict(zip(header, values))

        raw_key = (
            str(raw_row["DUID"]).strip().upper(),
            str(raw_row["BIDTYPE"]).strip().upper(),
            normalise_key_timestamp(raw_row["TRADINGDATE"]),
            normalise_key_timestamp(raw_row["OFFERDATETIME"]),
            int(raw_row["PERIODID"]),
        )

        if raw_key not in august_required_keys:
            continue

        if raw_key in boundary_records:
            raise RuntimeError(
                f"Duplicate July boundary identity: {raw_key}"
            )

        supplied_direction = str(
            raw_row.get("DIRECTION") or ""
        ).strip().upper()

        direction = (
            supplied_direction
            if supplied_direction
            else legacy_direction(raw_key[0])
        )

        first_august_interval = min(
            item["INTERVAL_DATETIME"]
            for item in august_targets[raw_key]
        )

        record = {
            "DUID": raw_key[0],
            "BIDTYPE": raw_key[1],
            "BIDSETTLEMENTDATE": pd.to_datetime(
                raw_key[2],
                utc=True,
            ),
            "OFFERDATE": pd.to_datetime(
                raw_key[3],
                utc=True,
            ),
            "PERIODID": raw_key[4],
            "DIRECTION": direction,
            "MAXAVAIL": to_number(raw_row.get("MAXAVAIL")),
            "FIXEDLOAD": to_number(raw_row.get("FIXEDLOAD")),
            "ROCUP": to_number(raw_row.get("RAMPUPRATE")),
            "ROCDOWN": to_number(raw_row.get("RAMPDOWNRATE")),
            "ENABLEMENTMIN": to_number(
                raw_row.get("ENABLEMENTMIN")
            ),
            "ENABLEMENTMAX": to_number(
                raw_row.get("ENABLEMENTMAX")
            ),
            "LOWBREAKPOINT": to_number(
                raw_row.get("LOWBREAKPOINT")
            ),
            "HIGHBREAKPOINT": to_number(
                raw_row.get("HIGHBREAKPOINT")
            ),
            "BANDAVAIL1": to_number(raw_row.get("BANDAVAIL1")),
            "BANDAVAIL2": to_number(raw_row.get("BANDAVAIL2")),
            "BANDAVAIL3": to_number(raw_row.get("BANDAVAIL3")),
            "BANDAVAIL4": to_number(raw_row.get("BANDAVAIL4")),
            "BANDAVAIL5": to_number(raw_row.get("BANDAVAIL5")),
            "BANDAVAIL6": to_number(raw_row.get("BANDAVAIL6")),
            "BANDAVAIL7": to_number(raw_row.get("BANDAVAIL7")),
            "BANDAVAIL8": to_number(raw_row.get("BANDAVAIL8")),
            "BANDAVAIL9": to_number(raw_row.get("BANDAVAIL9")),
            "BANDAVAIL10": to_number(raw_row.get("BANDAVAIL10")),
            "PASAAVAILABILITY": to_number(
                raw_row.get("PASAAVAILABILITY")
            ),
            "ENERGYLIMIT": to_number(raw_row.get("ENERGYLIMIT")),
            "SOURCE_FILE": july_file_name,
            "SOURCE_URL": july_source_url,
            "FIRST_SEEN_INTERVAL": first_august_interval,
            "ORIGIN_RUN_ID": (
                "bess-boundary-2021-07-to-2021-08"
            ),
            "CREATED_AT": pd.Timestamp.now(tz="UTC"),
        }

        boundary_records[raw_key] = record

finally:
    if temp_zip_path and os.path.exists(temp_zip_path):
        os.remove(temp_zip_path)
        print("✅ Temporary July boundary ZIP deleted.")

if not boundary_records:
    raise RuntimeError(
        "No July offers were found for August tracker identities."
    )

boundary_df = pd.DataFrame(
    list(boundary_records.values())
)

client.load_table_from_dataframe(
    boundary_df,
    BOUNDARY_LOOKUP_TABLE,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    ),
).result()

audit_df = client.query(
    f"""
    SELECT
      COUNT(*) AS boundary_offer_identity_count,
      COUNT(DISTINCT DUID) AS source_duid_count,
      MIN(FIRST_SEEN_INTERVAL) AS first_august_interval,
      MAX(FIRST_SEEN_INTERVAL) AS last_august_interval,
      COUNTIF(
        DUID IS NULL
        OR BIDTYPE IS NULL
        OR BIDSETTLEMENTDATE IS NULL
        OR OFFERDATE IS NULL
        OR PERIODID IS NULL
        OR DIRECTION IS NULL
      ) AS null_key_field_count
    FROM `{BOUNDARY_LOOKUP_TABLE}`
    """
).result().to_dataframe()

display(audit_df)

audit = audit_df.iloc[0]

if (
    int(audit["boundary_offer_identity_count"])
    != len(boundary_records)
    or int(audit["null_key_field_count"]) != 0
):
    raise RuntimeError(
        "Boundary-cache audit failed. Do not restart August yet."
    )

print("✅ July-to-August boundary cache seeded and audited.")
print(
    f"✅ Boundary offer identities available for August: "
    f"{len(boundary_records):,}"
)
print("✅ Production and existing stage tables remain unchanged.")

In [ ]:
# SECTION 36H — Automatic boundary capture + August 2021–July 2024 staging
# Staging only. Production is not modified.

BOUNDARY_CAPTURE_ACTIVE = False
BOUNDARY_CAPTURE_KEYS = set()
BOUNDARY_CAPTURE_RECORDS = {}

def read_aemo_records_fast(zip_path):
    global BOUNDARY_CAPTURE_RECORDS

    with zipfile.ZipFile(zip_path) as zip_file:
        csv_name = next(
            name for name in zip_file.namelist()
            if name.upper().endswith(".CSV")
        )

        with zip_file.open(csv_name) as binary_csv:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_csv,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            for row_number, record in enumerate(reader):
                if record and record[0].strip().upper() == "I":
                    header = [
                        value.strip().upper()
                        for value in record[4:]
                    ]
                    data_start_row = row_number + 1
                    break
            else:
                raise RuntimeError(f"No AEMO header found in {csv_name}")

        full_column_names = [
            "RECORD_TYPE",
            "_AEMO_GROUP",
            "_AEMO_TABLE",
            "_AEMO_VERSION",
        ] + header

        with zip_file.open(csv_name) as binary_csv:
            csv_reader = pa_csv.open_csv(
                binary_csv,
                read_options=pa_csv.ReadOptions(
                    skip_rows=data_start_row,
                    column_names=full_column_names,
                    block_size=64 * 1024 * 1024,
                ),
                parse_options=pa_csv.ParseOptions(
                    delimiter=",",
                    invalid_row_handler=skip_aemo_control_row,
                ),
                convert_options=pa_csv.ConvertOptions(
                    column_types={
                        name: pa.string()
                        for name in full_column_names
                    },
                    strings_can_be_null=False,
                ),
            )

            bess_values = pa.array(sorted(BESS_SOURCE_DUIDS))
            raw_offer_columns = {
                "DUID",
                "BIDTYPE",
                "TRADINGDATE",
                "OFFERDATETIME",
                "PERIODID",
            }

            for record_batch in csv_reader:
                mask = pc.and_(
                    pc.equal(
                        record_batch.column("RECORD_TYPE"),
                        pa.scalar("D"),
                    ),
                    pc.is_in(
                        record_batch.column("DUID"),
                        value_set=bess_values,
                    ),
                )

                bess_data = record_batch.filter(mask)

                if bess_data.num_rows == 0:
                    continue

                batch_values = bess_data.to_pydict()

                for index in range(bess_data.num_rows):
                    values = [
                        batch_values[column_name][index]
                        for column_name in header
                    ]

                    if (
                        BOUNDARY_CAPTURE_ACTIVE
                        and raw_offer_columns.issubset(set(header))
                    ):
                        raw_row = dict(zip(header, values))

                        raw_key = (
                            str(raw_row["DUID"]).strip().upper(),
                            str(raw_row["BIDTYPE"]).strip().upper(),
                            normalise_key_timestamp(
                                raw_row["TRADINGDATE"]
                            ),
                            normalise_key_timestamp(
                                raw_row["OFFERDATETIME"]
                            ),
                            int(raw_row["PERIODID"]),
                        )

                        if raw_key in BOUNDARY_CAPTURE_KEYS:
                            if raw_key in BOUNDARY_CAPTURE_RECORDS:
                                raise RuntimeError(
                                    f"Duplicate boundary key: {raw_key}"
                                )

                            BOUNDARY_CAPTURE_RECORDS[raw_key] = raw_row

                    yield header, values

read_aemo_records = read_aemo_records_fast


def lookup_carry_records(raw_keys):
    if not raw_keys:
        return {}

    temp_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BESS_LOOKUP_KEYS_TMP_{uuid.uuid4().hex[:12]}"
    )

    key_df = pd.DataFrame(
        [{
            "DUID": key[0],
            "BIDTYPE": key[1],
            "BIDSETTLEMENTDATE": pd.to_datetime(key[2], utc=True),
            "OFFERDATE": pd.to_datetime(key[3], utc=True),
            "PERIODID": key[4],
        } for key in raw_keys]
    )

    try:
        client.load_table_from_dataframe(
            key_df,
            temp_table,
            job_config=bigquery.LoadJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            ),
        ).result()

        lookup_df = client.query(
            f"""
            WITH combined_lookup AS (
              SELECT 1 AS source_priority, *
              FROM `{BOUNDARY_LOOKUP_TABLE}`

              UNION ALL

              SELECT 2 AS source_priority, *
              FROM `{CARRY_LOOKUP_TABLE}`
            )
            SELECT lookup.*
            FROM combined_lookup AS lookup
            INNER JOIN `{temp_table}` AS needed
              ON lookup.DUID = needed.DUID
              AND lookup.BIDTYPE = needed.BIDTYPE
              AND lookup.BIDSETTLEMENTDATE = needed.BIDSETTLEMENTDATE
              AND lookup.OFFERDATE = needed.OFFERDATE
              AND lookup.PERIODID = needed.PERIODID
            QUALIFY ROW_NUMBER() OVER (
              PARTITION BY
                lookup.DUID,
                lookup.BIDTYPE,
                lookup.BIDSETTLEMENTDATE,
                lookup.OFFERDATE,
                lookup.PERIODID
              ORDER BY lookup.source_priority DESC
            ) = 1
            """
        ).result().to_dataframe()

    finally:
        client.delete_table(temp_table, not_found_ok=True)

    records = {}

    for row in lookup_df.to_dict("records"):
        key = (
            row["DUID"].strip().upper(),
            row["BIDTYPE"].strip().upper(),
            normalise_key_timestamp(row["BIDSETTLEMENTDATE"]),
            normalise_key_timestamp(row["OFFERDATE"]),
            int(row["PERIODID"]),
        )
        records[key] = row

    return records


def persist_boundary_capture(month_start, next_targets):
    if not BOUNDARY_CAPTURE_RECORDS:
        print("ℹ️ No next-month boundary offers captured.")
        return 0

    source_file_names = " + ".join(
        name for name, _ in raw_source_urls(month_start)
    )
    source_urls = " + ".join(
        url for _, url in raw_source_urls(month_start)
    )

    records = []

    for raw_key, raw_row in BOUNDARY_CAPTURE_RECORDS.items():
        supplied_direction = str(
            raw_row.get("DIRECTION") or ""
        ).strip().upper()

        records.append({
            "DUID": raw_key[0],
            "BIDTYPE": raw_key[1],
            "BIDSETTLEMENTDATE": pd.to_datetime(raw_key[2], utc=True),
            "OFFERDATE": pd.to_datetime(raw_key[3], utc=True),
            "PERIODID": raw_key[4],
            "DIRECTION": (
                supplied_direction
                if supplied_direction
                else legacy_direction(raw_key[0])
            ),
            "MAXAVAIL": to_number(raw_row.get("MAXAVAIL")),
            "FIXEDLOAD": to_number(raw_row.get("FIXEDLOAD")),
            "ROCUP": to_number(raw_row.get("RAMPUPRATE")),
            "ROCDOWN": to_number(raw_row.get("RAMPDOWNRATE")),
            "ENABLEMENTMIN": to_number(raw_row.get("ENABLEMENTMIN")),
            "ENABLEMENTMAX": to_number(raw_row.get("ENABLEMENTMAX")),
            "LOWBREAKPOINT": to_number(raw_row.get("LOWBREAKPOINT")),
            "HIGHBREAKPOINT": to_number(raw_row.get("HIGHBREAKPOINT")),
            "BANDAVAIL1": to_number(raw_row.get("BANDAVAIL1")),
            "BANDAVAIL2": to_number(raw_row.get("BANDAVAIL2")),
            "BANDAVAIL3": to_number(raw_row.get("BANDAVAIL3")),
            "BANDAVAIL4": to_number(raw_row.get("BANDAVAIL4")),
            "BANDAVAIL5": to_number(raw_row.get("BANDAVAIL5")),
            "BANDAVAIL6": to_number(raw_row.get("BANDAVAIL6")),
            "BANDAVAIL7": to_number(raw_row.get("BANDAVAIL7")),
            "BANDAVAIL8": to_number(raw_row.get("BANDAVAIL8")),
            "BANDAVAIL9": to_number(raw_row.get("BANDAVAIL9")),
            "BANDAVAIL10": to_number(raw_row.get("BANDAVAIL10")),
            "PASAAVAILABILITY": to_number(
                raw_row.get("PASAAVAILABILITY")
            ),
            "ENERGYLIMIT": to_number(raw_row.get("ENERGYLIMIT")),
            "SOURCE_FILE": source_file_names,
            "SOURCE_URL": source_urls,
            "FIRST_SEEN_INTERVAL": min(
                item["INTERVAL_DATETIME"]
                for item in next_targets[raw_key]
            ),
            "ORIGIN_RUN_ID": (
                f"bess-boundary-{month_start:%Y-%m}-to-"
                f"{(month_start + pd.offsets.MonthBegin(1)):%Y-%m}"
            ),
            "CREATED_AT": pd.Timestamp.now(tz="UTC"),
        })

    client.load_table_from_dataframe(
        pd.DataFrame(records),
        BOUNDARY_LOOKUP_TABLE,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        ),
    ).result()

    print(f"✅ Boundary offers stored: {len(records):,}")
    return len(records)


automatic_months = pd.date_range(
    "2021-08-01",
    "2024-07-01",
    freq="MS",
)

automatic_results = []

print(
    f"Launching automatic staging batch: "
    f"{len(automatic_months):,} months"
)
print("Production will not be modified.")

for month_start in automatic_months:
    next_month = month_start + pd.offsets.MonthBegin(1)

    # Identify next month's target identities before scanning current raw data.
    BOUNDARY_CAPTURE_ACTIVE = False
    next_targets, _, _, _ = read_tracker_targets(next_month)

    BOUNDARY_CAPTURE_KEYS = set(next_targets)
    BOUNDARY_CAPTURE_RECORDS = {}
    BOUNDARY_CAPTURE_ACTIVE = True

    try:
        result = stage_month_with_carry(month_start)
        BOUNDARY_CAPTURE_ACTIVE = False

        result["BOUNDARY_ROWS_FOR_NEXT_MONTH"] = (
            persist_boundary_capture(month_start, next_targets)
        )

        automatic_results.append(result)

    except Exception as error:
        BOUNDARY_CAPTURE_ACTIVE = False

        automatic_results.append({
            "SETTLEMENT_MONTH": month_start,
            "RESULT": "FAILED",
            "ERROR": str(error),
        })

        print(f"\n❌ Batch stopped: {error}")
        break

display(pd.DataFrame(automatic_results))

In [ ]:
# SECTION 36I — Read-only audit of the retained August 2021 stage
# No deletes, no promotions, no production changes.

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

PROJECT_ID = "master-aemo"
DATASET_ID = "aemo_master_schema"

try:
    client
except NameError:
    client = bigquery.Client(project=PROJECT_ID)

STAGE_TABLE_202108 = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202108"
)
MANIFEST_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

try:
    client.get_table(STAGE_TABLE_202108)
except NotFound:
    print("August stage table does not exist.")
else:
    stage_audit = client.query(
        f"""
        SELECT
          COUNT(*) AS stage_row_count,
          COUNT(DISTINCT CONCAT(
            COALESCE(DUID, ''), '|',
            COALESCE(BIDTYPE, ''), '|',
            COALESCE(DIRECTION, ''), '|',
            CAST(INTERVAL_DATETIME AS STRING)
          )) AS distinct_effective_key_count,
          COUNT(DISTINCT DUID) AS source_duid_count,
          COUNTIF(
            DUID IS NULL OR BIDTYPE IS NULL OR
            INTERVAL_DATETIME IS NULL OR OFFERDATE IS NULL
          ) AS null_required_field_count,
          MIN(INTERVAL_DATETIME) AS first_interval,
          MAX(INTERVAL_DATETIME) AS last_interval
        FROM `{STAGE_TABLE_202108}`
        """
    ).result().to_dataframe()

    print("AUGUST 2021 RETAINED STAGE AUDIT")
    display(stage_audit)

manifest_audit = client.query(
    f"""
    SELECT
      RUN_ID,
      ARCHIVE_YEAR,
      ARCHIVE_MONTH,
      STATUS,
      SOURCE_ROWS,
      BESS_ROWS,
      LOADED_ROWS,
      CREATED_AT,
      COMPLETED_AT,
      ERROR_MESSAGE
    FROM `{MANIFEST_TABLE}`
    WHERE ARCHIVE_YEAR = 2021
      AND ARCHIVE_MONTH = 8
    ORDER BY CREATED_AT DESC
    """
).result().to_dataframe()

print("AUGUST 2021 MANIFEST ENTRIES")
display(manifest_audit)

In [ ]:
# SECTION 36J — Identify the exact missing August 2021 offer identities
# Read-only: no production changes, no staging writes, no deletes.

AUGUST_MONTH = pd.Timestamp("2021-08-01")

august_targets, august_expected_rows, _, _ = read_tracker_targets(AUGUST_MONTH)

stage_keys_df = client.query(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID
    FROM `{STAGE_TABLE_202108}`
    """
).result().to_dataframe()

august_stage_keys = {
    (
        str(row.DUID).strip().upper(),
        str(row.BIDTYPE).strip().upper(),
        normalise_key_timestamp(row.BIDSETTLEMENTDATE),
        normalise_key_timestamp(row.OFFERDATE),
        int(row.PERIODID),
    )
    for row in stage_keys_df.itertuples(index=False)
}

august_target_keys = set(august_targets)
missing_august_keys = august_target_keys - august_stage_keys

print("AUGUST 2021 MISSING-IDENTITY AUDIT")
print(f"Expected effective rows: {august_expected_rows:,}")
print(f"Rows already staged: {len(stage_keys_df):,}")
print(f"Effective rows still required: "
      f"{august_expected_rows - len(stage_keys_df):,}")
print(f"Missing raw-offer identities: {len(missing_august_keys):,}")

missing_examples = pd.DataFrame(
    [
        {
            "DUID": key[0],
            "BIDTYPE": key[1],
            "BIDSETTLEMENTDATE": key[2],
            "OFFERDATE": key[3],
            "PERIODID": key[4],
        }
        for key in sorted(missing_august_keys)[:30]
    ]
)

display(missing_examples)

In [ ]:
# SECTION 36K — Read-only proof that August's missing offers are in boundary cache
# No writes, deletes, promotions, or downloads.

BOUNDARY_LOOKUP_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

boundary_table = client.get_table(BOUNDARY_LOOKUP_TABLE)

print("BOUNDARY CACHE TABLE")
print(BOUNDARY_LOOKUP_TABLE)
print(f"Rows (metadata): {boundary_table.num_rows:,}")

print("\nBoundary-cache schema:")
for field in boundary_table.schema:
    print(f"  {field.name:<28} {field.field_type}")

boundary_preview = client.query(
    f"""
    SELECT *
    FROM `{BOUNDARY_LOOKUP_TABLE}`
    LIMIT 5
    """
).result().to_dataframe()

print("\nBoundary-cache sample:")
display(boundary_preview)

In [ ]:
# SECTION 36L — Inspect the already-built August tracker mapping
# Read-only. No downloads, writes, promotions, or deletes.

example_missing_key = sorted(missing_august_keys)[0]
example_targets = august_targets[example_missing_key]

print("Example missing offer identity:")
print(example_missing_key)

print("\nPython type of its tracker mapping:")
print(type(example_targets))

print("\nNumber of August effective intervals using this offer:")
print(len(example_targets))

print("\nFirst few target records:")
for item in example_targets[:5]:
    print(item)

In [ ]:
# SECTION 36M — Repair retained August 2021 stage from the verified boundary cache
# Writes ONLY to the existing August STAGE table.
# No AEMO downloads. No production promotion. No deletes.

from google.cloud import bigquery

BOUNDARY_LOOKUP_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

# Reconfirm the retained stage and August tracker requirements.
august_targets, august_expected_rows, _, _ = read_tracker_targets(
    pd.Timestamp("2021-08-01")
)

stage_table_obj = client.get_table(STAGE_TABLE_202108)
stage_field_names = [field.name for field in stage_table_obj.schema]

stage_raw_key_df = client.query(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      BIDSETTLEMENTDATE,
      OFFERDATE,
      PERIODID
    FROM `{STAGE_TABLE_202108}`
    """
).result().to_dataframe()

stage_raw_keys = {
    (
        str(row.DUID).strip().upper(),
        str(row.BIDTYPE).strip().upper(),
        normalise_key_timestamp(row.BIDSETTLEMENTDATE),
        normalise_key_timestamp(row.OFFERDATE),
        int(row.PERIODID),
    )
    for row in stage_raw_key_df.itertuples(index=False)
}

missing_keys = set(august_targets) - stage_raw_keys
rows_needed = august_expected_rows - len(stage_raw_key_df)
expanded_rows_needed = sum(len(august_targets[key]) for key in missing_keys)

print(f"Expected August effective rows: {august_expected_rows:,}")
print(f"Existing August stage rows: {len(stage_raw_key_df):,}")
print(f"Missing raw-offer identities: {len(missing_keys):,}")
print(f"Missing effective rows: {rows_needed:,}")

if len(missing_keys) != 1_152:
    raise RuntimeError(
        f"Safety stop: expected 1,152 missing identities, found "
        f"{len(missing_keys):,}."
    )

if expanded_rows_needed != rows_needed:
    raise RuntimeError(
        "Safety stop: tracker expansion does not equal the stage shortfall. "
        f"Expansion={expanded_rows_needed:,}; shortfall={rows_needed:,}."
    )

# Read the small, already-created boundary cache — no AEMO source download.
boundary_df = client.query(
    f"""
    SELECT *
    FROM `{BOUNDARY_LOOKUP_TABLE}`
    """
).result().to_dataframe()

boundary_records = {}

for row in boundary_df.itertuples(index=False):
    raw_key = (
        str(row.DUID).strip().upper(),
        str(row.BIDTYPE).strip().upper(),
        normalise_key_timestamp(row.BIDSETTLEMENTDATE),
        normalise_key_timestamp(row.OFFERDATE),
        int(row.PERIODID),
    )

    if raw_key in boundary_records:
        raise RuntimeError(f"Safety stop: duplicate boundary-cache key: {raw_key}")

    boundary_records[raw_key] = row._asdict()

cache_missing_keys = missing_keys - set(boundary_records)

print(f"Boundary-cache records available: {len(boundary_records):,}")
print(f"Missing identities found in boundary cache: "
      f"{len(missing_keys) - len(cache_missing_keys):,}")

if cache_missing_keys:
    raise RuntimeError(
        "Safety stop: boundary cache does not contain every required "
        f"August identity. Missing: {len(cache_missing_keys):,}."
    )

# Reuse the existing stage's pipeline/source metadata.
metadata_df = client.query(
    f"""
    SELECT
      _SOURCE,
      _PIPELINE,
      _SOURCE_FILE,
      _SOURCE_URL,
      _RUN_ID
    FROM `{STAGE_TABLE_202108}`
    LIMIT 1
    """
).result().to_dataframe()

if metadata_df.empty:
    raise RuntimeError("Safety stop: August stage has no metadata row.")

stage_metadata = metadata_df.iloc[0].to_dict()
load_timestamp = pd.Timestamp.now(tz="UTC")

# Expand each cached raw offer into its exact August tracker interval(s).
repair_rows = []

for raw_key in sorted(missing_keys):
    raw_record = boundary_records[raw_key]

    for target in august_targets[raw_key]:
        output_row = {field_name: None for field_name in stage_field_names}

        output_row.update({
            "SETTLEMENTDATE": target["SETTLEMENTDATE"],
            "DUID": raw_record["DUID"],
            "BIDTYPE": raw_record["BIDTYPE"],
            "DIRECTION": raw_record["DIRECTION"],
            "INTERVAL_DATETIME": target["INTERVAL_DATETIME"],
            "BIDSETTLEMENTDATE": raw_record["BIDSETTLEMENTDATE"],
            "OFFERDATE": raw_record["OFFERDATE"],
            "PERIODID": raw_record["PERIODID"],
            "MAXAVAIL": raw_record["MAXAVAIL"],
            "FIXEDLOAD": raw_record["FIXEDLOAD"],
            "ROCUP": raw_record["ROCUP"],
            "ROCDOWN": raw_record["ROCDOWN"],
            "ENABLEMENTMIN": raw_record["ENABLEMENTMIN"],
            "ENABLEMENTMAX": raw_record["ENABLEMENTMAX"],
            "LOWBREAKPOINT": raw_record["LOWBREAKPOINT"],
            "HIGHBREAKPOINT": raw_record["HIGHBREAKPOINT"],
            "PASAAVAILABILITY": raw_record["PASAAVAILABILITY"],
            "ENERGYLIMIT": raw_record["ENERGYLIMIT"],
            "_LOAD_DATE": load_timestamp,
            "_SOURCE": stage_metadata["_SOURCE"],
            "_PIPELINE": stage_metadata["_PIPELINE"],
            "_SOURCE_FILE": raw_record["SOURCE_FILE"],
            "_SOURCE_URL": raw_record["SOURCE_URL"],
            "_RUN_ID": stage_metadata["_RUN_ID"],
        })

        for band_number in range(1, 11):
            field_name = f"BANDAVAIL{band_number}"
            output_row[field_name] = raw_record[field_name]

        repair_rows.append(output_row)

repair_df = pd.DataFrame(repair_rows, columns=stage_field_names)

# Final in-memory safeguards before the append.
if len(repair_df) != rows_needed:
    raise RuntimeError(
        f"Safety stop: prepared {len(repair_df):,} rows; "
        f"expected {rows_needed:,}."
    )

if repair_df[
    ["DUID", "BIDTYPE", "DIRECTION", "INTERVAL_DATETIME"]
].duplicated().any():
    raise RuntimeError(
        "Safety stop: duplicate effective interval found in repair rows."
    )

existing_effective_df = client.query(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      DIRECTION,
      INTERVAL_DATETIME
    FROM `{STAGE_TABLE_202108}`
    """
).result().to_dataframe()

existing_effective_keys = {
    (
        str(row.DUID).strip().upper(),
        str(row.BIDTYPE).strip().upper(),
        str(row.DIRECTION).strip().upper(),
        pd.Timestamp(row.INTERVAL_DATETIME),
    )
    for row in existing_effective_df.itertuples(index=False)
}

repair_effective_keys = {
    (
        str(row.DUID).strip().upper(),
        str(row.BIDTYPE).strip().upper(),
        str(row.DIRECTION).strip().upper(),
        pd.Timestamp(row.INTERVAL_DATETIME),
    )
    for row in repair_df[
        ["DUID", "BIDTYPE", "DIRECTION", "INTERVAL_DATETIME"]
    ].itertuples(index=False)
}

overlap_count = len(existing_effective_keys & repair_effective_keys)

print(f"Repair rows prepared: {len(repair_df):,}")
print(f"Overlap with existing stage: {overlap_count:,}")

if overlap_count != 0:
    raise RuntimeError(
        "Safety stop: repair rows overlap existing August stage rows."
    )

# Append only the verified repair records.
load_job = client.load_table_from_dataframe(
    repair_df,
    STAGE_TABLE_202108,
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        schema=stage_table_obj.schema,
    ),
)
load_job.result()

# Final stage audit: it must now exactly equal the tracker requirement.
final_audit = client.query(
    f"""
    SELECT
      COUNT(*) AS stage_row_count,
      COUNT(DISTINCT CONCAT(
        COALESCE(DUID, ''), '|',
        COALESCE(BIDTYPE, ''), '|',
        COALESCE(DIRECTION, ''), '|',
        CAST(INTERVAL_DATETIME AS STRING)
      )) AS distinct_effective_key_count,
      COUNTIF(
        DUID IS NULL OR BIDTYPE IS NULL OR
        DIRECTION IS NULL OR INTERVAL_DATETIME IS NULL OR
        OFFERDATE IS NULL
      ) AS null_required_field_count,
      MIN(INTERVAL_DATETIME) AS first_interval,
      MAX(INTERVAL_DATETIME) AS last_interval
    FROM `{STAGE_TABLE_202108}`
    """
).result().to_dataframe()

display(final_audit)

final_rows = int(final_audit.iloc[0]["stage_row_count"])
final_distinct = int(final_audit.iloc[0]["distinct_effective_key_count"])
final_nulls = int(final_audit.iloc[0]["null_required_field_count"])

if (
    final_rows != august_expected_rows
    or final_distinct != august_expected_rows
    or final_nulls != 0
):
    raise RuntimeError(
        "Safety stop: August repair audit did not match the required "
        f"{august_expected_rows:,} rows. Stage retained for inspection."
    )

print("✅ AUGUST 2021 BOUNDARY-CACHE REPAIR PASSED")
print(f"✅ Stage now contains exactly {final_rows:,} effective-offer rows.")
print("✅ Production remains unchanged.")

In [ ]:
# SECTION 36N — Read-only audit of all ready April–August 2021 stages
# No downloads, writes, promotions, or deletes.

READY_MONTHS = [
    ("2021-04", 438_432),
    ("2021-05", 464_256),
    ("2021-06", 455_616),
    ("2021-07", 589_440),
    ("2021-08", 607_104),
]

PRODUCTION_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)

audit_results = []

for month_text, expected_count in READY_MONTHS:
    month_stamp = pd.Timestamp(f"{month_text}-01")
    stage_suffix = month_stamp.strftime("%Y%m")
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{stage_suffix}"
    )

    client.get_table(stage_table)  # Stops safely if a stage is missing.

    stage_result = client.query(
        f"""
        SELECT
          COUNT(*) AS stage_row_count,
          COUNT(DISTINCT CONCAT(
            COALESCE(DUID, ''), '|',
            COALESCE(BIDTYPE, ''), '|',
            COALESCE(DIRECTION, ''), '|',
            CAST(INTERVAL_DATETIME AS STRING)
          )) AS distinct_effective_key_count,
          COUNTIF(
            DUID IS NULL OR BIDTYPE IS NULL OR
            DIRECTION IS NULL OR INTERVAL_DATETIME IS NULL OR
            OFFERDATE IS NULL
          ) AS null_required_field_count
        FROM `{stage_table}`
        """
    ).result().to_dataframe().iloc[0]

    production_result = client.query(
        f"""
        SELECT COUNT(*) AS production_month_row_count
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
          AND SETTLEMENTDATE < TIMESTAMP(
              DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
          )
        """
    ).result().to_dataframe().iloc[0]

    audit_results.append({
        "SETTLEMENT_MONTH": month_text,
        "EXPECTED_STAGE_ROWS": expected_count,
        "STAGE_ROWS": int(stage_result["stage_row_count"]),
        "DISTINCT_EFFECTIVE_KEYS": int(
            stage_result["distinct_effective_key_count"]
        ),
        "NULL_REQUIRED_FIELDS": int(
            stage_result["null_required_field_count"]
        ),
        "PRODUCTION_ROWS_ALREADY_PRESENT": int(
            production_result["production_month_row_count"]
        ),
        "STAGE_TABLE": stage_table,
    })

ready_stage_audit = pd.DataFrame(audit_results)
display(ready_stage_audit)

all_valid = (
    (ready_stage_audit["STAGE_ROWS"]
     == ready_stage_audit["EXPECTED_STAGE_ROWS"]).all()
    and
    (ready_stage_audit["DISTINCT_EFFECTIVE_KEYS"]
     == ready_stage_audit["EXPECTED_STAGE_ROWS"]).all()
    and
    (ready_stage_audit["NULL_REQUIRED_FIELDS"] == 0).all()
    and
    (ready_stage_audit["PRODUCTION_ROWS_ALREADY_PRESENT"] == 0).all()
)

if not all_valid:
    raise RuntimeError(
        "Safety stop: one or more stages are not ready for promotion. "
        "No data was changed."
    )

print("✅ ALL APRIL–AUGUST 2021 STAGES PASSED PRE-PROMOTION AUDIT")
print(
    "✅ Total rows ready for promotion: "
    f"{ready_stage_audit['STAGE_ROWS'].sum():,}"
)
print("✅ Production remains unchanged.")

In [ ]:
# SECTION 36O — Promote audited April–August 2021 stages
# No AEMO downloads. No stage-table deletion.
# Each month is independently verified before and after promotion.

from datetime import datetime, timezone

PROMOTION_PLAN = [
    ("2021-04", 438_432),
    ("2021-05", 464_256),
    ("2021-06", 455_616),
    ("2021-07", 589_440),
    ("2021-08", 607_104),
]

PRODUCTION_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
MANIFEST_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)

promotion_results = []

for month_text, expected_count in PROMOTION_PLAN:
    month_stamp = pd.Timestamp(f"{month_text}-01")
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_stamp.strftime('%Y%m')}"
    )

    print("\n" + "=" * 100)
    print(f"PROMOTING {month_text}")
    print("=" * 100)

    # Recheck stage and production immediately before writing.
    preflight = client.query(
        f"""
        SELECT
          (SELECT COUNT(*) FROM `{stage_table}`) AS stage_row_count,
          (
            SELECT COUNT(*)
            FROM `{PRODUCTION_TABLE}`
            WHERE SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
              AND SETTLEMENTDATE < TIMESTAMP(
                DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
              )
          ) AS production_month_row_count
        """
    ).result().to_dataframe().iloc[0]

    stage_count = int(preflight["stage_row_count"])
    production_before = int(preflight["production_month_row_count"])

    print(f"Stage rows: {stage_count:,}")
    print(f"Production rows before: {production_before:,}")

    if stage_count != expected_count:
        raise RuntimeError(
            f"Safety stop for {month_text}: expected {expected_count:,} "
            f"stage rows, found {stage_count:,}."
        )

    if production_before != 0:
        raise RuntimeError(
            f"Safety stop for {month_text}: production already has "
            f"{production_before:,} rows. Nothing was promoted for this month."
        )

    # Obtain source metadata for a possible new manifest record.
    stage_metadata = client.query(
        f"""
        SELECT
          _RUN_ID,
          _SOURCE_FILE,
          _SOURCE_URL,
          _PIPELINE
        FROM `{stage_table}`
        LIMIT 1
        """
    ).result().to_dataframe().iloc[0].to_dict()

    # Copy the fully audited stage into production.
    client.query(
        f"""
        INSERT INTO `{PRODUCTION_TABLE}`
        SELECT *
        FROM `{stage_table}`
        """
    ).result()

    # Exact post-promotion verification.
    production_after = int(
        client.query(
            f"""
            SELECT COUNT(*) AS production_month_row_count
            FROM `{PRODUCTION_TABLE}`
            WHERE SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
              AND SETTLEMENTDATE < TIMESTAMP(
                DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
              )
            """
        ).result().to_dataframe().iloc[0]["production_month_row_count"]
    )

    print(f"Production rows after: {production_after:,}")

    if production_after != expected_count:
        raise RuntimeError(
            f"Safety stop for {month_text}: expected {expected_count:,} "
            f"production rows after promotion, found {production_after:,}. "
            "Stage table has been retained for reconciliation."
        )

    # Preserve an existing STAGED manifest row, or create one for August.
    manifest_count = int(
        client.query(
            f"""
            SELECT COUNT(*) AS manifest_entry_count
            FROM `{MANIFEST_TABLE}`
            WHERE ARCHIVE_YEAR = {month_stamp.year}
              AND ARCHIVE_MONTH = {month_stamp.month}
            """
        ).result().to_dataframe().iloc[0]["manifest_entry_count"]
    )

    if manifest_count > 1:
        raise RuntimeError(
            f"Safety stop for {month_text}: found {manifest_count} manifest "
            "entries. Production was verified, but manifest was not changed."
        )

    if manifest_count == 1:
        client.query(
            f"""
            UPDATE `{MANIFEST_TABLE}`
            SET
              STATUS = 'PROMOTED',
              LOADED_ROWS = {expected_count},
              COMPLETED_AT = CURRENT_TIMESTAMP(),
              ERROR_MESSAGE = NULL
            WHERE ARCHIVE_YEAR = {month_stamp.year}
              AND ARCHIVE_MONTH = {month_stamp.month}
            """
        ).result()
    else:
        manifest_row = {
            "RUN_ID": str(stage_metadata["_RUN_ID"]),
            "ARCHIVE_YEAR": month_stamp.year,
            "ARCHIVE_MONTH": month_stamp.month,
            "SOURCE_FILE_NAME": str(stage_metadata["_SOURCE_FILE"]),
            "SOURCE_URL": str(stage_metadata["_SOURCE_URL"]),
            "SOURCE_METHOD": str(stage_metadata["_PIPELINE"]),
            "STATUS": "PROMOTED",
            "SOURCE_ROWS": None,
            "BESS_ROWS": expected_count,
            "LOADED_ROWS": expected_count,
            "STARTED_AT": datetime.now(timezone.utc).isoformat(),
            "COMPLETED_AT": datetime.now(timezone.utc).isoformat(),
            "ERROR_MESSAGE": None,
            "CREATED_AT": datetime.now(timezone.utc).isoformat(),
        }

        errors = client.insert_rows_json(MANIFEST_TABLE, [manifest_row])

        if errors:
            raise RuntimeError(
                f"Manifest creation failed for {month_text}: {errors}"
            )

    promotion_results.append({
        "SETTLEMENT_MONTH": month_text,
        "PROMOTED_ROWS": production_after,
        "STAGE_TABLE_RETAINED": stage_table,
    })

promotion_summary = pd.DataFrame(promotion_results)
display(promotion_summary)

print("\n✅ APRIL–AUGUST 2021 PROMOTION COMPLETE")
print(f"✅ Total production rows promoted: "
      f"{promotion_summary['PROMOTED_ROWS'].sum():,}")
print("✅ All temporary stage tables were retained for final audit.")

In [ ]:
# SECTION 36P — Final production-versus-stage reconciliation
# Read-only: no downloads, writes, promotions, or deletes.

FINAL_AUDIT_MONTHS = [
    ("2021-04", 438_432),
    ("2021-05", 464_256),
    ("2021-06", 455_616),
    ("2021-07", 589_440),
    ("2021-08", 607_104),
]

final_audit_rows = []

for month_text, expected_count in FINAL_AUDIT_MONTHS:
    month_stamp = pd.Timestamp(f"{month_text}-01")
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_stamp.strftime('%Y%m')}"
    )

    result = client.query(
        f"""
        SELECT
          (SELECT COUNT(*) FROM `{stage_table}`) AS stage_row_count,

          (
            SELECT COUNT(*)
            FROM `{PRODUCTION_TABLE}`
            WHERE SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
              AND SETTLEMENTDATE < TIMESTAMP(
                DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
              )
          ) AS production_row_count,

          (
            SELECT COUNT(*)
            FROM `{stage_table}` AS stage_data
            WHERE NOT EXISTS (
              SELECT 1
              FROM `{PRODUCTION_TABLE}` AS production_data
              WHERE production_data.DUID = stage_data.DUID
                AND production_data.BIDTYPE = stage_data.BIDTYPE
                AND production_data.DIRECTION = stage_data.DIRECTION
                AND production_data.INTERVAL_DATETIME =
                    stage_data.INTERVAL_DATETIME
            )
          ) AS stage_missing_from_production_count,

          (
            SELECT COUNT(*)
            FROM `{PRODUCTION_TABLE}` AS production_data
            WHERE production_data.SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
              AND production_data.SETTLEMENTDATE < TIMESTAMP(
                DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
              )
              AND NOT EXISTS (
                SELECT 1
                FROM `{stage_table}` AS stage_data
                WHERE stage_data.DUID = production_data.DUID
                  AND stage_data.BIDTYPE = production_data.BIDTYPE
                  AND stage_data.DIRECTION = production_data.DIRECTION
                  AND stage_data.INTERVAL_DATETIME =
                      production_data.INTERVAL_DATETIME
              )
          ) AS production_missing_from_stage_count
        """
    ).result().to_dataframe().iloc[0]

    final_audit_rows.append({
        "SETTLEMENT_MONTH": month_text,
        "EXPECTED_ROWS": expected_count,
        "STAGE_ROWS": int(result["stage_row_count"]),
        "PRODUCTION_ROWS": int(result["production_row_count"]),
        "STAGE_MISSING_FROM_PRODUCTION": int(
            result["stage_missing_from_production_count"]
        ),
        "PRODUCTION_MISSING_FROM_STAGE": int(
            result["production_missing_from_stage_count"]
        ),
    })

final_promotion_audit = pd.DataFrame(final_audit_rows)
display(final_promotion_audit)

audit_passed = (
    (final_promotion_audit["STAGE_ROWS"]
     == final_promotion_audit["EXPECTED_ROWS"]).all()
    and
    (final_promotion_audit["PRODUCTION_ROWS"]
     == final_promotion_audit["EXPECTED_ROWS"]).all()
    and
    (final_promotion_audit["STAGE_MISSING_FROM_PRODUCTION"] == 0).all()
    and
    (final_promotion_audit["PRODUCTION_MISSING_FROM_STAGE"] == 0).all()
)

if not audit_passed:
    raise RuntimeError(
        "Final reconciliation did not pass. Stage tables have been retained "
        "and nothing was deleted."
    )

print("✅ APRIL–AUGUST 2021 FINAL PRODUCTION AUDIT PASSED")
print("✅ Every promoted effective-offer interval exactly matches its stage.")
print("✅ It is now safe to delete the five temporary stage tables.")

In [ ]:
# SECTION 36Q — Delete only the audited April–August 2021 temporary stages
# Production archive and manifest are preserved.

from google.api_core.exceptions import NotFound

CLEANUP_PLAN = [
    ("2021-04", 438_432),
    ("2021-05", 464_256),
    ("2021-06", 455_616),
    ("2021-07", 589_440),
    ("2021-08", 607_104),
]

stages_to_delete = []

# Final lightweight safety check before any deletion.
for month_text, expected_count in CLEANUP_PLAN:
    month_stamp = pd.Timestamp(f"{month_text}-01")
    stage_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_stamp.strftime('%Y%m')}"
    )

    stage_count = int(
        client.query(
            f"SELECT COUNT(*) AS stage_row_count FROM `{stage_table}`"
        ).result().to_dataframe().iloc[0]["stage_row_count"]
    )

    production_count = int(
        client.query(
            f"""
            SELECT COUNT(*) AS production_row_count
            FROM `{PRODUCTION_TABLE}`
            WHERE SETTLEMENTDATE >= TIMESTAMP('{month_text}-01')
              AND SETTLEMENTDATE < TIMESTAMP(
                DATE_ADD(DATE '{month_text}-01', INTERVAL 1 MONTH)
              )
            """
        ).result().to_dataframe().iloc[0]["production_row_count"]
    )

    print(
        f"{month_text} | stage: {stage_count:,} | "
        f"production: {production_count:,}"
    )

    if stage_count != expected_count or production_count != expected_count:
        raise RuntimeError(
            f"Safety stop: {month_text} no longer matches its approved "
            "production count. No tables were deleted."
        )

    stages_to_delete.append(stage_table)

# Exact deletion scope is printed before deletion.
print("\nDeleting only these temporary stage tables:")
for table_name in stages_to_delete:
    print(f" - {table_name}")

for table_name in stages_to_delete:
    client.delete_table(table_name, not_found_ok=False)
    print(f"✅ Deleted temporary stage: {table_name}")

print("\n✅ APRIL–AUGUST 2021 TEMPORARY STAGE CLEANUP COMPLETE")
print("✅ Production archive and manifest remain intact.")

In [ ]:
# SECTION 37A — Find the real existing AEMO Warehouse folder
# Read-only: does not create, move, download, or delete anything.

from pathlib import Path

NOTEBOOK_NAME = "AEMO_BESS_Backfill_2026_10_05.ipynb"

possible_folders = [
    Path.cwd(),
    Path.home() / "Desktop" / "AEMO Warehouse",
    Path.home() / "OneDrive" / "Desktop" / "AEMO Warehouse",
]

matches = []

for folder in possible_folders:
    if (folder / NOTEBOOK_NAME).exists():
        matches.append(folder)

if len(matches) != 1:
    print("Folders checked:")
    for folder in possible_folders:
        print(f" - {folder} | exists: {folder.exists()}")
    raise RuntimeError(
        "Could not identify one unique AEMO Warehouse folder. "
        "Nothing was changed."
    )

LOCAL_AEMO_WAREHOUSE = matches[0]

print("✅ Existing AEMO Warehouse folder found:")
print(LOCAL_AEMO_WAREHOUSE)

print("\nExisting contents:")
for item in sorted(LOCAL_AEMO_WAREHOUSE.iterdir()):
    print(f" - {item.name}")

In [ ]:
# SECTION 37B — Create BESS-only local cache inside the existing AEMO Warehouse
# Safe setup only: does not alter nemosis_cache, BigQuery, or existing notebooks.

import shutil

BESS_CACHE_ROOT = (
    LOCAL_AEMO_WAREHOUSE / "bess_effective_cache"
)

BESS_RAW_ZIPS = BESS_CACHE_ROOT / "raw_aemo_zips"
BESS_PARQUET = BESS_CACHE_ROOT / "bess_only_parquet"
BESS_STATE = BESS_CACHE_ROOT / "run_state"
BESS_LOGS = BESS_CACHE_ROOT / "logs"

for folder in [
    BESS_RAW_ZIPS,
    BESS_PARQUET,
    BESS_STATE,
    BESS_LOGS,
]:
    folder.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(BESS_CACHE_ROOT).free / (1024 ** 3)

print("✅ BESS local cache is ready")
print(f"Root: {BESS_CACHE_ROOT}")
print(f"Free SSD space: {free_gb:,.1f} GB")

print("\nNew folders created:")
print(f" - ZIP archive cache: {BESS_RAW_ZIPS}")
print(f" - BESS-only compact files: {BESS_PARQUET}")
print(f" - Resume/checkpoint state: {BESS_STATE}")
print(f" - Logs: {BESS_LOGS}")

print("\nExisting nemosis_cache and venv were not touched.")
print("No AEMO data has been downloaded yet.")


In [ ]:
# SECTION 37C — Cached AEMO ZIP downloader
# Setup only: no download begins until a later staging cell calls this function.

import os
import re
import time
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import requests

CACHE_DOWNLOAD_ATTEMPTS = 5
CACHE_DOWNLOAD_CHUNK_BYTES = 8 * 1024 * 1024  # 8 MB
CACHE_MIN_FREE_GB = 80


def cached_zip_path(source_file_name, source_url):
    """Return a safe local path for one AEMO ZIP source."""

    candidate_name = source_file_name or Path(
        urlparse(source_url).path
    ).name

    safe_name = re.sub(
        r'[<>:"/\\\\|?*]',
        "_",
        str(candidate_name),
    )

    if not safe_name.lower().endswith(".zip"):
        safe_name = f"{safe_name}.zip"

    return BESS_RAW_ZIPS / safe_name


def validate_cached_zip(zip_path):
    """Confirm the existing file is a readable ZIP without extracting it."""

    try:
        with zipfile.ZipFile(zip_path) as archive:
            csv_files = [
                name
                for name in archive.namelist()
                if name.upper().endswith(".CSV")
            ]

        return len(csv_files) > 0

    except (zipfile.BadZipFile, OSError):
        return False


def download_aemo_zip(source_url, label, source_file_name=None):
    """
    Download an AEMO ZIP once into the local cache.

    Existing valid ZIPs are reused.
    Incomplete transfers remain as .part files and are never mistaken
    for completed archives.
    """

    free_gb = shutil.disk_usage(BESS_RAW_ZIPS).free / (1024 ** 3)

    if free_gb < CACHE_MIN_FREE_GB:
        raise RuntimeError(
            f"Safety stop: only {free_gb:,.1f} GB free. "
            f"At least {CACHE_MIN_FREE_GB} GB is required."
        )

    final_path = cached_zip_path(source_file_name, source_url)
    partial_path = Path(f"{final_path}.part")

    if final_path.exists():
        if validate_cached_zip(final_path):
            print(f"♻️ Using cached ZIP: {final_path.name}")
            return str(final_path)

        raise RuntimeError(
            f"Existing cached file is not a valid ZIP: {final_path}\n"
            "It was not deleted automatically."
        )

    for attempt in range(1, CACHE_DOWNLOAD_ATTEMPTS + 1):
        response = None

        try:
            print(
                f"Downloading {label} — "
                f"attempt {attempt}/{CACHE_DOWNLOAD_ATTEMPTS}..."
            )

            if partial_path.exists():
                partial_path.unlink()

            response = requests.get(
                source_url,
                stream=True,
                timeout=(30, 900),
                headers={"User-Agent": "AEMO-BESS-Warehouse/1.0"},
            )
            response.raise_for_status()

            expected_bytes = int(
                response.headers.get("Content-Length", 0)
            )

            downloaded_bytes = 0
            next_progress_mark = 250 * 1024 * 1024

            with open(partial_path, "wb") as output_file:
                for chunk in response.iter_content(
                    chunk_size=CACHE_DOWNLOAD_CHUNK_BYTES
                ):
                    if not chunk:
                        continue

                    output_file.write(chunk)
                    downloaded_bytes += len(chunk)

                    if downloaded_bytes >= next_progress_mark:
                        print(
                            f"  Downloaded "
                            f"{downloaded_bytes / (1024 ** 2):,.0f} MB"
                        )
                        next_progress_mark += 250 * 1024 * 1024

            if expected_bytes and downloaded_bytes != expected_bytes:
                raise RuntimeError(
                    f"Incomplete download: got {downloaded_bytes:,} bytes; "
                    f"expected {expected_bytes:,}."
                )

            if not validate_cached_zip(partial_path):
                raise RuntimeError(
                    "Downloaded file is not a readable AEMO ZIP."
                )

            os.replace(partial_path, final_path)

            print(
                f"✅ Cached ZIP: {final_path.name} "
                f"({final_path.stat().st_size / (1024 ** 2):,.1f} MB)"
            )

            return str(final_path)

        except Exception as error:
            print(f"⚠️ Download attempt {attempt} failed: {error}")

            if attempt == CACHE_DOWNLOAD_ATTEMPTS:
                raise RuntimeError(
                    f"Could not cache {label} after "
                    f"{CACHE_DOWNLOAD_ATTEMPTS} attempts."
                ) from error

            time.sleep(min(120, 5 * (2 ** (attempt - 1))))

        finally:
            if response is not None:
                response.close()


print("✅ Cached AEMO downloader is ready.")
print("✅ Future downloads will be saved in:")
print(BESS_RAW_ZIPS)
print("✅ Existing valid ZIPs will be reused, not downloaded again.")


In [ ]:
# SECTION 37D — Make the proven Section 36 engine preserve cached ZIPs
# Setup only: no download, staging, promotion, or deletion of cached archives.

import uuid
from pathlib import Path

# Preserve the cached downloader created in Section 37C.
if "_download_aemo_zip_to_cache" not in globals():
    _download_aemo_zip_to_cache = download_aemo_zip

ACTIVE_ARCHIVE_LINKS = BESS_STATE / "active_archive_links"
ACTIVE_ARCHIVE_LINKS.mkdir(parents=True, exist_ok=True)


def download_aemo_zip(source_url, label, source_file_name=None):
    """
    Section 36-compatible downloader.

    It returns a temporary hard-link for the old Section 36 code to read.
    When Section 36 deletes that link, the real ZIP remains in raw_aemo_zips.
    """

    cached_path = Path(
        _download_aemo_zip_to_cache(
            source_url=source_url,
            label=label,
            source_file_name=source_file_name,
        )
    )

    working_path = (
        ACTIVE_ARCHIVE_LINKS
        / f"{uuid.uuid4().hex}_{cached_path.name}"
    )

    try:
        os.link(cached_path, working_path)
        return str(working_path)

    except OSError:
        # Safe fallback if Windows/OneDrive does not allow hard-links.
        shutil.copy2(cached_path, working_path)
        return str(working_path)


def lookup_carry_records(raw_keys):
    """
    Patched version of the Section 36H lookup.

    Same logic, but avoids using LOOKUP as a BigQuery SQL alias.
    """

    if not raw_keys:
        return {}

    temp_table = (
        f"{PROJECT_ID}.{DATASET_ID}."
        f"BESS_LOOKUP_KEYS_TMP_{uuid.uuid4().hex[:12]}"
    )

    key_df = pd.DataFrame(
        [{
            "DUID": key[0],
            "BIDTYPE": key[1],
            "BIDSETTLEMENTDATE": pd.to_datetime(key[2], utc=True),
            "OFFERDATE": pd.to_datetime(key[3], utc=True),
            "PERIODID": key[4],
        } for key in raw_keys]
    )

    try:
        client.load_table_from_dataframe(
            key_df,
            temp_table,
            job_config=bigquery.LoadJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            ),
        ).result()

        result_df = client.query(
            f"""
            WITH combined_sources AS (
              SELECT 1 AS source_priority, *
              FROM `{BOUNDARY_LOOKUP_TABLE}`

              UNION ALL

              SELECT 2 AS source_priority, *
              FROM `{CARRY_LOOKUP_TABLE}`
            )
            SELECT available_offer.*
            FROM combined_sources AS available_offer
            INNER JOIN `{temp_table}` AS needed
              ON available_offer.DUID = needed.DUID
              AND available_offer.BIDTYPE = needed.BIDTYPE
              AND available_offer.BIDSETTLEMENTDATE =
                  needed.BIDSETTLEMENTDATE
              AND available_offer.OFFERDATE = needed.OFFERDATE
              AND available_offer.PERIODID = needed.PERIODID
            QUALIFY ROW_NUMBER() OVER (
              PARTITION BY
                available_offer.DUID,
                available_offer.BIDTYPE,
                available_offer.BIDSETTLEMENTDATE,
                available_offer.OFFERDATE,
                available_offer.PERIODID
              ORDER BY available_offer.source_priority DESC
            ) = 1
            """
        ).result().to_dataframe()

    finally:
        client.delete_table(temp_table, not_found_ok=True)

    return {
        (
            row["DUID"].strip().upper(),
            row["BIDTYPE"].strip().upper(),
            normalise_key_timestamp(row["BIDSETTLEMENTDATE"]),
            normalise_key_timestamp(row["OFFERDATE"]),
            int(row["PERIODID"]),
        ): row
        for row in result_df.to_dict("records")
    }


print("✅ Section 36 cache-preservation patch is ready.")
print("✅ Cached ZIPs will remain in raw_aemo_zips after processing.")
print("✅ The BigQuery LOOKUP keyword error is fixed.")
print("✅ No data has been downloaded or changed yet.")

In [ ]:
# SECTION 37E — Cached local staging test: September 2021 only
# Writes: local ZIP cache + September temporary BigQuery stage + manifest/carry cache.
# Does NOT promote anything into production.

import json

MONTH_TO_STAGE = pd.Timestamp("2021-09-01")
NEXT_MONTH = MONTH_TO_STAGE + pd.offsets.MonthBegin(1)

RUN_ID_TO_STAGE = (
    f"bess-effective-reconstructed-{MONTH_TO_STAGE:%Y-%m}"
)
STAGE_TABLE_TO_STAGE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{MONTH_TO_STAGE:%Y%m}"
)

# Safety check: do not repeat an already staged/promoted month.
existing_manifest = client.query(
    f"""
    SELECT STATUS
    FROM `{MANIFEST_TABLE}`
    WHERE RUN_ID = @run_id
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter(
                "run_id",
                "STRING",
                RUN_ID_TO_STAGE,
            )
        ]
    ),
).result().to_dataframe()

if not existing_manifest.empty:
    raise RuntimeError(
        f"Safety stop: {MONTH_TO_STAGE:%Y-%m} already has manifest status "
        f"{existing_manifest.iloc[0]['STATUS']}."
    )

production_count = int(
    client.query(
        f"""
        SELECT COUNT(*) AS production_row_count
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-10-01')
        """
    ).result().to_dataframe().iloc[0]["production_row_count"]
)

if production_count != 0:
    raise RuntimeError(
        f"Safety stop: production already contains {production_count:,} "
        "September 2021 rows."
    )

try:
    client.get_table(STAGE_TABLE_TO_STAGE)
    raise RuntimeError(
        f"Safety stop: existing September stage found: {STAGE_TABLE_TO_STAGE}"
    )
except NotFound:
    pass

# Identify October's offers before reading September raw data.
BOUNDARY_CAPTURE_ACTIVE = False
next_targets, _, _, _ = read_tracker_targets(NEXT_MONTH)

BOUNDARY_CAPTURE_KEYS = set(next_targets)
BOUNDARY_CAPTURE_RECORDS = {}
BOUNDARY_CAPTURE_ACTIVE = True

print("=" * 100)
print("STARTING CACHED LOCAL STAGE: SEPTEMBER 2021")
print("=" * 100)
print("Production will not be modified.")
print(f"September stage table: {STAGE_TABLE_TO_STAGE}")
print(f"October boundary identities to capture: {len(BOUNDARY_CAPTURE_KEYS):,}")
print(f"Local ZIP cache: {BESS_RAW_ZIPS}")

try:
    september_result = stage_month_with_carry(MONTH_TO_STAGE)

    BOUNDARY_CAPTURE_ACTIVE = False

    boundary_row_count = persist_boundary_capture(
        MONTH_TO_STAGE,
        next_targets,
    )

    checkpoint_file = (
        BESS_STATE / "latest_completed_month.json"
    )

    checkpoint_payload = {
        "latest_completed_stage_month": "2021-09",
        "stage_table": STAGE_TABLE_TO_STAGE,
        "stage_rows": int(september_result["STAGE_ROWS"]),
        "boundary_rows_for_next_month": int(boundary_row_count),
        "status": "STAGED",
    }

    checkpoint_file.write_text(
        json.dumps(checkpoint_payload, indent=2),
        encoding="utf-8",
    )

    print("\n✅ SEPTEMBER 2021 CACHED LOCAL STAGE COMPLETE")
    print(
        f"✅ Stage rows: {september_result['STAGE_ROWS']:,}"
    )
    print(
        f"✅ Boundary offers captured for October: "
        f"{boundary_row_count:,}"
    )
    print(f"✅ Local checkpoint saved: {checkpoint_file}")
    print("✅ Production remains unchanged.")

except Exception:
    BOUNDARY_CAPTURE_ACTIVE = False
    print(
        "\n⚠️ September stage, if created, was retained for inspection."
    )
    print(
        "⚠️ Any completed AEMO ZIP remains in the local cache for reuse."
    )
    raise

In [ ]:
# SECTION 37F — Promote verified September 2021 stage
# No AEMO download. Stage table is retained for final audit.

SEPTEMBER_MONTH = "2021-09"
SEPTEMBER_EXPECTED_ROWS = 587_520
SEPTEMBER_STAGE_TABLE = (
    f"{PROJECT_ID}.{DATASET_ID}."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202109"
)
SEPTEMBER_RUN_ID = "bess-effective-reconstructed-2021-09"

preflight_df = client.query(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{SEPTEMBER_STAGE_TABLE}`)
        AS stage_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-10-01')
      ) AS production_month_row_count,

      (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = '{SEPTEMBER_RUN_ID}'
          AND STATUS = 'STAGED'
      ) AS staged_manifest_count
    """
).result().to_dataframe()

display(preflight_df)

preflight = preflight_df.iloc[0]

if int(preflight["stage_row_count"]) != SEPTEMBER_EXPECTED_ROWS:
    raise RuntimeError("Safety stop: September stage count is not exact.")

if int(preflight["production_month_row_count"]) != 0:
    raise RuntimeError(
        "Safety stop: September production rows already exist."
    )

if int(preflight["staged_manifest_count"]) != 1:
    raise RuntimeError(
        "Safety stop: expected exactly one STAGED September manifest entry."
    )

print("✅ September promotion preflight passed.")
print("Promoting verified September stage...")

client.query(
    f"""
    INSERT INTO `{PRODUCTION_TABLE}`
    SELECT *
    FROM `{SEPTEMBER_STAGE_TABLE}`
    """
).result()

post_promotion_df = client.query(
    f"""
    SELECT COUNT(*) AS production_month_row_count
    FROM `{PRODUCTION_TABLE}`
    WHERE SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
      AND SETTLEMENTDATE < TIMESTAMP('2021-10-01')
    """
).result().to_dataframe()

display(post_promotion_df)

production_rows = int(
    post_promotion_df.iloc[0]["production_month_row_count"]
)

if production_rows != SEPTEMBER_EXPECTED_ROWS:
    raise RuntimeError(
        "Safety stop: September promotion count does not match the stage."
    )

client.query(
    f"""
    UPDATE `{MANIFEST_TABLE}`
    SET
      STATUS = 'PROMOTED',
      LOADED_ROWS = {SEPTEMBER_EXPECTED_ROWS},
      COMPLETED_AT = CURRENT_TIMESTAMP(),
      ERROR_MESSAGE = NULL
    WHERE RUN_ID = '{SEPTEMBER_RUN_ID}'
    """
).result()

print("✅ SEPTEMBER 2021 PROMOTED")
print(f"✅ Production rows retained: {production_rows:,}")
print("✅ September stage table remains for final audit.")

In [ ]:
# SECTION 37G — Final September 2021 stage-versus-production audit
# Read-only: no downloads, writes, promotions, or deletions.

september_final_audit = client.query(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{SEPTEMBER_STAGE_TABLE}`)
        AS stage_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-10-01')
      ) AS production_row_count,

      (
        SELECT COUNT(*)
        FROM `{SEPTEMBER_STAGE_TABLE}` AS stage_data
        WHERE NOT EXISTS (
          SELECT 1
          FROM `{PRODUCTION_TABLE}` AS production_data
          WHERE production_data.DUID = stage_data.DUID
            AND production_data.BIDTYPE = stage_data.BIDTYPE
            AND production_data.DIRECTION = stage_data.DIRECTION
            AND production_data.INTERVAL_DATETIME =
                stage_data.INTERVAL_DATETIME
        )
      ) AS stage_missing_from_production_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}` AS production_data
        WHERE production_data.SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
          AND production_data.SETTLEMENTDATE < TIMESTAMP('2021-10-01')
          AND NOT EXISTS (
            SELECT 1
            FROM `{SEPTEMBER_STAGE_TABLE}` AS stage_data
            WHERE stage_data.DUID = production_data.DUID
              AND stage_data.BIDTYPE = production_data.BIDTYPE
              AND stage_data.DIRECTION = production_data.DIRECTION
              AND stage_data.INTERVAL_DATETIME =
                  production_data.INTERVAL_DATETIME
        )
      ) AS production_missing_from_stage_count
    """
).result().to_dataframe()

display(september_final_audit)

audit = september_final_audit.iloc[0]

if (
    int(audit["stage_row_count"]) != SEPTEMBER_EXPECTED_ROWS
    or int(audit["production_row_count"]) != SEPTEMBER_EXPECTED_ROWS
    or int(audit["stage_missing_from_production_count"]) != 0
    or int(audit["production_missing_from_stage_count"]) != 0
):
    raise RuntimeError(
        "September final audit did not pass. Stage retained; nothing changed."
    )

print("✅ SEPTEMBER 2021 FINAL PRODUCTION AUDIT PASSED")
print("✅ Safe to delete only the temporary September stage next.")

In [ ]:
# SECTION 37H — Delete only the verified September 2021 temporary stage

final_check = client.query(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{SEPTEMBER_STAGE_TABLE}`)
        AS stage_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-09-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-10-01')
      ) AS production_row_count
    """
).result().to_dataframe().iloc[0]

stage_count = int(final_check["stage_row_count"])
production_count = int(final_check["production_row_count"])

print(f"September stage rows: {stage_count:,}")
print(f"September production rows: {production_count:,}")

if (
    stage_count != SEPTEMBER_EXPECTED_ROWS
    or production_count != SEPTEMBER_EXPECTED_ROWS
):
    raise RuntimeError(
        "Safety stop: September stage and production no longer match. "
        "Nothing was deleted."
    )

client.delete_table(SEPTEMBER_STAGE_TABLE, not_found_ok=False)

print("✅ September 2021 temporary stage deleted.")
print("✅ Production archive, manifest, boundary cache, and local ZIP cache remain intact.")

In [ ]:
# SECTION 38A — Install/check DuckDB in the existing Jupyter environment
# Local software setup only: no AEMO download or BigQuery changes.

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("duckdb") is None:
    print("Installing DuckDB in the current Jupyter environment...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "duckdb",
    ])

import duckdb

print("✅ DuckDB is ready")
print(f"Version: {duckdb.__version__}")
print(f"Python environment: {sys.executable}")

duckdb_connection = duckdb.connect()
duckdb_connection.execute("PRAGMA threads=4")

print("✅ Local DuckDB connection opened.")
print("✅ No AEMO data or BigQuery table was changed.")

In [ ]:
# SECTION 38B — DuckDB September source preflight
# Read-only: no extraction, BigQuery write, or download.

import os
import zipfile

SEPTEMBER_RAW_ZIP = (
    BESS_RAW_ZIPS
    / "PUBLIC_DVD_BIDPEROFFER_202109010000.zip"
)

if not SEPTEMBER_RAW_ZIP.exists():
    raise FileNotFoundError(
        f"Cached September ZIP not found:\n{SEPTEMBER_RAW_ZIP}"
    )

with zipfile.ZipFile(SEPTEMBER_RAW_ZIP) as archive:
    csv_entries = [
        entry
        for entry in archive.infolist()
        if entry.filename.upper().endswith(".CSV")
    ]

if len(csv_entries) != 1:
    raise RuntimeError(
        f"Expected one CSV inside the September ZIP; found {len(csv_entries)}."
    )

SEPTEMBER_CSV_INFO = csv_entries[0]

available_cores = os.cpu_count() or 1
duckdb_threads = max(1, available_cores - 2)

duckdb_connection.execute(f"SET threads TO {duckdb_threads}")

free_gb = shutil.disk_usage(BESS_CACHE_ROOT).free / (1024 ** 3)
uncompressed_gb = SEPTEMBER_CSV_INFO.file_size / (1024 ** 3)
compressed_gb = SEPTEMBER_RAW_ZIP.stat().st_size / (1024 ** 3)

# Leave enough working room for a temporary CSV plus DuckDB output.
required_free_gb = max(30, uncompressed_gb * 1.5)

print("SEPTEMBER 2021 DUCKDB PREFLIGHT")
print(f"Cached ZIP: {SEPTEMBER_RAW_ZIP.name}")
print(f"Compressed ZIP size: {compressed_gb:,.2f} GB")
print(f"CSV inside ZIP: {SEPTEMBER_CSV_INFO.filename}")
print(f"Uncompressed CSV size: {uncompressed_gb:,.2f} GB")
print(f"Free SSD space: {free_gb:,.1f} GB")
print(f"Tank CPU cores visible to Python: {available_cores}")
print(f"DuckDB processing threads set to: {duckdb_threads}")

if free_gb < required_free_gb:
    raise RuntimeError(
        f"Safety stop: need at least {required_free_gb:,.1f} GB free "
        "for the temporary DuckDB working file."
    )

print("\n✅ DuckDB preflight passed.")
print("✅ Safe to extract and benchmark September locally.")
print("✅ No data was changed.")

In [ ]:
# SECTION 38C-RECOVERY — Inspect any interrupted DuckDB working files
# Read-only: does not delete or change anything.

possible_files = [
    SEPTEMBER_WORKING_CSV,
    Path(f"{SEPTEMBER_WORKING_CSV}.part"),
]

print("SEPTEMBER LOCAL WORKING-FILE CHECK")

for file_path in possible_files:
    if file_path.exists():
        print(
            f"FOUND: {file_path.name} | "
            f"{file_path.stat().st_size / (1024 ** 3):,.2f} GB"
        )
    else:
        print(f"NOT FOUND: {file_path.name}")

print("\n✅ September production remains complete.")
print("✅ No cleanup has been performed by this check.")

In [ ]:
# SECTION 38C-CLEANUP — Delete only the redundant September extracted CSV

from pathlib import Path

target_file = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache\duckdb_working"
    r"\PUBLIC_DVD_BIDPEROFFER_202109010000.CSV"
)

if not target_file.exists():
    print("Nothing deleted: the September working CSV is already absent.")
elif target_file.name != "PUBLIC_DVD_BIDPEROFFER_202109010000.CSV":
    raise RuntimeError("Safety stop: unexpected filename.")
else:
    size_gb = target_file.stat().st_size / (1024 ** 3)
    print(f"Deleting only: {target_file}")
    print(f"Space to recover: {size_gb:.2f} GB")
    target_file.unlink()
    print("✅ September temporary extracted CSV deleted.")
    print("✅ Cached ZIP, BigQuery production, manifest, and notebook were not changed.")

In [ ]:
# SECTION 38D1 — Scan the already-created DuckDB sample
# Read-only. Uses the sample created by Section 38D.

import time
import duckdb

if not SAMPLE_CSV.exists():
    raise FileNotFoundError(
        "The clean sample is missing. Re-run Section 38D first."
    )

con = duckdb.connect()
con.execute("PRAGMA threads=22")

scan_start = time.perf_counter()

summary = con.execute(
    """
    SELECT
      COUNT(*) AS sample_rows,
      COUNT(DISTINCT DUID) AS distinct_duids,
      MIN(TRADINGDATE) AS first_tradingdate,
      MAX(TRADINGDATE) AS last_tradingdate,
      COUNT(*) FILTER (
          WHERE DUID IS NULL OR TRIM(DUID) = ''
      ) AS blank_duid_rows
    FROM read_csv(
        ?,
        header = true,
        all_varchar = true,
        strict_mode = false,
        null_padding = true
    )
    """,
    [str(SAMPLE_CSV)],
).fetchdf()

scan_seconds = time.perf_counter() - scan_start

columns = con.execute(
    """
    SELECT column_name
    FROM (DESCRIBE SELECT * FROM read_csv(
        ?,
        header = true,
        all_varchar = true,
        strict_mode = false,
        null_padding = true
    ))
    """,
    [str(SAMPLE_CSV)],
).fetchdf()

con.close()

print("✅ DUCKDB SAMPLE SCAN PASSED")
print(f"DuckDB scan time: {scan_seconds:.2f} seconds")
print("\nSample summary:")
display(summary)

print("\nColumns found in this newer-format source:")
display(columns)

print("✅ No AEMO source data or BigQuery data was changed.")

In [ ]:
# SECTION 38E — DuckDB raw AEMO-format proof (corrected)
# Read-only against AEMO/BigQuery. Creates a small disposable raw-format test file.

import csv
import io
import time
import zipfile
from pathlib import Path
import duckdb

RAW_PROBE_CSV = DUCKDB_TEST_DIR / "bidperoffer_202109_raw_format_probe.csv"
RAW_PROBE_RECORDS = 100_000

if not SEPTEMBER_ZIP.exists():
    raise FileNotFoundError(f"Cached September ZIP not found:\n{SEPTEMBER_ZIP}")

with zipfile.ZipFile(SEPTEMBER_ZIP) as archive:
    csv_members = [
        name for name in archive.namelist()
        if name.upper().endswith(".CSV")
    ]
    if len(csv_members) != 1:
        raise RuntimeError(f"Expected one CSV in the ZIP; found {csv_members}")

    csv_member = csv_members[0]
    data_header = None

    with archive.open(csv_member) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
        )
        for record in reader:
            if (
                record
                and record[0].strip().upper() == "I"
                and len(record) > 4
                and record[3].strip() == "1"
                and "DUID" in record[4:]
            ):
                data_header = record[4:]
                break

if data_header is None:
    raise RuntimeError("Could not locate the AEMO BIDPEROFFER header record.")

start = time.perf_counter()

with zipfile.ZipFile(SEPTEMBER_ZIP) as archive:
    with archive.open(csv_member) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
        )

        with open(RAW_PROBE_CSV, "w", encoding="utf-8", newline="") as output_file:
            writer = csv.writer(output_file)
            copied_records = 0

            for record in reader:
                if not record:
                    continue

                writer.writerow(record)
                copied_records += 1

                if copied_records >= RAW_PROBE_RECORDS:
                    break

if copied_records != RAW_PROBE_RECORDS:
    raise RuntimeError(
        f"Only copied {copied_records:,} records; expected {RAW_PROBE_RECORDS:,}."
    )

raw_columns = [
    "RECORD_TYPE",
    "AEMO_GROUP",
    "AEMO_TABLE",
    "AEMO_VERSION",
] + data_header

if len(raw_columns) != len(set(raw_columns)):
    raise RuntimeError(f"Duplicate source column names found: {raw_columns}")

def sql_identifier(name):
    return '"' + name.replace('"', '""') + '"'

column_mapping_sql = ", ".join(
    f"{sql_identifier(name)}: 'VARCHAR'"
    for name in raw_columns
)

# Prepare the path before the f-string: avoids the backslash syntax issue.
raw_probe_sql_path = RAW_PROBE_CSV.as_posix()

con = duckdb.connect()
con.execute("PRAGMA threads=22")

scan_start = time.perf_counter()

raw_summary = con.execute(
    f"""
    SELECT
      COUNT(*) AS records_read,
      COUNT(*) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'C'
      ) AS control_records,
      COUNT(*) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'I'
      ) AS header_records,
      COUNT(*) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
      ) AS data_records,
      COUNT(DISTINCT DUID) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
      ) AS distinct_data_duids,
      MIN(TRADINGDATE) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
      ) AS first_tradingdate,
      MAX(TRADINGDATE) FILTER (
          WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
      ) AS last_tradingdate
    FROM read_csv(
        '{raw_probe_sql_path}',
        header = false,
        columns = {{{column_mapping_sql}}},
        strict_mode = false,
        null_padding = true,
        ignore_errors = true
    )
    """
).fetchdf()

scan_seconds = time.perf_counter() - scan_start
con.close()

probe_size_mb = RAW_PROBE_CSV.stat().st_size / (1024 ** 2)

print("✅ DUCKDB RAW AEMO-FORMAT TEST PASSED")
print(f"Raw records copied: {copied_records:,}")
print(f"Probe size: {probe_size_mb:.1f} MB")
print(f"DuckDB direct raw-format scan: {scan_seconds:.2f} seconds")
display(raw_summary)

print()
print("✅ No BigQuery tables, manifest rows, or production data were changed.")
print("✅ The full September CSV was not recreated.")

In [ ]:
# SECTION 38F — Prove DuckDB timestamp normalisation for raw-offer matching
# Local read-only test. No download, extraction, BigQuery write, or staging.

import duckdb

if not RAW_PROBE_CSV.exists():
    raise FileNotFoundError("Raw probe is missing. Re-run Section 38E.")

con = duckdb.connect()
con.execute("PRAGMA threads=22")

normalised_samples = con.execute(
    f"""
    SELECT
      DUID,
      BIDTYPE,
      TRADINGDATE AS source_tradingdate,
      OFFERDATETIME AS source_offerdatetime,

      strftime(
        coalesce(
          try_strptime(TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
          try_strptime(TRADINGDATE, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
      ) AS normalised_tradingdate,

      strftime(
        coalesce(
          try_strptime(OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
          try_strptime(OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
      ) AS normalised_offerdatetime,

      PERIODID

    FROM read_csv(
        '{raw_probe_sql_path}',
        header = false,
        columns = {{{column_mapping_sql}}},
        strict_mode = false,
        null_padding = true,
        ignore_errors = true
    )
    WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
    LIMIT 10
    """
).fetchdf()

normalisation_audit = con.execute(
    f"""
    SELECT
      COUNT(*) AS data_records_checked,

      COUNT(*) FILTER (
        WHERE coalesce(
          try_strptime(TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
          try_strptime(TRADINGDATE, '%Y/%m/%d %H:%M:%S')
        ) IS NULL
      ) AS unparseable_tradingdates,

      COUNT(*) FILTER (
        WHERE coalesce(
          try_strptime(OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
          try_strptime(OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
        ) IS NULL
      ) AS unparseable_offerdatetimes

    FROM read_csv(
        '{raw_probe_sql_path}',
        header = false,
        columns = {{{column_mapping_sql}}},
        strict_mode = false,
        null_padding = true,
        ignore_errors = true
    )
    WHERE UPPER(TRIM(RECORD_TYPE)) = 'D'
    """
).fetchdf()

con.close()

print("✅ DUCKDB RAW-OFFER TIMESTAMP TEST PASSED")
display(normalisation_audit)

print("\nExample source values and their matching keys:")
display(normalised_samples)

if int(normalisation_audit.iloc[0]["unparseable_tradingdates"]) != 0:
    raise RuntimeError("Safety stop: some TRADINGDATE values could not be normalised.")

if int(normalisation_audit.iloc[0]["unparseable_offerdatetimes"]) != 0:
    raise RuntimeError("Safety stop: some OFFERDATETIME values could not be normalised.")

print("✅ DuckDB can create the exact date/time keys required for October matching.")
print("✅ No data was changed.")

In [ ]:
# SECTION 38G — October 2021: local DuckDB raw-offer matching
# Local preparation only. No BigQuery writes, stage tables, promotions, or deletes.

import csv
import io
import os
import shutil
import time
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

OCTOBER_MONTH = pd.Timestamp("2021-10-01")

LOCAL_ROOT = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache"
)
RAW_ZIP_CACHE = LOCAL_ROOT / "raw_aemo_zips"
PARQUET_CACHE = LOCAL_ROOT / "bess_only_parquet"
RUN_STATE = LOCAL_ROOT / "run_state"
DUCKDB_WORKING = LOCAL_ROOT / "duckdb_working"

for folder in [RAW_ZIP_CACHE, PARQUET_CACHE, RUN_STATE, DUCKDB_WORKING]:
    folder.mkdir(parents=True, exist_ok=True)

OCTOBER_WORKING_CSV = (
    DUCKDB_WORKING / "PUBLIC_DVD_BIDPEROFFER_202110010000.CSV"
)
OCTOBER_WORKING_PART = Path(f"{OCTOBER_WORKING_CSV}.part")
OCTOBER_TARGET_CSV = RUN_STATE / "october_2021_tracker_offer_identities.csv"
OCTOBER_MATCH_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_202110_current_source_matches.parquet"
)

if OCTOBER_MATCH_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: October's compact match file already exists:\n"
        f"{OCTOBER_MATCH_PARQUET}\n"
        "Nothing was overwritten."
    )

if OCTOBER_WORKING_PART.exists():
    raise RuntimeError(
        "Safety stop: an interrupted October extraction exists:\n"
        f"{OCTOBER_WORKING_PART}\n"
        "Nothing was overwritten or deleted."
    )

if "_download_aemo_zip_to_cache" not in globals():
    raise RuntimeError(
        "Run the earlier cache-preservation section first: "
        "_download_aemo_zip_to_cache is unavailable."
    )

free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)
if free_gb < 70:
    raise RuntimeError(
        f"Safety stop: only {free_gb:.1f} GB free. "
        "At least 70 GB is required for the temporary October extraction."
    )

print("=" * 100)
print("OCTOBER 2021 — LOCAL DUCKDB RAW-OFFER PREPARATION")
print("=" * 100)
print(f"Free SSD space: {free_gb:.1f} GB")
print("BigQuery production and staging tables will not be changed.")

# Tracker requirements are small and use the existing cached downloader.
october_targets, october_expected_rows, _, _ = read_tracker_targets(OCTOBER_MONTH)

if not october_targets:
    raise RuntimeError("Safety stop: no October tracker targets were found.")

def canonical_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

target_rows = []
for key in october_targets:
    duid, bidtype, bidsettlementdate, offerdate, periodid = key

    target_rows.append(
        {
            "DUID": str(duid).strip().upper(),
            "BIDTYPE": str(bidtype).strip().upper(),
            "BIDSETTLEMENTDATE_KEY": canonical_timestamp(bidsettlementdate),
            "OFFERDATE_KEY": canonical_timestamp(offerdate),
            "PERIODID": int(periodid),
        }
    )

target_df = (
    pd.DataFrame(target_rows)
    .drop_duplicates()
    .sort_values(
        [
            "DUID",
            "BIDTYPE",
            "BIDSETTLEMENTDATE_KEY",
            "OFFERDATE_KEY",
            "PERIODID",
        ]
    )
    .reset_index(drop=True)
)

if target_df.empty:
    raise RuntimeError("Safety stop: October target-identity list is empty.")

target_df.to_csv(OCTOBER_TARGET_CSV, index=False)

print(f"Expected October effective rows: {october_expected_rows:,}")
print(f"Unique raw-offer identities required: {len(target_df):,}")
print(f"Tracker DUIDs in scope: {target_df['DUID'].nunique():,}")

# Download the October raw source once into the retained ZIP cache.
october_source_file, october_source_url = raw_source_urls(OCTOBER_MONTH)[0]

october_zip = Path(
    _download_aemo_zip_to_cache(
        october_source_url,
        f"October 2021 raw source {october_source_file}",
        source_file_name=october_source_file,
    )
)

if not october_zip.exists():
    raise RuntimeError("Safety stop: October ZIP was not placed in the cache.")

with zipfile.ZipFile(october_zip) as archive:
    csv_members = [
        member for member in archive.namelist()
        if member.upper().endswith(".CSV")
    ]

    if len(csv_members) != 1:
        raise RuntimeError(
            f"Expected exactly one CSV in October ZIP; found {csv_members}"
        )

    october_csv_member = csv_members[0]
    october_csv_info = archive.getinfo(october_csv_member)
    expected_csv_bytes = october_csv_info.file_size

    data_header = None
    with archive.open(october_csv_member) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
        )

        for record in reader:
            if (
                record
                and record[0].strip().upper() == "I"
                and len(record) > 4
                and record[3].strip() == "1"
                and "DUID" in record[4:]
            ):
                data_header = record[4:]
                break

if data_header is None:
    raise RuntimeError("Safety stop: could not find October's BIDPEROFFER header.")

required_source_columns = {
    "DUID",
    "BIDTYPE",
    "TRADINGDATE",
    "OFFERDATETIME",
    "PERIODID",
}
missing_source_columns = required_source_columns - set(data_header)

if missing_source_columns:
    raise RuntimeError(
        "Safety stop: October source schema differs from the proven format. "
        f"Missing: {sorted(missing_source_columns)}"
    )

print("October raw source schema verified.")
print(f"Compressed ZIP: {october_zip.stat().st_size / (1024 ** 3):.2f} GB")
print(f"Temporary extracted CSV required: {expected_csv_bytes / (1024 ** 3):.2f} GB")

# Extract only if a complete local CSV is not already available.
if OCTOBER_WORKING_CSV.exists():
    existing_bytes = OCTOBER_WORKING_CSV.stat().st_size

    if existing_bytes != expected_csv_bytes:
        raise RuntimeError(
            "Safety stop: an unexpected October working CSV already exists:\n"
            f"{OCTOBER_WORKING_CSV}\n"
            f"Expected bytes: {expected_csv_bytes:,}; found: {existing_bytes:,}.\n"
            "Nothing was overwritten."
        )

    print("♻️ Reusing complete local October extraction.")
else:
    print("Extracting October CSV locally — this is temporary and will be retained for audit.")
    copied_bytes = 0
    next_progress_bytes = 1024 ** 3

    with zipfile.ZipFile(october_zip) as archive:
        with archive.open(october_csv_member) as source_file:
            with open(OCTOBER_WORKING_PART, "wb") as destination_file:
                while True:
                    block = source_file.read(16 * 1024 * 1024)

                    if not block:
                        break

                    destination_file.write(block)
                    copied_bytes += len(block)

                    if copied_bytes >= next_progress_bytes:
                        print(
                            f"  Extracted {copied_bytes / (1024 ** 3):.1f} GB "
                            f"of {expected_csv_bytes / (1024 ** 3):.1f} GB"
                        )
                        next_progress_bytes += 1024 ** 3

    if copied_bytes != expected_csv_bytes:
        raise RuntimeError(
            f"Safety stop: incomplete extraction: {copied_bytes:,} of "
            f"{expected_csv_bytes:,} bytes. The .part file was retained."
        )

    os.replace(OCTOBER_WORKING_PART, OCTOBER_WORKING_CSV)
    print("✅ Complete temporary October CSV extracted.")

# Build an explicit DuckDB schema for the original AEMO C/I/D format.
raw_columns = [
    "RECORD_TYPE",
    "AEMO_GROUP",
    "AEMO_TABLE",
    "AEMO_VERSION",
] + data_header

if len(raw_columns) != len(set(raw_columns)):
    raise RuntimeError(
        f"Safety stop: duplicate October source column names: {raw_columns}"
    )

def quoted_identifier(name):
    return '"' + name.replace('"', '""') + '"'

column_mapping_sql = ", ".join(
    f"{quoted_identifier(name)}: 'VARCHAR'"
    for name in raw_columns
)

source_data_select_sql = ", ".join(
    f"raw.{quoted_identifier(name)}"
    for name in data_header
)

october_csv_sql_path = OCTOBER_WORKING_CSV.as_posix()
october_target_sql_path = OCTOBER_TARGET_CSV.as_posix()
october_parquet_sql_path = OCTOBER_MATCH_PARQUET.as_posix()

normalised_tradingdate_sql = """
strftime(
    coalesce(
        try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
        try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S')
    ),
    '%Y-%m-%d %H:%M:%S.%f'
)
"""

normalised_offerdatetime_sql = """
strftime(
    coalesce(
        try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
        try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
    ),
    '%Y-%m-%d %H:%M:%S.%f'
)
"""

con = duckdb.connect()
con.execute("PRAGMA threads=22")

print()
print("DuckDB is now scanning October once and writing only required raw offers...")

duckdb_start = time.perf_counter()

con.execute(
    f"""
    COPY (
        SELECT
            {source_data_select_sql},
            {normalised_tradingdate_sql} AS BIDSETTLEMENTDATE_KEY,
            {normalised_offerdatetime_sql} AS OFFERDATE_KEY

        FROM read_csv(
            '{october_csv_sql_path}',
            header = false,
            columns = {{{column_mapping_sql}}},
            strict_mode = false,
            null_padding = true,
            ignore_errors = true
        ) AS raw

        INNER JOIN read_csv(
            '{october_target_sql_path}',
            header = true,
            all_varchar = true
        ) AS target
          ON UPPER(TRIM(raw.DUID)) = target.DUID
         AND UPPER(TRIM(raw.BIDTYPE)) = target.BIDTYPE
         AND {normalised_tradingdate_sql} = target.BIDSETTLEMENTDATE_KEY
         AND {normalised_offerdatetime_sql} = target.OFFERDATE_KEY
         AND TRY_CAST(raw.PERIODID AS INTEGER) =
             TRY_CAST(target.PERIODID AS INTEGER)

        WHERE UPPER(TRIM(raw.RECORD_TYPE)) = 'D'
    )
    TO '{october_parquet_sql_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

duckdb_seconds = time.perf_counter() - duckdb_start

match_audit = con.execute(
    f"""
    SELECT
        COUNT(*) AS matched_raw_offer_rows,
        COUNT(*) AS distinct_raw_offer_keys,
        MIN(TRADINGDATE) AS first_tradingdate,
        MAX(TRADINGDATE) AS last_tradingdate
    FROM read_parquet('{october_parquet_sql_path}')
    """
).fetchdf()

missing_audit = con.execute(
    f"""
    SELECT
        COUNT(*) AS target_identities_not_in_october_raw
    FROM read_csv(
        '{october_target_sql_path}',
        header = true,
        all_varchar = true
    ) AS target
    LEFT JOIN read_parquet('{october_parquet_sql_path}') AS matched
      ON target.DUID = UPPER(TRIM(matched.DUID))
     AND target.BIDTYPE = UPPER(TRIM(matched.BIDTYPE))
     AND target.BIDSETTLEMENTDATE_KEY = matched.BIDSETTLEMENTDATE_KEY
     AND target.OFFERDATE_KEY = matched.OFFERDATE_KEY
     AND TRY_CAST(target.PERIODID AS INTEGER) =
         TRY_CAST(matched.PERIODID AS INTEGER)
    WHERE matched.DUID IS NULL
    """
).fetchdf()

duplicate_audit = con.execute(
    f"""
    SELECT COUNT(*) AS duplicate_raw_offer_keys
    FROM (
        SELECT
            DUID,
            BIDTYPE,
            BIDSETTLEMENTDATE_KEY,
            OFFERDATE_KEY,
            PERIODID,
            COUNT(*) AS occurrences
        FROM read_parquet('{october_parquet_sql_path}')
        GROUP BY 1, 2, 3, 4, 5
        HAVING COUNT(*) > 1
    )
    """
).fetchdf()

con.close()

parquet_size_mb = OCTOBER_MATCH_PARQUET.stat().st_size / (1024 ** 2)

print()
print("✅ OCTOBER 2021 LOCAL DUCKDB MATCH PREPARATION COMPLETE")
print(f"DuckDB source scan and compact write: {duckdb_seconds / 60:.1f} minutes")
print(f"Compact raw-match Parquet size: {parquet_size_mb:.1f} MB")
display(match_audit)
display(missing_audit)
display(duplicate_audit)

if int(duplicate_audit.iloc[0]["duplicate_raw_offer_keys"]) != 0:
    raise RuntimeError(
        "Safety stop: duplicate raw-offer identities were retained. "
        "The local Parquet file was kept for inspection; nothing reached BigQuery."
    )

print()
print("✅ No BigQuery data was changed.")
print("✅ October ZIP is retained in the local cache.")
print("✅ October's extracted CSV is retained temporarily for audit.")
print("✅ Next we will handle any boundary carry-in identities, then build the final local stage.")

In [ ]:
# SECTION 38H — Prove October's 576 carry-in identities exist in boundary cache
# Read-only audit. No BigQuery writes, no local deletes, no stage/promotion.

import duckdb
import pandas as pd

BOUNDARY_CACHE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

BQ = globals().get("client") or globals().get("bq_client")

if BQ is None:
    raise RuntimeError(
        "BigQuery client is unavailable. Run the notebook's existing "
        "BigQuery-client restore cell, then re-run this audit."
    )

if not OCTOBER_TARGET_CSV.exists():
    raise FileNotFoundError(f"October target file is missing:\n{OCTOBER_TARGET_CSV}")

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(
        f"October compact match file is missing:\n{OCTOBER_MATCH_PARQUET}"
    )

# Find the exact offer identities not present in October's own raw archive.
con = duckdb.connect()

missing_october_keys = con.execute(
    f"""
    SELECT
        target.DUID,
        target.BIDTYPE,
        target.BIDSETTLEMENTDATE_KEY,
        target.OFFERDATE_KEY,
        CAST(target.PERIODID AS INTEGER) AS PERIODID

    FROM read_csv(
        '{OCTOBER_TARGET_CSV.as_posix()}',
        header = true,
        all_varchar = true
    ) AS target

    LEFT JOIN read_parquet(
        '{OCTOBER_MATCH_PARQUET.as_posix()}'
    ) AS matched
      ON target.DUID = UPPER(TRIM(matched.DUID))
     AND target.BIDTYPE = UPPER(TRIM(matched.BIDTYPE))
     AND target.BIDSETTLEMENTDATE_KEY = matched.BIDSETTLEMENTDATE_KEY
     AND target.OFFERDATE_KEY = matched.OFFERDATE_KEY
     AND TRY_CAST(target.PERIODID AS INTEGER) =
         TRY_CAST(matched.PERIODID AS INTEGER)

    WHERE matched.DUID IS NULL
    """
).fetchdf()

con.close()

print("OCTOBER 2021 CARRY-IN AUDIT")
print(f"Raw-offer identities missing from October source: {len(missing_october_keys):,}")

if missing_october_keys.empty:
    raise RuntimeError(
        "Safety stop: no carry-in identities were found. "
        "This differs from the expected October boundary pattern."
    )

# Inspect the existing cache schema before reading it.
boundary_table_object = BQ.get_table(BOUNDARY_CACHE_TABLE)
boundary_columns = [field.name for field in boundary_table_object.schema]
boundary_columns_by_upper = {
    field.name.upper(): field.name
    for field in boundary_table_object.schema
}

required_key_columns = {
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
}

missing_cache_columns = required_key_columns - set(boundary_columns_by_upper)

if missing_cache_columns:
    raise RuntimeError(
        "Safety stop: boundary cache schema is not as expected. "
        f"Missing columns: {sorted(missing_cache_columns)}"
    )

print("Boundary-cache columns found:")
print(boundary_columns)

# The boundary cache is small; this is a read-only metadata/record query.
boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

def canonical_boundary_timestamp(value):
    if pd.isna(value):
        return None

    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

def make_key(duid, bidtype, bidsettlementdate, offerdate, periodid):
    if pd.isna(duid) or pd.isna(bidtype) or pd.isna(periodid):
        return None

    settlement_key = canonical_boundary_timestamp(bidsettlementdate)
    offer_key = canonical_boundary_timestamp(offerdate)

    if settlement_key is None or offer_key is None:
        return None

    return (
        str(duid).strip().upper(),
        str(bidtype).strip().upper(),
        settlement_key,
        offer_key,
        int(periodid),
    )

missing_key_set = {
    make_key(
        row.DUID,
        row.BIDTYPE,
        row.BIDSETTLEMENTDATE_KEY,
        row.OFFERDATE_KEY,
        row.PERIODID,
    )
    for row in missing_october_keys.itertuples(index=False)
}

boundary_key_series = boundary_df.apply(
    lambda row: make_key(
        row[boundary_columns_by_upper["DUID"]],
        row[boundary_columns_by_upper["BIDTYPE"]],
        row[boundary_columns_by_upper["BIDSETTLEMENTDATE"]],
        row[boundary_columns_by_upper["OFFERDATE"]],
        row[boundary_columns_by_upper["PERIODID"]],
    ),
    axis=1,
)

matching_boundary_df = boundary_df.loc[
    boundary_key_series.isin(missing_key_set)
].copy()

matched_key_set = {
    key for key in boundary_key_series.loc[
        boundary_key_series.isin(missing_key_set)
    ]
    if key is not None
}

still_missing = missing_key_set - matched_key_set
duplicate_boundary_rows = len(matching_boundary_df) - len(matched_key_set)

print()
print("BOUNDARY-CACHE RESULT")
print(f"Boundary-cache rows read: {len(boundary_df):,}")
print(f"Carry-in identities required: {len(missing_key_set):,}")
print(f"Carry-in identities found: {len(matched_key_set):,}")
print(f"Still missing: {len(still_missing):,}")
print(f"Duplicate matching boundary rows: {duplicate_boundary_rows:,}")

if still_missing:
    display(
        pd.DataFrame(
            [
                {
                    "DUID": key[0],
                    "BIDTYPE": key[1],
                    "BIDSETTLEMENTDATE": key[2],
                    "OFFERDATE": key[3],
                    "PERIODID": key[4],
                }
                for key in sorted(still_missing)[:20]
            ]
        )
    )
    raise RuntimeError(
        "October carry-in audit failed. No data has been changed."
    )

if duplicate_boundary_rows != 0:
    raise RuntimeError(
        "October carry-in audit found duplicate boundary-cache records. "
        "No data has been changed."
    )

print()
print("✅ OCTOBER 2021 CARRY-IN AUDIT PASSED")
print("✅ All October carry-in offers are present exactly once in the boundary cache.")
print("✅ Next: combine the 2.7 MB local October match file with these 576 carries")
print("   and construct October's final effective-offer stage locally.")

In [ ]:
# SECTION 38I — Build and audit the complete October 2021 stage locally
# Local write only: creates one compact Parquet stage file.
# No BigQuery stage, production, manifest, or cache table is changed.

import pandas as pd
from pathlib import Path

OCTOBER_MONTH = pd.Timestamp("2021-10-01")
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"

OCTOBER_LOCAL_STAGE_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110_LOCAL.parquet"
)

if OCTOBER_LOCAL_STAGE_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: the October local stage already exists:\n"
        f"{OCTOBER_LOCAL_STAGE_PARQUET}\n"
        "Nothing was overwritten."
    )

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(
        f"October compact raw-match file is missing:\n{OCTOBER_MATCH_PARQUET}"
    )

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

# Rebuild the tracker target map from the already cached October/November trackers.
october_targets_local, october_expected_rows, _, _ = read_tracker_targets(
    OCTOBER_MONTH
)

if not october_targets_local:
    raise RuntimeError("Safety stop: no October tracker targets were found.")

def canonical_key_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

OFFER_KEY_COLUMNS = [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
]

# Expand the tracker mapping: one row per required effective five-minute interval.
expanded_target_rows = []

for offer_key, applied_intervals in october_targets_local.items():
    duid, bidtype, bidsettlementdate, offerdate, periodid = offer_key

    for applied in applied_intervals:
        expanded_target_rows.append(
            {
                "DUID": str(duid).strip().upper(),
                "BIDTYPE": str(bidtype).strip().upper(),
                "BIDSETTLEMENTDATE_KEY": canonical_key_timestamp(
                    bidsettlementdate
                ),
                "OFFERDATE_KEY": canonical_key_timestamp(offerdate),
                "PERIODID": int(periodid),
                "SETTLEMENTDATE": pd.Timestamp(
                    applied["SETTLEMENTDATE"]
                ),
                "INTERVAL_DATETIME": pd.Timestamp(
                    applied["INTERVAL_DATETIME"]
                ),
            }
        )

target_intervals_df = pd.DataFrame(expanded_target_rows)

if len(target_intervals_df) != october_expected_rows:
    raise RuntimeError(
        f"Safety stop: tracker expansion created {len(target_intervals_df):,} "
        f"rows; expected {october_expected_rows:,}."
    )

print("OCTOBER 2021 LOCAL STAGE BUILD")
print(f"Expected effective rows: {october_expected_rows:,}")
print(f"Tracker offer identities: {len(october_targets_local):,}")

# October's own compact local raw matches.
raw_matches_df = pd.read_parquet(OCTOBER_MATCH_PARQUET)

required_raw_columns = {
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
    "MAXAVAIL",
    "FIXEDLOAD",
    "RAMPUPRATE",
    "RAMPDOWNRATE",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
}

missing_raw_columns = required_raw_columns - set(raw_matches_df.columns)
if missing_raw_columns:
    raise RuntimeError(
        "Safety stop: compact October raw file lacks columns: "
        f"{sorted(missing_raw_columns)}"
    )

raw_offer_df = pd.DataFrame(
    {
        "DUID": raw_matches_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            raw_matches_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": raw_matches_df["BIDSETTLEMENTDATE_KEY"],
        "OFFERDATE_KEY": raw_matches_df["OFFERDATE_KEY"],
        "PERIODID": pd.to_numeric(
            raw_matches_df["PERIODID"],
            errors="raise",
        ).astype("int64"),
        "DIRECTION": None,
        "VERSIONNO": None,
        "MAXAVAIL": raw_matches_df["MAXAVAIL"],
        "FIXEDLOAD": raw_matches_df["FIXEDLOAD"],
        "ROCUP": raw_matches_df["RAMPUPRATE"],
        "ROCDOWN": raw_matches_df["RAMPDOWNRATE"],
        "ENABLEMENTMIN": raw_matches_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": raw_matches_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": raw_matches_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": raw_matches_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": raw_matches_df["BANDAVAIL1"],
        "BANDAVAIL2": raw_matches_df["BANDAVAIL2"],
        "BANDAVAIL3": raw_matches_df["BANDAVAIL3"],
        "BANDAVAIL4": raw_matches_df["BANDAVAIL4"],
        "BANDAVAIL5": raw_matches_df["BANDAVAIL5"],
        "BANDAVAIL6": raw_matches_df["BANDAVAIL6"],
        "BANDAVAIL7": raw_matches_df["BANDAVAIL7"],
        "BANDAVAIL8": raw_matches_df["BANDAVAIL8"],
        "BANDAVAIL9": raw_matches_df["BANDAVAIL9"],
        "BANDAVAIL10": raw_matches_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": raw_matches_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": None,
        "SOURCE_FILE": october_source_file,
        "SOURCE_URL": october_source_url,
        "OFFER_ORIGIN": "OCTOBER_RAW_ARCHIVE",
    }
)

# Retrieve only the small verified carry-in cache, then retain October's 576 keys.
boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

boundary_required_columns = {
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "DIRECTION",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "ENERGYLIMIT",
    "SOURCE_FILE",
    "SOURCE_URL",
}

missing_boundary_columns = boundary_required_columns - set(boundary_df.columns)
if missing_boundary_columns:
    raise RuntimeError(
        "Safety stop: boundary cache is missing columns: "
        f"{sorted(missing_boundary_columns)}"
    )

boundary_offer_df = pd.DataFrame(
    {
        "DUID": boundary_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            boundary_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": boundary_df["BIDSETTLEMENTDATE"].map(
            canonical_key_timestamp
        ),
        "OFFERDATE_KEY": boundary_df["OFFERDATE"].map(
            canonical_key_timestamp
        ),
        "PERIODID": pd.to_numeric(
            boundary_df["PERIODID"],
            errors="raise",
        ).astype("int64"),
        "DIRECTION": boundary_df["DIRECTION"],
        "VERSIONNO": None,
        "MAXAVAIL": boundary_df["MAXAVAIL"],
        "FIXEDLOAD": boundary_df["FIXEDLOAD"],
        "ROCUP": boundary_df["ROCUP"],
        "ROCDOWN": boundary_df["ROCDOWN"],
        "ENABLEMENTMIN": boundary_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": boundary_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": boundary_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": boundary_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": boundary_df["BANDAVAIL1"],
        "BANDAVAIL2": boundary_df["BANDAVAIL2"],
        "BANDAVAIL3": boundary_df["BANDAVAIL3"],
        "BANDAVAIL4": boundary_df["BANDAVAIL4"],
        "BANDAVAIL5": boundary_df["BANDAVAIL5"],
        "BANDAVAIL6": boundary_df["BANDAVAIL6"],
        "BANDAVAIL7": boundary_df["BANDAVAIL7"],
        "BANDAVAIL8": boundary_df["BANDAVAIL8"],
        "BANDAVAIL9": boundary_df["BANDAVAIL9"],
        "BANDAVAIL10": boundary_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": boundary_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": boundary_df["ENERGYLIMIT"],
        "SOURCE_FILE": boundary_df["SOURCE_FILE"],
        "SOURCE_URL": boundary_df["SOURCE_URL"],
        "OFFER_ORIGIN": "SEPTEMBER_BOUNDARY_CACHE",
    }
)

target_key_set = {
    tuple(row)
    for row in target_intervals_df[OFFER_KEY_COLUMNS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
}

boundary_offer_df = boundary_offer_df.loc[
    boundary_offer_df[OFFER_KEY_COLUMNS]
    .apply(tuple, axis=1)
    .isin(target_key_set)
].copy()

print(f"October raw offer identities: {len(raw_offer_df):,}")
print(f"October carry-in offer identities: {len(boundary_offer_df):,}")

offer_df = pd.concat(
    [raw_offer_df, boundary_offer_df],
    ignore_index=True,
)

duplicate_offer_keys = int(
    offer_df.duplicated(OFFER_KEY_COLUMNS).sum()
)

if duplicate_offer_keys != 0:
    raise RuntimeError(
        f"Safety stop: {duplicate_offer_keys:,} duplicate offer keys found "
        "after combining raw and carry-in records."
    )

# Attach each applied five-minute tracker interval to its exact offer.
stage_df = target_intervals_df.merge(
    offer_df,
    how="left",
    on=OFFER_KEY_COLUMNS,
    validate="many_to_one",
    indicator=True,
)

unmatched_stage_rows = int((stage_df["_merge"] != "both").sum())

if unmatched_stage_rows != 0:
    display(
        stage_df.loc[
            stage_df["_merge"] != "both",
            OFFER_KEY_COLUMNS + ["SETTLEMENTDATE", "INTERVAL_DATETIME"],
        ].head(20)
    )
    raise RuntimeError(
        f"Safety stop: {unmatched_stage_rows:,} tracker intervals have no "
        "matching raw or carry-in offer."
    )

stage_df = stage_df.drop(columns="_merge")

# Standardise types for the production-table schema.
numeric_columns = [
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
]

for column in numeric_columns:
    stage_df[column] = pd.to_numeric(stage_df[column], errors="coerce")

for column in [
    "SETTLEMENTDATE",
    "INTERVAL_DATETIME",
]:
    stage_df[column] = pd.to_datetime(stage_df[column], utc=True)

stage_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
    stage_df["BIDSETTLEMENTDATE_KEY"],
    utc=True,
)
stage_df["OFFERDATE"] = pd.to_datetime(
    stage_df["OFFERDATE_KEY"],
    utc=True,
)

stage_df["LASTCHANGED"] = pd.to_datetime(
    stage_df["LASTCHANGED"],
    utc=True,
)

stage_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
stage_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
stage_df["_PIPELINE"] = "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_DUCKDB_LOCAL"
stage_df["_SOURCE_FILE"] = stage_df["SOURCE_FILE"]
stage_df["_SOURCE_URL"] = stage_df["SOURCE_URL"]
stage_df["_RUN_ID"] = OCTOBER_RUN_ID

FINAL_STAGE_COLUMNS = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
    "_LOAD_DATE",
    "_SOURCE",
    "_PIPELINE",
    "_SOURCE_FILE",
    "_SOURCE_URL",
    "_RUN_ID",
]

stage_df = stage_df[FINAL_STAGE_COLUMNS].copy()

effective_key_columns = [
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
]

required_stage_columns = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
]

stage_row_count = len(stage_df)
distinct_effective_keys = len(
    stage_df.drop_duplicates(effective_key_columns)
)
null_required_fields = int(
    stage_df[required_stage_columns].isna().sum().sum()
)

origin_summary = (
    stage_df.assign(
        OFFER_ORIGIN=stage_df.index.map(
            # The following source test is safe because SOURCE_FILE is retained.
            lambda index: (
                "SEPTEMBER_BOUNDARY_CACHE"
                if stage_df.at[index, "_SOURCE_FILE"]
                != october_source_file
                else "OCTOBER_RAW_ARCHIVE"
            )
        )
    )
    .groupby("OFFER_ORIGIN", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

print()
print("OCTOBER LOCAL STAGE AUDIT")
audit_df = pd.DataFrame(
    [
        {
            "expected_stage_rows": october_expected_rows,
            "stage_rows": stage_row_count,
            "distinct_effective_keys": distinct_effective_keys,
            "null_required_fields": null_required_fields,
            "first_interval": stage_df["INTERVAL_DATETIME"].min(),
            "last_interval": stage_df["INTERVAL_DATETIME"].max(),
        }
    ]
)
display(audit_df)
display(origin_summary)

if stage_row_count != october_expected_rows:
    raise RuntimeError(
        f"Safety stop: local stage has {stage_row_count:,} rows; "
        f"expected {october_expected_rows:,}."
    )

if distinct_effective_keys != stage_row_count:
    raise RuntimeError(
        "Safety stop: local stage contains duplicate effective-offer keys."
    )

if null_required_fields != 0:
    raise RuntimeError(
        "Safety stop: local stage contains null values in required fields."
    )

stage_df.to_parquet(
    OCTOBER_LOCAL_STAGE_PARQUET,
    index=False,
    compression="zstd",
)

local_stage_size_mb = (
    OCTOBER_LOCAL_STAGE_PARQUET.stat().st_size / (1024 ** 2)
)

print()
print("✅ OCTOBER 2021 LOCAL EFFECTIVE-OFFER STAGE PASSED")
print(f"Rows staged locally: {stage_row_count:,}")
print(f"Compact local stage size: {local_stage_size_mb:.1f} MB")
print(f"Local stage file: {OCTOBER_LOCAL_STAGE_PARQUET}")
print("✅ BigQuery remains unchanged.")
print("✅ The 46.94 GB temporary CSV was not rescanned.")

In [ ]:
# SECTION 38I0 — October direction-mapping preflight
# Read-only. No BigQuery writes and no local files changed.

import duckdb
import pandas as pd

if "legacy_direction" not in globals():
    raise RuntimeError(
        "legacy_direction() is not available in this runtime. "
        "Run the earlier legacy-direction helper cell before continuing."
    )

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(f"Missing: {OCTOBER_MATCH_PARQUET}")

raw_match_df = pd.read_parquet(OCTOBER_MATCH_PARQUET)

october_duids = sorted(
    raw_match_df["DUID"]
    .astype("string")
    .str.strip()
    .str.upper()
    .dropna()
    .unique()
)

direction_audit_df = pd.DataFrame({"DUID": october_duids})
direction_audit_df["LEGACY_DIRECTION"] = direction_audit_df["DUID"].map(
    legacy_direction
)
direction_audit_df["LEGACY_DIRECTION"] = (
    direction_audit_df["LEGACY_DIRECTION"]
    .astype("string")
    .str.strip()
    .str.upper()
)

invalid_direction_df = direction_audit_df.loc[
    ~direction_audit_df["LEGACY_DIRECTION"].isin(
        ["GEN", "LOAD", "BIDIRECTIONAL"]
    )
].copy()

print("OCTOBER 2021 DIRECTION PREFLIGHT")
print(f"Current-month raw DUIDs: {len(direction_audit_df):,}")
print(f"Unmapped/invalid directions: {len(invalid_direction_df):,}")
display(direction_audit_df.sort_values("DUID"))

if not invalid_direction_df.empty:
    display(invalid_direction_df)
    raise RuntimeError(
        "Safety stop: legacy_direction() does not provide a valid direction "
        "for every October DUID."
    )

# Recreate the 576 carry-in key set locally.
con = duckdb.connect()

missing_key_df = con.execute(
    f"""
    SELECT
        target.DUID,
        target.BIDTYPE,
        target.BIDSETTLEMENTDATE_KEY,
        target.OFFERDATE_KEY,
        CAST(target.PERIODID AS INTEGER) AS PERIODID
    FROM read_csv(
        '{OCTOBER_TARGET_CSV.as_posix()}',
        header = true,
        all_varchar = true
    ) AS target
    LEFT JOIN read_parquet(
        '{OCTOBER_MATCH_PARQUET.as_posix()}'
    ) AS matched
      ON target.DUID = UPPER(TRIM(matched.DUID))
     AND target.BIDTYPE = UPPER(TRIM(matched.BIDTYPE))
     AND target.BIDSETTLEMENTDATE_KEY = matched.BIDSETTLEMENTDATE_KEY
     AND target.OFFERDATE_KEY = matched.OFFERDATE_KEY
     AND TRY_CAST(target.PERIODID AS INTEGER) =
         TRY_CAST(matched.PERIODID AS INTEGER)
    WHERE matched.DUID IS NULL
    """
).fetchdf()

con.close()

def canonical_timestamp(value):
    if pd.isna(value):
        return None

    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

def offer_key(duid, bidtype, bidsettlementdate, offerdate, periodid):
    return (
        str(duid).strip().upper(),
        str(bidtype).strip().upper(),
        canonical_timestamp(bidsettlementdate),
        canonical_timestamp(offerdate),
        int(periodid),
    )

missing_key_set = {
    offer_key(
        row.DUID,
        row.BIDTYPE,
        row.BIDSETTLEMENTDATE_KEY,
        row.OFFERDATE_KEY,
        row.PERIODID,
    )
    for row in missing_key_df.itertuples(index=False)
}

boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

boundary_df["OFFER_KEY"] = boundary_df.apply(
    lambda row: offer_key(
        row["DUID"],
        row["BIDTYPE"],
        row["BIDSETTLEMENTDATE"],
        row["OFFERDATE"],
        row["PERIODID"],
    ),
    axis=1,
)

october_boundary_df = boundary_df.loc[
    boundary_df["OFFER_KEY"].isin(missing_key_set)
].copy()

october_boundary_df["CACHE_DIRECTION"] = (
    october_boundary_df["DIRECTION"]
    .astype("string")
    .str.strip()
    .str.upper()
)

october_boundary_df["LEGACY_DIRECTION"] = (
    october_boundary_df["DUID"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(legacy_direction)
    .astype("string")
    .str.strip()
    .str.upper()
)

boundary_direction_conflicts = october_boundary_df.loc[
    october_boundary_df["CACHE_DIRECTION"].notna()
    & (october_boundary_df["CACHE_DIRECTION"] != "")
    & (
        october_boundary_df["CACHE_DIRECTION"]
        != october_boundary_df["LEGACY_DIRECTION"]
    )
].copy()

print()
print("BOUNDARY DIRECTION CHECK")
print(f"October carry-in cache rows: {len(october_boundary_df):,}")
print(
    "Boundary rows whose supplied direction conflicts with "
    f"legacy_direction(): {len(boundary_direction_conflicts):,}"
)

if not boundary_direction_conflicts.empty:
    display(
        boundary_direction_conflicts[
            ["DUID", "BIDTYPE", "CACHE_DIRECTION", "LEGACY_DIRECTION"]
        ].head(20)
    )
    raise RuntimeError(
        "Safety stop: boundary-cache and legacy directions disagree."
    )

print()
print("✅ OCTOBER DIRECTION PREFLIGHT PASSED")
print("✅ Every current-month DUID has an explicit legacy direction.")
print("✅ Boundary-cache directions agree with the legacy mapping.")
print("✅ Next, I will give the corrected local-stage cell with:")
print("   - non-null direction enforcement,")
print("   - preserved OFFER_ORIGIN, and")
print("   - source-blank versus malformed-numeric quality checks.")

In [ ]:
# SECTION 38I2 — Corrected October local effective-offer stage
# Local writes only:
#   1) corrected production-ready V2 Parquet stage
#   2) compact origin-audit Parquet
# No BigQuery writes, promotions, deletes, or manifest changes.

import pandas as pd
from pathlib import Path

OCTOBER_MONTH = pd.Timestamp("2021-10-01")
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"

BOUNDARY_CACHE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

OCTOBER_LOCAL_STAGE_V2_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110_LOCAL_V2.parquet"
)
OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110_ORIGIN_AUDIT.parquet"
)

if OCTOBER_LOCAL_STAGE_V2_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: corrected October V2 stage already exists:\n"
        f"{OCTOBER_LOCAL_STAGE_V2_PARQUET}\n"
        "Nothing was overwritten."
    )

if OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: October origin-audit file already exists:\n"
        f"{OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET}\n"
        "Nothing was overwritten."
    )

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(f"Missing: {OCTOBER_MATCH_PARQUET}")

if "legacy_direction" not in globals():
    raise RuntimeError("legacy_direction() is unavailable.")

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

october_source_file, october_source_url = raw_source_urls(OCTOBER_MONTH)[0]

def canonical_key_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

OFFER_KEY_COLUMNS = [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
]

# Rebuild the tracker-to-effective-interval mapping from cached trackers.
october_targets_v2, october_expected_rows, _, _ = read_tracker_targets(
    OCTOBER_MONTH
)

expanded_target_rows = []

for offer_key, applied_intervals in october_targets_v2.items():
    duid, bidtype, bidsettlementdate, offerdate, periodid = offer_key

    for applied in applied_intervals:
        expanded_target_rows.append(
            {
                "DUID": str(duid).strip().upper(),
                "BIDTYPE": str(bidtype).strip().upper(),
                "BIDSETTLEMENTDATE_KEY": canonical_key_timestamp(
                    bidsettlementdate
                ),
                "OFFERDATE_KEY": canonical_key_timestamp(offerdate),
                "PERIODID": int(periodid),
                "SETTLEMENTDATE": pd.Timestamp(applied["SETTLEMENTDATE"]),
                "INTERVAL_DATETIME": pd.Timestamp(
                    applied["INTERVAL_DATETIME"]
                ),
            }
        )

target_intervals_df = pd.DataFrame(expanded_target_rows)

if len(target_intervals_df) != october_expected_rows:
    raise RuntimeError(
        f"Safety stop: tracker expansion created {len(target_intervals_df):,} "
        f"rows; expected {october_expected_rows:,}."
    )

target_key_set = {
    tuple(row)
    for row in target_intervals_df[OFFER_KEY_COLUMNS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
}

print("OCTOBER 2021 — CORRECTED LOCAL STAGE BUILD")
print(f"Expected effective rows: {october_expected_rows:,}")
print(f"Required offer identities: {len(target_key_set):,}")

# Standardise October's own compact raw matches.
raw_matches_df = pd.read_parquet(OCTOBER_MATCH_PARQUET)

required_raw_columns = {
    "DUID", "BIDTYPE", "BIDSETTLEMENTDATE_KEY", "OFFERDATE_KEY",
    "PERIODID", "MAXAVAIL", "FIXEDLOAD", "RAMPUPRATE", "RAMPDOWNRATE",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "PASAAVAILABILITY",
}

missing_raw_columns = required_raw_columns - set(raw_matches_df.columns)
if missing_raw_columns:
    raise RuntimeError(
        f"Safety stop: local October raw matches lack {sorted(missing_raw_columns)}"
    )

raw_offer_df = pd.DataFrame(
    {
        "DUID": raw_matches_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            raw_matches_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": raw_matches_df["BIDSETTLEMENTDATE_KEY"],
        "OFFERDATE_KEY": raw_matches_df["OFFERDATE_KEY"],
        "PERIODID": pd.to_numeric(
            raw_matches_df["PERIODID"], errors="raise"
        ).astype("int64"),
        "DIRECTION": (
            raw_matches_df["DUID"]
            .astype("string")
            .str.strip()
            .str.upper()
            .map(legacy_direction)
            .astype("string")
            .str.strip()
            .str.upper()
        ),
        "VERSIONNO": None,
        "MAXAVAIL": raw_matches_df["MAXAVAIL"],
        "FIXEDLOAD": raw_matches_df["FIXEDLOAD"],
        "ROCUP": raw_matches_df["RAMPUPRATE"],
        "ROCDOWN": raw_matches_df["RAMPDOWNRATE"],
        "ENABLEMENTMIN": raw_matches_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": raw_matches_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": raw_matches_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": raw_matches_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": raw_matches_df["BANDAVAIL1"],
        "BANDAVAIL2": raw_matches_df["BANDAVAIL2"],
        "BANDAVAIL3": raw_matches_df["BANDAVAIL3"],
        "BANDAVAIL4": raw_matches_df["BANDAVAIL4"],
        "BANDAVAIL5": raw_matches_df["BANDAVAIL5"],
        "BANDAVAIL6": raw_matches_df["BANDAVAIL6"],
        "BANDAVAIL7": raw_matches_df["BANDAVAIL7"],
        "BANDAVAIL8": raw_matches_df["BANDAVAIL8"],
        "BANDAVAIL9": raw_matches_df["BANDAVAIL9"],
        "BANDAVAIL10": raw_matches_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": raw_matches_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": None,
        "SOURCE_FILE": october_source_file,
        "SOURCE_URL": october_source_url,
        "OFFER_ORIGIN": "OCTOBER_RAW_ARCHIVE",
    }
)

# Add only boundary-cache offers needed for October.
boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

boundary_offer_df = pd.DataFrame(
    {
        "DUID": boundary_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            boundary_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": boundary_df["BIDSETTLEMENTDATE"].map(
            canonical_key_timestamp
        ),
        "OFFERDATE_KEY": boundary_df["OFFERDATE"].map(
            canonical_key_timestamp
        ),
        "PERIODID": pd.to_numeric(
            boundary_df["PERIODID"], errors="raise"
        ).astype("int64"),
        # Prefer cache direction, but fall back to the proven legacy mapping.
        "DIRECTION": (
            boundary_df["DIRECTION"]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
            .fillna(
                boundary_df["DUID"]
                .astype("string")
                .str.strip()
                .str.upper()
                .map(legacy_direction)
            )
        ),
        "VERSIONNO": None,
        "MAXAVAIL": boundary_df["MAXAVAIL"],
        "FIXEDLOAD": boundary_df["FIXEDLOAD"],
        "ROCUP": boundary_df["ROCUP"],
        "ROCDOWN": boundary_df["ROCDOWN"],
        "ENABLEMENTMIN": boundary_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": boundary_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": boundary_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": boundary_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": boundary_df["BANDAVAIL1"],
        "BANDAVAIL2": boundary_df["BANDAVAIL2"],
        "BANDAVAIL3": boundary_df["BANDAVAIL3"],
        "BANDAVAIL4": boundary_df["BANDAVAIL4"],
        "BANDAVAIL5": boundary_df["BANDAVAIL5"],
        "BANDAVAIL6": boundary_df["BANDAVAIL6"],
        "BANDAVAIL7": boundary_df["BANDAVAIL7"],
        "BANDAVAIL8": boundary_df["BANDAVAIL8"],
        "BANDAVAIL9": boundary_df["BANDAVAIL9"],
        "BANDAVAIL10": boundary_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": boundary_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": boundary_df["ENERGYLIMIT"],
        "SOURCE_FILE": boundary_df["SOURCE_FILE"],
        "SOURCE_URL": boundary_df["SOURCE_URL"],
        "OFFER_ORIGIN": "SEPTEMBER_BOUNDARY_CACHE",
    }
)

boundary_offer_df = boundary_offer_df.loc[
    boundary_offer_df[OFFER_KEY_COLUMNS]
    .apply(tuple, axis=1)
    .isin(target_key_set)
].copy()

offer_df = pd.concat(
    [raw_offer_df, boundary_offer_df],
    ignore_index=True,
)

duplicate_offer_keys = int(offer_df.duplicated(OFFER_KEY_COLUMNS).sum())
if duplicate_offer_keys != 0:
    raise RuntimeError(
        f"Safety stop: {duplicate_offer_keys:,} duplicate offer identities found."
    )

# Direction is required and must be explicit for every offer.
offer_df["DIRECTION"] = (
    offer_df["DIRECTION"].astype("string").str.strip().str.upper()
)

invalid_direction_rows = offer_df.loc[
    ~offer_df["DIRECTION"].isin(["GEN", "LOAD", "BIDIRECTIONAL"])
].copy()

if not invalid_direction_rows.empty:
    display(
        invalid_direction_rows[
            ["DUID", "BIDTYPE", "DIRECTION", "OFFER_ORIGIN"]
        ].head(20)
    )
    raise RuntimeError(
        "Safety stop: missing or invalid directions remain after mapping."
    )

# Numeric-quality audit: blank source values are allowed;
# nonblank values that fail conversion are not.
NUMERIC_SOURCE_FIELDS = [
    "MAXAVAIL", "FIXEDLOAD", "ROCUP", "ROCDOWN",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "PASAAVAILABILITY", "ENERGYLIMIT",
]

quality_rows = []

for origin, origin_df in offer_df.groupby("OFFER_ORIGIN", dropna=False):
    origin_index = origin_df.index

    for field in NUMERIC_SOURCE_FIELDS:
        source_text = origin_df[field].astype("string").str.strip()
        source_blank = source_text.isna() | source_text.eq("")
        parsed = pd.to_numeric(origin_df[field], errors="coerce")
        malformed = (~source_blank) & parsed.isna()

        quality_rows.append(
            {
                "OFFER_ORIGIN": origin,
                "FIELD_NAME": field,
                "SOURCE_BLANK_VALUES": int(source_blank.sum()),
                "VALID_NUMERIC_VALUES": int((~source_blank & ~parsed.isna()).sum()),
                "MALFORMED_NONBLANK_VALUES": int(malformed.sum()),
            }
        )

        offer_df.loc[origin_index, field] = parsed

numeric_quality_df = pd.DataFrame(quality_rows)
malformed_numeric_count = int(
    numeric_quality_df["MALFORMED_NONBLANK_VALUES"].sum()
)

print(f"Current-month offer identities: {len(raw_offer_df):,}")
print(f"Boundary carry-in offer identities: {len(boundary_offer_df):,}")
print(f"Malformed nonblank numeric values: {malformed_numeric_count:,}")
display(numeric_quality_df)

if malformed_numeric_count != 0:
    raise RuntimeError(
        "Safety stop: malformed nonblank numeric source values were found. "
        "No V2 stage was written."
    )

# Expand each matched offer across the tracker intervals that used it.
stage_work_df = target_intervals_df.merge(
    offer_df,
    how="left",
    on=OFFER_KEY_COLUMNS,
    validate="many_to_one",
    indicator=True,
)

unmatched_rows = int((stage_work_df["_merge"] != "both").sum())
if unmatched_rows != 0:
    display(
        stage_work_df.loc[
            stage_work_df["_merge"] != "both",
            OFFER_KEY_COLUMNS + ["SETTLEMENTDATE", "INTERVAL_DATETIME"],
        ].head(20)
    )
    raise RuntimeError(
        f"Safety stop: {unmatched_rows:,} effective intervals lack an offer."
    )

stage_work_df = stage_work_df.drop(columns="_merge")

# Preserve explicit source origin in a separate compact audit file.
origin_audit_df = stage_work_df[
    [
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "OFFER_ORIGIN",
        "SOURCE_FILE",
        "SOURCE_URL",
    ]
].copy()

origin_summary = (
    origin_audit_df
    .groupby("OFFER_ORIGIN", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

# Standardise final table types.
for column in ["SETTLEMENTDATE", "INTERVAL_DATETIME"]:
    stage_work_df[column] = pd.to_datetime(
        stage_work_df[column],
        utc=True,
    )

stage_work_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
    stage_work_df["BIDSETTLEMENTDATE_KEY"],
    utc=True,
)
stage_work_df["OFFERDATE"] = pd.to_datetime(
    stage_work_df["OFFERDATE_KEY"],
    utc=True,
)
stage_work_df["LASTCHANGED"] = pd.to_datetime(
    stage_work_df["LASTCHANGED"],
    utc=True,
)

stage_work_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
stage_work_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
stage_work_df["_PIPELINE"] = (
    "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_DUCKDB_LOCAL"
)
stage_work_df["_SOURCE_FILE"] = stage_work_df["SOURCE_FILE"]
stage_work_df["_SOURCE_URL"] = stage_work_df["SOURCE_URL"]
stage_work_df["_RUN_ID"] = OCTOBER_RUN_ID

FINAL_STAGE_COLUMNS = [
    "SETTLEMENTDATE", "DUID", "BIDTYPE", "DIRECTION",
    "INTERVAL_DATETIME", "BIDSETTLEMENTDATE", "OFFERDATE", "PERIODID",
    "VERSIONNO", "MAXAVAIL", "FIXEDLOAD", "ROCUP", "ROCDOWN",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "LASTCHANGED", "PASAAVAILABILITY",
    "MR_CAPACITY", "ENERGYLIMIT", "_LOAD_DATE", "_SOURCE", "_PIPELINE",
    "_SOURCE_FILE", "_SOURCE_URL", "_RUN_ID",
]

stage_final_df = stage_work_df[FINAL_STAGE_COLUMNS].copy()

required_fields = [
    "SETTLEMENTDATE", "DUID", "BIDTYPE", "DIRECTION",
    "INTERVAL_DATETIME", "BIDSETTLEMENTDATE", "OFFERDATE", "PERIODID",
]

effective_key_columns = [
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
]

stage_rows = len(stage_final_df)
distinct_effective_keys = len(
    stage_final_df.drop_duplicates(effective_key_columns)
)
null_required_fields = int(
    stage_final_df[required_fields].isna().sum().sum()
)

direction_summary = (
    stage_final_df
    .groupby("DIRECTION", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

audit_df = pd.DataFrame(
    [
        {
            "expected_stage_rows": october_expected_rows,
            "stage_rows": stage_rows,
            "distinct_effective_keys": distinct_effective_keys,
            "null_required_fields_including_direction": null_required_fields,
            "first_interval": stage_final_df["INTERVAL_DATETIME"].min(),
            "last_interval": stage_final_df["INTERVAL_DATETIME"].max(),
        }
    ]
)

print()
print("OCTOBER 2021 CORRECTED LOCAL STAGE AUDIT")
display(audit_df)

print("Direction coverage:")
display(direction_summary)

print("Explicit provenance:")
display(origin_summary)

if stage_rows != october_expected_rows:
    raise RuntimeError("Safety stop: stage row count differs from tracker target.")

if distinct_effective_keys != stage_rows:
    raise RuntimeError("Safety stop: duplicate effective keys were found.")

if null_required_fields != 0:
    raise RuntimeError(
        "Safety stop: a required final-stage field, including DIRECTION, is null."
    )

# Write only after every gate has passed.
stage_final_df.to_parquet(
    OCTOBER_LOCAL_STAGE_V2_PARQUET,
    index=False,
    compression="zstd",
)
origin_audit_df.to_parquet(
    OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET,
    index=False,
    compression="zstd",
)

print()
print("✅ OCTOBER 2021 CORRECTED LOCAL STAGE PASSED")
print(f"Corrected local stage: {OCTOBER_LOCAL_STAGE_V2_PARQUET}")
print(f"Origin audit file: {OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET}")
print("✅ Direction is explicit and non-null for every final row.")
print("✅ Numeric conversion has no malformed nonblank source values.")
print("✅ BigQuery remains unchanged.")# SECTION 38I2 — Corrected October local effective-offer stage
# Local writes only:
#   1) corrected production-ready V2 Parquet stage
#   2) compact origin-audit Parquet
# No BigQuery writes, promotions, deletes, or manifest changes.

import pandas as pd
from pathlib import Path

OCTOBER_MONTH = pd.Timestamp("2021-10-01")
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"

BOUNDARY_CACHE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

OCTOBER_LOCAL_STAGE_V2_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110_LOCAL_V2.parquet"
)
OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110_ORIGIN_AUDIT.parquet"
)

if OCTOBER_LOCAL_STAGE_V2_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: corrected October V2 stage already exists:\n"
        f"{OCTOBER_LOCAL_STAGE_V2_PARQUET}\n"
        "Nothing was overwritten."
    )

if OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: October origin-audit file already exists:\n"
        f"{OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET}\n"
        "Nothing was overwritten."
    )

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(f"Missing: {OCTOBER_MATCH_PARQUET}")

if "legacy_direction" not in globals():
    raise RuntimeError("legacy_direction() is unavailable.")

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

october_source_file, october_source_url = raw_source_urls(OCTOBER_MONTH)[0]

def canonical_key_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

OFFER_KEY_COLUMNS = [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
]

# Rebuild the tracker-to-effective-interval mapping from cached trackers.
october_targets_v2, october_expected_rows, _, _ = read_tracker_targets(
    OCTOBER_MONTH
)

expanded_target_rows = []

for offer_key, applied_intervals in october_targets_v2.items():
    duid, bidtype, bidsettlementdate, offerdate, periodid = offer_key

    for applied in applied_intervals:
        expanded_target_rows.append(
            {
                "DUID": str(duid).strip().upper(),
                "BIDTYPE": str(bidtype).strip().upper(),
                "BIDSETTLEMENTDATE_KEY": canonical_key_timestamp(
                    bidsettlementdate
                ),
                "OFFERDATE_KEY": canonical_key_timestamp(offerdate),
                "PERIODID": int(periodid),
                "SETTLEMENTDATE": pd.Timestamp(applied["SETTLEMENTDATE"]),
                "INTERVAL_DATETIME": pd.Timestamp(
                    applied["INTERVAL_DATETIME"]
                ),
            }
        )

target_intervals_df = pd.DataFrame(expanded_target_rows)

if len(target_intervals_df) != october_expected_rows:
    raise RuntimeError(
        f"Safety stop: tracker expansion created {len(target_intervals_df):,} "
        f"rows; expected {october_expected_rows:,}."
    )

target_key_set = {
    tuple(row)
    for row in target_intervals_df[OFFER_KEY_COLUMNS]
    .drop_duplicates()
    .itertuples(index=False, name=None)
}

print("OCTOBER 2021 — CORRECTED LOCAL STAGE BUILD")
print(f"Expected effective rows: {october_expected_rows:,}")
print(f"Required offer identities: {len(target_key_set):,}")

# Standardise October's own compact raw matches.
raw_matches_df = pd.read_parquet(OCTOBER_MATCH_PARQUET)

required_raw_columns = {
    "DUID", "BIDTYPE", "BIDSETTLEMENTDATE_KEY", "OFFERDATE_KEY",
    "PERIODID", "MAXAVAIL", "FIXEDLOAD", "RAMPUPRATE", "RAMPDOWNRATE",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "PASAAVAILABILITY",
}

missing_raw_columns = required_raw_columns - set(raw_matches_df.columns)
if missing_raw_columns:
    raise RuntimeError(
        f"Safety stop: local October raw matches lack {sorted(missing_raw_columns)}"
    )

raw_offer_df = pd.DataFrame(
    {
        "DUID": raw_matches_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            raw_matches_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": raw_matches_df["BIDSETTLEMENTDATE_KEY"],
        "OFFERDATE_KEY": raw_matches_df["OFFERDATE_KEY"],
        "PERIODID": pd.to_numeric(
            raw_matches_df["PERIODID"], errors="raise"
        ).astype("int64"),
        "DIRECTION": (
            raw_matches_df["DUID"]
            .astype("string")
            .str.strip()
            .str.upper()
            .map(legacy_direction)
            .astype("string")
            .str.strip()
            .str.upper()
        ),
        "VERSIONNO": None,
        "MAXAVAIL": raw_matches_df["MAXAVAIL"],
        "FIXEDLOAD": raw_matches_df["FIXEDLOAD"],
        "ROCUP": raw_matches_df["RAMPUPRATE"],
        "ROCDOWN": raw_matches_df["RAMPDOWNRATE"],
        "ENABLEMENTMIN": raw_matches_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": raw_matches_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": raw_matches_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": raw_matches_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": raw_matches_df["BANDAVAIL1"],
        "BANDAVAIL2": raw_matches_df["BANDAVAIL2"],
        "BANDAVAIL3": raw_matches_df["BANDAVAIL3"],
        "BANDAVAIL4": raw_matches_df["BANDAVAIL4"],
        "BANDAVAIL5": raw_matches_df["BANDAVAIL5"],
        "BANDAVAIL6": raw_matches_df["BANDAVAIL6"],
        "BANDAVAIL7": raw_matches_df["BANDAVAIL7"],
        "BANDAVAIL8": raw_matches_df["BANDAVAIL8"],
        "BANDAVAIL9": raw_matches_df["BANDAVAIL9"],
        "BANDAVAIL10": raw_matches_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": raw_matches_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": None,
        "SOURCE_FILE": october_source_file,
        "SOURCE_URL": october_source_url,
        "OFFER_ORIGIN": "OCTOBER_RAW_ARCHIVE",
    }
)

# Add only boundary-cache offers needed for October.
boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

boundary_offer_df = pd.DataFrame(
    {
        "DUID": boundary_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": (
            boundary_df["BIDTYPE"].astype("string").str.strip().str.upper()
        ),
        "BIDSETTLEMENTDATE_KEY": boundary_df["BIDSETTLEMENTDATE"].map(
            canonical_key_timestamp
        ),
        "OFFERDATE_KEY": boundary_df["OFFERDATE"].map(
            canonical_key_timestamp
        ),
        "PERIODID": pd.to_numeric(
            boundary_df["PERIODID"], errors="raise"
        ).astype("int64"),
        # Prefer cache direction, but fall back to the proven legacy mapping.
        "DIRECTION": (
            boundary_df["DIRECTION"]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
            .fillna(
                boundary_df["DUID"]
                .astype("string")
                .str.strip()
                .str.upper()
                .map(legacy_direction)
            )
        ),
        "VERSIONNO": None,
        "MAXAVAIL": boundary_df["MAXAVAIL"],
        "FIXEDLOAD": boundary_df["FIXEDLOAD"],
        "ROCUP": boundary_df["ROCUP"],
        "ROCDOWN": boundary_df["ROCDOWN"],
        "ENABLEMENTMIN": boundary_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": boundary_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": boundary_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": boundary_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": boundary_df["BANDAVAIL1"],
        "BANDAVAIL2": boundary_df["BANDAVAIL2"],
        "BANDAVAIL3": boundary_df["BANDAVAIL3"],
        "BANDAVAIL4": boundary_df["BANDAVAIL4"],
        "BANDAVAIL5": boundary_df["BANDAVAIL5"],
        "BANDAVAIL6": boundary_df["BANDAVAIL6"],
        "BANDAVAIL7": boundary_df["BANDAVAIL7"],
        "BANDAVAIL8": boundary_df["BANDAVAIL8"],
        "BANDAVAIL9": boundary_df["BANDAVAIL9"],
        "BANDAVAIL10": boundary_df["BANDAVAIL10"],
        "LASTCHANGED": None,
        "PASAAVAILABILITY": boundary_df["PASAAVAILABILITY"],
        "MR_CAPACITY": None,
        "ENERGYLIMIT": boundary_df["ENERGYLIMIT"],
        "SOURCE_FILE": boundary_df["SOURCE_FILE"],
        "SOURCE_URL": boundary_df["SOURCE_URL"],
        "OFFER_ORIGIN": "SEPTEMBER_BOUNDARY_CACHE",
    }
)

boundary_offer_df = boundary_offer_df.loc[
    boundary_offer_df[OFFER_KEY_COLUMNS]
    .apply(tuple, axis=1)
    .isin(target_key_set)
].copy()

offer_df = pd.concat(
    [raw_offer_df, boundary_offer_df],
    ignore_index=True,
)

duplicate_offer_keys = int(offer_df.duplicated(OFFER_KEY_COLUMNS).sum())
if duplicate_offer_keys != 0:
    raise RuntimeError(
        f"Safety stop: {duplicate_offer_keys:,} duplicate offer identities found."
    )

# Direction is required and must be explicit for every offer.
offer_df["DIRECTION"] = (
    offer_df["DIRECTION"].astype("string").str.strip().str.upper()
)

invalid_direction_rows = offer_df.loc[
    ~offer_df["DIRECTION"].isin(["GEN", "LOAD", "BIDIRECTIONAL"])
].copy()

if not invalid_direction_rows.empty:
    display(
        invalid_direction_rows[
            ["DUID", "BIDTYPE", "DIRECTION", "OFFER_ORIGIN"]
        ].head(20)
    )
    raise RuntimeError(
        "Safety stop: missing or invalid directions remain after mapping."
    )

# Numeric-quality audit: blank source values are allowed;
# nonblank values that fail conversion are not.
NUMERIC_SOURCE_FIELDS = [
    "MAXAVAIL", "FIXEDLOAD", "ROCUP", "ROCDOWN",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "PASAAVAILABILITY", "ENERGYLIMIT",
]

quality_rows = []

for origin, origin_df in offer_df.groupby("OFFER_ORIGIN", dropna=False):
    origin_index = origin_df.index

    for field in NUMERIC_SOURCE_FIELDS:
        source_text = origin_df[field].astype("string").str.strip()
        source_blank = source_text.isna() | source_text.eq("")
        parsed = pd.to_numeric(origin_df[field], errors="coerce")
        malformed = (~source_blank) & parsed.isna()

        quality_rows.append(
            {
                "OFFER_ORIGIN": origin,
                "FIELD_NAME": field,
                "SOURCE_BLANK_VALUES": int(source_blank.sum()),
                "VALID_NUMERIC_VALUES": int((~source_blank & ~parsed.isna()).sum()),
                "MALFORMED_NONBLANK_VALUES": int(malformed.sum()),
            }
        )

        offer_df.loc[origin_index, field] = parsed

numeric_quality_df = pd.DataFrame(quality_rows)
malformed_numeric_count = int(
    numeric_quality_df["MALFORMED_NONBLANK_VALUES"].sum()
)

print(f"Current-month offer identities: {len(raw_offer_df):,}")
print(f"Boundary carry-in offer identities: {len(boundary_offer_df):,}")
print(f"Malformed nonblank numeric values: {malformed_numeric_count:,}")
display(numeric_quality_df)

if malformed_numeric_count != 0:
    raise RuntimeError(
        "Safety stop: malformed nonblank numeric source values were found. "
        "No V2 stage was written."
    )

# Expand each matched offer across the tracker intervals that used it.
stage_work_df = target_intervals_df.merge(
    offer_df,
    how="left",
    on=OFFER_KEY_COLUMNS,
    validate="many_to_one",
    indicator=True,
)

unmatched_rows = int((stage_work_df["_merge"] != "both").sum())
if unmatched_rows != 0:
    display(
        stage_work_df.loc[
            stage_work_df["_merge"] != "both",
            OFFER_KEY_COLUMNS + ["SETTLEMENTDATE", "INTERVAL_DATETIME"],
        ].head(20)
    )
    raise RuntimeError(
        f"Safety stop: {unmatched_rows:,} effective intervals lack an offer."
    )

stage_work_df = stage_work_df.drop(columns="_merge")

# Preserve explicit source origin in a separate compact audit file.
origin_audit_df = stage_work_df[
    [
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "OFFER_ORIGIN",
        "SOURCE_FILE",
        "SOURCE_URL",
    ]
].copy()

origin_summary = (
    origin_audit_df
    .groupby("OFFER_ORIGIN", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

# Standardise final table types.
for column in ["SETTLEMENTDATE", "INTERVAL_DATETIME"]:
    stage_work_df[column] = pd.to_datetime(
        stage_work_df[column],
        utc=True,
    )

stage_work_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
    stage_work_df["BIDSETTLEMENTDATE_KEY"],
    utc=True,
)
stage_work_df["OFFERDATE"] = pd.to_datetime(
    stage_work_df["OFFERDATE_KEY"],
    utc=True,
)
stage_work_df["LASTCHANGED"] = pd.to_datetime(
    stage_work_df["LASTCHANGED"],
    utc=True,
)

stage_work_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
stage_work_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
stage_work_df["_PIPELINE"] = (
    "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_DUCKDB_LOCAL"
)
stage_work_df["_SOURCE_FILE"] = stage_work_df["SOURCE_FILE"]
stage_work_df["_SOURCE_URL"] = stage_work_df["SOURCE_URL"]
stage_work_df["_RUN_ID"] = OCTOBER_RUN_ID

FINAL_STAGE_COLUMNS = [
    "SETTLEMENTDATE", "DUID", "BIDTYPE", "DIRECTION",
    "INTERVAL_DATETIME", "BIDSETTLEMENTDATE", "OFFERDATE", "PERIODID",
    "VERSIONNO", "MAXAVAIL", "FIXEDLOAD", "ROCUP", "ROCDOWN",
    "ENABLEMENTMIN", "ENABLEMENTMAX", "LOWBREAKPOINT", "HIGHBREAKPOINT",
    "BANDAVAIL1", "BANDAVAIL2", "BANDAVAIL3", "BANDAVAIL4",
    "BANDAVAIL5", "BANDAVAIL6", "BANDAVAIL7", "BANDAVAIL8",
    "BANDAVAIL9", "BANDAVAIL10", "LASTCHANGED", "PASAAVAILABILITY",
    "MR_CAPACITY", "ENERGYLIMIT", "_LOAD_DATE", "_SOURCE", "_PIPELINE",
    "_SOURCE_FILE", "_SOURCE_URL", "_RUN_ID",
]

stage_final_df = stage_work_df[FINAL_STAGE_COLUMNS].copy()

required_fields = [
    "SETTLEMENTDATE", "DUID", "BIDTYPE", "DIRECTION",
    "INTERVAL_DATETIME", "BIDSETTLEMENTDATE", "OFFERDATE", "PERIODID",
]

effective_key_columns = [
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
]

stage_rows = len(stage_final_df)
distinct_effective_keys = len(
    stage_final_df.drop_duplicates(effective_key_columns)
)
null_required_fields = int(
    stage_final_df[required_fields].isna().sum().sum()
)

direction_summary = (
    stage_final_df
    .groupby("DIRECTION", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

audit_df = pd.DataFrame(
    [
        {
            "expected_stage_rows": october_expected_rows,
            "stage_rows": stage_rows,
            "distinct_effective_keys": distinct_effective_keys,
            "null_required_fields_including_direction": null_required_fields,
            "first_interval": stage_final_df["INTERVAL_DATETIME"].min(),
            "last_interval": stage_final_df["INTERVAL_DATETIME"].max(),
        }
    ]
)

print()
print("OCTOBER 2021 CORRECTED LOCAL STAGE AUDIT")
display(audit_df)

print("Direction coverage:")
display(direction_summary)

print("Explicit provenance:")
display(origin_summary)

if stage_rows != october_expected_rows:
    raise RuntimeError("Safety stop: stage row count differs from tracker target.")

if distinct_effective_keys != stage_rows:
    raise RuntimeError("Safety stop: duplicate effective keys were found.")

if null_required_fields != 0:
    raise RuntimeError(
        "Safety stop: a required final-stage field, including DIRECTION, is null."
    )

# Write only after every gate has passed.
stage_final_df.to_parquet(
    OCTOBER_LOCAL_STAGE_V2_PARQUET,
    index=False,
    compression="zstd",
)
origin_audit_df.to_parquet(
    OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET,
    index=False,
    compression="zstd",
)

print()
print("✅ OCTOBER 2021 CORRECTED LOCAL STAGE PASSED")
print(f"Corrected local stage: {OCTOBER_LOCAL_STAGE_V2_PARQUET}")
print(f"Origin audit file: {OCTOBER_LOCAL_ORIGIN_AUDIT_PARQUET}")
print("✅ Direction is explicit and non-null for every final row.")
print("✅ Numeric conversion has no malformed nonblank source values.")
print("✅ BigQuery remains unchanged.")

In [ ]:
# SECTION 38J2 — Upload and audit corrected October V2 stage
# Creates only a temporary BigQuery stage table.
# Production and manifest remain unchanged.

from google.cloud import bigquery
from google.api_core.exceptions import NotFound
import pandas as pd

PRODUCTION_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)
OCTOBER_STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110"
)
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"
EXPECTED_OCTOBER_ROWS = 610_656

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

if not OCTOBER_LOCAL_STAGE_V2_PARQUET.exists():
    raise FileNotFoundError(
        f"Corrected V2 local stage is missing:\n"
        f"{OCTOBER_LOCAL_STAGE_V2_PARQUET}"
    )

# Never overwrite a BigQuery stage table.
try:
    BQ.get_table(OCTOBER_STAGE_TABLE)
    raise RuntimeError(
        f"Safety stop: October stage table already exists:\n"
        f"{OCTOBER_STAGE_TABLE}\n"
        "Nothing was overwritten."
    )
except NotFound:
    print("✅ No existing October BigQuery stage found.")

production_october_rows = int(
    BQ.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-10-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-11-01')
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

if production_october_rows != 0:
    raise RuntimeError(
        f"Safety stop: production already contains "
        f"{production_october_rows:,} October rows."
    )

manifest_run_rows = int(
    BQ.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = '{OCTOBER_RUN_ID}'
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

if manifest_run_rows != 0:
    raise RuntimeError(
        f"Safety stop: manifest already has {manifest_run_rows:,} "
        f"row(s) for this October run."
    )

local_stage_df = pd.read_parquet(OCTOBER_LOCAL_STAGE_V2_PARQUET)

production_schema = BQ.get_table(PRODUCTION_TABLE).schema
production_columns = [field.name for field in production_schema]

if set(local_stage_df.columns) != set(production_columns):
    raise RuntimeError(
        "Safety stop: V2 local-stage schema differs from production.\n"
        f"Missing: {sorted(set(production_columns) - set(local_stage_df.columns))}\n"
        f"Unexpected: {sorted(set(local_stage_df.columns) - set(production_columns))}"
    )

local_stage_df = local_stage_df[production_columns].copy()

if len(local_stage_df) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError(
        f"Safety stop: local V2 stage has {len(local_stage_df):,} rows; "
        f"expected {EXPECTED_OCTOBER_ROWS:,}."
    )

if local_stage_df["DIRECTION"].isna().any():
    raise RuntimeError("Safety stop: local V2 stage has null DIRECTION values.")

if not local_stage_df["DIRECTION"].isin(
    ["GEN", "LOAD", "BIDIRECTIONAL"]
).all():
    raise RuntimeError("Safety stop: local V2 stage has invalid DIRECTION values.")

print("OCTOBER 2021 V2 BIGQUERY STAGE PREFLIGHT PASSED")
print(f"Local V2 rows ready: {len(local_stage_df):,}")
print(f"Production October rows: {production_october_rows:,}")
print(f"Existing manifest rows for this run: {manifest_run_rows:,}")
print(f"Temporary stage to create: {OCTOBER_STAGE_TABLE}")

load_config = bigquery.LoadJobConfig(
    schema=production_schema,
    create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED,
    write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
)

load_job = BQ.load_table_from_dataframe(
    local_stage_df,
    OCTOBER_STAGE_TABLE,
    job_config=load_config,
)
load_job.result()

stage_audit = BQ.query(
    f"""
    SELECT
      COUNT(*) AS stage_row_count,

      COUNT(DISTINCT CONCAT(
        CAST(INTERVAL_DATETIME AS STRING), '|',
        DUID, '|',
        BIDTYPE, '|',
        DIRECTION
      )) AS distinct_effective_key_count,

      COUNTIF(
        SETTLEMENTDATE IS NULL
        OR DUID IS NULL
        OR BIDTYPE IS NULL
        OR DIRECTION IS NULL
        OR INTERVAL_DATETIME IS NULL
        OR BIDSETTLEMENTDATE IS NULL
        OR OFFERDATE IS NULL
        OR PERIODID IS NULL
      ) AS null_required_field_count,

      COUNTIF(
        DIRECTION NOT IN ('GEN', 'LOAD', 'BIDIRECTIONAL')
      ) AS invalid_direction_count,

      MIN(INTERVAL_DATETIME) AS first_interval,
      MAX(INTERVAL_DATETIME) AS last_interval

    FROM `{OCTOBER_STAGE_TABLE}`
    """
).result().to_dataframe()

display(stage_audit)

audit = stage_audit.iloc[0]

if int(audit["stage_row_count"]) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: uploaded stage row count differs.")

if int(audit["distinct_effective_key_count"]) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: uploaded stage has duplicate effective keys.")

if int(audit["null_required_field_count"]) != 0:
    raise RuntimeError("Safety stop: uploaded stage has null required fields.")

if int(audit["invalid_direction_count"]) != 0:
    raise RuntimeError("Safety stop: uploaded stage has invalid directions.")

print()
print("✅ OCTOBER 2021 CORRECTED BIGQUERY STAGE PASSED")
print(f"Stage rows: {int(audit['stage_row_count']):,}")
print("✅ Production archive remains unchanged.")
print("✅ Manifest remains unchanged.")
print("✅ Next step is a separate promotion preflight and promotion.")

In [ ]:
# SECTION 38K — Promote verified October 2021 V2 stage into production
# Writes production rows and one manifest entry together.
# Does NOT delete the temporary stage or any local files.

PRODUCTION_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)
OCTOBER_STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110"
)
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"
EXPECTED_OCTOBER_ROWS = 610_656

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

# One final read-only preflight.
preflight = BQ.query(
    f"""
    SELECT
      (SELECT COUNT(*) FROM `{OCTOBER_STAGE_TABLE}`) AS stage_rows,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-10-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-11-01')
      ) AS production_october_rows,

      (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = '{OCTOBER_RUN_ID}'
      ) AS manifest_run_rows,

      (
        SELECT COUNT(*)
        FROM `{OCTOBER_STAGE_TABLE}`
        WHERE _RUN_ID != '{OCTOBER_RUN_ID}'
           OR _RUN_ID IS NULL
      ) AS unexpected_stage_run_rows
    """
).result().to_dataframe()

display(preflight)

check = preflight.iloc[0]

if int(check["stage_rows"]) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: stage row count is no longer correct.")

if int(check["production_october_rows"]) != 0:
    raise RuntimeError(
        "Safety stop: October production rows already exist."
    )

if int(check["manifest_run_rows"]) != 0:
    raise RuntimeError(
        "Safety stop: an October manifest entry already exists."
    )

if int(check["unexpected_stage_run_rows"]) != 0:
    raise RuntimeError(
        "Safety stop: stage contains an unexpected run identifier."
    )

print("✅ October promotion preflight passed.")
print("Promoting stage and creating manifest entry together...")

promotion_sql = f"""
BEGIN TRANSACTION;

INSERT INTO `{PRODUCTION_TABLE}`
SELECT *
FROM `{OCTOBER_STAGE_TABLE}`;

INSERT INTO `{MANIFEST_TABLE}` (
  RUN_ID,
  ARCHIVE_YEAR,
  ARCHIVE_MONTH,
  SOURCE_FILE_NAME,
  SOURCE_URL,
  SOURCE_METHOD,
  STATUS,
  SOURCE_ROWS,
  BESS_ROWS,
  LOADED_ROWS,
  STARTED_AT,
  COMPLETED_AT,
  ERROR_MESSAGE,
  CREATED_AT
)
VALUES (
  '{OCTOBER_RUN_ID}',
  2021,
  10,
  'PUBLIC_DVD_BIDPEROFFER_202110010000.zip + September boundary cache',
  'https://www.nemweb.com.au/',
  'TRACKER_RECONSTRUCTED_DUCKDB_LOCAL_WITH_BOUNDARY_CACHE',
  'PROMOTED',
  NULL,
  {EXPECTED_OCTOBER_ROWS},
  {EXPECTED_OCTOBER_ROWS},
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP(),
  NULL,
  CURRENT_TIMESTAMP()
);

COMMIT TRANSACTION;
"""

BQ.query(promotion_sql).result()

promotion_audit = BQ.query(
    f"""
    SELECT
      (
        SELECT COUNT(*)
        FROM `{OCTOBER_STAGE_TABLE}`
      ) AS stage_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE _RUN_ID = '{OCTOBER_RUN_ID}'
      ) AS production_run_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE SETTLEMENTDATE >= TIMESTAMP('2021-10-01')
          AND SETTLEMENTDATE < TIMESTAMP('2021-11-01')
      ) AS production_october_row_count,

      (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = '{OCTOBER_RUN_ID}'
          AND STATUS = 'PROMOTED'
      ) AS promoted_manifest_row_count
    """
).result().to_dataframe()

display(promotion_audit)

audit = promotion_audit.iloc[0]

if int(audit["production_run_row_count"]) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: promoted run count is incorrect.")

if int(audit["production_october_row_count"]) != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: October production count is incorrect.")

if int(audit["promoted_manifest_row_count"]) != 1:
    raise RuntimeError("Safety stop: promoted manifest entry is incorrect.")

print()
print("✅ OCTOBER 2021 PROMOTED")
print(f"Production rows retained: {EXPECTED_OCTOBER_ROWS:,}")
print("✅ Manifest entry created.")
print("✅ Temporary BigQuery stage and local audit files remain for final audit.")

In [ ]:
# SECTION 38L — Final exact audit of October 2021 promotion
# Read-only. Does not delete the BigQuery stage or local files.

OCTOBER_STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110"
)
PRODUCTION_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
MANIFEST_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_MANIFEST"
)
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"
EXPECTED_OCTOBER_ROWS = 610_656

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

final_audit = BQ.query(
    f"""
    SELECT
      (
        SELECT COUNT(*)
        FROM `{OCTOBER_STAGE_TABLE}`
      ) AS stage_row_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE _RUN_ID = '{OCTOBER_RUN_ID}'
      ) AS production_run_row_count,

      (
        SELECT COUNT(*)
        FROM (
          SELECT *
          FROM `{OCTOBER_STAGE_TABLE}`

          EXCEPT DISTINCT

          SELECT *
          FROM `{PRODUCTION_TABLE}`
          WHERE _RUN_ID = '{OCTOBER_RUN_ID}'
        )
      ) AS stage_missing_from_production_count,

      (
        SELECT COUNT(*)
        FROM (
          SELECT *
          FROM `{PRODUCTION_TABLE}`
          WHERE _RUN_ID = '{OCTOBER_RUN_ID}'

          EXCEPT DISTINCT

          SELECT *
          FROM `{OCTOBER_STAGE_TABLE}`
        )
      ) AS production_missing_from_stage_count,

      (
        SELECT COUNT(*)
        FROM `{PRODUCTION_TABLE}`
        WHERE _RUN_ID = '{OCTOBER_RUN_ID}'
          AND (
            DIRECTION IS NULL
            OR DIRECTION NOT IN ('GEN', 'LOAD', 'BIDIRECTIONAL')
          )
      ) AS invalid_direction_rows,

      (
        SELECT COUNT(*)
        FROM `{MANIFEST_TABLE}`
        WHERE RUN_ID = '{OCTOBER_RUN_ID}'
          AND STATUS = 'PROMOTED'
      ) AS promoted_manifest_rows
    """
).result().to_dataframe()

display(final_audit)

audit = final_audit.iloc[0]

checks = {
    "stage row count": int(audit["stage_row_count"]) == EXPECTED_OCTOBER_ROWS,
    "production row count": (
        int(audit["production_run_row_count"]) == EXPECTED_OCTOBER_ROWS
    ),
    "stage missing from production": (
        int(audit["stage_missing_from_production_count"]) == 0
    ),
    "production missing from stage": (
        int(audit["production_missing_from_stage_count"]) == 0
    ),
    "invalid direction rows": int(audit["invalid_direction_rows"]) == 0,
    "promoted manifest rows": int(audit["promoted_manifest_rows"]) == 1,
}

failed_checks = [name for name, passed in checks.items() if not passed]

if failed_checks:
    raise RuntimeError(
        "October final audit failed: " + ", ".join(failed_checks) + ". "
        "Nothing was deleted."
    )

print()
print("✅ OCTOBER 2021 FINAL PRODUCTION AUDIT PASSED")
print("✅ Production exactly matches the corrected temporary stage.")
print("✅ Direction values are complete and valid.")
print("✅ Safe next step: delete only the temporary BigQuery stage table.")

In [ ]:
# SECTION 38M — Delete only the verified temporary October BigQuery stage

from google.api_core.exceptions import NotFound

OCTOBER_STAGE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202110"
)
PRODUCTION_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_ARCHIVE_V1"
)
OCTOBER_RUN_ID = "bess-effective-reconstructed-duckdb-2021-10"
EXPECTED_OCTOBER_ROWS = 610_656

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

stage_rows = int(
    BQ.query(
        f"SELECT COUNT(*) AS row_count FROM `{OCTOBER_STAGE_TABLE}`"
    ).result().to_dataframe().iloc[0]["row_count"]
)

production_rows = int(
    BQ.query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM `{PRODUCTION_TABLE}`
        WHERE _RUN_ID = '{OCTOBER_RUN_ID}'
        """
    ).result().to_dataframe().iloc[0]["row_count"]
)

print(f"October stage rows: {stage_rows:,}")
print(f"October production rows: {production_rows:,}")

if stage_rows != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: unexpected temporary-stage row count.")

if production_rows != EXPECTED_OCTOBER_ROWS:
    raise RuntimeError("Safety stop: production is not complete.")

BQ.delete_table(OCTOBER_STAGE_TABLE, not_found_ok=False)

try:
    BQ.get_table(OCTOBER_STAGE_TABLE)
    raise RuntimeError("Safety stop: stage table still exists after deletion.")
except NotFound:
    pass

print()
print("✅ October 2021 temporary BigQuery stage deleted.")
print("✅ Production archive and manifest remain intact.")
print("✅ Local V2 stage, origin audit, cached ZIP, and raw October CSV were not changed.")

In [ ]:
# SECTION 38N — Delete only the verified temporary October extracted CSV

from pathlib import Path

october_working_csv = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache\duckdb_working"
    r"\PUBLIC_DVD_BIDPEROFFER_202110010000.CSV"
)

expected_name = "PUBLIC_DVD_BIDPEROFFER_202110010000.CSV"

if not october_working_csv.exists():
    print("Nothing deleted: the October working CSV is already absent.")
elif october_working_csv.name != expected_name:
    raise RuntimeError("Safety stop: unexpected filename.")
else:
    recovered_gb = october_working_csv.stat().st_size / (1024 ** 3)

    print(f"Deleting only: {october_working_csv}")
    print(f"Space to recover: {recovered_gb:.2f} GB")

    october_working_csv.unlink()

    if october_working_csv.exists():
        raise RuntimeError("Safety stop: file still exists after deletion.")

    print("✅ October temporary extracted CSV deleted.")
    print("✅ Cached ZIP, local V2 stage, origin audit, production, and manifest remain intact.")

In [ ]:
# SECTION 38O0 — One-time recovery: seed the local October → November boundary cache
# Local work only. No BigQuery writes, promotions, or deletions.
# The temporary 46.94 GB CSV is deleted only after a successful compact boundary file is verified.

import csv
import io
import os
import shutil
import time
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

OCTOBER_MONTH = pd.Timestamp("2021-10-01")
NOVEMBER_MONTH = pd.Timestamp("2021-11-01")

LOCAL_ROOT = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache"
)
PARQUET_CACHE = LOCAL_ROOT / "bess_only_parquet"
RUN_STATE = LOCAL_ROOT / "run_state"
DUCKDB_WORKING = LOCAL_ROOT / "duckdb_working"

for folder in [PARQUET_CACHE, RUN_STATE, DUCKDB_WORKING]:
    folder.mkdir(parents=True, exist_ok=True)

RECOVERY_WORKING_CSV = (
    DUCKDB_WORKING / "PUBLIC_DVD_BIDPEROFFER_202110010000.CSV"
)
RECOVERY_WORKING_PART = Path(f"{RECOVERY_WORKING_CSV}.part")

NOVEMBER_TARGET_CSV = (
    RUN_STATE / "november_2021_tracker_offer_identities.csv"
)
NOVEMBER_BOUNDARY_PARQUET = (
    PARQUET_CACHE / "BIDPEROFFER_BOUNDARY_FOR_202111_LOCAL.parquet"
)

if NOVEMBER_BOUNDARY_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: the local November boundary file already exists:\n"
        f"{NOVEMBER_BOUNDARY_PARQUET}\n"
        "Nothing was overwritten."
    )

if RECOVERY_WORKING_PART.exists():
    raise RuntimeError(
        "Safety stop: an incomplete October recovery extraction exists:\n"
        f"{RECOVERY_WORKING_PART}\n"
        "Nothing was overwritten or deleted."
    )

if RECOVERY_WORKING_CSV.exists():
    raise RuntimeError(
        "Safety stop: an October working CSV already exists:\n"
        f"{RECOVERY_WORKING_CSV}\n"
        "Nothing was overwritten or deleted."
    )

free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)
if free_gb < 70:
    raise RuntimeError(
        f"Safety stop: only {free_gb:.1f} GB free; at least 70 GB is required."
    )

def canonical_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

print("=" * 100)
print("RECOVERING OCTOBER → NOVEMBER LOCAL BOUNDARY CACHE")
print("=" * 100)
print("No BigQuery data will be changed.")

# November uses the November and December tracker archives.
november_targets, november_expected_rows, _, _ = read_tracker_targets(
    NOVEMBER_MONTH
)

if not november_targets:
    raise RuntimeError("Safety stop: no November tracker targets were found.")

target_rows = []

for key in november_targets:
    duid, bidtype, bidsettlementdate, offerdate, periodid = key

    target_rows.append(
        {
            "DUID": str(duid).strip().upper(),
            "BIDTYPE": str(bidtype).strip().upper(),
            "BIDSETTLEMENTDATE_KEY": canonical_timestamp(bidsettlementdate),
            "OFFERDATE_KEY": canonical_timestamp(offerdate),
            "PERIODID": int(periodid),
        }
    )

november_target_df = (
    pd.DataFrame(target_rows)
    .drop_duplicates()
    .sort_values(
        [
            "DUID",
            "BIDTYPE",
            "BIDSETTLEMENTDATE_KEY",
            "OFFERDATE_KEY",
            "PERIODID",
        ]
    )
    .reset_index(drop=True)
)

november_target_df.to_csv(NOVEMBER_TARGET_CSV, index=False)

print(f"Expected November effective rows: {november_expected_rows:,}")
print(f"November offer identities to check: {len(november_target_df):,}")

# Reuse the retained October ZIP cache.
october_source_file, october_source_url = raw_source_urls(OCTOBER_MONTH)[0]

if "_download_aemo_zip_to_cache" not in globals():
    raise RuntimeError(
        "Cached downloader is unavailable. Run the cache-preservation cell first."
    )

october_zip = Path(
    _download_aemo_zip_to_cache(
        october_source_url,
        f"October recovery source {october_source_file}",
        source_file_name=october_source_file,
    )
)

with zipfile.ZipFile(october_zip) as archive:
    csv_members = [
        member for member in archive.namelist()
        if member.upper().endswith(".CSV")
    ]

    if len(csv_members) != 1:
        raise RuntimeError(f"Expected one CSV inside October ZIP; found {csv_members}")

    csv_member = csv_members[0]
    csv_info = archive.getinfo(csv_member)
    expected_bytes = csv_info.file_size

    data_header = None

    with archive.open(csv_member) as raw_file:
        reader = csv.reader(
            io.TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
        )

        for record in reader:
            if (
                record
                and record[0].strip().upper() == "I"
                and len(record) > 4
                and record[3].strip() == "1"
                and "DUID" in record[4:]
            ):
                data_header = record[4:]
                break

if data_header is None:
    raise RuntimeError("Safety stop: October source header was not found.")

required_columns = {
    "DUID",
    "BIDTYPE",
    "TRADINGDATE",
    "OFFERDATETIME",
    "PERIODID",
}

if required_columns - set(data_header):
    raise RuntimeError(
        "Safety stop: October source schema differs from the tested format."
    )

print(
    f"Temporarily extracting October source: {expected_bytes / (1024 ** 3):.2f} GB"
)

copied_bytes = 0
next_progress_bytes = 5 * (1024 ** 3)
recovery_succeeded = False

try:
    with zipfile.ZipFile(october_zip) as archive:
        with archive.open(csv_member) as source_file:
            with open(RECOVERY_WORKING_PART, "wb") as destination_file:
                while True:
                    block = source_file.read(16 * 1024 * 1024)

                    if not block:
                        break

                    destination_file.write(block)
                    copied_bytes += len(block)

                    if copied_bytes >= next_progress_bytes:
                        print(
                            f"  Extracted {copied_bytes / (1024 ** 3):.1f} GB "
                            f"of {expected_bytes / (1024 ** 3):.1f} GB"
                        )
                        next_progress_bytes += 5 * (1024 ** 3)

    if copied_bytes != expected_bytes:
        raise RuntimeError(
            f"Incomplete extraction: {copied_bytes:,} of {expected_bytes:,} bytes."
        )

    os.replace(RECOVERY_WORKING_PART, RECOVERY_WORKING_CSV)

    raw_columns = [
        "RECORD_TYPE",
        "AEMO_GROUP",
        "AEMO_TABLE",
        "AEMO_VERSION",
    ] + data_header

    def quoted_identifier(name):
        return '"' + name.replace('"', '""') + '"'

    column_mapping_sql = ", ".join(
        f"{quoted_identifier(name)}: 'VARCHAR'"
        for name in raw_columns
    )

    data_select_sql = ", ".join(
        f"raw.{quoted_identifier(name)}"
        for name in data_header
    )

    working_csv_path = RECOVERY_WORKING_CSV.as_posix()
    target_csv_path = NOVEMBER_TARGET_CSV.as_posix()
    boundary_parquet_path = NOVEMBER_BOUNDARY_PARQUET.as_posix()

    normalised_tradingdate_sql = """
    strftime(
        coalesce(
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    normalised_offerdatetime_sql = """
    strftime(
        coalesce(
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    con = duckdb.connect()
    con.execute("PRAGMA threads=22")

    print("DuckDB is selecting October offers that November may require...")

    con.execute(
        f"""
        COPY (
            SELECT
                {data_select_sql},
                {normalised_tradingdate_sql} AS BIDSETTLEMENTDATE_KEY,
                {normalised_offerdatetime_sql} AS OFFERDATE_KEY

            FROM read_csv(
                '{working_csv_path}',
                header = false,
                columns = {{{column_mapping_sql}}},
                strict_mode = false,
                null_padding = true,
                ignore_errors = true
            ) AS raw

            INNER JOIN read_csv(
                '{target_csv_path}',
                header = true,
                all_varchar = true
            ) AS target
              ON UPPER(TRIM(raw.DUID)) = target.DUID
             AND UPPER(TRIM(raw.BIDTYPE)) = target.BIDTYPE
             AND {normalised_tradingdate_sql} = target.BIDSETTLEMENTDATE_KEY
             AND {normalised_offerdatetime_sql} = target.OFFERDATE_KEY
             AND TRY_CAST(raw.PERIODID AS INTEGER) =
                 TRY_CAST(target.PERIODID AS INTEGER)

            WHERE UPPER(TRIM(raw.RECORD_TYPE)) = 'D'
        )
        TO '{boundary_parquet_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    boundary_audit = con.execute(
        f"""
        SELECT
            COUNT(*) AS boundary_offer_identities,
            COUNT(DISTINCT DUID) AS source_duids,
            MIN(TRADINGDATE) AS first_tradingdate,
            MAX(TRADINGDATE) AS last_tradingdate
        FROM read_parquet('{boundary_parquet_path}')
        """
    ).fetchdf()

    duplicate_boundary_keys = con.execute(
        f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT
                DUID,
                BIDTYPE,
                BIDSETTLEMENTDATE_KEY,
                OFFERDATE_KEY,
                PERIODID,
                COUNT(*) AS occurrences
            FROM read_parquet('{boundary_parquet_path}')
            GROUP BY 1, 2, 3, 4, 5
            HAVING COUNT(*) > 1
        )
        """
    ).fetchdf()

    con.close()

    display(boundary_audit)
    display(duplicate_boundary_keys)

    if int(boundary_audit.iloc[0]["boundary_offer_identities"]) == 0:
        raise RuntimeError(
            "Safety stop: no October offers were found for November."
        )

    if int(duplicate_boundary_keys.iloc[0]["duplicate_keys"]) != 0:
        raise RuntimeError(
            "Safety stop: duplicate October→November boundary offers found."
        )

    recovery_succeeded = True

finally:
    # Delete only the temporary file created by this recovery after success.
    if recovery_succeeded and RECOVERY_WORKING_CSV.exists():
        recovered_gb = RECOVERY_WORKING_CSV.stat().st_size / (1024 ** 3)
        RECOVERY_WORKING_CSV.unlink()
        print(f"✅ Temporary October recovery CSV deleted: {recovered_gb:.2f} GB")

print()
print("✅ OCTOBER → NOVEMBER LOCAL BOUNDARY CACHE READY")
print(f"Boundary file: {NOVEMBER_BOUNDARY_PARQUET}")
print("✅ Future continuous processing will capture the next boundary before")
print("   each month’s temporary raw CSV is deleted.")
print("✅ No BigQuery data was changed.")

In [ ]:
# SECTION 38O1 — Diagnose November carry-in coverage from persistent offer state
# Read-only. No local deletions and no BigQuery writes.

import pandas as pd

NOVEMBER_MONTH = pd.Timestamp("2021-11-01")

if "lookup_carry_records" not in globals():
    raise RuntimeError(
        "lookup_carry_records() is unavailable. "
        "Run the earlier carry-lookup helper cell first."
    )

print("NOVEMBER 2021 CARRY-STATE DIAGNOSIS")
print("No data will be changed.")

november_targets, november_expected_rows, _, _ = read_tracker_targets(
    NOVEMBER_MONTH
)

november_target_keys = set(november_targets)

print(f"November expected effective rows: {november_expected_rows:,}")
print(f"November unique offer identities: {len(november_target_keys):,}")

# This queries only the compact persistent offer-state lookup.
carry_records = lookup_carry_records(november_target_keys)
carry_key_set = set(carry_records)

unresolved_keys = november_target_keys - carry_key_set

print()
print("PERSISTENT CARRY-LOOKUP COVERAGE")
print(f"Offer identities found in carry lookup: {len(carry_key_set):,}")
print(f"Not in carry lookup: {len(unresolved_keys):,}")

def key_to_row(key):
    duid, bidtype, bidsettlementdate, offerdate, periodid = key

    return {
        "DUID": duid,
        "BIDTYPE": bidtype,
        "BIDSETTLEMENTDATE": bidsettlementdate,
        "OFFERDATE": offerdate,
        "PERIODID": periodid,
    }

carry_date_profile = pd.DataFrame(
    [key_to_row(key) for key in carry_key_set]
)

if not carry_date_profile.empty:
    carry_date_profile["BIDSETTLEMENTDATE"] = pd.to_datetime(
        carry_date_profile["BIDSETTLEMENTDATE"],
        utc=True,
    )

    carry_month_profile = (
        carry_date_profile
        .assign(
            bidsettlement_month=(
                carry_date_profile["BIDSETTLEMENTDATE"]
                .dt.to_period("M")
                .astype(str)
            )
        )
        .groupby("bidsettlement_month", dropna=False)
        .size()
        .reset_index(name="offer_identities_found")
        .sort_values("bidsettlement_month")
    )

    print("\nCarry-lookup identities by their bid-settlement month:")
    display(carry_month_profile)

if unresolved_keys:
    unresolved_preview = pd.DataFrame(
        [key_to_row(key) for key in sorted(unresolved_keys)[:30]]
    )

    unresolved_preview["BIDSETTLEMENTDATE"] = pd.to_datetime(
        unresolved_preview["BIDSETTLEMENTDATE"],
        utc=True,
    )
    unresolved_preview["OFFERDATE"] = pd.to_datetime(
        unresolved_preview["OFFERDATE"],
        utc=True,
    )

    print("\nExamples not currently in the carry lookup:")
    display(unresolved_preview)

print()
print("✅ Diagnosis complete.")
print("This determines whether November needs an older local source scan,")
print("or whether the existing carry state already supplies its pre-November offers.")

In [ ]:
# SECTION 38O2 — Delete only the redundant October recovery CSV

from pathlib import Path

recovery_csv = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache\duckdb_working"
    r"\PUBLIC_DVD_BIDPEROFFER_202110010000.CSV"
)

if not recovery_csv.exists():
    print("Nothing deleted: the October recovery CSV is already absent.")
elif recovery_csv.name != "PUBLIC_DVD_BIDPEROFFER_202110010000.CSV":
    raise RuntimeError("Safety stop: unexpected filename.")
else:
    size_gb = recovery_csv.stat().st_size / (1024 ** 3)

    print(f"Deleting only: {recovery_csv}")
    print(f"Space to recover: {size_gb:.2f} GB")

    recovery_csv.unlink()

    if recovery_csv.exists():
        raise RuntimeError("Safety stop: file still exists after deletion.")

    print("✅ Redundant October recovery CSV deleted.")
    print("✅ Cached ZIP, compact Parquets, BigQuery production, and manifest were not changed.")

In [ ]:
# SECTION 38P — Seed persistent local offer-state cache for continuous backfill
# Local writes only. BigQuery is read only.

import pandas as pd
from pathlib import Path

LOCAL_ROOT = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache"
)

STATE_DIR = LOCAL_ROOT / "continuous_offer_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)

STATE_SEED_PARQUET = STATE_DIR / "offer_state_seed_through_202110.parquet"

BOUNDARY_CACHE_TABLE = (
    "master-aemo.aemo_master_schema."
    "BIDPEROFFER_BESS_EFFECTIVE_BOUNDARY_LOOKUP_V1"
)

if STATE_SEED_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: continuous offer-state seed already exists:\n"
        f"{STATE_SEED_PARQUET}\n"
        "Nothing was overwritten."
    )

if not OCTOBER_MATCH_PARQUET.exists():
    raise FileNotFoundError(f"Missing October compact matches:\n{OCTOBER_MATCH_PARQUET}")

if "legacy_direction" not in globals():
    raise RuntimeError("legacy_direction() is unavailable.")

BQ = globals().get("client") or globals().get("bq_client")
if BQ is None:
    raise RuntimeError("BigQuery client is unavailable.")

def canonical_key_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

STATE_KEY_COLUMNS = [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
]

STATE_VALUE_COLUMNS = [
    "DIRECTION",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "ENERGYLIMIT",
    "SOURCE_FILE",
    "SOURCE_URL",
    "STATE_SOURCE_MONTH",
    "STATE_ORIGIN",
]

october_raw_df = pd.read_parquet(OCTOBER_MATCH_PARQUET)

october_state_df = pd.DataFrame(
    {
        "DUID": october_raw_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": october_raw_df["BIDTYPE"].astype("string").str.strip().str.upper(),
        "BIDSETTLEMENTDATE_KEY": october_raw_df["BIDSETTLEMENTDATE_KEY"],
        "OFFERDATE_KEY": october_raw_df["OFFERDATE_KEY"],
        "PERIODID": pd.to_numeric(
            october_raw_df["PERIODID"], errors="raise"
        ).astype("int64"),
        "DIRECTION": (
            october_raw_df["DUID"]
            .astype("string")
            .str.strip()
            .str.upper()
            .map(legacy_direction)
        ),
        "MAXAVAIL": october_raw_df["MAXAVAIL"],
        "FIXEDLOAD": october_raw_df["FIXEDLOAD"],
        "ROCUP": october_raw_df["RAMPUPRATE"],
        "ROCDOWN": october_raw_df["RAMPDOWNRATE"],
        "ENABLEMENTMIN": october_raw_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": october_raw_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": october_raw_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": october_raw_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": october_raw_df["BANDAVAIL1"],
        "BANDAVAIL2": october_raw_df["BANDAVAIL2"],
        "BANDAVAIL3": october_raw_df["BANDAVAIL3"],
        "BANDAVAIL4": october_raw_df["BANDAVAIL4"],
        "BANDAVAIL5": october_raw_df["BANDAVAIL5"],
        "BANDAVAIL6": october_raw_df["BANDAVAIL6"],
        "BANDAVAIL7": october_raw_df["BANDAVAIL7"],
        "BANDAVAIL8": october_raw_df["BANDAVAIL8"],
        "BANDAVAIL9": october_raw_df["BANDAVAIL9"],
        "BANDAVAIL10": october_raw_df["BANDAVAIL10"],
        "PASAAVAILABILITY": october_raw_df["PASAAVAILABILITY"],
        "ENERGYLIMIT": None,
        "SOURCE_FILE": "PUBLIC_DVD_BIDPEROFFER_202110010000.zip",
        "SOURCE_URL": october_source_url,
        "STATE_SOURCE_MONTH": "2021-10",
        "STATE_ORIGIN": "OCTOBER_RAW_MATCHES",
    }
)

boundary_df = BQ.query(
    f"SELECT * FROM `{BOUNDARY_CACHE_TABLE}`"
).result().to_dataframe()

boundary_state_df = pd.DataFrame(
    {
        "DUID": boundary_df["DUID"].astype("string").str.strip().str.upper(),
        "BIDTYPE": boundary_df["BIDTYPE"].astype("string").str.strip().str.upper(),
        "BIDSETTLEMENTDATE_KEY": boundary_df["BIDSETTLEMENTDATE"].map(
            canonical_key_timestamp
        ),
        "OFFERDATE_KEY": boundary_df["OFFERDATE"].map(
            canonical_key_timestamp
        ),
        "PERIODID": pd.to_numeric(
            boundary_df["PERIODID"], errors="raise"
        ).astype("int64"),
        "DIRECTION": (
            boundary_df["DIRECTION"]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
            .fillna(
                boundary_df["DUID"]
                .astype("string")
                .str.strip()
                .str.upper()
                .map(legacy_direction)
            )
        ),
        "MAXAVAIL": boundary_df["MAXAVAIL"],
        "FIXEDLOAD": boundary_df["FIXEDLOAD"],
        "ROCUP": boundary_df["ROCUP"],
        "ROCDOWN": boundary_df["ROCDOWN"],
        "ENABLEMENTMIN": boundary_df["ENABLEMENTMIN"],
        "ENABLEMENTMAX": boundary_df["ENABLEMENTMAX"],
        "LOWBREAKPOINT": boundary_df["LOWBREAKPOINT"],
        "HIGHBREAKPOINT": boundary_df["HIGHBREAKPOINT"],
        "BANDAVAIL1": boundary_df["BANDAVAIL1"],
        "BANDAVAIL2": boundary_df["BANDAVAIL2"],
        "BANDAVAIL3": boundary_df["BANDAVAIL3"],
        "BANDAVAIL4": boundary_df["BANDAVAIL4"],
        "BANDAVAIL5": boundary_df["BANDAVAIL5"],
        "BANDAVAIL6": boundary_df["BANDAVAIL6"],
        "BANDAVAIL7": boundary_df["BANDAVAIL7"],
        "BANDAVAIL8": boundary_df["BANDAVAIL8"],
        "BANDAVAIL9": boundary_df["BANDAVAIL9"],
        "BANDAVAIL10": boundary_df["BANDAVAIL10"],
        "PASAAVAILABILITY": boundary_df["PASAAVAILABILITY"],
        "ENERGYLIMIT": boundary_df["ENERGYLIMIT"],
        "SOURCE_FILE": boundary_df["SOURCE_FILE"],
        "SOURCE_URL": boundary_df["SOURCE_URL"],
        "STATE_SOURCE_MONTH": "PRE_2021_10",
        "STATE_ORIGIN": "EXISTING_BOUNDARY_CACHE",
    }
)

state_df = pd.concat(
    [october_state_df, boundary_state_df],
    ignore_index=True,
)[STATE_KEY_COLUMNS + STATE_VALUE_COLUMNS]

duplicate_key_count = int(state_df.duplicated(STATE_KEY_COLUMNS).sum())

if duplicate_key_count != 0:
    raise RuntimeError(
        f"Safety stop: seed contains {duplicate_key_count:,} duplicate offer keys."
    )

invalid_direction_count = int(
    (~state_df["DIRECTION"].isin(["GEN", "LOAD", "BIDIRECTIONAL"])).sum()
)

if invalid_direction_count != 0:
    raise RuntimeError(
        f"Safety stop: seed contains {invalid_direction_count:,} invalid directions."
    )

state_df.to_parquet(
    STATE_SEED_PARQUET,
    index=False,
    compression="zstd",
)

origin_summary = (
    state_df.groupby("STATE_ORIGIN")
    .size()
    .reset_index(name="offer_identities")
)

print("✅ CONTINUOUS LOCAL OFFER-STATE CACHE SEEDED")
print(f"Offer identities retained: {len(state_df):,}")
print(f"State file: {STATE_SEED_PARQUET}")
display(origin_summary)

print("✅ No raw archive was scanned.")
print("✅ No BigQuery data was changed.")

In [ ]:
# SECTION 38P1 — Fix local offer-state Parquet types and write the seed
# Local write only. No BigQuery changes.

import pandas as pd

if STATE_SEED_PARQUET.exists():
    raise RuntimeError(
        "Safety stop: a state seed file now exists:\n"
        f"{STATE_SEED_PARQUET}\n"
        "Nothing was overwritten."
    )

if "state_df" not in globals():
    raise RuntimeError(
        "state_df is unavailable. Re-run Section 38P, then run this patch."
    )

NUMERIC_STATE_COLUMNS = [
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "ENERGYLIMIT",
]

quality_rows = []

for column in NUMERIC_STATE_COLUMNS:
    source_text = state_df[column].astype("string").str.strip()
    source_blank = source_text.isna() | source_text.eq("")
    parsed = pd.to_numeric(state_df[column], errors="coerce")
    malformed = (~source_blank) & parsed.isna()

    quality_rows.append(
        {
            "FIELD_NAME": column,
            "SOURCE_BLANK_VALUES": int(source_blank.sum()),
            "VALID_NUMERIC_VALUES": int(
                (~source_blank & ~parsed.isna()).sum()
            ),
            "MALFORMED_NONBLANK_VALUES": int(malformed.sum()),
        }
    )

    # Float64 is deliberately used so blanks remain nullable.
    state_df[column] = parsed.astype("Float64")

state_quality_df = pd.DataFrame(quality_rows)
display(state_quality_df)

malformed_count = int(
    state_quality_df["MALFORMED_NONBLANK_VALUES"].sum()
)

if malformed_count != 0:
    raise RuntimeError(
        f"Safety stop: {malformed_count:,} malformed nonblank numeric "
        "values found. State seed was not written."
    )

# Make all remaining text fields explicitly consistent for Parquet.
for column in [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "DIRECTION",
    "SOURCE_FILE",
    "SOURCE_URL",
    "STATE_SOURCE_MONTH",
    "STATE_ORIGIN",
]:
    state_df[column] = state_df[column].astype("string")

state_df["PERIODID"] = pd.to_numeric(
    state_df["PERIODID"],
    errors="raise",
).astype("int64")

state_df.to_parquet(
    STATE_SEED_PARQUET,
    index=False,
    compression="zstd",
)

origin_summary = (
    state_df.groupby("STATE_ORIGIN", dropna=False)
    .size()
    .reset_index(name="offer_identities")
)

print("✅ CONTINUOUS LOCAL OFFER-STATE CACHE SEEDED")
print(f"Offer identities retained: {len(state_df):,}")
print(f"State seed: {STATE_SEED_PARQUET}")
display(origin_summary)

print("✅ Numeric fields are now stored consistently as nullable numbers.")
print("✅ No BigQuery data was changed.")

In [ ]:
# SECTION 38Q — Resumable continuous local BESS effective-offer backfill
# Scope: November 2021 through July 2024.
# Local work only: no BigQuery stage, production, manifest, or cache writes.
#
# Behaviour:
# - processes one month at a time;
# - downloads each raw ZIP once and retains it;
# - uses DuckDB to extract only required offer identities;
# - uses local historical offer state only when needed;
# - saves compact local stage/state/checkpoint files;
# - deletes a raw CSV only after that month has passed every local audit;
# - stops safely at the first unresolved source/schema/data issue.

import csv
import io
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

# ---------- Configuration ----------

BATCH_START_MONTH = pd.Timestamp("2021-11-01")
BATCH_END_MONTH = pd.Timestamp("2024-07-01")

LOCAL_ROOT = Path(
    r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse"
    r"\bess_effective_cache"
)
PARQUET_CACHE = LOCAL_ROOT / "bess_only_parquet"
RUN_STATE = LOCAL_ROOT / "run_state"
DUCKDB_WORKING = LOCAL_ROOT / "duckdb_working"
STATE_DIR = LOCAL_ROOT / "continuous_offer_state"

for folder in [PARQUET_CACHE, RUN_STATE, DUCKDB_WORKING, STATE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

LOCAL_BATCH_MANIFEST = RUN_STATE / "continuous_local_backfill_manifest.jsonl"
LOCAL_BATCH_CHECKPOINT = RUN_STATE / "continuous_local_backfill_checkpoint.json"

STATE_SEED_PARQUET = STATE_DIR / "offer_state_seed_through_202110.parquet"

if not STATE_SEED_PARQUET.exists():
    raise RuntimeError(
        "Run Section 38P/38P1 first: the local offer-state seed is missing."
    )

if "_download_aemo_zip_to_cache" not in globals():
    raise RuntimeError(
        "Cached downloader is unavailable. Run the cache-preservation cell first."
    )

if "legacy_direction" not in globals():
    raise RuntimeError("legacy_direction() is unavailable.")

# ---------- Constants ----------

OFFER_KEY_COLUMNS = [
    "DUID",
    "BIDTYPE",
    "BIDSETTLEMENTDATE_KEY",
    "OFFERDATE_KEY",
    "PERIODID",
]

NUMERIC_OFFER_COLUMNS = [
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "PASAAVAILABILITY",
    "ENERGYLIMIT",
]

STATE_COLUMNS = OFFER_KEY_COLUMNS + [
    "DIRECTION",
    *NUMERIC_OFFER_COLUMNS,
    "SOURCE_FILE",
    "SOURCE_URL",
    "STATE_SOURCE_MONTH",
    "STATE_ORIGIN",
]

FINAL_STAGE_COLUMNS = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
    "VERSIONNO",
    "MAXAVAIL",
    "FIXEDLOAD",
    "ROCUP",
    "ROCDOWN",
    "ENABLEMENTMIN",
    "ENABLEMENTMAX",
    "LOWBREAKPOINT",
    "HIGHBREAKPOINT",
    "BANDAVAIL1",
    "BANDAVAIL2",
    "BANDAVAIL3",
    "BANDAVAIL4",
    "BANDAVAIL5",
    "BANDAVAIL6",
    "BANDAVAIL7",
    "BANDAVAIL8",
    "BANDAVAIL9",
    "BANDAVAIL10",
    "LASTCHANGED",
    "PASAAVAILABILITY",
    "MR_CAPACITY",
    "ENERGYLIMIT",
    "_LOAD_DATE",
    "_SOURCE",
    "_PIPELINE",
    "_SOURCE_FILE",
    "_SOURCE_URL",
    "_RUN_ID",
]

# ---------- Helpers ----------

def canonical_timestamp(value):
    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is not None:
        timestamp = timestamp.tz_convert("UTC").tz_localize(None)

    return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")


def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def target_identity_dataframe(target_map):
    rows = []

    for key in target_map:
        duid, bidtype, bidsettlementdate, offerdate, periodid = key

        rows.append(
            {
                "DUID": str(duid).strip().upper(),
                "BIDTYPE": str(bidtype).strip().upper(),
                "BIDSETTLEMENTDATE_KEY": canonical_timestamp(
                    bidsettlementdate
                ),
                "OFFERDATE_KEY": canonical_timestamp(offerdate),
                "PERIODID": int(periodid),
            }
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
        .sort_values(OFFER_KEY_COLUMNS)
        .reset_index(drop=True)
    )


def expanded_interval_dataframe(target_map):
    rows = []

    for key, applied_intervals in target_map.items():
        duid, bidtype, bidsettlementdate, offerdate, periodid = key

        for applied in applied_intervals:
            rows.append(
                {
                    "DUID": str(duid).strip().upper(),
                    "BIDTYPE": str(bidtype).strip().upper(),
                    "BIDSETTLEMENTDATE_KEY": canonical_timestamp(
                        bidsettlementdate
                    ),
                    "OFFERDATE_KEY": canonical_timestamp(offerdate),
                    "PERIODID": int(periodid),
                    "SETTLEMENTDATE": pd.Timestamp(
                        applied["SETTLEMENTDATE"]
                    ),
                    "INTERVAL_DATETIME": pd.Timestamp(
                        applied["INTERVAL_DATETIME"]
                    ),
                }
            )

    return pd.DataFrame(rows)


def standardise_raw_offer_matches(
    raw_matches_df,
    source_file_name,
    source_url,
    source_month_text,
):
    required = {
        "DUID",
        "BIDTYPE",
        "BIDSETTLEMENTDATE_KEY",
        "OFFERDATE_KEY",
        "PERIODID",
        "MAXAVAIL",
        "FIXEDLOAD",
        "RAMPUPRATE",
        "RAMPDOWNRATE",
        "ENABLEMENTMIN",
        "ENABLEMENTMAX",
        "LOWBREAKPOINT",
        "HIGHBREAKPOINT",
        "BANDAVAIL1",
        "BANDAVAIL2",
        "BANDAVAIL3",
        "BANDAVAIL4",
        "BANDAVAIL5",
        "BANDAVAIL6",
        "BANDAVAIL7",
        "BANDAVAIL8",
        "BANDAVAIL9",
        "BANDAVAIL10",
        "PASAAVAILABILITY",
    }

    missing = required - set(raw_matches_df.columns)

    if missing:
        raise RuntimeError(
            f"{source_month_text}: raw match file lacks {sorted(missing)}"
        )

    output = pd.DataFrame(
        {
            "DUID": (
                raw_matches_df["DUID"]
                .astype("string")
                .str.strip()
                .str.upper()
            ),
            "BIDTYPE": (
                raw_matches_df["BIDTYPE"]
                .astype("string")
                .str.strip()
                .str.upper()
            ),
            "BIDSETTLEMENTDATE_KEY": raw_matches_df[
                "BIDSETTLEMENTDATE_KEY"
            ].astype("string"),
            "OFFERDATE_KEY": raw_matches_df["OFFERDATE_KEY"].astype("string"),
            "PERIODID": pd.to_numeric(
                raw_matches_df["PERIODID"],
                errors="raise",
            ).astype("int64"),
            "DIRECTION": (
                raw_matches_df["DUID"]
                .astype("string")
                .str.strip()
                .str.upper()
                .map(legacy_direction)
                .astype("string")
                .str.strip()
                .str.upper()
            ),
            "MAXAVAIL": raw_matches_df["MAXAVAIL"],
            "FIXEDLOAD": raw_matches_df["FIXEDLOAD"],
            "ROCUP": raw_matches_df["RAMPUPRATE"],
            "ROCDOWN": raw_matches_df["RAMPDOWNRATE"],
            "ENABLEMENTMIN": raw_matches_df["ENABLEMENTMIN"],
            "ENABLEMENTMAX": raw_matches_df["ENABLEMENTMAX"],
            "LOWBREAKPOINT": raw_matches_df["LOWBREAKPOINT"],
            "HIGHBREAKPOINT": raw_matches_df["HIGHBREAKPOINT"],
            "BANDAVAIL1": raw_matches_df["BANDAVAIL1"],
            "BANDAVAIL2": raw_matches_df["BANDAVAIL2"],
            "BANDAVAIL3": raw_matches_df["BANDAVAIL3"],
            "BANDAVAIL4": raw_matches_df["BANDAVAIL4"],
            "BANDAVAIL5": raw_matches_df["BANDAVAIL5"],
            "BANDAVAIL6": raw_matches_df["BANDAVAIL6"],
            "BANDAVAIL7": raw_matches_df["BANDAVAIL7"],
            "BANDAVAIL8": raw_matches_df["BANDAVAIL8"],
            "BANDAVAIL9": raw_matches_df["BANDAVAIL9"],
            "BANDAVAIL10": raw_matches_df["BANDAVAIL10"],
            "PASAAVAILABILITY": raw_matches_df["PASAAVAILABILITY"],
            "ENERGYLIMIT": None,
            "SOURCE_FILE": source_file_name,
            "SOURCE_URL": source_url,
            "STATE_SOURCE_MONTH": source_month_text,
            "STATE_ORIGIN": "CURRENT_RAW_ARCHIVE",
        }
    )

    invalid_directions = output.loc[
        ~output["DIRECTION"].isin(["GEN", "LOAD", "BIDIRECTIONAL"])
    ]

    if not invalid_directions.empty:
        raise RuntimeError(
            f"{source_month_text}: raw matches contain unmapped directions."
        )

    quality_rows = []

    for field in NUMERIC_OFFER_COLUMNS:
        text_values = output[field].astype("string").str.strip()
        source_blank = text_values.isna() | text_values.eq("")
        parsed = pd.to_numeric(output[field], errors="coerce")
        malformed = (~source_blank) & parsed.isna()

        quality_rows.append(
            {
                "FIELD_NAME": field,
                "SOURCE_BLANK_VALUES": int(source_blank.sum()),
                "VALID_NUMERIC_VALUES": int(
                    (~source_blank & ~parsed.isna()).sum()
                ),
                "MALFORMED_NONBLANK_VALUES": int(malformed.sum()),
            }
        )

        output[field] = parsed.astype("Float64")

    quality_df = pd.DataFrame(quality_rows)
    malformed_count = int(
        quality_df["MALFORMED_NONBLANK_VALUES"].sum()
    )

    if malformed_count != 0:
        display(quality_df)
        raise RuntimeError(
            f"{source_month_text}: malformed nonblank numeric values found."
        )

    return output[STATE_COLUMNS].copy(), quality_df


def existing_state_matches(current_target_csv, carry_output_parquet):
    state_glob = (STATE_DIR / "*.parquet").as_posix()
    target_csv_path = current_target_csv.as_posix()
    carry_path = carry_output_parquet.as_posix()

    con = duckdb.connect()
    con.execute("PRAGMA threads=22")

    con.execute(
        f"""
        COPY (
            SELECT state.*
            FROM read_parquet(
                '{state_glob}',
                union_by_name = true
            ) AS state
            INNER JOIN read_csv(
                '{target_csv_path}',
                header = true,
                all_varchar = true
            ) AS target
              ON state.DUID = target.DUID
             AND state.BIDTYPE = target.BIDTYPE
             AND state.BIDSETTLEMENTDATE_KEY =
                 target.BIDSETTLEMENTDATE_KEY
             AND state.OFFERDATE_KEY = target.OFFERDATE_KEY
             AND TRY_CAST(state.PERIODID AS INTEGER) =
                 TRY_CAST(target.PERIODID AS INTEGER)
        )
        TO '{carry_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    con.close()

    return pd.read_parquet(carry_output_parquet)


def append_local_manifest(record):
    with open(
        LOCAL_BATCH_MANIFEST,
        "a",
        encoding="utf-8",
    ) as output_file:
        output_file.write(json.dumps(record, default=str) + "\n")


def process_one_month(month, next_month=None):
    month_text = month.strftime("%Y-%m")
    month_compact = month.strftime("%Y%m")

    raw_match_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_{month_compact}_RAW_MATCHES.parquet"
    )
    local_stage_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_compact}_LOCAL.parquet"
    )
    origin_audit_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_compact}_ORIGIN_AUDIT.parquet"
    )
    state_month_parquet = (
        STATE_DIR / f"offer_state_{month_compact}.parquet"
    )
    current_target_csv = (
        RUN_STATE / f"{month_compact}_current_target_identities.csv"
    )
    union_target_csv = (
        RUN_STATE / f"{month_compact}_current_and_lookahead_identities.csv"
    )
    carry_match_parquet = (
        RUN_STATE / f"{month_compact}_carry_state_matches.parquet"
    )
    working_csv = (
        DUCKDB_WORKING /
        f"PUBLIC_DVD_BIDPEROFFER_{month_compact}010000.CSV"
    )
    working_part = Path(f"{working_csv}.part")

    if local_stage_parquet.exists():
        print(f"♻️ {month_text}: local stage already exists; skipping.")
        return {
            "month": month_text,
            "result": "SKIPPED_EXISTING_LOCAL_STAGE",
        }

    if state_month_parquet.exists():
        raise RuntimeError(
            f"{month_text}: state file exists but local stage does not. "
            "Nothing was overwritten."
        )

    if working_part.exists():
        raise RuntimeError(
            f"{month_text}: incomplete extraction exists: {working_part}"
        )

    free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)
    if free_gb < 70:
        raise RuntimeError(
            f"{month_text}: only {free_gb:.1f} GB free; need at least 70 GB."
        )

    print("\n" + "=" * 100)
    print(f"STARTING LOCAL MONTH: {month_text}")
    print("=" * 100)

    current_targets, expected_rows, _, _ = read_tracker_targets(month)

    if not current_targets:
        raise RuntimeError(f"{month_text}: tracker returned no targets.")

    current_identity_df = target_identity_dataframe(current_targets)
    current_identity_df.to_csv(current_target_csv, index=False)

    # Add next month's identities to the current raw scan.
    # This lets the state cache preserve any offer needed next month
    # without rescanning this source later.
    if next_month is not None:
        next_targets, _, _, _ = read_tracker_targets(next_month)
        next_identity_df = target_identity_dataframe(next_targets)

        union_identity_df = (
            pd.concat(
                [current_identity_df, next_identity_df],
                ignore_index=True,
            )
            .drop_duplicates()
            .sort_values(OFFER_KEY_COLUMNS)
            .reset_index(drop=True)
        )
    else:
        union_identity_df = current_identity_df.copy()

    union_identity_df.to_csv(union_target_csv, index=False)

    print(f"Expected effective rows: {expected_rows:,}")
    print(f"Current offer identities: {len(current_identity_df):,}")
    print(f"Current + look-ahead identities: {len(union_identity_df):,}")

    source_file_name, source_url = raw_source_urls(month)[0]

    source_zip = Path(
        _download_aemo_zip_to_cache(
            source_url,
            f"{month_text} raw source {source_file_name}",
            source_file_name=source_file_name,
        )
    )

    if not raw_match_parquet.exists():
        with zipfile.ZipFile(source_zip) as archive:
            csv_members = [
                name for name in archive.namelist()
                if name.upper().endswith(".CSV")
            ]

            if len(csv_members) != 1:
                raise RuntimeError(
                    f"{month_text}: expected one CSV; found {csv_members}"
                )

            csv_member = csv_members[0]
            csv_info = archive.getinfo(csv_member)
            expected_raw_bytes = csv_info.file_size

            data_header = None

            with archive.open(csv_member) as source_file:
                reader = csv.reader(
                    io.TextIOWrapper(
                        source_file,
                        encoding="utf-8-sig",
                        newline="",
                    )
                )

                for record in reader:
                    if (
                        record
                        and record[0].strip().upper() == "I"
                        and len(record) > 4
                        and record[3].strip() == "1"
                        and "DUID" in record[4:]
                    ):
                        data_header = record[4:]
                        break

        required_raw_fields = {
            "DUID",
            "BIDTYPE",
            "TRADINGDATE",
            "OFFERDATETIME",
            "PERIODID",
            "MAXAVAIL",
            "FIXEDLOAD",
            "RAMPUPRATE",
            "RAMPDOWNRATE",
            "ENABLEMENTMIN",
            "ENABLEMENTMAX",
            "LOWBREAKPOINT",
            "HIGHBREAKPOINT",
            "BANDAVAIL1",
            "BANDAVAIL2",
            "BANDAVAIL3",
            "BANDAVAIL4",
            "BANDAVAIL5",
            "BANDAVAIL6",
            "BANDAVAIL7",
            "BANDAVAIL8",
            "BANDAVAIL9",
            "BANDAVAIL10",
            "PASAAVAILABILITY",
        }

        if data_header is None or required_raw_fields - set(data_header):
            raise RuntimeError(
                f"{month_text}: unsupported raw-source schema. "
                "The batch stopped before writing a local stage."
            )

        if working_csv.exists():
            existing_bytes = working_csv.stat().st_size

            if existing_bytes != expected_raw_bytes:
                raise RuntimeError(
                    f"{month_text}: unexpected existing working CSV. "
                    "Nothing was overwritten."
                )

            print("♻️ Reusing a complete temporary raw CSV.")
        else:
            print(
                f"Extracting temporary raw CSV: "
                f"{expected_raw_bytes / (1024 ** 3):.2f} GB"
            )

            copied_bytes = 0
            next_notice = 5 * (1024 ** 3)

            with zipfile.ZipFile(source_zip) as archive:
                with archive.open(csv_member) as source_file:
                    with open(working_part, "wb") as destination_file:
                        while True:
                            block = source_file.read(16 * 1024 * 1024)

                            if not block:
                                break

                            destination_file.write(block)
                            copied_bytes += len(block)

                            if copied_bytes >= next_notice:
                                print(
                                    f"  {month_text}: extracted "
                                    f"{copied_bytes / (1024 ** 3):.1f} GB"
                                )
                                next_notice += 5 * (1024 ** 3)

            if copied_bytes != expected_raw_bytes:
                raise RuntimeError(
                    f"{month_text}: incomplete extraction retained for inspection."
                )

            os.replace(working_part, working_csv)

        raw_columns = [
            "RECORD_TYPE",
            "AEMO_GROUP",
            "AEMO_TABLE",
            "AEMO_VERSION",
        ] + data_header

        mapping_sql = ", ".join(
            f"{quote_identifier(name)}: 'VARCHAR'"
            for name in raw_columns
        )

        raw_data_sql = ", ".join(
            f"raw.{quote_identifier(name)}"
            for name in data_header
        )

        raw_csv_path = working_csv.as_posix()
        union_csv_path = union_target_csv.as_posix()
        raw_match_path = raw_match_parquet.as_posix()

        tradingdate_sql = """
        strftime(
            coalesce(
                try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
                try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S')
            ),
            '%Y-%m-%d %H:%M:%S.%f'
        )
        """

        offerdatetime_sql = """
        strftime(
            coalesce(
                try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
                try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
            ),
            '%Y-%m-%d %H:%M:%S.%f'
        )
        """

        con = duckdb.connect()
        con.execute("PRAGMA threads=22")

        print(f"{month_text}: DuckDB matching required offers...")

        con.execute(
            f"""
            COPY (
                SELECT
                    {raw_data_sql},
                    {tradingdate_sql} AS BIDSETTLEMENTDATE_KEY,
                    {offerdatetime_sql} AS OFFERDATE_KEY

                FROM read_csv(
                    '{raw_csv_path}',
                    header = false,
                    columns = {{{mapping_sql}}},
                    strict_mode = false,
                    null_padding = true,
                    ignore_errors = true
                ) AS raw

                INNER JOIN read_csv(
                    '{union_csv_path}',
                    header = true,
                    all_varchar = true
                ) AS target
                  ON UPPER(TRIM(raw.DUID)) = target.DUID
                 AND UPPER(TRIM(raw.BIDTYPE)) = target.BIDTYPE
                 AND {tradingdate_sql} =
                     target.BIDSETTLEMENTDATE_KEY
                 AND {offerdatetime_sql} =
                     target.OFFERDATE_KEY
                 AND TRY_CAST(raw.PERIODID AS INTEGER) =
                     TRY_CAST(target.PERIODID AS INTEGER)

                WHERE UPPER(TRIM(raw.RECORD_TYPE)) = 'D'
            )
            TO '{raw_match_path}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
            """
        )

        con.close()

    raw_matches_df = pd.read_parquet(raw_match_parquet)

    raw_state_df, numeric_quality_df = standardise_raw_offer_matches(
        raw_matches_df,
        source_file_name,
        source_url,
        month_text,
    )

    if raw_state_df.duplicated(OFFER_KEY_COLUMNS).any():
        raise RuntimeError(
            f"{month_text}: duplicate raw offer identities found."
        )

    # Current raw matches for this month.
    current_key_set = {
        tuple(row)
        for row in current_identity_df[
            OFFER_KEY_COLUMNS
        ].itertuples(index=False, name=None)
    }

    current_raw_df = raw_state_df.loc[
        raw_state_df[OFFER_KEY_COLUMNS]
        .apply(tuple, axis=1)
        .isin(current_key_set)
    ].copy()

    # Retrieve only relevant historical offers from local state.
    carry_state_df = existing_state_matches(
        current_target_csv,
        carry_match_parquet,
    )

    if not carry_state_df.empty:
        carry_state_df["STATE_ORIGIN"] = "PERSISTENT_LOCAL_STATE"

    available_offer_df = pd.concat(
        [current_raw_df, carry_state_df],
        ignore_index=True,
    )

    # Current source has priority if an identity exists in both places.
    available_offer_df = available_offer_df.drop_duplicates(
        OFFER_KEY_COLUMNS,
        keep="first",
    )

    available_key_set = {
        tuple(row)
        for row in available_offer_df[
            OFFER_KEY_COLUMNS
        ].itertuples(index=False, name=None)
    }

    unresolved_keys = current_key_set - available_key_set

    if unresolved_keys:
        display(
            pd.DataFrame(
                [
                    {
                        "DUID": key[0],
                        "BIDTYPE": key[1],
                        "BIDSETTLEMENTDATE_KEY": key[2],
                        "OFFERDATE_KEY": key[3],
                        "PERIODID": key[4],
                    }
                    for key in sorted(unresolved_keys)[:30]
                ]
            )
        )

        raise RuntimeError(
            f"{month_text}: {len(unresolved_keys):,} required offer identities "
            "were absent from both the current archive and local offer state. "
            "Temporary raw CSV was retained; no local stage was written."
        )

    interval_df = expanded_interval_dataframe(current_targets)

    if len(interval_df) != expected_rows:
        raise RuntimeError(
            f"{month_text}: tracker expansion differs from expected row count."
        )

    stage_work_df = interval_df.merge(
        available_offer_df,
        how="left",
        on=OFFER_KEY_COLUMNS,
        validate="many_to_one",
        indicator=True,
    )

    missing_intervals = int((stage_work_df["_merge"] != "both").sum())

    if missing_intervals != 0:
        raise RuntimeError(
            f"{month_text}: {missing_intervals:,} intervals have no offer."
        )

    stage_work_df = stage_work_df.drop(columns="_merge")

    stage_work_df["DIRECTION"] = (
        stage_work_df["DIRECTION"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_direction_count = int(
        (~stage_work_df["DIRECTION"].isin(
            ["GEN", "LOAD", "BIDIRECTIONAL"]
        )).sum()
    )

    if invalid_direction_count != 0:
        raise RuntimeError(
            f"{month_text}: invalid or null directions in final stage."
        )

    # Save explicit origin before selecting production fields.
    stage_work_df["OFFER_ORIGIN"] = stage_work_df["STATE_ORIGIN"]

    origin_audit_df = stage_work_df[
        [
            "INTERVAL_DATETIME",
            "DUID",
            "BIDTYPE",
            "DIRECTION",
            "OFFER_ORIGIN",
            "SOURCE_FILE",
            "SOURCE_URL",
        ]
    ].copy()

    for field in ["SETTLEMENTDATE", "INTERVAL_DATETIME"]:
        stage_work_df[field] = pd.to_datetime(
            stage_work_df[field],
            utc=True,
        )

    stage_work_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
        stage_work_df["BIDSETTLEMENTDATE_KEY"],
        utc=True,
    )
    stage_work_df["OFFERDATE"] = pd.to_datetime(
        stage_work_df["OFFERDATE_KEY"],
        utc=True,
    )
    stage_work_df["VERSIONNO"] = pd.Series(
        pd.NA,
        index=stage_work_df.index,
        dtype="Float64",
    )
    stage_work_df["LASTCHANGED"] = pd.to_datetime(
        pd.Series(pd.NaT, index=stage_work_df.index),
        utc=True,
    )
    stage_work_df["MR_CAPACITY"] = pd.Series(
        pd.NA,
        index=stage_work_df.index,
        dtype="Float64",
    )

    run_id = f"bess-effective-reconstructed-local-{month_compact}"

    stage_work_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
    stage_work_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    stage_work_df["_PIPELINE"] = (
        "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_DUCKDB_CONTINUOUS_LOCAL"
    )
    stage_work_df["_SOURCE_FILE"] = stage_work_df["SOURCE_FILE"]
    stage_work_df["_SOURCE_URL"] = stage_work_df["SOURCE_URL"]
    stage_work_df["_RUN_ID"] = run_id

    stage_df = stage_work_df[FINAL_STAGE_COLUMNS].copy()

    required_stage_columns = [
        "SETTLEMENTDATE",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "INTERVAL_DATETIME",
        "BIDSETTLEMENTDATE",
        "OFFERDATE",
        "PERIODID",
    ]

    null_required_count = int(
        stage_df[required_stage_columns].isna().sum().sum()
    )

    effective_key_columns = [
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
    ]

    distinct_effective_keys = len(
        stage_df.drop_duplicates(effective_key_columns)
    )

    if len(stage_df) != expected_rows:
        raise RuntimeError(
            f"{month_text}: stage rows differ from tracker expectation."
        )

    if distinct_effective_keys != expected_rows:
        raise RuntimeError(
            f"{month_text}: duplicate effective keys in local stage."
        )

    if null_required_count != 0:
        raise RuntimeError(
            f"{month_text}: null required values in local stage."
        )

    # Write compact local outputs only after all audits pass.
    stage_df.to_parquet(
        local_stage_parquet,
        index=False,
        compression="zstd",
    )
    origin_audit_df.to_parquet(
        origin_audit_parquet,
        index=False,
        compression="zstd",
    )

    # Save every raw identity found for this month or next month's look-ahead.
    raw_state_df.to_parquet(
        state_month_parquet,
        index=False,
        compression="zstd",
    )

    origin_summary = (
        origin_audit_df
        .groupby("OFFER_ORIGIN", dropna=False)
        .size()
        .reset_index(name="effective_rows")
    )

    append_local_manifest(
        {
            "settlement_month": month_text,
            "run_id": run_id,
            "status": "LOCAL_STAGED",
            "expected_effective_rows": int(expected_rows),
            "stage_rows": int(len(stage_df)),
            "raw_offer_identities_found": int(len(raw_state_df)),
            "current_raw_offer_identities": int(len(current_raw_df)),
            "historical_state_offer_identities": int(len(carry_state_df)),
            "stage_file": str(local_stage_parquet),
            "origin_audit_file": str(origin_audit_parquet),
            "state_file": str(state_month_parquet),
            "source_file": source_file_name,
            "source_url": source_url,
            "completed_at_utc": str(pd.Timestamp.now(tz="UTC")),
        }
    )

    LOCAL_BATCH_CHECKPOINT.write_text(
        json.dumps(
            {
                "latest_completed_month": month_text,
                "status": "LOCAL_STAGED",
                "stage_file": str(local_stage_parquet),
                "state_file": str(state_month_parquet),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    # Delete only the temporary CSV created for this successful month.
    if working_csv.exists():
        temporary_gb = working_csv.stat().st_size / (1024 ** 3)
        working_csv.unlink()
        print(
            f"✅ {month_text}: temporary raw CSV deleted "
            f"({temporary_gb:.2f} GB recovered)."
        )

    print(f"✅ {month_text}: LOCAL STAGE PASSED — {len(stage_df):,} rows")
    display(origin_summary)

    return {
        "month": month_text,
        "result": "LOCAL_STAGED",
        "stage_rows": int(len(stage_df)),
        "raw_offer_identities": int(len(raw_state_df)),
        "current_raw_identities": int(len(current_raw_df)),
        "historical_state_identities": int(len(carry_state_df)),
    }


# ---------- Run the continuous batch ----------

month_starts = list(
    pd.date_range(
        BATCH_START_MONTH,
        BATCH_END_MONTH,
        freq="MS",
    )
)

print("=" * 100)
print("STARTING RESUMABLE LOCAL BESS BACKFILL")
print(
    f"Months: {BATCH_START_MONTH.strftime('%Y-%m')} "
    f"to {BATCH_END_MONTH.strftime('%Y-%m')} "
    f"({len(month_starts)} months)"
)
print("BigQuery production remains unchanged during this batch.")
print("=" * 100)

batch_results = []

for index, month in enumerate(month_starts):
    next_month = (
        month_starts[index + 1]
        if index + 1 < len(month_starts)
        else None
    )

    try:
        result = process_one_month(month, next_month)
        batch_results.append(result)

    except Exception as error:
        batch_results.append(
            {
                "month": month.strftime("%Y-%m"),
                "result": "STOPPED_FOR_REVIEW",
                "error": str(error),
            }
        )

        print()
        print(f"⚠️ Batch stopped safely at {month.strftime('%Y-%m')}.")
        print("No later month was started.")
        print("Any temporary raw CSV for this failed month was retained.")
        break

batch_summary_df = pd.DataFrame(batch_results)
print()
print("CONTINUOUS LOCAL BACKFILL SUMMARY")
display(batch_summary_df)

print()
print("✅ Successful months have compact local stages, state files, ZIP caches,")
print("   provenance audits, checkpoints, and manifest records.")
print("✅ No BigQuery production data was changed by this batch.")

In [ ]:
# SECTION 38R — June 2022 BIDPEROFFER1 source-format diagnosis
# Read-only. Uses the retained local CSV if present; otherwise reads the cached ZIP.
# No BigQuery writes, downloads, or deletions.

from pathlib import Path
import csv
import io
import zipfile
import pandas as pd

WAREHOUSE = Path(r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse")
CACHE_ROOT = WAREHOUSE / "bess_effective_cache"
ZIP_PATH = CACHE_ROOT / "raw_aemo_zips" / "PUBLIC_DVD_BIDPEROFFER1_202206010000.zip"
CSV_PATH = (
    CACHE_ROOT / "duckdb_working"
    / "PUBLIC_DVD_BIDPEROFFER1_202206010000.CSV"
)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Cached June ZIP not found:\n{ZIP_PATH}")

def inspect_aemo_stream(text_stream, sample_data_rows=10):
    reader = csv.reader(text_stream)

    control_record = None
    header_record = None
    sample_rows = []

    for record_number, row in enumerate(reader, start=1):
        if not row:
            continue

        record_type = row[0].strip().upper()

        if record_type == "C" and control_record is None:
            control_record = row

        elif record_type == "I" and header_record is None:
            header_record = row
            source_columns = header_record[4:]

        elif record_type == "D":
            if header_record is None:
                raise RuntimeError(
                    "Found data records before the AEMO I/header record."
                )

            values = row[4:]
            sample_rows.append(values)

            if len(sample_rows) >= sample_data_rows:
                break

    if header_record is None:
        raise RuntimeError("No AEMO I/header record was found.")

    if not sample_rows:
        raise RuntimeError("No AEMO D/data records were found.")

    sample_df = pd.DataFrame(sample_rows, columns=source_columns)

    return control_record, header_record, sample_df


print("=" * 100)
print("JUNE 2022 BIDPEROFFER1 FORMAT DIAGNOSIS")
print("=" * 100)
print(f"Cached ZIP: {ZIP_PATH.name}")
print(f"Retained working CSV exists: {CSV_PATH.exists()}")

if CSV_PATH.exists():
    print(f"Reading first records from retained CSV: {CSV_PATH.name}")

    with CSV_PATH.open(
        "r",
        encoding="utf-8-sig",
        newline="",
        errors="replace",
    ) as source_file:
        control, header, june_sample = inspect_aemo_stream(source_file)

else:
    print("Retained CSV was not found; reading first records directly from cached ZIP.")

    with zipfile.ZipFile(ZIP_PATH) as archive:
        csv_members = [
            member
            for member in archive.namelist()
            if member.upper().endswith(".CSV")
        ]

        if len(csv_members) != 1:
            raise RuntimeError(
                f"Expected exactly one CSV inside the ZIP; found: {csv_members}"
            )

        csv_member = csv_members[0]
        print(f"CSV inside ZIP: {csv_member}")

        with archive.open(csv_member) as binary_file:
            with io.TextIOWrapper(
                binary_file,
                encoding="utf-8-sig",
                newline="",
                errors="replace",
            ) as source_file:
                control, header, june_sample = inspect_aemo_stream(source_file)

print("\nControl record:")
print(control)

print("\nHeader record:")
print(header)

print("\nSource columns:")
display(pd.DataFrame({"SOURCE_COLUMN": header[4:]}))

important_names = {
    "DUID",
    "BIDTYPE",
    "SETTLEMENTDATE",
    "TRADINGDATE",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "OFFERDATETIME",
    "PERIODID",
    "VERSIONNO",
    "INTERVAL_DATETIME",
    "LASTCHANGED",
}

present_key_columns = [
    column for column in june_sample.columns
    if column.strip().upper() in important_names
]

print("\nKey/timestamp columns present:")
print(present_key_columns)

print("\nFirst 10 June data records — key fields only:")
display(june_sample[present_key_columns])

print("\nAll source columns in sample:")
print(list(june_sample.columns))

print("\n✅ JUNE 2022 FORMAT DIAGNOSIS COMPLETE")
print("✅ Nothing was changed.")

In [ ]:
# SECTION 38S — Discover June 2022 BIDPEROFFER1 companion archive
# Read-only. No BigQuery writes, no ZIP downloads, no deletions.

import requests
import pandas as pd

BASE_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2022/MMSDM_2022_06/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

candidate_names = [
    "PUBLIC_DVD_BIDPEROFFER2_202206010000.zip",
    "PUBLIC_DVD_BIDPEROFFER2_202206160000.zip",
    "PUBLIC_DVD_BIDPEROFFER_202206010000.zip",
    "PUBLIC_DVD_BIDPEROFFER_202206160000.zip",
    "PUBLIC_DVD_BIDOFFERPERIOD_202206010000.zip",
    "PUBLIC_DVD_BIDOFFERPERIOD_202206160000.zip",
    "PUBLIC_ARCHIVE#BIDPEROFFER#FILE02#202206010000.zip",
    "PUBLIC_ARCHIVE#BIDPEROFFER1#FILE02#202206010000.zip",
    "PUBLIC_ARCHIVE#BIDOFFERPERIOD#FILE02#202206010000.zip",
]

results = []

for file_name in candidate_names:
    url = BASE_URL + file_name

    try:
        response = requests.get(
            url,
            headers={"Range": "bytes=0-0", "User-Agent": "Mozilla/5.0"},
            timeout=60,
            stream=True,
        )

        available = response.status_code in (200, 206)

        results.append(
            {
                "SOURCE_FILE": file_name,
                "HTTP_STATUS": response.status_code,
                "AVAILABLE": available,
                "CONTENT_LENGTH": response.headers.get("Content-Length"),
                "CONTENT_RANGE": response.headers.get("Content-Range"),
                "SOURCE_URL": url,
            }
        )

    except Exception as error:
        results.append(
            {
                "SOURCE_FILE": file_name,
                "HTTP_STATUS": None,
                "AVAILABLE": False,
                "CONTENT_LENGTH": None,
                "CONTENT_RANGE": None,
                "SOURCE_URL": url,
                "ERROR": str(error),
            }
        )

    finally:
        if "response" in locals():
            response.close()

availability_df = pd.DataFrame(results)

print("JUNE 2022 COMPANION-ARCHIVE AVAILABILITY")
display(
    availability_df[
        [
            "SOURCE_FILE",
            "HTTP_STATUS",
            "AVAILABLE",
            "CONTENT_LENGTH",
            "CONTENT_RANGE",
        ]
    ]
)

available = availability_df[availability_df["AVAILABLE"]].copy()

print(f"\nAvailable candidates found: {len(available):,}")

if available.empty:
    print("⚠️ No candidate selected yet. Nothing has been downloaded or changed.")
else:
    print("✅ Candidate archive(s) found. Do not download them yet.")
    display(available[["SOURCE_FILE", "SOURCE_URL"]])


In [ ]:
# SECTION 38T — Cache and inspect June 2022 BIDPEROFFER2 companion
# Local ZIP-cache write only. No BigQuery activity and no CSV extraction.

from pathlib import Path
import csv
import io
import os
import time
import zipfile
import requests
import pandas as pd

WAREHOUSE = Path(r"C:\Users\carme\OneDrive\Desktop\AEMO Warehouse")
RAW_ZIP_CACHE = WAREHOUSE / "bess_effective_cache" / "raw_aemo_zips"
RAW_ZIP_CACHE.mkdir(parents=True, exist_ok=True)

JUNE_PART2_NAME = "PUBLIC_DVD_BIDPEROFFER2_202206010000.zip"
JUNE_PART2_URL = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/2022/MMSDM_2022_06/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
    + JUNE_PART2_NAME
)

part2_zip = RAW_ZIP_CACHE / JUNE_PART2_NAME
part2_partial = RAW_ZIP_CACHE / f"{JUNE_PART2_NAME}.part"

def valid_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            return archive.testzip() is None
    except (zipfile.BadZipFile, OSError):
        return False

if part2_zip.exists() and valid_zip(part2_zip):
    size_mb = part2_zip.stat().st_size / (1024 ** 2)
    print(f"♻️ Using cached ZIP: {part2_zip.name} ({size_mb:,.1f} MB)")

else:
    if part2_partial.exists():
        part2_partial.unlink()

    print(f"Downloading June companion source: {JUNE_PART2_NAME}")

    last_error = None

    for attempt in range(1, 6):
        try:
            with requests.get(
                JUNE_PART2_URL,
                headers={"User-Agent": "Mozilla/5.0"},
                stream=True,
                timeout=(30, 300),
            ) as response:
                response.raise_for_status()

                expected_bytes = int(
                    response.headers.get("Content-Length", 0)
                )
                downloaded_bytes = 0
                next_report = 256 * 1024 ** 2

                with part2_partial.open("wb") as output_file:
                    for chunk in response.iter_content(
                        chunk_size=8 * 1024 ** 2
                    ):
                        if not chunk:
                            continue

                        output_file.write(chunk)
                        downloaded_bytes += len(chunk)

                        if downloaded_bytes >= next_report:
                            print(
                                f"  Downloaded "
                                f"{downloaded_bytes / (1024 ** 2):,.0f} MB"
                            )
                            next_report += 256 * 1024 ** 2

                if expected_bytes and downloaded_bytes != expected_bytes:
                    raise RuntimeError(
                        "Incomplete download: "
                        f"expected {expected_bytes:,} bytes, "
                        f"received {downloaded_bytes:,}."
                    )

            if not valid_zip(part2_partial):
                raise RuntimeError("Downloaded file failed ZIP validation.")

            os.replace(part2_partial, part2_zip)

            print(
                f"✅ Cached ZIP: {part2_zip.name} "
                f"({part2_zip.stat().st_size / (1024 ** 2):,.1f} MB)"
            )
            break

        except Exception as error:
            last_error = error

            if part2_partial.exists():
                part2_partial.unlink()

            if attempt == 5:
                raise RuntimeError(
                    f"Could not cache {JUNE_PART2_NAME} after 5 attempts."
                ) from last_error

            wait_seconds = 5 * attempt
            print(
                f"⚠️ Download attempt {attempt}/5 failed; "
                f"retrying in {wait_seconds}s..."
            )
            time.sleep(wait_seconds)

print("\nInspecting the companion ZIP; no extraction will occur.")

with zipfile.ZipFile(part2_zip) as archive:
    csv_members = [
        name for name in archive.namelist()
        if name.upper().endswith(".CSV")
    ]

    if len(csv_members) != 1:
        raise RuntimeError(
            f"Expected one CSV inside {JUNE_PART2_NAME}; found: {csv_members}"
        )

    csv_member = csv_members[0]

    with archive.open(csv_member) as binary_file:
        with io.TextIOWrapper(
            binary_file,
            encoding="utf-8-sig",
            newline="",
            errors="replace",
        ) as source_file:
            reader = csv.reader(source_file)

            control_record = None
            header_record = None
            sample_rows = []

            for row in reader:
                if not row:
                    continue

                if row[0].strip().upper() == "C" and control_record is None:
                    control_record = row

                elif row[0].strip().upper() == "I" and header_record is None:
                    header_record = row
                    source_columns = header_record[4:]

                elif row[0].strip().upper() == "D":
                    sample_rows.append(row[4:])

                    if len(sample_rows) == 10:
                        break

if header_record is None or not sample_rows:
    raise RuntimeError("Could not locate valid AEMO header and data records.")

sample_df = pd.DataFrame(sample_rows, columns=source_columns)

key_columns = [
    column for column in [
        "DUID",
        "BIDTYPE",
        "TRADINGDATE",
        "OFFERDATETIME",
        "PERIODID",
    ]
    if column in sample_df.columns
]

print("\nJUNE 2022 BIDPEROFFER2 INSPECTION")
print(f"CSV inside ZIP: {csv_member}")
print("Control record:")
print(control_record)

print("\nSource columns:")
display(pd.DataFrame({"SOURCE_COLUMN": source_columns}))

print("\nFirst 10 data records — matching fields:")
display(sample_df[key_columns])

print("\n✅ June companion ZIP is cached and ready.")
print("✅ No CSV was extracted and no BigQuery data was changed.")

In [ ]:
# SECTION 38U — Repair June 2022 split BIDPEROFFER archive locally
# Uses:
#   - BIDPEROFFER1: 1–15 June
#   - BIDPEROFFER2: 16–30 June
#
# Local-only. No BigQuery writes, promotions, manifests, or deletions.
# It creates the normal compact June local stage/state outputs only after
# every identity, direction, numeric, and effective-key audit passes.

import csv
import io
import json
import os
import shutil
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

JUNE_MONTH = pd.Timestamp("2022-06-01")
JULY_MONTH = pd.Timestamp("2022-07-01")
JUNE_TEXT = "2022-06"
JUNE_COMPACT = "202206"

if "read_tracker_targets" not in globals():
    raise RuntimeError("read_tracker_targets() is unavailable.")
if "standardise_raw_offer_matches" not in globals():
    raise RuntimeError("Run Section 38Q first so its helpers are available.")
if "existing_state_matches" not in globals():
    raise RuntimeError("Run Section 38Q first so its helpers are available.")
if "expanded_interval_dataframe" not in globals():
    raise RuntimeError("Run Section 38Q first so its helpers are available.")
if "target_identity_dataframe" not in globals():
    raise RuntimeError("Run Section 38Q first so its helpers are available.")
if "_download_aemo_zip_to_cache" not in globals():
    raise RuntimeError("Cached downloader is unavailable.")

JUNE_PARTS = [
    {
        "file_name": "PUBLIC_DVD_BIDPEROFFER1_202206010000.zip",
        "url": (
            "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
            "MMSDM/2022/MMSDM_2022_06/"
            "MMSDM_Historical_Data_SQLLoader/DATA/"
            "PUBLIC_DVD_BIDPEROFFER1_202206010000.zip"
        ),
    },
    {
        "file_name": "PUBLIC_DVD_BIDPEROFFER2_202206010000.zip",
        "url": (
            "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
            "MMSDM/2022/MMSDM_2022_06/"
            "MMSDM_Historical_Data_SQLLoader/DATA/"
            "PUBLIC_DVD_BIDPEROFFER2_202206010000.zip"
        ),
    },
]

split_match_parquet = (
    PARQUET_CACHE /
    "BIDPEROFFER_202206_SPLIT_RAW_MATCHES.parquet"
)
local_stage_parquet = (
    PARQUET_CACHE /
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202206_LOCAL.parquet"
)
origin_audit_parquet = (
    PARQUET_CACHE /
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202206_ORIGIN_AUDIT.parquet"
)
state_month_parquet = (
    STATE_DIR / "offer_state_202206.parquet"
)
current_target_csv = (
    RUN_STATE / "202206_current_target_identities.csv"
)
union_target_csv = (
    RUN_STATE / "202206_current_and_lookahead_identities.csv"
)
carry_match_parquet = (
    RUN_STATE / "202206_split_carry_state_matches.parquet"
)

if local_stage_parquet.exists():
    raise RuntimeError(
        "June local stage already exists. Nothing was overwritten."
    )

if state_month_parquet.exists():
    raise RuntimeError(
        "June state file exists but June local stage does not. "
        "Nothing was overwritten."
    )

free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)

if free_gb < 140:
    raise RuntimeError(
        f"Only {free_gb:.1f} GB free. At least 140 GB is required "
        "to safely process both June source halves."
    )

print("=" * 100)
print("REPAIRING JUNE 2022 SPLIT BIDPEROFFER ARCHIVE")
print("=" * 100)
print(f"Free SSD space: {free_gb:.1f} GB")
print("BigQuery remains unchanged.")
print("June sources:")
for item in JUNE_PARTS:
    print(f" - {item['file_name']}")

current_targets, expected_rows, _, _ = read_tracker_targets(JUNE_MONTH)
july_targets, _, _, _ = read_tracker_targets(JULY_MONTH)

if not current_targets:
    raise RuntimeError("June tracker returned no targets.")

current_identity_df = target_identity_dataframe(current_targets)
july_identity_df = target_identity_dataframe(july_targets)

union_identity_df = (
    pd.concat(
        [current_identity_df, july_identity_df],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(OFFER_KEY_COLUMNS)
    .reset_index(drop=True)
)

current_identity_df.to_csv(current_target_csv, index=False)
union_identity_df.to_csv(union_target_csv, index=False)

print(f"Expected June effective rows: {expected_rows:,}")
print(f"June offer identities: {len(current_identity_df):,}")
print(f"June + July look-ahead identities: {len(union_identity_df):,}")

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def source_schema_from_zip(source_zip, source_label):
    with zipfile.ZipFile(source_zip) as archive:
        csv_members = [
            name for name in archive.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_members) != 1:
            raise RuntimeError(
                f"{source_label}: expected exactly one CSV; found {csv_members}"
            )

        csv_member = csv_members[0]
        csv_info = archive.getinfo(csv_member)
        data_header = None

        with archive.open(csv_member) as binary_file:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_file,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            for record in reader:
                if (
                    record
                    and record[0].strip().upper() == "I"
                    and len(record) > 4
                    and "DUID" in record[4:]
                ):
                    data_header = record[4:]
                    break

    required_fields = {
        "DUID",
        "BIDTYPE",
        "TRADINGDATE",
        "OFFERDATETIME",
        "PERIODID",
        "MAXAVAIL",
        "FIXEDLOAD",
        "RAMPUPRATE",
        "RAMPDOWNRATE",
        "ENABLEMENTMIN",
        "ENABLEMENTMAX",
        "LOWBREAKPOINT",
        "HIGHBREAKPOINT",
        "BANDAVAIL1",
        "BANDAVAIL2",
        "BANDAVAIL3",
        "BANDAVAIL4",
        "BANDAVAIL5",
        "BANDAVAIL6",
        "BANDAVAIL7",
        "BANDAVAIL8",
        "BANDAVAIL9",
        "BANDAVAIL10",
        "PASAAVAILABILITY",
    }

    if data_header is None or required_fields - set(data_header):
        raise RuntimeError(
            f"{source_label}: unsupported source schema."
        )

    return csv_member, csv_info.file_size, data_header


def extract_zip_csv(source_zip, csv_member, expected_bytes, working_csv):
    working_part = Path(f"{working_csv}.part")

    if working_part.exists():
        raise RuntimeError(
            f"Incomplete extraction exists and was preserved:\n{working_part}"
        )

    if working_csv.exists():
        if working_csv.stat().st_size != expected_bytes:
            raise RuntimeError(
                f"Unexpected existing working CSV was preserved:\n{working_csv}"
            )

        print(f"♻️ Reusing complete temporary CSV: {working_csv.name}")
        return

    print(
        f"Extracting {working_csv.name}: "
        f"{expected_bytes / (1024 ** 3):.2f} GB"
    )

    copied_bytes = 0
    next_notice = 5 * (1024 ** 3)

    with zipfile.ZipFile(source_zip) as archive:
        with archive.open(csv_member) as source_file:
            with working_part.open("wb") as destination_file:
                while True:
                    block = source_file.read(16 * 1024 * 1024)

                    if not block:
                        break

                    destination_file.write(block)
                    copied_bytes += len(block)

                    if copied_bytes >= next_notice:
                        print(
                            f"  extracted "
                            f"{copied_bytes / (1024 ** 3):.1f} GB"
                        )
                        next_notice += 5 * (1024 ** 3)

    if copied_bytes != expected_bytes:
        raise RuntimeError(
            f"Extraction did not complete; partial file was retained:\n"
            f"{working_part}"
        )

    os.replace(working_part, working_csv)


def duckdb_match_part(
    working_csv,
    data_header,
    source_file_name,
    source_url,
    part_match_parquet,
):
    raw_columns = [
        "RECORD_TYPE",
        "AEMO_GROUP",
        "AEMO_TABLE",
        "AEMO_VERSION",
    ] + data_header

    mapping_sql = ", ".join(
        f'{quote_identifier(name)}: \'VARCHAR\''
        for name in raw_columns
    )

    raw_data_sql = ", ".join(
        f"raw.{quote_identifier(name)}"
        for name in data_header
    )

    tradingdate_sql = """
    strftime(
        coalesce(
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    offerdatetime_sql = """
    strftime(
        coalesce(
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    con = duckdb.connect()
    con.execute("PRAGMA threads=22")

    print(f"DuckDB matching required offers in {source_file_name}...")

    try:
        con.execute(
            f"""
            COPY (
                SELECT
                    {raw_data_sql},
                    {tradingdate_sql} AS BIDSETTLEMENTDATE_KEY,
                    {offerdatetime_sql} AS OFFERDATE_KEY,
                    {sql_literal(source_file_name)} AS SOURCE_FILE,
                    {sql_literal(source_url)} AS SOURCE_URL

                FROM read_csv(
                    '{working_csv.as_posix()}',
                    header = false,
                    columns = {{{mapping_sql}}},
                    strict_mode = false,
                    null_padding = true,
                    ignore_errors = true
                ) AS raw

                INNER JOIN read_csv(
                    '{union_target_csv.as_posix()}',
                    header = true,
                    all_varchar = true
                ) AS target
                  ON UPPER(TRIM(raw.DUID)) = target.DUID
                 AND UPPER(TRIM(raw.BIDTYPE)) = target.BIDTYPE
                 AND {tradingdate_sql} =
                     target.BIDSETTLEMENTDATE_KEY
                 AND {offerdatetime_sql} =
                     target.OFFERDATE_KEY
                 AND TRY_CAST(raw.PERIODID AS INTEGER) =
                     TRY_CAST(target.PERIODID AS INTEGER)

                WHERE UPPER(TRIM(raw.RECORD_TYPE)) = 'D'
            )
            TO '{part_match_parquet.as_posix()}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
            """
        )
    finally:
        con.close()

    return pd.read_parquet(part_match_parquet)


if split_match_parquet.exists():
    print("♻️ Reusing previously created combined June split-match Parquet.")
    raw_matches_df = pd.read_parquet(split_match_parquet)

    expected_source_files = {
        item["file_name"] for item in JUNE_PARTS
    }

    if (
        "SOURCE_FILE" not in raw_matches_df.columns
        or set(raw_matches_df["SOURCE_FILE"].dropna().unique())
        != expected_source_files
    ):
        raise RuntimeError(
            "Existing combined June match file does not prove that both "
            "source halves were used. Nothing was overwritten."
        )

else:
    part_frames = []
    temporary_csvs = []

    for part_number, item in enumerate(JUNE_PARTS, start=1):
        source_zip = Path(
            _download_aemo_zip_to_cache(
                item["url"],
                f"June split source {item['file_name']}",
                source_file_name=item["file_name"],
            )
        )

        csv_member, expected_raw_bytes, data_header = source_schema_from_zip(
            source_zip,
            item["file_name"],
        )

        working_csv = (
            DUCKDB_WORKING /
            f"{Path(csv_member).stem}.CSV"
        )
        temporary_csvs.append(working_csv)

        extract_zip_csv(
            source_zip,
            csv_member,
            expected_raw_bytes,
            working_csv,
        )

        part_match_parquet = (
            RUN_STATE /
            f"202206_split_part{part_number}_raw_matches.parquet"
        )

        if part_match_parquet.exists():
            part_match_parquet.unlink()

        part_frames.append(
            duckdb_match_part(
                working_csv,
                data_header,
                item["file_name"],
                item["url"],
                part_match_parquet,
            )
        )

    raw_matches_df = pd.concat(
        part_frames,
        ignore_index=True,
    )

    if raw_matches_df.empty:
        raise RuntimeError(
            "June split source scan found no required offers."
        )

    raw_matches_df.to_parquet(
        split_match_parquet,
        index=False,
        compression="zstd",
    )

    print(
        f"✅ Combined June raw-match file written: "
        f"{len(raw_matches_df):,} rows"
    )

# Standardise each source independently so row-level provenance is preserved.
part_state_frames = []
quality_frames = []

for item in JUNE_PARTS:
    part_raw_df = raw_matches_df.loc[
        raw_matches_df["SOURCE_FILE"].eq(item["file_name"])
    ].copy()

    if part_raw_df.empty:
        raise RuntimeError(
            f"No required raw offers were found in {item['file_name']}."
        )

    standardised_part_df, numeric_quality_df = (
        standardise_raw_offer_matches(
            part_raw_df,
            item["file_name"],
            item["url"],
            JUNE_TEXT,
        )
    )

    numeric_quality_df["SOURCE_FILE"] = item["file_name"]
    part_state_frames.append(standardised_part_df)
    quality_frames.append(numeric_quality_df)

raw_state_df = pd.concat(part_state_frames, ignore_index=True)
numeric_quality_summary = pd.concat(
    quality_frames,
    ignore_index=True,
)

if raw_state_df.duplicated(OFFER_KEY_COLUMNS).any():
    duplicate_examples = raw_state_df.loc[
        raw_state_df.duplicated(OFFER_KEY_COLUMNS, keep=False),
        OFFER_KEY_COLUMNS + ["SOURCE_FILE"],
    ].head(20)

    display(duplicate_examples)

    raise RuntimeError(
        "Duplicate offer identities were found across the June source halves."
    )

current_key_set = {
    tuple(row)
    for row in current_identity_df[
        OFFER_KEY_COLUMNS
    ].itertuples(index=False, name=None)
}

current_raw_df = raw_state_df.loc[
    raw_state_df[OFFER_KEY_COLUMNS]
    .apply(tuple, axis=1)
    .isin(current_key_set)
].copy()

if carry_match_parquet.exists():
    carry_match_parquet.unlink()

carry_state_df = existing_state_matches(
    current_target_csv,
    carry_match_parquet,
)

if not carry_state_df.empty:
    carry_state_df["STATE_ORIGIN"] = "PERSISTENT_LOCAL_STATE"

available_offer_df = pd.concat(
    [current_raw_df, carry_state_df],
    ignore_index=True,
).drop_duplicates(
    OFFER_KEY_COLUMNS,
    keep="first",
)

available_key_set = {
    tuple(row)
    for row in available_offer_df[
        OFFER_KEY_COLUMNS
    ].itertuples(index=False, name=None)
}

unresolved_keys = current_key_set - available_key_set

if unresolved_keys:
    display(
        pd.DataFrame(
            [
                {
                    "DUID": key[0],
                    "BIDTYPE": key[1],
                    "BIDSETTLEMENTDATE_KEY": key[2],
                    "OFFERDATE_KEY": key[3],
                    "PERIODID": key[4],
                }
                for key in sorted(unresolved_keys)[:30]
            ]
        )
    )

    raise RuntimeError(
        f"June still has {len(unresolved_keys):,} unresolved identities. "
        "Temporary CSVs were retained; no June stage was written."
    )

interval_df = expanded_interval_dataframe(current_targets)

if len(interval_df) != expected_rows:
    raise RuntimeError(
        "June tracker expansion differs from its expected row count."
    )

stage_work_df = interval_df.merge(
    available_offer_df,
    how="left",
    on=OFFER_KEY_COLUMNS,
    validate="many_to_one",
    indicator=True,
)

missing_interval_count = int(
    (stage_work_df["_merge"] != "both").sum()
)

if missing_interval_count != 0:
    raise RuntimeError(
        f"{missing_interval_count:,} June effective intervals have no offer."
    )

stage_work_df = stage_work_df.drop(columns="_merge")

stage_work_df["DIRECTION"] = (
    stage_work_df["DIRECTION"]
    .astype("string")
    .str.strip()
    .str.upper()
)

invalid_direction_count = int(
    (~stage_work_df["DIRECTION"].isin(
        ["GEN", "LOAD", "BIDIRECTIONAL"]
    )).sum()
)

if invalid_direction_count != 0:
    raise RuntimeError(
        "June final stage contains invalid or null directions."
    )

stage_work_df["OFFER_ORIGIN"] = stage_work_df["STATE_ORIGIN"]

origin_audit_df = stage_work_df[
    [
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "OFFER_ORIGIN",
        "SOURCE_FILE",
        "SOURCE_URL",
    ]
].copy()

for field in ["SETTLEMENTDATE", "INTERVAL_DATETIME"]:
    stage_work_df[field] = pd.to_datetime(
        stage_work_df[field],
        utc=True,
    )

stage_work_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
    stage_work_df["BIDSETTLEMENTDATE_KEY"],
    utc=True,
)
stage_work_df["OFFERDATE"] = pd.to_datetime(
    stage_work_df["OFFERDATE_KEY"],
    utc=True,
)
stage_work_df["VERSIONNO"] = pd.Series(
    pd.NA,
    index=stage_work_df.index,
    dtype="Float64",
)
stage_work_df["LASTCHANGED"] = pd.to_datetime(
    pd.Series(pd.NaT, index=stage_work_df.index),
    utc=True,
)
stage_work_df["MR_CAPACITY"] = pd.Series(
    pd.NA,
    index=stage_work_df.index,
    dtype="Float64",
)

run_id = "bess-effective-reconstructed-local-202206"

stage_work_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
stage_work_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
stage_work_df["_PIPELINE"] = (
    "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_DUCKDB_CONTINUOUS_LOCAL"
)
stage_work_df["_SOURCE_FILE"] = stage_work_df["SOURCE_FILE"]
stage_work_df["_SOURCE_URL"] = stage_work_df["SOURCE_URL"]
stage_work_df["_RUN_ID"] = run_id

stage_df = stage_work_df[FINAL_STAGE_COLUMNS].copy()

required_stage_columns = [
    "SETTLEMENTDATE",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
    "INTERVAL_DATETIME",
    "BIDSETTLEMENTDATE",
    "OFFERDATE",
    "PERIODID",
]

effective_key_columns = [
    "INTERVAL_DATETIME",
    "DUID",
    "BIDTYPE",
    "DIRECTION",
]

null_required_count = int(
    stage_df[required_stage_columns].isna().sum().sum()
)
distinct_effective_keys = len(
    stage_df.drop_duplicates(effective_key_columns)
)

audit_df = pd.DataFrame(
    [
        {
            "expected_stage_rows": expected_rows,
            "stage_rows": len(stage_df),
            "distinct_effective_keys": distinct_effective_keys,
            "null_required_field_count": null_required_count,
            "invalid_direction_count": invalid_direction_count,
            "first_interval": stage_df["INTERVAL_DATETIME"].min(),
            "last_interval": stage_df["INTERVAL_DATETIME"].max(),
        }
    ]
)

display(audit_df)

if len(stage_df) != expected_rows:
    raise RuntimeError("June stage row count differs from tracker expectation.")
if distinct_effective_keys != expected_rows:
    raise RuntimeError("June stage contains duplicate effective keys.")
if null_required_count != 0:
    raise RuntimeError("June stage contains null required values.")

# Write local outputs only after every audit has passed.
stage_df.to_parquet(
    local_stage_parquet,
    index=False,
    compression="zstd",
)
origin_audit_df.to_parquet(
    origin_audit_parquet,
    index=False,
    compression="zstd",
)
raw_state_df.to_parquet(
    state_month_parquet,
    index=False,
    compression="zstd",
)

origin_summary = (
    origin_audit_df
    .groupby("OFFER_ORIGIN", dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

source_summary = (
    origin_audit_df
    .groupby(["OFFER_ORIGIN", "SOURCE_FILE"], dropna=False)
    .size()
    .reset_index(name="effective_rows")
)

append_local_manifest(
    {
        "settlement_month": JUNE_TEXT,
        "run_id": run_id,
        "status": "LOCAL_STAGED_SPLIT_ARCHIVE_REPAIR",
        "expected_effective_rows": int(expected_rows),
        "stage_rows": int(len(stage_df)),
        "raw_offer_identities_found": int(len(raw_state_df)),
        "current_raw_offer_identities": int(len(current_raw_df)),
        "historical_state_offer_identities": int(len(carry_state_df)),
        "stage_file": str(local_stage_parquet),
        "origin_audit_file": str(origin_audit_parquet),
        "state_file": str(state_month_parquet),
        "source_files": [item["file_name"] for item in JUNE_PARTS],
        "source_urls": [item["url"] for item in JUNE_PARTS],
        "completed_at_utc": str(pd.Timestamp.now(tz="UTC")),
    }
)

LOCAL_BATCH_CHECKPOINT.write_text(
    json.dumps(
        {
            "latest_completed_month": JUNE_TEXT,
            "status": "LOCAL_STAGED_SPLIT_ARCHIVE_REPAIR",
            "stage_file": str(local_stage_parquet),
            "state_file": str(state_month_parquet),
        },
        indent=2,
    ),
    encoding="utf-8",
)

# Delete only temporary CSVs after the complete June stage has passed.
for item in JUNE_PARTS:
    csv_name = Path(
        item["file_name"].replace(".zip", ".CSV")
    ).name
    working_csv = DUCKDB_WORKING / csv_name

    if working_csv.exists():
        recovered_gb = working_csv.stat().st_size / (1024 ** 3)
        working_csv.unlink()
        print(
            f"✅ Deleted successful temporary CSV: "
            f"{working_csv.name} ({recovered_gb:.2f} GB recovered)"
        )

print("\nJUNE 2022 SPLIT-ARCHIVE REPAIR COMPLETE")
display(origin_summary)
display(source_summary)

print("\n✅ June local stage passed.")
print("✅ June state is ready for July.")
print("✅ Checkpoint advanced to June 2022.")
print("✅ No BigQuery data was changed.")

In [ ]:
# SECTION 38V — Check which remaining months use split BIDPEROFFER archives
# Read-only: checks one byte per possible file. No downloads, extraction,
# BigQuery writes, local-stage changes, or deletions.

import requests
import pandas as pd

CHECK_START = pd.Timestamp("2022-07-01")
CHECK_END = pd.Timestamp("2024-07-01")

BASE_URL_TEMPLATE = (
    "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
    "MMSDM/{year}/MMSDM_{year}_{month:02d}/"
    "MMSDM_Historical_Data_SQLLoader/DATA/"
)

months_to_check = pd.date_range(
    CHECK_START,
    CHECK_END,
    freq="MS",
)

rows = []

with requests.Session() as session:
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    for month_start in months_to_check:
        year = month_start.year
        month_number = month_start.month
        month_text = month_start.strftime("%Y-%m")

        base_url = BASE_URL_TEMPLATE.format(
            year=year,
            month=month_number,
        )

        for archive_part in [1, 2]:
            file_name = (
                f"PUBLIC_DVD_BIDPEROFFER{archive_part}_"
                f"{year}{month_number:02d}010000.zip"
            )
            source_url = base_url + file_name

            try:
                response = session.get(
                    source_url,
                    headers={"Range": "bytes=0-0"},
                    timeout=60,
                    stream=True,
                )

                rows.append(
                    {
                        "SETTLEMENT_MONTH": month_text,
                        "ARCHIVE_PART": archive_part,
                        "SOURCE_FILE": file_name,
                        "HTTP_STATUS": response.status_code,
                        "AVAILABLE": response.status_code in (200, 206),
                        "SOURCE_URL": source_url,
                    }
                )

                response.close()

            except Exception as error:
                rows.append(
                    {
                        "SETTLEMENT_MONTH": month_text,
                        "ARCHIVE_PART": archive_part,
                        "SOURCE_FILE": file_name,
                        "HTTP_STATUS": None,
                        "AVAILABLE": False,
                        "SOURCE_URL": source_url,
                        "ERROR": str(error),
                    }
                )

split_inventory_df = pd.DataFrame(rows)

summary_df = (
    split_inventory_df
    .pivot_table(
        index="SETTLEMENT_MONTH",
        columns="ARCHIVE_PART",
        values="AVAILABLE",
        aggfunc="first",
        fill_value=False,
    )
    .rename(
        columns={
            1: "PART1_AVAILABLE",
            2: "PART2_AVAILABLE",
        }
    )
    .reset_index()
)

print("REMAINING BIDPEROFFER SPLIT-ARCHIVE INVENTORY")
display(summary_df)

print("\nMonths with a confirmed Part 2 companion:")
display(
    split_inventory_df.loc[
        split_inventory_df["ARCHIVE_PART"].eq(2)
        & split_inventory_df["AVAILABLE"],
        [
            "SETTLEMENT_MONTH",
            "SOURCE_FILE",
            "HTTP_STATUS",
        ],
    ].reset_index(drop=True)
)

print("\n✅ Inventory complete. Nothing was downloaded or changed.")

In [ ]:
# SECTION 39A — Split-archive continuous backfill preflight
# Scope: July 2022 to July 2024.
# Read-only: no downloads, BigQuery writes, file writes, or deletions.

from pathlib import Path
import shutil
import pandas as pd

SECTION_39_START = pd.Timestamp("2022-07-01")
SECTION_39_END = pd.Timestamp("2024-07-01")

required_globals = [
    "read_tracker_targets",
    "target_identity_dataframe",
    "expanded_interval_dataframe",
    "standardise_raw_offer_matches",
    "existing_state_matches",
    "legacy_direction",
    "_download_aemo_zip_to_cache",
    "quote_identifier",
    "append_local_manifest",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Required earlier notebook helpers are unavailable: "
        f"{missing_globals}. Do not continue until they are restored."
    )

june_stage = (
    PARQUET_CACHE /
    "BIDPEROFFER_BESS_EFFECTIVE_STAGE_202206_LOCAL.parquet"
)
june_state = STATE_DIR / "offer_state_202206.parquet"
june_split_matches = (
    PARQUET_CACHE /
    "BIDPEROFFER_202206_SPLIT_RAW_MATCHES.parquet"
)

required_files = [
    june_stage,
    june_state,
    june_split_matches,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise RuntimeError(
        "June 2022 repair artifacts are missing. Nothing was changed:\n"
        + "\n".join(f" - {path}" for path in missing_files)
    )

existing_stage_files = sorted(
    PARQUET_CACHE.glob(
        "BIDPEROFFER_BESS_EFFECTIVE_STAGE_*_LOCAL.parquet"
    )
)

existing_months = sorted(
    path.name
    .replace("BIDPEROFFER_BESS_EFFECTIVE_STAGE_", "")
    .replace("_LOCAL.parquet", "")
    for path in existing_stage_files
)

free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)

if free_gb < 140:
    raise RuntimeError(
        f"Only {free_gb:.1f} GB free. Section 39 needs at least 140 GB "
        "for two temporary raw CSVs."
    )

remaining_months = pd.date_range(
    SECTION_39_START,
    SECTION_39_END,
    freq="MS",
)

print("=" * 100)
print("SECTION 39 SPLIT-ARCHIVE BACKFILL PREFLIGHT")
print("=" * 100)
print(
    f"Planned scope: {SECTION_39_START.strftime('%Y-%m')} "
    f"to {SECTION_39_END.strftime('%Y-%m')} "
    f"({len(remaining_months)} months)"
)
print("Source pattern: BIDPEROFFER1 + BIDPEROFFER2 for every month")
print("BigQuery production will remain unchanged.")
print(f"Free SSD space: {free_gb:.1f} GB")

print("\nJune hand-off artifacts confirmed:")
for path in required_files:
    print(f" ✅ {path.name}")

print(
    f"\nExisting local effective-offer stages: "
    f"{len(existing_months):,}"
)
print("Stages through June 2022:")
print(existing_months)

print("\n✅ SECTION 39A PREFLIGHT PASSED")
print("✅ Ready to launch the new split-archive local runner from July 2022.")
print("✅ Nothing was downloaded, changed, or deleted.")

In [ ]:
# SECTION 39B — Continuous split-archive local BESS backfill
# Scope: July 2022 through July 2024.
# Local-only. No BigQuery writes, promotions, or deletions.
#
# Each month:
# - uses BIDPEROFFER1 + BIDPEROFFER2;
# - matches only required current + next-month offer identities;
# - uses the persistent local offer state only when needed;
# - writes compact local stage/state/provenance files after every audit passes;
# - deletes temporary extracted CSVs only after success;
# - stops safely at the first unresolved issue.

import csv
import io
import json
import os
import shutil
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

SECTION_39_START = pd.Timestamp("2022-07-01")
SECTION_39_END = pd.Timestamp("2024-07-01")

for required_name in [
    "PARQUET_CACHE",
    "RUN_STATE",
    "DUCKDB_WORKING",
    "STATE_DIR",
    "LOCAL_ROOT",
    "OFFER_KEY_COLUMNS",
    "NUMERIC_OFFER_COLUMNS",
    "STATE_COLUMNS",
    "FINAL_STAGE_COLUMNS",
    "target_identity_dataframe",
    "expanded_interval_dataframe",
    "standardise_raw_offer_matches",
    "existing_state_matches",
    "append_local_manifest",
    "legacy_direction",
    "quote_identifier",
    "_download_aemo_zip_to_cache",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. Run the earlier setup cells first."
        )

def section39_sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def section39_source_parts(month):
    year = month.year
    month_number = month.month

    base_url = (
        "https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/"
        f"MMSDM/{year}/MMSDM_{year}_{month_number:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    return [
        {
            "file_name": (
                f"PUBLIC_DVD_BIDPEROFFER1_{year}{month_number:02d}010000.zip"
            ),
            "url": (
                base_url
                + f"PUBLIC_DVD_BIDPEROFFER1_{year}{month_number:02d}010000.zip"
            ),
        },
        {
            "file_name": (
                f"PUBLIC_DVD_BIDPEROFFER2_{year}{month_number:02d}010000.zip"
            ),
            "url": (
                base_url
                + f"PUBLIC_DVD_BIDPEROFFER2_{year}{month_number:02d}010000.zip"
            ),
        },
    ]


def section39_inspect_source(source_zip, source_label):
    with zipfile.ZipFile(source_zip) as archive:
        csv_members = [
            name for name in archive.namelist()
            if name.upper().endswith(".CSV")
        ]

        if len(csv_members) != 1:
            raise RuntimeError(
                f"{source_label}: expected one CSV; found {csv_members}"
            )

        csv_member = csv_members[0]
        expected_bytes = archive.getinfo(csv_member).file_size
        data_header = None

        with archive.open(csv_member) as binary_file:
            reader = csv.reader(
                io.TextIOWrapper(
                    binary_file,
                    encoding="utf-8-sig",
                    newline="",
                )
            )

            for record in reader:
                if (
                    record
                    and record[0].strip().upper() == "I"
                    and len(record) > 4
                    and "DUID" in record[4:]
                ):
                    data_header = record[4:]
                    break

    required_fields = {
        "DUID",
        "BIDTYPE",
        "TRADINGDATE",
        "OFFERDATETIME",
        "PERIODID",
        "MAXAVAIL",
        "FIXEDLOAD",
        "RAMPUPRATE",
        "RAMPDOWNRATE",
        "ENABLEMENTMIN",
        "ENABLEMENTMAX",
        "LOWBREAKPOINT",
        "HIGHBREAKPOINT",
        "BANDAVAIL1",
        "BANDAVAIL2",
        "BANDAVAIL3",
        "BANDAVAIL4",
        "BANDAVAIL5",
        "BANDAVAIL6",
        "BANDAVAIL7",
        "BANDAVAIL8",
        "BANDAVAIL9",
        "BANDAVAIL10",
        "PASAAVAILABILITY",
    }

    if data_header is None or required_fields - set(data_header):
        raise RuntimeError(
            f"{source_label}: unsupported raw-source schema."
        )

    return csv_member, expected_bytes, data_header


def section39_extract_csv(
    source_zip,
    csv_member,
    expected_bytes,
    working_csv,
    month_text,
):
    working_part = Path(f"{working_csv}.part")

    if working_part.exists():
        raise RuntimeError(
            f"{month_text}: incomplete extraction was retained:\n{working_part}"
        )

    if working_csv.exists():
        if working_csv.stat().st_size != expected_bytes:
            raise RuntimeError(
                f"{month_text}: existing CSV has an unexpected size:\n"
                f"{working_csv}"
            )

        print(f"♻️ Reusing complete temporary CSV: {working_csv.name}")
        return

    print(
        f"Extracting {working_csv.name}: "
        f"{expected_bytes / (1024 ** 3):.2f} GB"
    )

    copied_bytes = 0
    next_notice = 5 * (1024 ** 3)

    with zipfile.ZipFile(source_zip) as archive:
        with archive.open(csv_member) as source_file:
            with working_part.open("wb") as destination_file:
                while True:
                    block = source_file.read(16 * 1024 * 1024)

                    if not block:
                        break

                    destination_file.write(block)
                    copied_bytes += len(block)

                    if copied_bytes >= next_notice:
                        print(
                            f"  {month_text}: extracted "
                            f"{copied_bytes / (1024 ** 3):.1f} GB"
                        )
                        next_notice += 5 * (1024 ** 3)

    if copied_bytes != expected_bytes:
        raise RuntimeError(
            f"{month_text}: incomplete extraction retained:\n{working_part}"
        )

    os.replace(working_part, working_csv)


def section39_match_one_part(
    working_csv,
    data_header,
    union_target_csv,
    source_file_name,
    source_url,
    output_parquet,
    month_text,
):
    if output_parquet.exists():
        existing_df = pd.read_parquet(output_parquet)

        if (
            "SOURCE_FILE" in existing_df.columns
            and set(existing_df["SOURCE_FILE"].dropna().unique())
            == {source_file_name}
        ):
            print(f"♻️ {month_text}: reusing matched {source_file_name}")
            return existing_df

        raise RuntimeError(
            f"{month_text}: existing partial-match file is unsafe to reuse:\n"
            f"{output_parquet}"
        )

    raw_columns = [
        "RECORD_TYPE",
        "AEMO_GROUP",
        "AEMO_TABLE",
        "AEMO_VERSION",
    ] + data_header

    mapping_sql = ", ".join(
        f"{quote_identifier(name)}: 'VARCHAR'"
        for name in raw_columns
    )

    raw_data_sql = ", ".join(
        f"raw.{quote_identifier(name)}"
        for name in data_header
    )

    tradingdate_sql = """
    strftime(
        coalesce(
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.TRADINGDATE, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    offerdatetime_sql = """
    strftime(
        coalesce(
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S.%f'),
            try_strptime(raw.OFFERDATETIME, '%Y/%m/%d %H:%M:%S')
        ),
        '%Y-%m-%d %H:%M:%S.%f'
    )
    """

    con = duckdb.connect()
    con.execute("PRAGMA threads=22")

    print(f"{month_text}: DuckDB matching {source_file_name}...")

    try:
        con.execute(
            f"""
            COPY (
                SELECT
                    {raw_data_sql},
                    {tradingdate_sql} AS BIDSETTLEMENTDATE_KEY,
                    {offerdatetime_sql} AS OFFERDATE_KEY,
                    {section39_sql_literal(source_file_name)} AS SOURCE_FILE,
                    {section39_sql_literal(source_url)} AS SOURCE_URL

                FROM read_csv(
                    '{working_csv.as_posix()}',
                    header = false,
                    columns = {{{mapping_sql}}},
                    strict_mode = false,
                    null_padding = true,
                    ignore_errors = true
                ) AS raw

                INNER JOIN read_csv(
                    '{union_target_csv.as_posix()}',
                    header = true,
                    all_varchar = true
                ) AS target
                  ON UPPER(TRIM(raw.DUID)) = target.DUID
                 AND UPPER(TRIM(raw.BIDTYPE)) = target.BIDTYPE
                 AND {tradingdate_sql} =
                     target.BIDSETTLEMENTDATE_KEY
                 AND {offerdatetime_sql} =
                     target.OFFERDATE_KEY
                 AND TRY_CAST(raw.PERIODID AS INTEGER) =
                     TRY_CAST(target.PERIODID AS INTEGER)

                WHERE UPPER(TRIM(raw.RECORD_TYPE)) = 'D'
            )
            TO '{output_parquet.as_posix()}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
            """
        )
    finally:
        con.close()

    return pd.read_parquet(output_parquet)


def section39_process_one_month(month, next_month):
    month_text = month.strftime("%Y-%m")
    month_compact = month.strftime("%Y%m")
    parts = section39_source_parts(month)

    combined_match_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_{month_compact}_SPLIT_RAW_MATCHES.parquet"
    )
    local_stage_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_compact}_LOCAL.parquet"
    )
    origin_audit_parquet = (
        PARQUET_CACHE /
        f"BIDPEROFFER_BESS_EFFECTIVE_STAGE_{month_compact}_ORIGIN_AUDIT.parquet"
    )
    state_month_parquet = (
        STATE_DIR / f"offer_state_{month_compact}.parquet"
    )
    current_target_csv = (
        RUN_STATE / f"{month_compact}_current_target_identities.csv"
    )
    union_target_csv = (
        RUN_STATE /
        f"{month_compact}_current_and_lookahead_identities.csv"
    )
    carry_match_parquet = (
        RUN_STATE / f"{month_compact}_split_carry_state_matches.parquet"
    )

    if local_stage_parquet.exists():
        print(f"♻️ {month_text}: local stage already exists; skipping.")
        return {
            "month": month_text,
            "result": "SKIPPED_EXISTING_LOCAL_STAGE",
        }

    if state_month_parquet.exists():
        raise RuntimeError(
            f"{month_text}: state exists but local stage does not. "
            "Nothing was overwritten."
        )

    free_gb = shutil.disk_usage(LOCAL_ROOT).free / (1024 ** 3)

    if free_gb < 140:
        raise RuntimeError(
            f"{month_text}: only {free_gb:.1f} GB free; need at least 140 GB."
        )

    print("\n" + "=" * 100)
    print(f"STARTING SECTION 39 LOCAL MONTH: {month_text}")
    print("=" * 100)

    current_targets, expected_rows, _, _ = read_tracker_targets(month)

    if not current_targets:
        raise RuntimeError(f"{month_text}: tracker returned no targets.")

    current_identity_df = target_identity_dataframe(current_targets)

    if next_month is not None:
        next_targets, _, _, _ = read_tracker_targets(next_month)
        next_identity_df = target_identity_dataframe(next_targets)

        union_identity_df = (
            pd.concat(
                [current_identity_df, next_identity_df],
                ignore_index=True,
            )
            .drop_duplicates()
            .sort_values(OFFER_KEY_COLUMNS)
            .reset_index(drop=True)
        )
    else:
        union_identity_df = current_identity_df.copy()

    current_identity_df.to_csv(current_target_csv, index=False)
    union_identity_df.to_csv(union_target_csv, index=False)

    print(f"Expected effective rows: {expected_rows:,}")
    print(f"Current offer identities: {len(current_identity_df):,}")
    print(
        f"Current + look-ahead identities: {len(union_identity_df):,}"
    )

    expected_source_files = {
        item["file_name"] for item in parts
    }

    temporary_csvs = [
        DUCKDB_WORKING / item["file_name"].replace(".zip", ".CSV")
        for item in parts
    ]

    if combined_match_parquet.exists():
        raw_matches_df = pd.read_parquet(combined_match_parquet)

        if (
            "SOURCE_FILE" not in raw_matches_df.columns
            or set(raw_matches_df["SOURCE_FILE"].dropna().unique())
            != expected_source_files
        ):
            raise RuntimeError(
                f"{month_text}: existing combined match file does not prove "
                "both source halves were used."
            )

        print(f"♻️ {month_text}: reusing combined split-match Parquet.")

    else:
        part_frames = []

        for part_number, item in enumerate(parts, start=1):
            source_zip = Path(
                _download_aemo_zip_to_cache(
                    item["url"],
                    f"{month_text} raw source {item['file_name']}",
                    source_file_name=item["file_name"],
                )
            )

            csv_member, expected_bytes, data_header = (
                section39_inspect_source(
                    source_zip,
                    item["file_name"],
                )
            )

            working_csv = DUCKDB_WORKING / Path(csv_member).name

            section39_extract_csv(
                source_zip,
                csv_member,
                expected_bytes,
                working_csv,
                month_text,
            )

            part_match_parquet = (
                RUN_STATE /
                f"{month_compact}_part{part_number}_raw_matches.parquet"
            )

            part_frames.append(
                section39_match_one_part(
                    working_csv,
                    data_header,
                    union_target_csv,
                    item["file_name"],
                    item["url"],
                    part_match_parquet,
                    month_text,
                )
            )

        raw_matches_df = pd.concat(
            part_frames,
            ignore_index=True,
        )

        if raw_matches_df.empty:
            raise RuntimeError(
                f"{month_text}: both raw sources found no required offers."
            )

        raw_matches_df.to_parquet(
            combined_match_parquet,
            index=False,
            compression="zstd",
        )

        print(
            f"✅ {month_text}: combined raw-match file written "
            f"({len(raw_matches_df):,} rows)."
        )

    part_state_frames = []

    for item in parts:
        part_raw_df = raw_matches_df.loc[
            raw_matches_df["SOURCE_FILE"].eq(item["file_name"])
        ].copy()

        if part_raw_df.empty:
            raise RuntimeError(
                f"{month_text}: no required offers found in "
                f"{item['file_name']}."
            )

        standardised_part_df, _ = standardise_raw_offer_matches(
            part_raw_df,
            item["file_name"],
            item["url"],
            month_text,
        )

        part_state_frames.append(standardised_part_df)

    raw_state_df = pd.concat(
        part_state_frames,
        ignore_index=True,
    )

    if raw_state_df.duplicated(OFFER_KEY_COLUMNS).any():
        raise RuntimeError(
            f"{month_text}: duplicate raw offer identities across source parts."
        )

    current_raw_df = raw_state_df.merge(
        current_identity_df[OFFER_KEY_COLUMNS],
        on=OFFER_KEY_COLUMNS,
        how="inner",
        validate="one_to_one",
    )

    if carry_match_parquet.exists():
        carry_match_parquet.unlink()

    carry_state_df = existing_state_matches(
        current_target_csv,
        carry_match_parquet,
    )

    if not carry_state_df.empty:
        carry_state_df["STATE_ORIGIN"] = "PERSISTENT_LOCAL_STATE"

    available_offer_df = pd.concat(
        [current_raw_df, carry_state_df],
        ignore_index=True,
    ).drop_duplicates(
        OFFER_KEY_COLUMNS,
        keep="first",
    )

    available_keys_df = available_offer_df[
        OFFER_KEY_COLUMNS
    ].drop_duplicates()

    unresolved_df = current_identity_df.merge(
        available_keys_df,
        on=OFFER_KEY_COLUMNS,
        how="left",
        indicator=True,
    )

    unresolved_df = unresolved_df.loc[
        unresolved_df["_merge"].ne("both"),
        OFFER_KEY_COLUMNS,
    ]

    if not unresolved_df.empty:
        display(unresolved_df.head(30))

        raise RuntimeError(
            f"{month_text}: {len(unresolved_df):,} required offer identities "
            "are absent from both the two-part raw archive and local state."
        )

    interval_df = expanded_interval_dataframe(current_targets)

    if len(interval_df) != expected_rows:
        raise RuntimeError(
            f"{month_text}: tracker expansion differs from expected rows."
        )

    stage_work_df = interval_df.merge(
        available_offer_df,
        on=OFFER_KEY_COLUMNS,
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    missing_intervals = int(
        stage_work_df["_merge"].ne("both").sum()
    )

    if missing_intervals != 0:
        raise RuntimeError(
            f"{month_text}: {missing_intervals:,} intervals have no offer."
        )

    stage_work_df = stage_work_df.drop(columns="_merge")

    stage_work_df["DIRECTION"] = (
        stage_work_df["DIRECTION"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_direction_count = int(
        (~stage_work_df["DIRECTION"].isin(
            ["GEN", "LOAD", "BIDIRECTIONAL"]
        )).sum()
    )

    if invalid_direction_count != 0:
        raise RuntimeError(
            f"{month_text}: final stage has invalid/null directions."
        )

    stage_work_df["OFFER_ORIGIN"] = stage_work_df["STATE_ORIGIN"]

    origin_audit_df = stage_work_df[
        [
            "INTERVAL_DATETIME",
            "DUID",
            "BIDTYPE",
            "DIRECTION",
            "OFFER_ORIGIN",
            "SOURCE_FILE",
            "SOURCE_URL",
        ]
    ].copy()

    for field in ["SETTLEMENTDATE", "INTERVAL_DATETIME"]:
        stage_work_df[field] = pd.to_datetime(
            stage_work_df[field],
            utc=True,
        )

    stage_work_df["BIDSETTLEMENTDATE"] = pd.to_datetime(
        stage_work_df["BIDSETTLEMENTDATE_KEY"],
        utc=True,
    )
    stage_work_df["OFFERDATE"] = pd.to_datetime(
        stage_work_df["OFFERDATE_KEY"],
        utc=True,
    )
    stage_work_df["VERSIONNO"] = pd.Series(
        pd.NA,
        index=stage_work_df.index,
        dtype="Float64",
    )
    stage_work_df["LASTCHANGED"] = pd.to_datetime(
        pd.Series(pd.NaT, index=stage_work_df.index),
        utc=True,
    )
    stage_work_df["MR_CAPACITY"] = pd.Series(
        pd.NA,
        index=stage_work_df.index,
        dtype="Float64",
    )

    run_id = (
        f"bess-effective-reconstructed-local-split-{month_compact}"
    )

    stage_work_df["_LOAD_DATE"] = pd.Timestamp.now(tz="UTC")
    stage_work_df["_SOURCE"] = "AEMO_NEMWEB_ARCHIVE"
    stage_work_df["_PIPELINE"] = (
        "BIDPEROFFER_EFFECTIVE_RECONSTRUCTED_"
        "DUCKDB_CONTINUOUS_LOCAL_SPLIT_ARCHIVES"
    )
    stage_work_df["_SOURCE_FILE"] = stage_work_df["SOURCE_FILE"]
    stage_work_df["_SOURCE_URL"] = stage_work_df["SOURCE_URL"]
    stage_work_df["_RUN_ID"] = run_id

    stage_df = stage_work_df[FINAL_STAGE_COLUMNS].copy()

    required_stage_columns = [
        "SETTLEMENTDATE",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
        "INTERVAL_DATETIME",
        "BIDSETTLEMENTDATE",
        "OFFERDATE",
        "PERIODID",
    ]

    effective_key_columns = [
        "INTERVAL_DATETIME",
        "DUID",
        "BIDTYPE",
        "DIRECTION",
    ]

    null_required_count = int(
        stage_df[required_stage_columns].isna().sum().sum()
    )
    distinct_effective_keys = len(
        stage_df.drop_duplicates(effective_key_columns)
    )

    audit_df = pd.DataFrame(
        [
            {
                "expected_stage_rows": expected_rows,
                "stage_rows": len(stage_df),
                "distinct_effective_keys": distinct_effective_keys,
                "null_required_field_count": null_required_count,
                "invalid_direction_count": invalid_direction_count,
                "first_interval": stage_df["INTERVAL_DATETIME"].min(),
                "last_interval": stage_df["INTERVAL_DATETIME"].max(),
            }
        ]
    )
    display(audit_df)

    if len(stage_df) != expected_rows:
        raise RuntimeError(
            f"{month_text}: stage rows differ from expected rows."
        )

    if distinct_effective_keys != expected_rows:
        raise RuntimeError(
            f"{month_text}: duplicate effective keys in local stage."
        )

    if null_required_count != 0:
        raise RuntimeError(
            f"{month_text}: local stage contains null required fields."
        )

    stage_df.to_parquet(
        local_stage_parquet,
        index=False,
        compression="zstd",
    )
    origin_audit_df.to_parquet(
        origin_audit_parquet,
        index=False,
        compression="zstd",
    )
    raw_state_df.to_parquet(
        state_month_parquet,
        index=False,
        compression="zstd",
    )

    origin_summary = (
        origin_audit_df
        .groupby("OFFER_ORIGIN", dropna=False)
        .size()
        .reset_index(name="effective_rows")
    )

    source_summary = (
        origin_audit_df
        .groupby(
            ["OFFER_ORIGIN", "SOURCE_FILE"],
            dropna=False,
        )
        .size()
        .reset_index(name="effective_rows")
    )

    append_local_manifest(
        {
            "settlement_month": month_text,
            "run_id": run_id,
            "status": "LOCAL_STAGED_SPLIT_ARCHIVES",
            "expected_effective_rows": int(expected_rows),
            "stage_rows": int(len(stage_df)),
            "raw_offer_identities_found": int(len(raw_state_df)),
            "current_raw_offer_identities": int(len(current_raw_df)),
            "historical_state_offer_identities": int(len(carry_state_df)),
            "stage_file": str(local_stage_parquet),
            "origin_audit_file": str(origin_audit_parquet),
            "state_file": str(state_month_parquet),
            "source_files": [
                item["file_name"] for item in parts
            ],
            "source_urls": [
                item["url"] for item in parts
            ],
            "completed_at_utc": str(pd.Timestamp.now(tz="UTC")),
        }
    )

    LOCAL_BATCH_CHECKPOINT.write_text(
        json.dumps(
            {
                "latest_completed_month": month_text,
                "status": "LOCAL_STAGED_SPLIT_ARCHIVES",
                "stage_file": str(local_stage_parquet),
                "state_file": str(state_month_parquet),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    # Temporary extracts are deleted only after the month fully passes.
    for working_csv in temporary_csvs:
        if working_csv.exists():
            recovered_gb = (
                working_csv.stat().st_size / (1024 ** 3)
            )
            working_csv.unlink()
            print(
                f"✅ {month_text}: temporary CSV deleted "
                f"({working_csv.name}, {recovered_gb:.2f} GB recovered)."
            )

    print(
        f"✅ {month_text}: LOCAL SPLIT-ARCHIVE STAGE PASSED — "
        f"{len(stage_df):,} rows"
    )
    display(origin_summary)
    display(source_summary)

    return {
        "month": month_text,
        "result": "LOCAL_STAGED",
        "stage_rows": int(len(stage_df)),
        "raw_offer_identities": int(len(raw_state_df)),
        "current_raw_identities": int(len(current_raw_df)),
        "historical_state_identities": int(len(carry_state_df)),
    }


# ---------- Run Section 39 ----------

section39_months = list(
    pd.date_range(
        SECTION_39_START,
        SECTION_39_END,
        freq="MS",
    )
)

print("=" * 100)
print("STARTING SECTION 39 SPLIT-ARCHIVE LOCAL BACKFILL")
print(
    f"Months: {SECTION_39_START.strftime('%Y-%m')} "
    f"to {SECTION_39_END.strftime('%Y-%m')} "
    f"({len(section39_months)} months)"
)
print("Every month uses BIDPEROFFER1 + BIDPEROFFER2.")
print("BigQuery production remains unchanged.")
print("=" * 100)

section39_results = []

for index, month in enumerate(section39_months):
    next_month = (
        section39_months[index + 1]
        if index + 1 < len(section39_months)
        else None
    )

    try:
        section39_results.append(
            section39_process_one_month(month, next_month)
        )

    except Exception as error:
        section39_results.append(
            {
                "month": month.strftime("%Y-%m"),
                "result": "STOPPED_FOR_REVIEW",
                "error": str(error),
            }
        )

        print()
        print(
            f"⚠️ Section 39 stopped safely at "
            f"{month.strftime('%Y-%m')}."
        )
        print("No later month was started.")
        print("Any temporary CSVs for that month were retained.")
        break

section39_summary_df = pd.DataFrame(section39_results)

print()
print("SECTION 39 LOCAL BACKFILL SUMMARY")
display(section39_summary_df)

print()
print("✅ Successful months have compact local stages, offer-state files,")
print("   ZIP caches, provenance audits, checkpoints, and manifest records.")
print("✅ No BigQuery production data was changed.")